# Reproduce the current Part 1 candidate

This notebook embeds all experiment Python source. It creates an isolated replay folder, regenerates the frozen candidate, checks raw hashes and gold-label preservation, and compares the submission hash. It does not rerun model selection.

In [1]:
from pathlib import Path
import shutil,sys,subprocess,json,hashlib
root=Path.cwd(); replay=root/'current_reproduction';replay.mkdir(exist_ok=True)
shutil.copytree(root/'data',replay/'data',dirs_exist_ok=True)
(replay/'experiments').mkdir(exist_ok=True)
for name in ['current-best.json','data-audit.json']:
    shutil.copy2(root/'experiments'/name,replay/'experiments'/name)

In [2]:
%%writefile current_reproduction/audit_candidates.py
"""Nested shortlist audit and temporal stress test. Does NOT open validation or test labels."""
import experiment as e
import numpy as np,json,time
from pathlib import Path
from threadpoolctl import threadpool_limits
rank=json.loads(Path('experiments/robust-ranking.json').read_text())
ens=json.loads(Path('experiments/ensemble-ranking.json').read_text())
ssl=next(x['config'] for x in rank if x['config']['model']=='pseudo')
ssl_ensemble=next(x['config'] for x in ens if len(x['config']['members'])==3 and x['config']['members'][0]['model']=='svc')
candidates=[dict(model='logistic',C=.1,scale='global'),dict(model='logistic',C=.3,scale='global'),dict(model='svc',C=10,gamma=.001,scale='standard'),dict(model='svc',C=10,gamma=.001,scale='standard',features=['drug_targets'],feature_weight=1.),ssl,ens[0]['config'],ssl_ensemble]
e.save_json('experiments/nested-candidates.json',candidates)
outer=e.make_splits(seed=6172026,n_splits=5);pred=np.zeros_like(e.Y);prob=np.zeros_like(e.Y,float);records=[]
with threadpool_limits(limits=1):
 for fold,(a,b) in enumerate(outer):
  print('OUTER',fold,flush=True);inner=e.make_splits(e.Y[a],e.GROUP[e.LAB[a]],seed=1200+fold,n_splits=3)
  pool=[]
  for ci,c in enumerate(candidates):
   ip=np.zeros((len(a),5))
   for ia,ib in inner:
    exclude=np.r_[e.GROUP[e.LAB[b]],e.GROUP[e.LAB[a[ib]]]]
    ip[ib]=e.predict(a[ia],e.DATA.iloc[e.LAB[a[ib]]],c,excluded_groups=exclude)
   rule,score=e.choose_rule(e.Y[a],ip);pool.append(dict(config=c,rule=rule,score=score))
   print('INNER',fold,ci,score,rule,flush=True)
  # A small tie favors an earlier, simpler candidate.
  maximum=max(r['score'] for r in pool);best=next(r for r in pool if r['score']>=maximum-.003)
  pp=e.predict(a,e.DATA.iloc[e.LAB[b]],best['config'],excluded_groups=e.GROUP[e.LAB[b]])
  prob[b]=pp;pred[b]=e.decisions(pp,best['rule'])
  rec=dict(fold=fold,selected=best,inner=pool,outer=e.metrics(e.Y[b],pp,best['rule']))
  records.append(rec);e.save_json('experiments/nested-progress.json',records);print('OUTER RESULT',rec['outer'],flush=True)
 from sklearn.metrics import f1_score
 result=dict(sample_f1=f1_score(e.Y,pred,average='samples'),macro_f1=f1_score(e.Y,pred,average='macro'),per_label=dict(zip(e.LABELS,f1_score(e.Y,pred,average=None))),folds=records,limitation='Conditional shortlist audit. Candidate shortlist was informed by full-training exploratory CV; not an unbiased nested rerun of the complete adaptive search. External validation remains unopened.')
 e.save_json('experiments/nested-audit.json',result);np.savez_compressed('experiments/nested-predictions.npz',p=prob,pred=pred,y=e.Y)
 # Chronology tests are deliberately hard: most extra rare-reason labels occur in later years.
 temporal=[]
 for cutoff in [2002,2005]:
  years=e.DATA.iloc[e.LAB].start_year.to_numpy();a=np.flatnonzero(years<=cutoff);b=np.flatnonzero(years>cutoff)
  a=a[~np.isin(e.GROUP[e.LAB[a]],e.GROUP[e.LAB[b]])]
  for c in candidates:
   pp=e.predict(a,e.DATA.iloc[e.LAB[b]],c,excluded_groups=e.GROUP[e.LAB[b]])
   # Fixed default decision: no tuning on temporal holdout.
   rec=dict(cutoff=cutoff,config=c,n_train=len(a),n_test=len(b),train_positive=e.Y[a].sum(0).tolist(),test_positive=e.Y[b].sum(0).tolist(),**e.metrics(e.Y[b],pp,'t0.5'))
   temporal.append(rec);print('TEMPORAL',rec,flush=True)
 e.save_json('experiments/temporal-audit.json',temporal)
print('DONE',flush=True)


Overwriting current_reproduction/audit_candidates.py


In [3]:
%%writefile current_reproduction/audit_search_gaps.py
"""Record coverage gaps found after the independent search; no validation fitting."""
from pathlib import Path
import json,datetime
rows=[json.loads(s) for s in Path('experiments/trials.jsonl').read_text().splitlines()]
old={r['id']:r['config'] for r in rows if r['tag']!='phase20'}
feature=[c for c in old.values() if c['model']=='lowdensity_v2' and c.get('features')]
audit=dict(created_at=datetime.datetime.now(datetime.timezone.utc).isoformat(),previous_logged_configurations=len(old),winning_family_feature_configurations=len(feature),winning_family_multiblock_configs=sum(len(c['features'])>1 for c in feature),gaps=[dict(name='Joint feature kernels',finding='The strongest low-density SVM family received 22 single-block additions and no multiblock configurations. Earlier concatenation-based RBF models combine block distances multiplicatively in similarity space. Separate additive cosine/Tanimoto kernels and their pairs were not evaluated.',action='phase20 screens single auxiliary kernels, five feature pairs, and training-label-alignment weights.'),dict(name='Effective sample mass in class balancing',finding='Installed SVC._validate_targets computes balanced class weights from y without sample_weight. Downweighted pseudo-labels still count equally in automatic class-frequency estimates. Earlier trials covered raw joint counts and fixed gold/initial priors, but not effective weighted counts.',action='phase20 compares effective weighted class counts and repeated-vector weighting. This is a modeling alternative, not evidence that old evaluation scores are invalid.'),dict(name='Tri-training',finding='Previous static consensus, self-training, bagging and model averaging did not implement iterative three-classifier cross-teaching with agreement/error controls.',action='Still an untested candidate; no claim of a measured improvement.'),dict(name='Validation-matched selection',finding='Default grouped CV mixes early gold and deliberately rare-enriched later gold. It does not recreate the randomly sampled validation distribution or later Kaggle distribution. Some chronological stress tests and covariate weighting were tried, but no unbiased target-distribution risk estimate is available.',action='Do not infer the leader algorithm or our latest Kaggle score from separate validation scores.')],references=[dict(title='Cortes, Mohri, Rostamizadeh: Algorithms for Learning Kernels Based on Centered Alignment',url='https://www.jmlr.org/papers/v13/cortes12a.html'),dict(title='Zhou and Li: Tri-Training',url='https://www.lamda.nju.edu.cn/publication/tkde05.pdf'),dict(title='SVC documentation',url='https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html')])
Path('experiments/search-gap-audit.json').write_text(json.dumps(audit,indent=2))
print(json.dumps({k:audit[k] for k in ['previous_logged_configurations','winning_family_feature_configurations','winning_family_multiblock_configs']},indent=2))


Overwriting current_reproduction/audit_search_gaps.py


In [4]:
%%writefile current_reproduction/benchmark_cv.py
"""Frozen, common-fold Part 1 comparisons. Never reads validation/test data.

Run: .venv/bin/python benchmark_cv.py
Register a future frozen recipe: --register candidate.json
The candidate needs name, config and rule; an existing validation metric is optional.
"""
from pathlib import Path
import argparse,datetime,hashlib,json,re,time,traceback
import numpy as np
from threadpoolctl import threadpool_limits
import phase26,experiment as e

ROOT=Path(__file__).resolve().parent;OUT=ROOT/'experiments/benchmark_v1'
SEEDS=[20261126,20261127,20261128]

def digest(x):return hashlib.sha256(json.dumps(x,sort_keys=True,separators=(',',':')).encode()).hexdigest()
def file_hash(p):return hashlib.sha256(p.read_bytes()).hexdigest()
def now():return datetime.datetime.now(datetime.timezone.utc).isoformat()
def save(p,x):p.write_text(json.dumps(x,indent=2))

def validate_protocol(p):
    assert p['training_sha256']==file_hash(ROOT/'data/part1_train.csv'),'Training data changed; create a new protocol version.'
    assert p['gold_ids']==e.DATA.iloc[e.LAB].ID.tolist()
    assert p['label_order']==e.LABELS and p['seeds']==SEEDS
    assert p['folds_sha256']==digest(p['folds'])
    for seed in SEEDS:
        folds=[f for f in p['folds'] if f['seed']==seed]
        assert len(folds)==5
        all_held=np.concatenate([f['heldout_gold_positions'] for f in folds])
        assert np.array_equal(np.sort(all_held),np.arange(len(e.Y)))
        for f in folds:
            a=np.array(f['train_gold_positions']);b=np.array(f['heldout_gold_positions'])
            assert not set(a)&set(b)
            assert set(a)|set(b)==set(range(len(e.Y)))
            assert not set(e.GROUP[e.LAB[a]])&set(e.GROUP[e.LAB[b]])
            assert f['heldout_ids']==e.DATA.iloc[e.LAB[b]].ID.tolist()
            excluded=np.isin(e.GROUP[e.UNLAB],e.GROUP[e.LAB[b]])
            assert f['masked_duplicates_excluded']==int(excluded.sum())

def protocol():
    OUT.mkdir(exist_ok=True)
    path=OUT/'protocol.json'
    if path.exists():
        p=json.loads(path.read_text());validate_protocol(p);return p
    folds=[]
    for seed in SEEDS:
        for i,(a,b) in enumerate(e.make_splits(seed=seed,n_splits=5)):
            folds.append(dict(seed=seed,fold=i,train_gold_positions=a.tolist(),heldout_gold_positions=b.tolist(),
                heldout_ids=e.DATA.iloc[e.LAB[b]].ID.tolist(),
                masked_duplicates_excluded=int(np.isin(e.GROUP[e.UNLAB],e.GROUP[e.LAB[b]]).sum())))
    p=dict(version='part1-common-cv-v1',created_at=now(),training_sha256=file_hash(ROOT/'data/part1_train.csv'),
        gold_ids=e.DATA.iloc[e.LAB].ID.tolist(),label_order=e.LABELS,seeds=SEEDS,folds_per_repeat=5,
        grouping='Hash all 150 reason embedding coordinates rounded to 10 decimal places; iterative multilabel stratification over groups.',
        metric='Mean of three complete-OOF sample-wise F1 scores; equal weight per gold trial within each repeat.',
        dispersion='Population standard deviation of three repeat scores, not a confidence interval.',
        fitting='Only current outer-training gold and original masked training rows; remove masked duplicates of held-out groups. No held-out inputs participate in fitting.',
        tuning='Freeze recipe and final decoder before benchmarking. Any adaptive tuning/calibration/feature selection must occur inside the outer training fold.',
        validation='Benchmark reads no validation or test CSV. Copy existing validation metrics with provenance; evaluate new validation candidates only after CV shortlisting and freezing.',
        shortlist='Positive mean paired CV difference versus the fixed incumbent and improvement in at least 2 of 3 repeats; descriptive screen, not a significance test or automatic promotion.',
        limitation='Historical tuning used these training labels and repeatedly consulted validation. This is a common development benchmark, not an independent test. Fixed folds can also be overfit through repeated search.',
        chronology='Any chronological stress test is supplementary and separately labeled; never substitute a favorable split for this primary benchmark.',
        folds=folds,folds_sha256=digest(folds))
    validate_protocol(p);save(path,p);return p

def transductive(x):
    if isinstance(x,dict):return bool(x.get('transductive',False)) or any(transductive(v) for v in x.values())
    if isinstance(x,list):return any(transductive(v) for v in x)
    return False

def entry(r,name,source,incumbent=False):
    return dict(name=name,config=r['config'],rule=r['rule'],incumbent=incumbent,
        recorded_validation=r.get('validation'),validation_source=source,
        validation_status='Previously measured; not rerun by this benchmark',registered_at=now())

def registry(add=None):
    path=OUT/'registry.json'
    if path.exists():rs=json.loads(path.read_text())
    else:
        current=json.loads((ROOT/'experiments/current-best.json').read_text())['candidate']
        old=json.loads((ROOT/'experiments/phase21-prior-incumbent.json').read_text())['candidate']
        rs=[entry(current,'Current best hybrid','experiments/current-best.json',True),
            entry(old,'Previous cluster-guided SVM','experiments/phase21-prior-incumbent.json')]
        names={'label_shift26':'Label-frequency correction','lap_svm26':'Graph-regularized SVM','other_subclasses26':'Other subgroups'}
        for r in json.loads((ROOT/'experiments/phase26-validation-results.json').read_text()):
            if not r['matched_control']:rs.append(entry(r,names[r['config']['learner']['model']], 'experiments/phase26-validation-results.json'))
    if add:
        r=json.loads(Path(add).read_text())
        assert all(k in r for k in ['name','config','rule'])
        assert not any(x['name']==r['name'] for x in rs),'Use a unique versioned candidate name.'
        rs.append(entry(r,r['name'],str(add)))
    for r in rs:
        assert not transductive(r['config']),'Transductive recipes belong in a separate benchmark.'
        if r.get('recorded_validation') is None:r['validation_status']='Not evaluated'
    assert sum(r.get('incumbent',False) for r in rs)==1
    save(path,rs);return rs

def evaluate(r,p):
    source_files=[ROOT/'experiment.py',Path(__file__)]+[f for f in ROOT.glob('phase*.py') if re.fullmatch(r'phase\d+\.py',f.name)]
    source_files += sorted((ROOT/'experiments').glob('phase*-baseline.json'))
    sources={str(f.relative_to(ROOT)):file_hash(f) for f in source_files}
    key=digest(dict(config=r['config'],rule=r['rule'],protocol=digest(p),sources=sources))[:20]
    path=OUT/(key+'.json')
    if path.exists():return json.loads(path.read_text())
    started=time.time();all_scores=[];all_pred=[];fold_scores=[]
    try:
        with threadpool_limits(limits=1):
            for seed in p['seeds']:
                scores=np.full(e.Y.shape,np.nan)
                for f in [x for x in p['folds'] if x['seed']==seed]:
                    a=np.array(f['train_gold_positions']);b=np.array(f['heldout_gold_positions'])
                    query=e.DATA.iloc[e.LAB[b]].drop(columns=e.LABELS)
                    pred=e.predict(a,query,r['config'],excluded_groups=e.GROUP[e.LAB[b]])
                    assert pred.shape==(len(b),len(e.LABELS)) and np.isfinite(pred).all()
                    scores[b]=pred
                    # Use only the frozen rule. Never choose a rule on outer OOF labels.
                    fold_scores.append(dict(seed=seed,fold=f['fold'],rows=len(b),**e.metrics(e.Y[b],pred,r['rule'])))
                assert np.isfinite(scores).all()
                all_scores.append(scores);all_pred.append(e.decisions(scores,r['rule']))
                print(json.dumps(dict(candidate=r['name'],seed=seed,**e.metrics(e.Y,scores,r['rule']))),flush=True)
        repeat_scores=[e.metrics(e.Y,s,r['rule']) for s in all_scores]
        values=[v['sample_f1'] for v in repeat_scores]
        result=dict(name=r['name'],status='ok',key=key,config=r['config'],rule=r['rule'],protocol_sha256=digest(p),
            source_sha256=sources,cv_mean=float(np.mean(values)),repeat_std=float(np.std(values)),
            cv_macro=float(np.mean([v['macro_f1'] for v in repeat_scores])),repeat_scores=repeat_scores,
            fold_scores=fold_scores,seconds=round(time.time()-started,3),finished_at=now())
        np.savez_compressed(OUT/(key+'.npz'),scores=all_scores,predictions=all_pred,truth=e.Y,ids=e.DATA.iloc[e.LAB].ID.to_numpy())
        save(path,result);return result
    except Exception as error:
        result=dict(name=r['name'],status='error',error=str(error),traceback=traceback.format_exc(),finished_at=now())
        with (OUT/'errors.jsonl').open('a') as f:f.write(json.dumps(result)+'\n')
        raise

def main():
    parser=argparse.ArgumentParser();parser.add_argument('--register');parser.add_argument('--verify-only',action='store_true')
    args=parser.parse_args();p=protocol()
    if args.verify_only:print('Verified frozen folds, training hash, group separation and masked duplicate counts.');return
    rs=registry(args.register);outputs=[]
    for r in rs:outputs.append(evaluate(r,p))
    base=outputs[next(i for i,r in enumerate(rs) if r['incumbent'])]
    rows=[]
    for r,out in zip(rs,outputs):
        deltas=np.array([x['sample_f1'] for x in out['repeat_scores']])-np.array([x['sample_f1'] for x in base['repeat_scores']])
        row=dict(candidate=r['name'],incumbent=r['incumbent'],cv_mean=out['cv_mean'],repeat_std=out['repeat_std'],
            cv_macro=out['cv_macro'],paired_cv_difference=float(deltas.mean()),repeats_improved=int(np.sum(deltas>1e-12)),
            cv_shortlist=bool(deltas.mean()>1e-12 and np.sum(deltas>1e-12)>=2),
            validation_f1=(r.get('recorded_validation') or {}).get('sample_f1'),validation_status=r['validation_status'],
            validation_source=r['validation_source'],result_key=out['key'])
        rows.append(row)
    e.pd.DataFrame(rows).to_csv(OUT/'comparison.csv',index=False)
    save(OUT/'summary.json',dict(protocol='part1-common-cv-v1',protocol_sha256=digest(p),folds_sha256=p['folds_sha256'],
        updated_at=now(),seeds=SEEDS,models=rows,validation_evaluations_performed=0,model_promotions_performed=0))
    print(e.pd.DataFrame(rows)[['candidate','cv_mean','repeat_std','paired_cv_difference','validation_f1']].to_string(index=False))

if __name__=='__main__':main()


Overwriting current_reproduction/benchmark_cv.py


In [5]:
%%writefile current_reproduction/benchmark_pooled_cv.py
"""Separate repeated grouped CV on 157 training + 78 validation gold labels.

Reuses frozen recipes and scoring code, with an isolated in-memory data context.
No full-data refit, Kaggle prediction, or current-model replacement is performed.
"""
from contextlib import contextmanager
from pathlib import Path
import argparse,json,sys
import numpy as np
from threadpoolctl import threadpool_limits
import benchmark_cv as b
e=b.e;ROOT=b.ROOT;OUT=ROOT/'experiments/pooled_benchmark_v1'

def clear_caches():
    for name,module in list(sys.modules.items()):
        if name=='experiment' or (name.startswith('phase') and name[5:].isdigit()):
            for key,value in vars(module).items():
                if 'CACHE' in key and isinstance(value,dict):value.clear()

@contextmanager
def pooled_context():
    keys=['DATA','LAB','UNLAB','Y','GROUP','make_splits'];original={k:getattr(e,k) for k in keys}
    old_out=b.OUT
    try:
        clear_caches()
        val=e.pd.read_csv(ROOT/'data/part1_val.csv')
        assert not set(e.DATA.ID)&set(val.ID)
        e.DATA=e.pd.concat([e.DATA,val],ignore_index=True)
        e.LAB=np.flatnonzero(e.DATA[e.LABELS].ge(0).all(axis=1))
        e.UNLAB=np.flatnonzero(e.DATA[e.LABELS].eq(-100).all(axis=1))
        e.Y=e.DATA.iloc[e.LAB][e.LABELS].to_numpy(int)
        e.GROUP=e.pd.util.hash_pandas_object(e.DATA[e.EMB].round(10),index=False).to_numpy()
        # The original function's default Y was bound at import; replace that
        # default for this context while preserving explicitly supplied inner folds.
        def make_splits(y=None,groups=None,seed=20261104,n_splits=5):
            if y is None:y=e.Y
            if groups is None:groups=e.GROUP[e.LAB]
            return original['make_splits'](y,groups,seed,n_splits)
        e.make_splits=make_splits;b.OUT=OUT
        assert len(e.Y)==235 and len(e.UNLAB)==1014 and len(e.DATA)==1249
        yield
    finally:
        clear_caches()
        for k,v in original.items():setattr(e,k,v)
        b.OUT=old_out

def protected_hashes():
    paths=list((ROOT/'data').glob('*.csv'))+[ROOT/'experiments/current-best.json']
    paths+=list((ROOT/'predictions/current_best').glob('*'))
    # The primary fold assignments and retained model are immutable here.
    paths += [ROOT/'experiments/benchmark_v1/protocol.json']
    return {str(p.relative_to(ROOT)):b.file_hash(p) for p in paths if p.is_file()}

def validate(p):
    assert p['training_sha256']==b.file_hash(ROOT/'data/part1_train.csv')
    assert p['validation_sha256']==b.file_hash(ROOT/'data/part1_val.csv')
    assert p['runner_sha256']==b.file_hash(Path(__file__))
    assert p['gold_ids']==e.DATA.iloc[e.LAB].ID.tolist() and p['label_order']==e.LABELS
    assert p['folds_sha256']==b.digest(p['folds']) and p['seeds']==b.SEEDS
    for seed in p['seeds']:
        folds=[f for f in p['folds'] if f['seed']==seed];assert len(folds)==5
        held=np.concatenate([f['heldout_gold_positions'] for f in folds])
        assert np.array_equal(np.sort(held),np.arange(len(e.Y)))
        for f in folds:
            a=np.array(f['train_gold_positions']);h=np.array(f['heldout_gold_positions'])
            assert len(np.unique(a))==len(a) and not set(a)&set(h)
            assert set(a)|set(h)==set(range(len(e.Y)))
            assert not set(e.GROUP[e.LAB[a]])&set(e.GROUP[e.LAB[h]])
            assert f['heldout_ids']==e.DATA.iloc[e.LAB[h]].ID.tolist()
            assert f['masked_duplicates_excluded']==int(np.isin(e.GROUP[e.UNLAB],e.GROUP[e.LAB[h]]).sum())

def protocol():
    path=OUT/'protocol.json'
    if path.exists():p=json.loads(path.read_text());validate(p);return p
    original_train_rows=len(e.pd.read_csv(ROOT/'data/part1_train.csv'))
    folds=[]
    for seed in b.SEEDS:
        for i,(a,h) in enumerate(e.make_splits(y=e.Y,groups=e.GROUP[e.LAB],seed=seed,n_splits=5)):
            folds.append(dict(seed=seed,fold=i,train_gold_positions=a.tolist(),heldout_gold_positions=h.tolist(),
                heldout_ids=e.DATA.iloc[e.LAB[h]].ID.tolist(),training_gold_rows=len(a),heldout_gold_rows=len(h),
                original_validation_rows_in_training=int(np.sum(e.LAB[a]>=original_train_rows)),
                original_validation_rows_heldout=int(np.sum(e.LAB[h]>=original_train_rows)),
                masked_duplicates_excluded=int(np.isin(e.GROUP[e.UNLAB],e.GROUP[e.LAB[h]]).sum())))
    p=dict(version='part1-pooled-cv-v1',created_at=b.now(),runner_sha256=b.file_hash(Path(__file__)),
        training_sha256=b.file_hash(ROOT/'data/part1_train.csv'),validation_sha256=b.file_hash(ROOT/'data/part1_val.csv'),
        gold_ids=e.DATA.iloc[e.LAB].ID.tolist(),label_order=e.LABELS,gold_rows=len(e.Y),
        gold_embedding_groups=int(len(np.unique(e.GROUP[e.LAB]))),masked_rows=len(e.UNLAB),
        seeds=b.SEEDS,folds_per_repeat=5,folds=folds,folds_sha256=b.digest(folds),
        purpose='Supplementary comparison of unchanged recipes trained with more labels; preserve training-only primary benchmark.',
        parameter_policy='Identical configs and frozen decoding rules to training-only registry. Refit learned parameters per fold. No new hyperparameter, feature, or threshold search.',
        fit_policy='Only outer-training gold (from either source) plus original masked training rows; exclude masked duplicates of held-out embedding groups. Held-out features and labels do not enter fitting. No Kaggle test inputs.',
        metric='Mean of three complete out-of-fold sample-wise F1 scores across all 235 gold rows; repeat std is split sensitivity, not a confidence interval.',
        validation_interpretation='Previously reported validation F1 applies to the original training-only fit. It is not an independent validation score for these pooled-fold models.',
        limitation='Prior model selection used both label sources, so pooled CV is a development estimate. Temporal population shift remains. It is not a third independent test.',
        final_all_label_refit=False,model_promotions=False)
    validate(p);b.save(path,p);return p

def registry():
    path=OUT/'registry.json'
    current=json.loads((ROOT/'experiments/benchmark_v1/registry.json').read_text())
    signature=lambda rs:[dict(name=r['name'],config=r['config'],rule=r['rule']) for r in rs]
    if path.exists():
        rs=json.loads(path.read_text());assert signature(rs)==signature(current),'Cohort changed; freeze a new comparison version.'
    else:rs=current;b.save(path,rs)
    assert not any(b.transductive(r['config']) for r in rs)
    return rs

def integrity(p,rs):
    path=OUT/'integrity.json'
    if path.exists():
        old=json.loads(path.read_text());assert old['runner_sha256']==p['runner_sha256'];return old
    f=p['folds'][0];a=np.array(f['train_gold_positions']);h=np.array(f['heldout_gold_positions'])
    q=e.DATA.iloc[e.LAB[h]].drop(columns=e.LABELS);ex=e.GROUP[e.LAB[h]]
    # Check incumbent plus the nested calibrator, the two most important paths
    # affected by the enlarged gold pool. Other methods retain previous audits.
    chosen=[r for r in rs if r['incumbent'] or r['name']=='Label-frequency correction']
    checks=[]
    with threadpool_limits(limits=1):
        for r in chosen:
            clear_caches();pred=e.predict(a,q,r['config'],ex)
            y=e.Y.copy();raw=e.DATA.loc[e.LAB[h],e.LABELS].copy()
            try:
                e.Y[h]=1-e.Y[h];e.DATA.loc[e.LAB[h],e.LABELS]=1-raw.to_numpy()
                clear_caches();changed=e.predict(a,q,r['config'],ex)
            finally:
                e.Y[:]=y;e.DATA.loc[e.LAB[h],e.LABELS]=raw.to_numpy();clear_caches()
            one=e.predict(a,q.iloc[:1],r['config'],ex)
            assert np.allclose(pred,changed,atol=1e-9,rtol=0)
            assert np.allclose(pred[:1],one,atol=1e-9,rtol=0)
            checks.append(dict(model=r['name'],heldout_labels_cannot_change_predictions=True,query_batch_invariance=True))
    result=dict(passed=True,runner_sha256=p['runner_sha256'],pooled_gold_rows=235,gold_embedding_groups=222,
        no_conflicting_gold_embedding_groups=True,checks=checks,folds_verified=True,all_configs_and_rules_unchanged=True)
    b.save(path,result);clear_caches();return result

def compare(rs,outputs,p):
    base=outputs[next(i for i,r in enumerate(rs) if r['incumbent'])]
    train=e.pd.read_csv(ROOT/'experiments/benchmark_v1/comparison.csv').set_index('candidate')
    rows=[]
    for r,out in zip(rs,outputs):
        delta=np.array([x['sample_f1'] for x in out['repeat_scores']])-np.array([x['sample_f1'] for x in base['repeat_scores']])
        rows.append(dict(candidate=r['name'],incumbent=r['incumbent'],training_cv_mean=float(train.loc[r['name'],'cv_mean']),
            training_cv_std=float(train.loc[r['name'],'repeat_std']),pooled_cv_mean=out['cv_mean'],pooled_cv_std=out['repeat_std'],
            original_validation_f1=(r.get('recorded_validation') or {}).get('sample_f1'),
            pooled_paired_difference=float(delta.mean()),pooled_repeats_improved=int(np.sum(delta>1e-12)),
            pooled_result_key=out['key']))
    table=e.pd.DataFrame(rows)
    for column in ['training_cv_mean','pooled_cv_mean','original_validation_f1']:
        table[column+'_rank']=table[column].rank(method='min',ascending=False).astype(int)
    table.to_csv(OUT/'comparison.csv',index=False)
    b.save(OUT/'summary.json',dict(protocol=p['version'],protocol_sha256=b.digest(p),folds_sha256=p['folds_sha256'],
        updated_at=b.now(),model_count=len(rs),seeds=b.SEEDS,models=table.to_dict('records'),
        final_all_label_refit_performed=False,current_submission_replaced=False,
        original_validation_scores_reused=True,standalone_validation_evaluations_performed=0))
    print(table[['candidate','training_cv_mean','pooled_cv_mean','original_validation_f1']].sort_values('pooled_cv_mean',ascending=False).to_string(index=False))

def main():
    parser=argparse.ArgumentParser();parser.add_argument('--verify-only',action='store_true');parser.add_argument('--integrity-only',action='store_true');args=parser.parse_args()
    OUT.mkdir(exist_ok=True);before=protected_hashes();original_ids=e.DATA.ID.tolist()
    with pooled_context():
        p=protocol();rs=registry()
        if args.verify_only:print('Verified pooled protocol and unchanged recipe cohort.')
        elif args.integrity_only:print(json.dumps(integrity(p,rs),indent=2))
        else:
            integrity(p,rs);outputs=[b.evaluate(r,p) for r in rs];compare(rs,outputs,p)
    assert e.DATA.ID.tolist()==original_ids and len(e.Y)==157
    assert protected_hashes()==before,'A protected model, primary fold file, or input changed.'
    b.save(OUT/'preservation.json',dict(raw_inputs_and_current_model_unchanged=True,primary_folds_unchanged=True,
        original_engine_context_restored=True,protected_sha256=before))

if __name__=='__main__':main()


Writing current_reproduction/benchmark_pooled_cv.py


In [6]:
%%writefile current_reproduction/build_benchmark_report.py
"""Document the common CV protocol and side-by-side existing validation results."""
from pathlib import Path
import json,nbformat as nbf

ROOT=Path(__file__).resolve().parent;B=ROOT/'experiments/benchmark_v1'
s=json.loads((B/'summary.json').read_text());p=json.loads((B/'protocol.json').read_text())
base=next(r for r in s['models'] if r['incumbent'])
nb=nbf.v4.new_notebook();nb.metadata.kernelspec=dict(display_name='Python (AML Midterm)',language='python',name='aml-midterm')
nb.cells=[nbf.v4.new_markdown_cell(f'''# One common CV benchmark for model comparison

The primary comparison now uses the same saved grouped five-fold splits, repeated with seeds **20261126, 20261127 and 20261128**, for every model. The current hybrid scores **{base['cv_mean']:.6f} mean CV sample-wise F1**, with repeat standard deviation **{base['repeat_std']:.6f}**. Its previously measured validation F1 remains **{base['validation_f1']:.6f}**.

This benchmark runs no additional validation evaluation and makes no model promotion. Historical CV values are preserved but are not substitutes for this common comparison score. The Kaggle score is unmeasured.'''),
nbf.v4.new_code_cell("from pathlib import Path\nimport json,pandas as pd,numpy as np\nroot=Path.cwd();bench=root/'experiments/benchmark_v1'\nsummary=json.loads((bench/'summary.json').read_text())\ntable=pd.read_csv(bench/'comparison.csv')\ntable[['candidate','cv_mean','repeat_std','paired_cv_difference','repeats_improved','validation_f1']]"),
nbf.v4.new_markdown_cell('''## What is fixed

The protocol stores the actual fold memberships and trial IDs, not only random seeds. A SHA-256 hash identifies the training CSV and another identifies the complete fold assignments. Identical embeddings, rounded to ten decimal places, form one group; multilabel stratification balances the groups across folds. Each gold row is held out exactly once per repeat. Matching originally masked duplicates are removed from the fold's training pool.

Recipes and final decoding rules are frozen before benchmarking. Each fitted transform sees only the outer training gold and permitted unlabeled training examples. Inner calibration, feature selection or parameter tuning, when part of a recipe, must remain inside that outer training fold. The runner passes query features without label columns and does not select a threshold from outer-fold outcomes.

For each repeat, the metric is sample-wise F1 over the complete out-of-fold prediction matrix, giving equal weight to each trial. The headline is the mean of the three repeat scores. The repeat standard deviation describes split sensitivity; it is not a confidence interval. Macro F1, per-reason F1, every fold's score, raw scores and decoded OOF predictions are also saved.

All models use the same outer folds, so their repeat-by-repeat differences are paired. A positive mean difference with improvement in at least two of three repeats is a descriptive shortlist criterion, not a significance test or an automatic replacement rule.'''),
nbf.v4.new_code_cell("rows=[]\nfor r in summary['models']:\n    result=json.loads((bench/(r['result_key']+'.json')).read_text())\n    for seed,score in zip(summary['seeds'],result['repeat_scores']):\n        rows.append(dict(candidate=r['candidate'],seed=seed,sample_f1=score['sample_f1'],macro_f1=score['macro_f1']))\npd.DataFrame(rows).pivot(index='candidate',columns='seed',values='sample_f1')"),
nbf.v4.new_markdown_cell('''## Validation and honest interpretation

The comparison table copies already measured validation scores from the frozen recipe records. These recipes have not changed, so this benchmark does not refit or rescore validation. Incoming recipes should first pass through the same CV comparison; shortlist them, freeze them, and then evaluate validation in a small recorded batch.

Agreement between CV and validation is useful evidence, but these are development results. Earlier work used the training labels and repeatedly consulted validation. New split seeds do not turn those labels into independent test data. Continued search can also overfit the fixed CV folds. The later-year Kaggle population can differ from both the labeled training population and validation, so any chronological stress test should be reported separately and should not replace the primary CV metric.

The retained hybrid keeps its 0.912393 validation score. There is no final all-label refit and no transductive fitting on validation/test inputs in this main benchmark. The separately authorized pooled CV in notebook 11 uses original validation rows only within the training portion of a pooled fold; its results remain a distinct supplementary measurement.'''),
nbf.v4.new_markdown_cell('''## Adding a candidate

Save a JSON object with `name`, `config`, and `rule` for an implemented fold-local recipe, then run:

```sh
.venv/bin/python benchmark_cv.py --register candidate.json
```

An optional `validation` object can reference an already recorded evaluation. Otherwise its validation entry stays empty. New methods first need an adapter compatible with the experiment engine's training-position, query-feature and excluded-group interface; registration alone cannot implement an arbitrary external model.

Running `.venv/bin/python benchmark_cv.py` regenerates the comparison table from cached results where the protocol, recipe, rule and source hashes match. `--verify-only` checks the saved folds, training hash, group separation and duplicate-exclusion counts. Changing a recipe requires a new versioned name. Keep the protocol fixed across searches.

Source is embedded in notebook 04. `AGENTS.md` records this convention for future work in the project. For the methodological background, see [scikit-learn's cross-validation guidance](https://scikit-learn.org/stable/modules/cross_validation.html).'''),
nbf.v4.new_code_cell("protocol=json.loads((bench/'protocol.json').read_text())\n{k:protocol[k] for k in ['version','training_sha256','folds_sha256','seeds','folds_per_repeat','metric','limitation']}"),
nbf.v4.new_code_cell("assert summary['validation_evaluations_performed']==0\nassert summary['model_promotions_performed']==0\nprint('Common comparison complete; existing validation metrics reused; current submission retained.')")]
nbf.write(nb,ROOT/'midterm/10_common_cv_benchmark.ipynb')
print(json.dumps(base,indent=2))


Overwriting current_reproduction/build_benchmark_report.py


In [7]:
%%writefile current_reproduction/build_current_report.py
"""Build a current, auditable Part 1 report without rerunning model selection."""
from pathlib import Path
import json,hashlib,shutil,zipfile,re,datetime
import numpy as np,pandas as pd,nbformat as nbf
from nbconvert import HTMLExporter
ROOT=Path(__file__).resolve().parent
P=ROOT/'experiments'
rows=[json.loads(s) for s in (P/'trials.jsonl').read_text().splitlines()]
flat=[]
for i,r in enumerate(rows):
 z={k:v for k,v in r.items() if not isinstance(v,(list,dict))};z['log_row']=i;z['config_json']=json.dumps(r['config'],sort_keys=True);z['config']=z['config_json'];flat.append(z)
pd.DataFrame(flat).to_csv(P/'trials.csv',index=False)
validation=[]
files=[P/'validation-once.json',P/'final-validation.json',P/'phase3-reference-validation.json']+sorted(P.glob('phase*-validation-results.json'))
for path in files:
 if not path.exists():continue
 obj=json.loads(path.read_text())
 if path.name=='validation-once.json':items=obj['candidates']
 elif path.name=='phase3-reference-validation.json':items=obj['results']
 elif isinstance(obj,list):items=obj
 else:items=[obj]
 for i,r in enumerate(items):
  m=r.get('validation',r)
  score=m.get('sample_f1',r.get('validation_sample_f1'))
  if score is None:continue
  config=r.get('config',{'members_by_label':r.get('members_by_label')});name=r.get('name',r.get('family',config.get('model','candidate_'+str(i))))
  validation.append(dict(batch=path.stem,candidate=name,rule=r.get('rule'),sample_f1=score,macro_f1=m.get('macro_f1',r.get('validation_macro_f1')),config_json=json.dumps(config,sort_keys=True)))
val=pd.DataFrame(validation);val.to_csv(P/'validation-comparisons.csv',index=False)
frozen=json.loads((P/'current-best.json').read_text());candidate=frozen['candidate'];m=candidate['validation']
assert abs(val.sample_f1.max()-frozen['best_validation_f1'])<1e-12,'Update current-best.json before reporting a new winner.'
# Descriptive uncertainty only: repeated model selection invalidates a pristine-test interpretation.
v=pd.read_csv(ROOT/'data/part1_val.csv');a=pd.read_csv(ROOT/'predictions/current_best/validation_audit.csv')
g=pd.util.hash_pandas_object(v[[f'emb_{i}' for i in range(150)]].round(10),index=False).to_numpy();groups=np.unique(g);tot=np.array([a.loc[g==k,'sample_f1'].sum() for k in groups]);counts=np.array([(g==k).sum() for k in groups]);rng=np.random.default_rng(20261104);draw=rng.integers(len(groups),size=(5000,len(groups)));boot=tot[draw].sum(1)/counts[draw].sum(1)
ci=np.quantile(boot,[.025,.975]).tolist()
summary=dict(updated_at=datetime.datetime.now(datetime.timezone.utc).isoformat(),logged_cv_runs=len(rows),successful_cv_runs=sum(r['status']=='ok' for r in rows),failed_attempts=sum(r['status']!='ok' for r in rows),distinct_configs=len({r['id'] for r in rows}),validation_candidate_evaluations=len(val),validation_batches=len(val.batch.unique()),best_validation_f1=m['sample_f1'],validation_macro_f1=m['macro_f1'],descriptive_group_bootstrap_95=ci,validation_embedding_groups=len(groups),loo_current_selected=frozen.get('leave_one_embedding_group_out_f1'),target=.92735,target_met=m['sample_f1']>.92735,kaggle_score=None,limitation='Repeated CV and validation selection cause optimism; bootstrap interval is descriptive and does not adjust for selection. Equivalent model variants and verification comparisons can recur in the validation ledger.')
summary['additional_fixed_policy_cv']=frozen.get('additional_fixed_policy_cv')
summary['reported_leader']='0.93xx; exact score not provided'
summary['above_reported_leader_lower_bound']=m['sample_f1']>.93
primary_cv_text=f"Historical model-selection repeated CV F1: {candidate['ordinary_cv']:.8f}."
pooled_cv_text=''
benchmark_path=P/'benchmark_v1/summary.json'
if benchmark_path.exists():
 bench=json.loads(benchmark_path.read_text())
 registry=json.loads((P/'benchmark_v1/registry.json').read_text())
 matching=[r for r in registry if r['config']==candidate['config'] and r['rule']==candidate['rule']]
 if matching:
  common=next(r for r in bench['models'] if r['candidate']==matching[0]['name'])
  summary['common_cv_benchmark']={**common,'protocol':bench['protocol'],'seeds':bench['seeds']}
  primary_cv_text=f"Primary common grouped five-fold CV F1: {common['cv_mean']:.8f}; repeat standard deviation: {common['repeat_std']:.5f}.\nSame saved folds and seeds {bench['seeds']} for every candidate. See notebook 10 and experiments/benchmark_v1/comparison.csv."
pooled_path=P/'pooled_benchmark_v1/summary.json'
if pooled_path.exists():
 pooled=json.loads(pooled_path.read_text())
 pooled_registry=json.loads((P/'pooled_benchmark_v1/registry.json').read_text())
 matching=[r for r in pooled_registry if r['config']==candidate['config'] and r['rule']==candidate['rule']]
 if matching:
  pooled_row=next(r for r in pooled['models'] if r['candidate']==matching[0]['name'])
  summary['supplementary_pooled_cv']=pooled_row
  pooled_cv_text=f"Supplementary pooled train-plus-validation CV F1: {pooled_row['pooled_cv_mean']:.8f}; repeat standard deviation: {pooled_row['pooled_cv_std']:.5f}. This evaluates fresh fold fits of the same recipe on 235 pooled labels, not the exact saved model. See notebook 11."
(P/'current-summary.json').write_text(json.dumps(summary,indent=2))
additional_cv_text=''
if frozen.get('additional_fixed_policy_cv'):
 audit=frozen['additional_fixed_policy_cv']
 additional_cv_text=f"Additional fixed-policy grouped five-fold audit on seeds {audit['seeds']}: {audit['mean_sample_f1']:.8f} (split standard deviation {audit['split_std']:.5f}). Different split sets give different estimates; these are development audits, not a nested replay of the full search.\n"
# Add technique references; these sources contain no added trial data.
refs=json.loads((P/'references.json').read_text())
add=[('Joachims (1999), Transductive Inference for Text Classification using Support Vector Machines','https://www.cs.cornell.edu/people/tj/publications/joachims_99c.pdf','Motivation for low-density semi-supervised margins. Our alternating pseudo-label procedure is a heuristic, not an implementation of the exact TSVM optimization algorithm.'),('Huang et al., Correcting Sample Selection Bias by Unlabeled Data','https://papers.nips.cc/paper_files/paper/2006/hash/a2186aa7c086b46ad4e8bf81e2a3a19b-Abstract.html','Kernel mean matching weights, estimated only against original masked training features.'),('On Mixup Regularization','https://www.jmlr.org/papers/v23/20-1385.html','Motivation for synthetic interpolation; evaluated with classical logistic/SVM learners only.'),('Nystroem kernel approximation, scikit-learn','https://scikit-learn.org/stable/modules/generated/sklearn.kernel_approximation.Nystroem.html','Classical finite kernel features for comparing squared hinge, logistic and squared losses.')]
for title,url,use in add:
 if not any(r['url']==url for r in refs):refs.append(dict(title=title,url=url,use=use))
(P/'references.json').write_text(json.dumps(refs,indent=2))
text=f'''AML MIDTERM — PART 1 — CURRENT RESULTS

Best measured validation sample-wise F1: {m['sample_f1']:.8f}
Validation macro F1: {m['macro_f1']:.8f}
Earlier validation target: strictly above 0.92735. Target met: {summary['target_met']}.
Latest reported Kaggle leader: 0.93xx, exact score not provided. Local validation above 0.93: {summary['above_reported_leader_lower_bound']}.
This modestly exceeds the user's reported 0.9103 validation result; it does not establish a statistically reliable improvement.
{primary_cv_text}
{pooled_cv_text}
Kaggle score: not measured. Validation and CV are not leaderboard scores.

CURRENT FILES
predictions/current_best/part1_submission.csv — 312 rows, ID and Predicted.
predictions/current_best/part1_train_augmented.csv — all 157 original gold labels preserved; 1,014 missing-label rows predicted.
predictions/current_best/manifest.json — exact model, metrics, source hashes and submission hash.
midterm/03_current_report.ipynb and .html — current report and figures.
midterm/04_reproduce_current.ipynb — embedded source and executed reproduction check.
midterm/05_tri_training_and_shift.ipynb — tri-training, target-aware selection, matched controls and results (when present).
midterm/06_stability_and_specialists.ipynb — bootstrap stability, feature additions and nested score specialists (when present).
midterm/07_cotraining_and_distances.ipynb — separate-view co-training, learned distances and matched controls (when present).
midterm/08_svm_refinements.ipynb — per-reason tuning, reliability weights and local SVM specialists (when present).
midterm/09_shift_graph_and_subgroups.ipynb — label-frequency correction, graph SVMs and Other subgroups (when present).
midterm/10_common_cv_benchmark.ipynb — primary common-fold model comparison with separate existing validation scores (when present).
midterm/11_pooled_cv_comparison.ipynb — supplementary train-plus-validation CV of frozen leading recipes (when present).
experiments/benchmark_v1/protocol.json and comparison.csv — saved fold assignments and primary comparison table (when present).
experiments/validation-comparisons.csv — every recorded validation comparison, including failures to improve.
experiments/trials.csv and trials.jsonl — {len(rows)} logged CV runs, {summary['distinct_configs']} distinct configurations, {summary['failed_attempts']} failed attempts retained.
Full-size CV and ensemble-only screens have separate files; they are not all counted in the main trial log.

REPRODUCE
From this directory (or the portable midterm directory):
  .venv/bin/python reproduce_current.py
For a new environment, use Python 3.12 and requirements-lock.txt.
The script reads the frozen experiments/current-best.json configuration, trains only on supplied training rows, writes predictions, and then verifies its validation score. It does not tune on validation labels.

CURRENT METHOD
A fixed per-reason combination uses the previous semi-supervised SVM for Safety, Funding and Efficacy, and classical tri-training for Recruitment and Other. This combination was selected from 32 masks using training-fold predictions and the frozen robust weighting criterion before validation evaluation. Empty combined predictions fall back to the base SVM's decoded labels.
Both components fit StandardScaler to gold and original masked training embeddings, retaining all 150 coordinates. No validation/test features enter the selected inductive fitters.
Base component: KMeans with 10 clusters, 10 initializations and seed 42, then five balanced RBF SVMs with C=1 and gamma=0.001. At each pseudo-label weight 0.003, 0.01 and 0.03, perform three label/refit iterations. Original gold labels remain fixed. If a cluster has at least two gold rows and at least 80% agreement for a binary reason, conflicting pseudo-labels receive one tenth the normal weight. Transform margins with sigmoid(2*margin), threshold at 0.55, and use the largest score if all labels would be empty.
Tri-training component: ordinary bootstrap samples initialize an RBF SVM (C=1, gamma=0.001), logistic regression (C=0.3), and ridge classifier (alpha=10). Peer agreement and the classical error/size conditions govern pseudo-label updates, with unit pseudo-label weight and no cluster gate in this component. Majority voting at threshold 0.5 decodes predictions; smoothed vote scores break ties for empty-label fallback. At most eight update rounds are used.
All 157 gold and 1,014 originally masked training rows are available for final inference. Validation remains excluded. Final hybrid scores are binary decisions; component margin/vote scores are not calibrated probabilities.
Earlier screens tested auxiliary criteria, disease, drug, target, action, pathway, structure and date features. They did not improve the earlier incumbent on validation; drug-target additions tied it. The new hybrid uses embeddings only. Exhaustive feature-subset testing of this new hybrid has not been performed.

EVIDENCE AND LIMITATIONS
{len(val)} recorded standalone validation candidate evaluations across {summary['validation_batches']} batches, including equivalent variants. Their hyperparameters and decision rules were chosen from training-side experiments before each small validation batch. Those original fits excluded validation labels and made no per-row corrections. Repeated aggregate comparisons still create selection bias: this is no longer a pristine holdout. The separately authorized pooled CV deliberately uses original validation labels only in the training portion of each fold, with all held-out groups excluded; it does not replace the saved training-only model or reinterpret its original validation score.
Five-fold iterative multilabel splits group identical embeddings and exclude matching masked duplicates from training. Repeats use three seeds. Leave-one-group-out evaluates 150 distinct gold embedding groups. All learned transformations stay inside their permitted folds. Temporal stress tests are affected by the intentional rare-label enrichment of later gold rows.
Descriptive grouped validation bootstrap interval: {ci[0]:.4f}–{ci[1]:.4f}; this does not correct selection bias.
The search is broad, not an exhaustive enumeration of every possible algorithm or feature subset. No external trial data, extra labels, ID predictors or deep learning were used. Literature was consulted only for methods. Some explicitly marked research variants used query features transductively; the selected candidate does not.

HISTORY
Historical model-selection repeated CV F1: {candidate['ordinary_cv']:.8f}; leave-one-embedding-group-out development F1: {summary['loo_current_selected']:.8f}. These historical diagnostics do not replace the primary common benchmark.
{additional_cv_text}
The earlier delivered full-fit ensemble scored 0.850427 on validation; its 0.936306 training CV did not transfer. It is preserved under the older filenames and in notebooks 01/02, which describe that historical stage. The previous 0.910256 validation incumbent is preserved under predictions/checkpoints/pre_phase21_091026 and experiments/phase21-prior-incumbent.json. Use the current_best directory and notebooks 03/04/05 for the current candidate.
Part 2 and the combined five-page course report remain outside this Part 1 work.
'''
(ROOT/'READ_ME_FIRST.txt').write_text(text)
# Current report contains executable tables/figures, while full source is embedded separately.
nb=nbf.v4.new_notebook();nb.metadata.kernelspec=dict(display_name='Python (AML Midterm)',language='python',name='aml-midterm')
nb.cells=[nbf.v4.new_markdown_cell('# Part 1: current independent experiments\n\n'+text.split('CURRENT FILES')[0]),
 nbf.v4.new_code_cell("from pathlib import Path\nimport json, pandas as pd, numpy as np\nimport matplotlib.pyplot as plt\nroot=Path.cwd()\nsummary=json.loads((root/'experiments/current-summary.json').read_text())\nsummary"),
 nbf.v4.new_markdown_cell('## Selected method and validation discipline\n\n'+text.split('CURRENT METHOD\n')[1].split('\nHISTORY')[0]),
 nbf.v4.new_code_cell("ledger=pd.read_csv(root/'experiments/validation-comparisons.csv')\nledger.groupby('batch',sort=False).agg(candidates=('sample_f1','size'),best_sample_f1=('sample_f1','max'),best_macro_f1=('macro_f1','max'))"),
 nbf.v4.new_markdown_cell('## Current validation performance\n\nPer-reason F1 complements the competition’s sample-wise F1. The latest reported Kaggle leader is in the 0.93xx range; its exact score is unknown. Our earlier 0.92735 target and the newer leaderboard reference both concern sample-wise F1, measured on different evaluation datasets.'),
 nbf.v4.new_code_cell("current=json.loads((root/'experiments/current-best.json').read_text())\nm=current['candidate']['validation']\nlabels=['Safety','Funding','Efficacy','Recruitment','Other']\nfig,ax=plt.subplots(figsize=(7,3.4))\nax.bar(labels,[m['f1_'+x] for x in labels],color='#2563a0')\nax.set(ylim=(0,1.04),ylabel='Validation label F1',title=f\"Sample F1 {m['sample_f1']:.5f}; macro F1 {m['macro_f1']:.5f}\")\nfig.tight_layout()\n(root/'figures').mkdir(exist_ok=True)\nfig.savefig(root/'figures/current_validation_labels.png',dpi=180)\nplt.show()"),
 nbf.v4.new_markdown_cell('## Feature additions and model coverage\n\nFeature settings below use training CV, including multiple weights. Their maxima are development scores, not independent estimates. Broad feature groups were added individually; selected groups were combined in earlier forward searches. No auxiliary group improved the earlier incumbent on validation. The new tri-training hybrid uses embeddings only; it has not undergone an exhaustive feature-subset search.'),
 nbf.v4.new_code_cell("trials=[json.loads(s) for s in (root/'experiments/trials.jsonl').read_text().splitlines()]\nfeatures=[dict(features=', '.join(r['config'].get('features',[])),weight=r['config'].get('feature_weight'),cv=r['score']) for r in trials if r['tag']=='phase6_features' and r['status']=='ok']\npd.DataFrame(features).sort_values('cv',ascending=False)"),
 nbf.v4.new_code_cell("ok=[r for r in trials if r['status']=='ok']\npd.DataFrame([dict(model=r['config']['model'],cv=r['score']) for r in ok]).groupby('model').agg(logged_runs=('cv','size'),best_development_cv=('cv','max')).sort_values('best_development_cv',ascending=False)"),
 nbf.v4.new_markdown_cell('## Reproduction and historical results\n\nRun notebook 04 to regenerate the current predictions with embedded source. All raw data hashes and original gold labels are checked. Notebooks 01/02 preserve the earlier 0.850427 validation model; their final-candidate wording is historical. The newer notebooks and current_best directory supersede that candidate.\n\nThe full experiment log retains unsuccessful trials and error records. Phase 10 initially had a non-contiguous sample-weight array error, which was corrected before valid results. The first phase 17 XGBoost run emitted an unused `lambda_` warning and used native default L2=1; the source now spells the same effective value as `reg_lambda=1`. No measured hyperparameter changed in this correction.'),
 nbf.v4.new_markdown_cell('## Technique references\n\n'+'\n\n'.join(f"[{r['title']}]({r['url']}) — {r['use']}" for r in refs))]
mid=ROOT/'midterm';mid.mkdir(exist_ok=True)
nbf.write(nb,mid/'03_current_report.ipynb')
# Reproduction notebook embeds every Python source, including search runners and this report builder.
source_files=sorted(p.name for p in ROOT.glob('*.py'))
rep=nbf.v4.new_notebook();rep.metadata.kernelspec=nb.metadata.kernelspec
rep.cells=[nbf.v4.new_markdown_cell('# Reproduce the current Part 1 candidate\n\nThis notebook embeds all experiment Python source. It creates an isolated replay folder, regenerates the frozen candidate, checks raw hashes and gold-label preservation, and compares the submission hash. It does not rerun model selection.'),nbf.v4.new_code_cell("from pathlib import Path\nimport shutil,sys,subprocess,json,hashlib\nroot=Path.cwd(); replay=root/'current_reproduction';replay.mkdir(exist_ok=True)\nshutil.copytree(root/'data',replay/'data',dirs_exist_ok=True)\n(replay/'experiments').mkdir(exist_ok=True)\nfor name in ['current-best.json','data-audit.json']:\n    shutil.copy2(root/'experiments'/name,replay/'experiments'/name)")]
for name in source_files:rep.cells.append(nbf.v4.new_code_cell('%%writefile current_reproduction/'+name+'\n'+(ROOT/name).read_text()))
rep.cells.extend([nbf.v4.new_code_cell("result=subprocess.run([sys.executable,'reproduce_current.py','--output','result'],cwd=replay,check=True,text=True,capture_output=True)\nprint(result.stdout)\nexpected=json.loads((root/'predictions/current_best/manifest.json').read_text())['submission_sha256']\nactual=hashlib.sha256((replay/'result/part1_submission.csv').read_bytes()).hexdigest()\nassert actual==expected,(actual,expected)\nprint('Exact current submission reproduced:',actual)"),nbf.v4.new_markdown_cell('To replay searches, use the phase scripts and their frozen config JSON files. `experiment.py --help` lists the original search entry point. Most later `phaseN.py` scripts run their own training-CV screen; `continued_select.py phaseN` performs documented repeat-CV selection and validation. Running these intentionally expands the experiment log and repeats validation; it is not required to reproduce the frozen candidate.')])
nbf.write(rep,mid/'04_reproduce_current.ipynb')
# Portable directory mirrors inputs and artifacts; keep historical notebooks and outputs.
for name in ['data','experiments','predictions']:
 shutil.copytree(ROOT/name,mid/name,dirs_exist_ok=True)
for file in list(ROOT.glob('*.py'))+[ROOT/'READ_ME_FIRST.txt',ROOT/'requirements-lock.txt']:
 shutil.copy2(file,mid/file.name)
if (ROOT/'AGENTS.md').exists():shutil.copy2(ROOT/'AGENTS.md',mid/'AGENTS.md')
print(json.dumps(summary,indent=2))


Overwriting current_reproduction/build_current_report.py


In [8]:
%%writefile current_reproduction/build_deliverables.py
"""Compile the experiment ledger, figures, executed notebooks and portable coursework folder."""
from pathlib import Path
import json, shutil, hashlib, os
import numpy as np,pandas as pd
import nbformat as nbf
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
ROOT=Path(__file__).resolve().parent
os.chdir(ROOT)
rows=[json.loads(s) for s in Path('experiments/trials.jsonl').read_text().splitlines()]
trial=pd.DataFrame([{**r,'config':json.dumps(r['config'],sort_keys=True)} for r in rows]);trial.to_csv('experiments/trials.csv',index=False)
ok=[r for r in rows if r['status']=='ok']
def family(r):
 c=r['config'];return 'cluster: '+c['cluster'] if c['model']=='cluster' else c['model']
summary=pd.DataFrame([dict(family=family(r),score=r['score'],default_score=r['default_score'],id=r['id'],seconds=r['seconds']) for r in ok]).groupby('family').agg(best_development_f1=('score','max'),best_default_f1=('default_score','max'),configurations=('id','nunique'),cv_runs=('id','size'),seconds=('seconds','sum')).sort_values('best_development_f1',ascending=False)
summary.to_csv('experiments/model-family-summary.csv')
# Individual feature screening selected configurations on the ORIGINAL seed only.
features=[]
base=next(r for r in ok if r['tag']=='screen' and r['config']==dict(model='svc',C=10,gamma=.001,scale='standard'))
for block in sorted({r['config']['features'][0] for r in ok if r['tag']=='features'}):
 r=max([r for r in ok if r['tag']=='features' and r['config'].get('features')==[block]],key=lambda r:r['score'])
 fresh=[x for x in ok if x['id']==r['id'] and x['tag']=='robustness']
 from experiment import metrics,Y
 freshscores=[]
 for x in fresh:
  p=np.load(f"experiments/{x['tag']}-{x['id']}-{x['seed']}.npz",allow_pickle=True)['p'];freshscores.append(metrics(Y,p,r['rule'])['sample_f1'])
 features.append(dict(feature=block,model=r['model'],weight=r['config']['feature_weight'],first_seed_f1=r['score'],rule_frozen_on_first_seed=r['rule'],fresh_seed_mean_f1=float(np.mean(freshscores)) if freshscores else np.nan,n_fresh_seeds=len(freshscores)))
pd.DataFrame(features).to_csv('experiments/feature-ablation.csv',index=False)
val=json.loads(Path('experiments/validation-once.json').read_text());final=json.loads(Path('experiments/final-candidate.json').read_text());nested=json.loads(Path('experiments/nested-audit.json').read_text());audit=json.loads(Path('experiments/data-audit.json').read_text())
readme=f'''AML MIDTERM — PART 1

CURRENT STATUS
The requested winning Kaggle score has NOT been established.
The first frozen 3-candidate holdout comparison achieved best sample-F1 {val['selected_sample_f1']:.6f} on 78 validation trials. It did not improve on the user's approximate previous validation result. Its grouped-bootstrap 95% interval is {val['bootstrap_95_interval'][0]:.4f}–{val['bootstrap_95_interval'][1]:.4f}.
The final FULL-FIT candidate was subsequently selected using TRAINING DATA ONLY. It achieves {final['selected']['fullsize_oof_f1']:.6f} leave-one-embedding-group-out development F1. This is NOT a Kaggle score or a new external validation result. No validation-error-driven retuning was performed. The conditional nested shortlist audit scored {nested['sample_f1']:.6f}; the candidate shortlist itself was informed by exploratory CV, so this is not an unbiased nested rerun of the entire search.

FILE TO TRY ON KAGGLE
predictions/part1_fullfit_candidate.csv
312 rows, columns ID and Predicted, matching sample-submission order. Awaiting a user-run Kaggle evaluation; credentials were not configured here. Do not use a development score to claim a leaderboard win.

PRIMARY READING
midterm/01_experiment_report.ipynb — executed report with tables, figures, limitations and references.
midterm/01_experiment_report.html — browser-readable version.
midterm/02_reproduce.ipynb — all experiment source embedded and exact commands.
experiments/trials.csv / trials.jsonl — {len(rows)} logged CV runs, {len({r['id'] for r in rows})} distinct configurations; {sum(r['status']=='error' for r in rows)} failed attempt(s) retained. Repeated seeds count as additional runs. Nested and 600 leave-group-out member fits are recorded separately.
experiments/protocol.txt, phase2-protocol.txt — evaluation rules and chronological changes.
experiments/final-candidate.json — final configuration, provenance and checksum.
predictions/part1_train_augmented_fullfit.csv — original 157 labeled rows preserved; 1,014 masked rows receive predicted labels, with label_source recorded.

REPRODUCE THE FINAL CANDIDATE
From the AMLMidterm project directory, or the portable midterm directory:
  uv venv --python 3.12 .venv
  uv pip install --python .venv/bin/python -r requirements-lock.txt
  .venv/bin/python make_candidate.py
On macOS, XGBoost/LightGBM additionally require Homebrew libomp for the full model screen; the selected final model does not require these two packages at inference.
The executed notebook lists replay commands. Raw data SHA256 hashes are in experiments/data-audit.json. The first validation pipeline is retained as experiment_v1.py and uses its own recorded engine hash.

FINAL METHOD
50%: independent logistic models (C=0.3) with rotation-preserving centered global scaling of all 150 embeddings and 1024-bit Morgan fingerprints (radius 2; max across matched molecules; zero if unavailable). Fingerprint block has relative weight 0.5. HDBSCAN on 25-dimensional, train-fitted standardized PCA supplies a cluster-label teacher. Teacher and supervised predictions must agree; their mean confidence must reach 0.95. Accepted pseudo-labels enter per-label logistic fits at weight 0.05. True labels are never replaced.
25%: standardized-embedding RBF SVMs (C=10, gamma=0.001) plus word/bigram TF-IDF drug names and criteria (relative weight 0.5 each).
25%: the same SVM family plus drug-target TF-IDF (relative weight 1.0).
Each member trains on all 157 original labeled rows and legitimate masked training rows as applicable. Probabilities are combined with fixed 0.50/0.25/0.25 weights. Threshold 0.35, with argmax fallback when all five probabilities fall below threshold.

INTERPRETATION
The search was broad and structured, not an exhaustive enumeration of every possible model, feature subset or hyperparameter. Individual rotated embedding coordinates have no semantic meaning; we retained the block and compared scaling/PCA instead of claiming coordinate-level feature explanations. Source fields were added individually, with several weights, followed by selected forward combinations and an all-features stress test.
External validation was used once for a frozen 3-candidate comparison. Later work did not recheck it. All search/CV values are development estimates and can be optimistic after extensive selection. The training labeled subset is temporally biased and enriched for rare reasons; chronological stress tests are also confounded by this enrichment. The small data do not support a guaranteed score.
No new clinical-trial records, external labels, test-label reverse engineering, ID predictors, or deep learning were used. Only provided CSV inputs and classical methods. Supplied pretrained embeddings are input data, not a newly trained deep model.
Part 2 and the final combined five-page course report are not completed by this Part 1 work.
'''
Path('READ_ME_FIRST.txt').write_text(readme)
# Notebook with directly executable plotting source.
cells=[]
def md(s):cells.append(nbf.v4.new_markdown_cell(s))
def code(s):cells.append(nbf.v4.new_code_cell(s))
md('# AML Midterm — Part 1 experiment report\n\n**Objective not yet verified:** beat Kaggle 0.92735 without external trial records, deep learning, or validation-set overfitting. The user’s prior Kaggle score was 0.89423. Neither number is comparable directly to a training CV estimate.\n\n**Evidence:** the first frozen validation comparison scored **0.8504**. A later full-training candidate has **0.9363 grouped leave-one-out development F1**, but no new external score. The first result is preserved; the report does not present a development result as a leaderboard win.')
code('''from pathlib import Path
import json, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
root = Path.cwd()
if not (root / 'experiments').exists():
    root = root.parent
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})
figdir = root / 'figures'; figdir.mkdir(exist_ok=True)
labels = ['Safety','Funding','Efficacy','Recruitment','Other']
train = pd.read_csv(root/'data/part1_train.csv')
known = train[labels].ge(0).all(axis=1)
trials = pd.read_csv(root/'experiments/trials.csv')
val = json.loads((root/'experiments/validation-once.json').read_text())
final = json.loads((root/'experiments/final-candidate.json').read_text())
''')
md('## Data and constraints\n\n1,171 training trials: 157 labeled, 1,014 wholly masked. The 78 validation trials are separate. The 312 test trials start later, in 2010–2016. The main target is sample-wise multilabel F1; per-reason and macro F1 are also retained. There are 150 unique embedding groups among the 157 labeled rows. Exact duplicate groups remain together in CV, and their masked counterparts are excluded from fold fitting. IDs are identifiers only.\n\nFunding has only eight positive training examples. The known labels mostly come from early years, plus a selected rare-reason enrichment sample. Random-fold CV does not reproduce the future-test distribution.')
code('''display(pd.DataFrame({'positive_training_examples': train.loc[known,labels].sum()}))
fig, axes = plt.subplots(1,2,figsize=(11,3.6))
train.loc[known,labels].sum().plot.bar(ax=axes[0], color='#326b8c', rot=25)
axes[0].set(title='Visible training labels', ylabel='Positive trials')
axes[1].hist([train.loc[known,'start_year'], train.loc[~known,'start_year']], bins=range(1993,2012), label=['Labeled','Masked'], color=['#326b8c','#d79a45'])
axes[1].set(title='Training time distribution', xlabel='Start year', ylabel='Trials'); axes[1].legend()
fig.tight_layout(); fig.savefig(figdir/'data_distribution.png'); fig.savefig(figdir/'data_distribution.pdf'); plt.show()
''')
md('## Evaluation design\n\nThe broad screen used grouped multilabel-stratified five-fold CV. Scaling, TF-IDF vocabularies, PCA, clustering, label mapping, feature transforms and pseudo-labeling are fitted on training-fold labels plus permitted masked training rows. Held-out rows never participate in preprocessing or pseudo-label generation. A test verifies that changing held-out labels, query values, or IDs cannot change fitted predictions for other rows.\n\nPromising methods were repeated on three fresh split seeds. Added fields were screened individually, followed by selected combinations and an all-fields comparison. Decision policies were a small fixed threshold grid, argmax, or exact expected-F1 enumeration under independent marginal probabilities. Extensive selection makes these scores optimistic.\n\nA five-outer/three-inner conditional nested audit selected both models and decision rules inside each outer fold. It scored 0.89384. Because the shortlist was informed by earlier full-training exploration, this is not an unbiased assessment of the entire adaptive search. Chronological cutoffs add a separate stress test, but later labeled rows are intentionally enriched for rare classes.\n\nA three-candidate shortlist was frozen before the first external validation evaluation. After its disappointing result, later experiments remained training-only; no individual validation-error retuning or repeated validation scoring occurred. Leave-one-embedding-group-out CV then checked a nearly full-size training regime (150 groups, around 156 labeled rows per fit).')
md('## Model-family comparisons\n\nThe table reports each family’s **best development result**, not an unbiased ranking. Counts include successful unique configurations and CV runs; repeated seeds are separate runs. The failed full-dimensional QDA attempt is retained in the ledger; reduced-dimensional variants were subsequently evaluated.\n\nClustering includes centroid-based K-Means and bisecting K-Means, Gaussian mixtures, hierarchical agglomeration, spectral clustering, BIRCH, DBSCAN and HDBSCAN. Label propagation uses normalized graph diffusion or hard-clamped graph inference. Supervised comparisons include logistic/linear and kernel SVM, neighbors, ridge/kernel ridge, shrinkage discriminants, Gaussian NB, random/extra trees, three boosting libraries, classifier chains, powerset and soft multiclass models, Gaussian processes, NCA and PLS. Additional tests include semi-supervised EM, Laplacian-regularized kernels, prototype dictionaries and consensus pseudo-labeling.')
code('''families = pd.read_csv(root/'experiments/model-family-summary.csv')
display(families[['family','best_development_f1','configurations','cv_runs']].round(4))
fig, ax = plt.subplots(figsize=(9,9))
families.sort_values('best_development_f1').plot.barh(x='family',y='best_development_f1',ax=ax,legend=False,color='#326b8c')
ax.set(xlim=(0.55,1),xlabel='Best development sample-F1 (selection-biased)',ylabel='')
fig.tight_layout(); fig.savefig(figdir/'model_families.png'); plt.show()
''')
md('## Features and preprocessing\n\nAll 150 rotated reason coordinates are kept as a block. Standard scaling, rotation-preserving global scaling, row normalization, PCA dimensions and whitening strengths were compared. Auxiliary groups include year, eligibility word/bigram and character TF-IDF, disease names, drugs, ICD codes, matched drugs, targets, actions, pathways, broad pathway filtering, pathway-depth counts, Morgan fingerprints, molecular descriptors, missingness and field counts. Each group was initially added alone at several weights.\n\nThe first-seed result is the selection score. When available, the fresh-seed mean below uses that first-seed decision rule unchanged. A missing fresh-seed value means the configuration was screened out or another configuration was repeated; it is not evidence of a gain. Some apparent improvements disappeared across splits. Chemistry and selected drug/text blocks were retained in the final candidate because of repeated-CV and combination checks, not per-validation-row edits. More features can overwhelm a tiny labeled set, alter kernel distances and encode fragile disease/year associations. Fingerprint missingness also reflects incomplete drug matching.')
code('''ablation = pd.read_csv(root/'experiments/feature-ablation.csv')
display(ablation.round(4))
repeat = ablation.dropna(subset=['fresh_seed_mean_f1']).sort_values('fresh_seed_mean_f1')
fig, ax = plt.subplots(figsize=(9,4.5))
repeat.plot.barh(x='feature',y=['first_seed_f1','fresh_seed_mean_f1'],ax=ax,color=['#b8cdd9','#326b8c'])
ax.set(xlim=(.75,1),xlabel='Development sample-F1',ylabel=''); ax.legend(['Selected first-seed score','Fresh seeds, frozen rule'],loc='lower right')
fig.tight_layout(); fig.savefig(figdir/'feature_robustness.png'); plt.show()
''')
md('## External validation — preserve the disappointing result\n\nThese are the only three external-validation model evaluations. All model definitions and thresholds were fixed beforehand. The selected validation result has selection optimism, and 78 rows leave wide uncertainty. It is lower than the user’s approximate previous validation performance. **No leaderboard improvement is established.**')
code('''vtable = pd.DataFrame([{k:r[k] for k in ['name','sample_f1','macro_f1']+[f'f1_{l}' for l in labels]} for r in val['candidates']])
display(vtable.round(5))
print('Selected-model grouped bootstrap 95% interval:', val['bootstrap_95_interval'])
nested = json.loads((root/'experiments/nested-audit.json').read_text())
print('Conditional nested shortlist sample-F1:', nested['sample_f1'])
temporal = pd.read_json(root/'experiments/temporal-audit.json')
display(temporal[['cutoff','n_train','n_test','sample_f1','macro_f1']].round(4))
''')
md('## Final full-training candidate and augmentation\n\nThe final candidate uses all 157 true labeled rows, unlike the earlier 15-model fold-bagging prediction strategy. It is a 50% HDBSCAN-consensus pseudo-label logistic model with Morgan fingerprints, 25% RBF SVM with criteria/drug TF-IDF, and 25% RBF SVM with target TF-IDF.\n\nFor the pseudo-label component, HDBSCAN is fitted to 25 PCA coordinates of standardized reason embeddings, without labels. Cluster labels are mapped using gold fold-training rows; noise/unlabeled clusters fall back to a distance-weighted labeled-neighbor estimate. A separate logistic model supplies the second view. Only agreed binary labels with average confidence at least 0.95 are accepted into the per-label fit, each at weight 0.05. Original true labels are retained. The auxiliary fingerprint uses radius 2 and 1,024 bits, max-pooled across valid supplied molecules; unmatched/invalid molecules contribute zero.\n\nFor final predictions, member probabilities use fixed weights 0.50/0.25/0.25 and a 0.35 threshold, with highest-probability fallback for empty predictions. Every masked row receives five binary labels. Pseudo-labels are predictions, not newly verified ground truth. The final prediction recipe and SHA256 are saved in `final-candidate.json`. The final candidate was **not re-evaluated on the used validation set**. Its 0.93631 leave-group-out score remains a selection-biased development estimate, awaiting external confirmation.')
code('''display(pd.DataFrame([{'candidate':r['name'],'leave_group_out_f1':r['fullsize_oof_f1'],'rule':r['rule']} for r in json.loads((root/'experiments/fullsize-ranking.json').read_text())]))
print('Final CSV:', final['submission'])
print('Kaggle score:', final['kaggle_score'])
aug = pd.read_csv(root/'predictions/part1_train_augmented_fullfit.csv')
display(aug.groupby('label_source')[labels].sum())
print('Ground truth rows unchanged:', np.array_equal(aug.loc[known,labels].to_numpy(),train.loc[known,labels].to_numpy()))
print('Generated masked rows:', (~known).sum())
''')
md('## Limitations and next step\n\nThe requested winning score is **not achieved or verified**. Submit `predictions/part1_fullfit_candidate.csv` once and record its Kaggle result before claiming improvement. The first holdout failure is evidence against assuming that the higher CV values transfer. The final full-fit regime addresses training-size differences but does not erase that evidence.\n\nThe broad search was structured, not exhaustive. Reusing tiny training folds for many choices creates selection bias despite strict per-fit leakage controls. Rare-label estimates are unstable; the temporal sampling scheme differs from both ordinary CV and future tests. Strong uncertainty, pseudo-label error and class-distribution shifts remain. Part 2 and the final combined five-page assignment report are outside this Part 1 package.')
refs=json.loads(Path('experiments/references.json').read_text())
md('## References\n\n'+'\n\n'.join(f"- [{r['title']}]({r['url']}) — {r['use']}" for r in refs))
nb=nbf.v4.new_notebook(cells=cells,metadata={'kernelspec':{'name':'aml-midterm','display_name':'AML Midterm','language':'python'}})
nbf.write(nb,'midterm/01_experiment_report.ipynb')
# Source notebook embeds every script and every static configuration, so the source is retained in .ipynb form.
sourcecells=[nbf.v4.new_markdown_cell('# Reproduce the Part 1 experiments\n\nThis notebook embeds the complete implementation and helper scripts. It can recreate them in an isolated `reproduction` directory; original results remain unchanged. Full search is opt-in because it runs hundreds of fits. The report notebook executes directly from saved evidence. No deep learning or external trial records are used.\n\nInstall the locked Python 3.12 environment using the commands in `READ_ME_FIRST.txt`. On macOS the XGBoost/LightGBM screen needs libomp. The selected model does not. The first validation result remains separate from the later unvalidated full-fit candidate.')]
sourcecells.append(nbf.v4.new_code_cell("from pathlib import Path\nimport shutil, subprocess, sys, json\nroot = Path.cwd()\nreplica = root / 'reproduction'\nreplica.mkdir(exist_ok=True)\n(replica/'experiments').mkdir(exist_ok=True)\n(replica/'predictions').mkdir(exist_ok=True)\nshutil.copytree(root/'data', replica/'data', dirs_exist_ok=True)\n"))
for f in sorted(Path('.').glob('*.py')):
 if f.name=='build_deliverables.py':continue
 sourcecells.append(nbf.v4.new_markdown_cell('### '+f.name+'\n\nComplete source:'))
 sourcecells.append(nbf.v4.new_code_cell('source = '+repr(f.read_text())+'\n(replica / '+repr(f.name)+').write_text(source)\n'))
configs={p.name:json.loads(p.read_text()) for p in Path('experiments').glob('*-input.json')}
sourcecells.append(nbf.v4.new_code_cell('static_configs = '+repr(configs)+'\nfor name, content in static_configs.items():\n    (replica/"experiments"/name).write_text(json.dumps(content, indent=2))\n'))
sourcecells.append(nbf.v4.new_markdown_cell('## Fast exact prediction reproduction\n\nThis fits the frozen final model recipe using only training labels and the provided masked training rows. It does not read validation labels. Set `RUN_FINAL = True` to run. A fresh output checksum is compared with the observed candidate.'))
sourcecells.append(nbf.v4.new_code_cell('''RUN_FINAL = False
if RUN_FINAL:
    for name in ['fullsize-ranking.json','final-candidate.json']:
        shutil.copy2(root/'experiments'/name, replica/'experiments'/name)
    subprocess.run([sys.executable,'make_candidate.py'],cwd=replica,check=True)
    import hashlib
    observed = hashlib.sha256((root/'predictions/part1_fullfit_candidate.csv').read_bytes()).hexdigest()
    reproduced = hashlib.sha256((replica/'predictions/part1_fullfit_candidate.csv').read_bytes()).hexdigest()
    assert observed == reproduced, (observed,reproduced)
    print('Exact prediction checksum matched:', observed)
'''))
sourcecells.append(nbf.v4.new_markdown_cell('## Full search replay\n\nThe executable command list below covers all logged search stages. Repeated seeds and adaptive stage configuration files are retained in the evidence. For an exact historical replay, copy the recorded JSON input/configuration files; do not re-select model families after seeing external validation. Nested and leave-group-out audits can be run separately. `finalize.py` deliberately refuses a second external validation comparison when its result already exists.'))
commands=[['experiment.py'],['prepare_refinement.py'],['experiment.py','--stage','refine','--configs','experiments/refine-input.json'],['experiment.py','--stage','features','--configs','experiments/features-input.json'],['prepare_consensus.py'],['experiment.py','--stage','consensus','--configs','experiments/consensus-input.json'],['robustness.py'],['prepare_metric.py'],['experiment.py','--stage','metric','--configs','experiments/metric-input.json']]
for stage,cf in [('ssl_features','ssl-features'),('phase2_repeat','phase2-repeat')]:
 for seed in ([20261104,20261105,20261106] if stage=='ssl_features' else [20261105,20261106]):commands.append(['experiment.py','--stage',stage,'--configs',f'experiments/{cf}-input.json','--seed',str(seed)])
commands.extend([['experiment.py','--stage','dictionary','--configs','experiments/dictionary-input.json'],['experiment.py','--stage','phase2','--configs','experiments/phase2-input.json']])
sourcecells.append(nbf.v4.new_code_cell('RUN_SEARCH = False\ncommands = '+repr(commands)+'\nif RUN_SEARCH:\n    for command in commands:\n        subprocess.run([sys.executable,*command],cwd=replica,check=True)\n'))
sourcecells.append(nbf.v4.new_code_cell("RUN_AUDITS = False\nif RUN_AUDITS:\n    shutil.copytree(root/'experiments',replica/'experiments',dirs_exist_ok=True)\n    for script in ['verify_engine.py','audit_candidates.py','fullsize_cv.py']:\n        subprocess.run([sys.executable,script],cwd=replica,check=True)\n"))
nbf.write(nbf.v4.new_notebook(cells=sourcecells,metadata={'kernelspec':{'name':'aml-midterm','display_name':'AML Midterm','language':'python'}}),'midterm/02_reproduce.ipynb')
# Portable folder contains the notebook-visible evidence and source files, but never .venv or credentials.
for folder in ['data','experiments','predictions']:
 shutil.copytree(folder,Path('midterm')/folder,dirs_exist_ok=True)
for p in list(Path('.').glob('*.py'))+[Path('requirements-lock.txt'),Path('READ_ME_FIRST.txt')]:shutil.copy2(p,Path('midterm')/p.name)
print('Deliverables built',len(rows),'runs',trial.id.nunique(),'configurations',len(summary),'families')


Overwriting current_reproduction/build_deliverables.py


In [9]:
%%writefile current_reproduction/build_phase21_report.py
"""Document the user-requested tri-training / selection-weighting comparison."""
from pathlib import Path
import json,nbformat as nbf
ROOT=Path(__file__).resolve().parent;E=ROOT/'experiments'
results=sum([json.loads((E/f'{b}-validation-results.json').read_text()) for b in ['phase21','phase21b','phase21c','phase21d']],[])
inc=json.loads((E/'phase21-prior-incumbent.json').read_text())
current=json.loads((E/'current-best.json').read_text())
by_name={r['name']:r for r in results}
def val(name):return by_name[name]['validation']['sample_f1']
summary=dict(tri_configurations_screened=44,initial_fresh_cv_configurations=len(json.loads((E/'phase21-refit-shortlist.json').read_text())['candidates']),fresh_cv_seeds=[20261107,20261108,20261109],validation_candidates=len(results),best_tri_validation_f1=val('classical_tri_majority'),guarded_tri_validation_f1=val('tri_ordinary_cv'),target_weighted_selection_validation_f1=val('existing_target_weighted_cv + robust_weighting_cv'),initial_reason_hybrid_validation_f1=val('training_selected_reason_hybrid'),reason_hybrid_validation_f1=val('classical_tri_reason_hybrid'),exact_control_validation_f1=val('exact_no_teaching_ablation'),incumbent_validation_f1=inc['best_validation_f1'],current_validation_f1=current['best_validation_f1'],validation_improvement=current['best_validation_f1']-inc['best_validation_f1'],current_validation_macro_f1=current['candidate']['validation']['macro_f1'],selected_hybrid_ordinary_cv=current['candidate']['ordinary_cv'],selected_hybrid_leave_group_out_cv=current['leave_one_embedding_group_out_f1'],incumbent_retained=False,target_met=False,validation_used_for_training=False,final_train_plus_validation_refit_performed=False)
(E/'phase21-summary.json').write_text(json.dumps(summary,indent=2))
refs=json.loads((E/'references.json').read_text())
url='https://www.jmlr.org/papers/v8/sugiyama07a.html'
if not any(r['url']==url for r in refs):refs.append(dict(title='Sugiyama, Krauledat, Müller (2007): Covariate Shift Adaptation by Importance Weighted Cross Validation',url=url,use='Motivation for target-aware model selection. Estimated, clipped weights and label-dependent sampling here do not establish an unbiased target-risk guarantee.'))
(E/'references.json').write_text(json.dumps(refs,indent=2))
p=E/'search-gap-audit.json';audit=json.loads(p.read_text());audit['phase21_followup']=summary;p.write_text(json.dumps(audit,indent=2))
nb=nbf.v4.new_notebook();nb.metadata.kernelspec=dict(display_name='Python (AML Midterm)',language='python',name='aml-midterm')
nb.cells=[nbf.v4.new_markdown_cell('''# Tri-training and model selection under sampling bias

The two requested approaches produced a modest new best: **0.912393 validation sample-wise F1**, compared with the previous **0.910256**. Macro F1 rose from **0.780000 to 0.810444**. The winner combines the previous SVM and classical tri-training, with per-reason choices made on training folds. It remains below the requested **0.92735 validation target**. This small improvement does not establish statistical significance or a better Kaggle score.

All 78 validation rows remain outside fitting. The final train-plus-validation refit is deferred.

We screened 40 initial tri-training configurations and reranked prior configurations with training-only risk weights. Eleven diverse candidates were reevaluated on three fresh grouped five-fold splits. Follow-ups tested an exact no-teaching control, classical tri-training without added safeguards (four additional configurations), and per-reason combinations. Across these batches, eight frozen candidates were evaluated on validation. Follow-ups were informed by earlier results; each candidate's settings and combination mask were fixed from training-side evidence before its validation evaluation.'''),
nbf.v4.new_code_cell("from pathlib import Path\nimport json,numpy as np,pandas as pd\nimport matplotlib.pyplot as plt\nroot=Path.cwd()\nsummary=json.loads((root/'experiments/phase21-summary.json').read_text())\nsummary"),
nbf.v4.new_markdown_cell('''## Tri-training implementation

Each reason is a binary task with three bootstrapped classical learners. A learner receives pseudo-labels from its two peers only when they agree and the peer-error / sample-size update conditions permit an update. Updates within a round are simultaneous. True gold labels stay fixed.

Four committees were tested: three RBF SVMs; SVM/logistic/ridge; three RBF bandwidths; and RBF/polynomial/linear SVMs. Comparisons included pseudo-label weights 0.03, 0.1 and 1; conservative cross-fitted error floors; confidence filtering; pseudo-label mass caps; and no-teaching controls. KMeans cluster agreement downweights inconsistent pseudo-labels. Stratified bootstrapping keeps the rare class present.

The classical follow-up used ordinary bootstrap samples, unit pseudo-label weights, no cluster gate, and a fixed majority-vote decoder. Its selected committee combines an RBF SVM (C=1, gamma=0.001), logistic regression (C=0.3), and ridge classification (alpha=10), with class balancing and at most eight update rounds. Its standalone validation sample F1 is 0.903846.

These are explicit adaptations of [Zhou and Li's tri-training](https://www.lamda.nju.edu.cn/publication/tkde05.pdf). Weighted labels, cluster gates, confidence filters, and cross-fitted error floors do not automatically inherit its theoretical guarantees. Smoothed vote scores are heuristic values, not calibrated probabilities; threshold 0.5 preserves majority voting, and margin scores break ties if the multilabel decoder would otherwise be empty.'''),
nbf.v4.new_markdown_cell('''## Selection weights

The source contains 157 gold-status rows; the target is the feature distribution of all 1,171 training rows. No validation/test rows or reason-label values are used to estimate weights.

A logistic propensity model and a tree propensity model predict gold versus masked status using embeddings and year, with identical-embedding groups kept together in cross-fitting. Estimated risk weights are proportional to P(gold) / P(gold | features). They are clipped, normalized, and shrunk halfway toward uniform weights. Kernel mean matching provides a third estimate.

These weights change the scoring of held-out training predictions and therefore the selected model/decoder. They do not fit validation labels. Strong label-dependent selection, imperfect propensity estimates, and poor overlap mean that an unbiased target-risk guarantee is not justified. The weighting models' AUCs below measure prediction of **selection status**, not trial-reason classification.'''),
nbf.v4.new_code_cell("weighting=json.loads((root/'experiments/phase21-weighting.json').read_text())\npd.DataFrame(weighting['diagnostics']).T"),
nbf.v4.new_code_cell("weights=pd.read_csv(root/'experiments/phase21-weights.csv')\ntrain=pd.read_csv(root/'data/part1_train.csv')\nplot=weights.merge(train[['ID','start_year']],on='ID',validate='one_to_one')\nfig,ax=plt.subplots(figsize=(7,3.2))\nax.scatter(plot.start_year,plot.propensity_logistic,s=25,alpha=.7,label='Logistic propensity')\nax.scatter(plot.start_year,plot.kmm,s=20,alpha=.5,label='Kernel mean matching')\nax.axhline(1,color='gray',linestyle='--',linewidth=1)\nax.set(xlabel='Start year',ylabel='Stabilized risk weight',title='Training-only weights for CV model selection')\nax.legend()\nfig.tight_layout()\n(root/'figures').mkdir(exist_ok=True)\nfig.savefig(root/'figures/current_phase21_weights.png',dpi=180)\nplt.show()"),
nbf.v4.new_markdown_cell('''## Frozen validation results

The adjusted selector favored an HDBSCAN-consensus logistic model with drug-name features. It had target-weighted training CV around 0.9276 but only 0.86966 on external validation. This is not a demonstrated generalization improvement.

The initial per-reason hybrid was selected from 32 combinations of the incumbent and tri-training predictions on fresh training folds. Training-only selection chose tri-training for Recruitment and the incumbent for the other reasons. It scored 0.90171 on validation and was rejected.

The initial bootstrap control matched the learner kinds but differed in aggregation and decoder. The exact ablation matches the initial tri-training winner's settings, including aggregation and its fixed decoder, with cross-teaching disabled. On that exact comparison, tri-training improves 0.85470 to 0.88205. The separately selected bootstrap control scored 0.88462. These initial candidates did not beat the incumbent.

Classical tri-training without the added safeguards scored 0.903846. Applying the unchanged hybrid selection rule to that candidate yielded the new best. The selector considered 32 masks on three fresh sets of grouped training folds, favored stability and robust weighted risk, and penalized extra switches. It chose **the previous SVM for Safety, Funding and Efficacy; tri-training for Recruitment and Other**. The mask and both decoders were frozen before validation. No per-reason choices came from validation errors.

The hybrid's mean repeated grouped five-fold F1 is **0.917481**. A subsequent leave-one-embedding-group-out audit of the fixed policy scored **0.928662**. Both are training-data development scores; neither is a validation or Kaggle score. The latter audit does not rerun the complete model-selection process inside each fold.'''),
nbf.v4.new_code_cell("batches=['phase21','phase21b','phase21c','phase21d']\nresults=sum([json.loads((root/f'experiments/{b}-validation-results.json').read_text()) for b in batches],[])\nrows=[dict(candidate=r['name'],ordinary_cv=r.get('ordinary_cv'),validation_f1=r['validation']['sample_f1'],validation_macro=r['validation']['macro_f1'],rule=r['rule']) for r in results]\npd.DataFrame(rows)"),
nbf.v4.new_code_cell("names=['New SVM / tri-training hybrid','Previous best','Classical tri-training','Target-weighted selection','Exact no-teaching control']\nscores=[summary[k] for k in ['current_validation_f1','incumbent_validation_f1','best_tri_validation_f1','target_weighted_selection_validation_f1','exact_control_validation_f1']]\nfig,ax=plt.subplots(figsize=(8,3.5))\nax.scatter(scores,np.arange(len(names)),s=65,color=['#197144']+['#2563a0']*4)\nfor i,s in enumerate(scores): ax.annotate(f'{s:.5f}',(s,i),xytext=(6,5),textcoords='offset points')\nax.axvline(.92735,color='#9b3e27',linestyle='--',label='Requested validation target')\nax.set(yticks=np.arange(len(names)),yticklabels=names,xlim=(.81,1),xlabel='Validation sample-wise F1')\nax.invert_yaxis();ax.legend(loc='lower right')\nfig.tight_layout();fig.savefig(root/'figures/current_phase21_comparison.png',dpi=180)\nplt.show()"),
nbf.v4.new_markdown_cell('''## Integrity and artifacts

The current prediction file and frozen configuration now contain the new hybrid. The previous candidate is preserved under `predictions/checkpoints/pre_phase21_091026`. No train-plus-validation refit was performed. Every validation comparison is logged; this repeatedly used selection set is not a pristine test set. Kaggle performance remains unmeasured. No external trial data, extra labels or deep learning were used.

Source is embedded in notebook 04: `phase21.py`, `phase21_weights.py`, `phase21_select.py`, `phase21_hybrid.py`, `phase21_plain.py`, `phase21_classical_hybrid.py`, `phase21_fullsize_current.py` and `phase21_integrity.py`. Protocol, frozen weights, fold rankings, update diagnostics, predictions and results are under `experiments/phase21*`.

The final hybrid passed held-out-label perturbation, query-batch invariance, masked-duplicate exclusion and nonempty-output checks. Reproduction verifies raw-data hashes, all original gold labels and the exact submission checksum.

The weighting approach is motivated by [importance-weighted cross-validation](https://www.jmlr.org/papers/v8/sugiyama07a.html); its assumptions and limitations are stated above.'''),
nbf.v4.new_code_cell("json.loads((root/'experiments/phase21-integrity-checks.json').read_text())")]
nbf.write(nb,ROOT/'midterm/05_tri_training_and_shift.ipynb')
print(json.dumps(summary,indent=2))


Overwriting current_reproduction/build_phase21_report.py


In [10]:
%%writefile current_reproduction/build_phase22_23_report.py
"""Summarize the completed stability, feature and nested specialist follow-ups."""
from pathlib import Path
import json,nbformat as nbf
ROOT=Path(__file__).resolve().parent;E=ROOT/'experiments'
baseline=json.loads((E/'phase22-baseline.json').read_text())
current=json.loads((E/'current-best.json').read_text())
results=sum([json.loads((E/f'phase{n}-validation-results.json').read_text()) for n in [22,23]],[])
screen22=json.loads((E/'phase22-screen.json').read_text())
screen23=json.loads((E/'phase23-screen.json').read_text())
repeat22=json.loads((E/'phase22-repeat-results.json').read_text())
repeat23=json.loads((E/'phase23-repeat-results.json').read_text())
control=next(r for r in repeat22 if r['bucket']=='control')
summary=dict(screened_configurations=len(screen22)+len(screen23),phase22_screened=len(screen22),
    phase23_screened=len(screen23),validation_candidates=len(results),
    baseline_validation_f1=baseline['best_validation_f1'],current_validation_f1=current['best_validation_f1'],
    best_followup_validation_f1=max((r['validation']['sample_f1'] for r in results),default=None),
    previous_baseline_retained=current['candidate']['config']==baseline['candidate']['config'],
    baseline_additional_cv=control['ordinary_cv'],baseline_additional_cv_std=control['cv_std'],
    additional_cv_seeds=[20261111,20261112,20261113],validation_labels_used_for_training=0,
    final_train_plus_validation_refit_performed=False,target_met=current['best_validation_f1']>.92735)
(E/'phase22-23-summary.json').write_text(json.dumps(summary,indent=2))
refs=json.loads((E/'references.json').read_text())
for title,url,use in [
 ('Breiman: Bagging Predictors','https://statistics.berkeley.edu/tech-reports/421','Motivation for averaging fixed bootstrap runs; no best-seed selection.'),
 ('scikit-learn: StackingClassifier','https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.StackingClassifier.html','Meta-learners must learn from out-of-fold component predictions; custom implementation uses embedding-grouped inner folds.'),
 ('Dembczynski et al.: An Exact Algorithm for F-Measure Maximization','https://proceedings.neurips.cc/paper/2011/hash/71ad16ad2c4d81f348082ff6c4b20768-Abstract.html','Exact expected sample-F1 decoding under an estimated label-pattern distribution; estimated probabilities need not be correct.')]:
 if not any(r['url']==url for r in refs):refs.append(dict(title=title,url=url,use=use))
(E/'references.json').write_text(json.dumps(refs,indent=2))
nb=nbf.v4.new_notebook();nb.metadata.kernelspec=dict(display_name='Python (AML Midterm)',language='python',name='aml-midterm')
nb.cells=[nbf.v4.new_markdown_cell(f'''# Follow-up: stability, auxiliary features and nested specialists

The starting hybrid scored **{baseline['best_validation_f1']:.6f} validation sample-wise F1**. The current best is **{current['best_validation_f1']:.6f}**. Requested target: strictly above 0.92735. These scores have not been measured on Kaggle.

This follow-up tested {len(screen22)} stability/feature/score-combination configurations and {len(screen23)} nested specialists. All 78 validation rows remain excluded from fitting. The final train-plus-validation refit remains deferred.

The full search is adaptive and the validation set has been consulted repeatedly. Fixed-candidate CV audits and small frozen validation batches reduce some leakage risks but do not make the entire model-selection process unbiased.'''),
nbf.v4.new_code_cell("from pathlib import Path\nimport json,pandas as pd,numpy as np\nimport matplotlib.pyplot as plt\nroot=Path.cwd()\nsummary=json.loads((root/'experiments/phase22-23-summary.json').read_text())\nsummary"),
nbf.v4.new_markdown_cell('''## Why these experiments

The new tri-training component previously had no auxiliary-feature trials and only one bootstrap initialization. It also combined hard labels with the SVM, discarding confidence information. The earlier broad search did not test these exact combinations.

Training-only OOF errors identified Other as the largest remaining weakness: approximately 12.3 false positives and 8.3 false negatives per complete OOF pass, averaged over the original three split seeds. This motivated a dedicated Other classifier without selecting individual validation mistakes.

We used fixed ensembles of the first three or all five predetermined bootstrap seeds, never selecting the best seed. Feature blocks were added separately at two weights: eligibility criteria (word and character TF-IDF), diseases, drugs, ICD codes, drug targets/actions/pathways, molecular fingerprints/descriptors, year and field counts. Small changes to regularization, bandwidth, pseudo-label weight, confidence gates and preprocessing were also evaluated.

The score-combination variants align the original component thresholds before averaging. Their values are heuristic scores, not calibrated probabilities. The previous per-reason mask remained fixed during the feature and parameter screens.'''),
nbf.v4.new_code_cell("screen=json.loads((root/'experiments/phase22-screen.json').read_text())\nrows=[]\nfor r in screen:\n t=r['config']['tri']\n if r['bucket']=='features': rows.append(dict(feature=t['features'][0],weight=t['feature_weight'],ordinary_cv=r['ordinary'],weighted_robust_cv=r['robust']))\npd.DataFrame(rows).sort_values('weighted_robust_cv',ascending=False)"),
nbf.v4.new_markdown_cell('''## Nested specialists

Each outer training fold contains its own 15-fold grouped inner CV. The inner models generate ten score features: five from the SVM and five from tri-training. No row's label is fitted by the model that generates its inner OOF score features. Matching masked embedding duplicates of both inner and outer held-out groups are excluded.

Regularized logistic classifiers use these scores to predict Other alone, Recruitment and Other, or all five reasons. Other-only variants also test score interactions and conservative blending. A multinomial pattern model supplies a separate decoder that chooses the label set with maximum expected sample-wise F1. Earlier calibration experiments used only the old SVM; these experiments use both current components.

The specialists' complexity settings are selected from outer training CV. This is nested feature generation, not an independent nested rerun of the entire historical hyperparameter search.'''),
nbf.v4.new_code_cell("rep22=json.loads((root/'experiments/phase22-repeat-results.json').read_text())\nrep23=json.loads((root/'experiments/phase23-repeat-results.json').read_text())\nrows=[dict(batch=batch,variant=r['bucket'],mean_cv=r['ordinary_cv'],split_std=r['cv_std'],robust_cv=r['robust_cv']) for batch,rs in [('22',rep22),('23',rep23)] for r in rs]\npd.DataFrame(rows)"),
nbf.v4.new_markdown_cell('''## Validation and interpretation

Candidate configurations and decision rules were written to frozen JSON files before loading validation labels. Only finalists meeting the prespecified training-CV screen were evaluated on validation. All results, including failures to improve, appear below.

The control's scores on the additional split seeds differ from its earlier three-seed CV. That variability is useful evidence of the uncertainty created by just 157 gold rows; do not compare scores from different split sets as if they were paired measurements.'''),
nbf.v4.new_code_cell("results=sum([json.loads((root/f'experiments/phase{n}-validation-results.json').read_text()) for n in [22,23]],[])\nvalidation=pd.DataFrame([dict(candidate=r['name'],cv=r['ordinary_cv'],validation_f1=r['validation']['sample_f1'],macro_f1=r['validation']['macro_f1']) for r in results])\nvalidation"),
nbf.v4.new_code_cell("fig,ax=plt.subplots(figsize=(7,3.5))\ny=np.arange(len(validation))\nax.scatter(validation.validation_f1,y,s=65,label='Follow-up candidate')\nax.axvline(summary['baseline_validation_f1'],color='#197144',label='Starting hybrid')\nax.axvline(.92735,color='#9b3e27',linestyle='--',label='Requested target')\nfor i,s in enumerate(validation.validation_f1):ax.annotate(f'{s:.5f}',(s,i),xytext=(5,7),textcoords='offset points')\nax.set(yticks=y,yticklabels=validation.candidate,xlim=(.8,.98),xlabel='Validation sample-wise F1')\nax.margins(y=.25);ax.invert_yaxis();ax.legend(loc='lower left',fontsize=8)\nfig.tight_layout();fig.savefig(root/'figures/current_phase22_23_comparison.png',dpi=180)\nplt.show()"),
nbf.v4.new_markdown_cell('''## Reproduction and integrity

Protocols, exact configurations, OOF scores and frozen validation batches are under `experiments/phase22*` and `experiments/phase23*`. All source is embedded in notebook 04. The phase22 core was checked against phase21 with matching defaults. The nested specialist passed an uncached held-out-label perturbation check; caches were cleared to prevent them from masking leakage. Raw inputs and gold labels remain unchanged.

Method references: [bootstrap aggregation](https://statistics.berkeley.edu/tech-reports/421), [out-of-fold stacking](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.StackingClassifier.html), and [exact F-measure decoding](https://proceedings.neurips.cc/paper/2011/hash/71ad16ad2c4d81f348082ff6c4b20768-Abstract.html). No external trial records or labels were retrieved.'''),
nbf.v4.new_code_cell("json.loads((root/'experiments/phase23-integrity-checks.json').read_text())")]
nbf.write(nb,ROOT/'midterm/06_stability_and_specialists.ipynb')
print(json.dumps(summary,indent=2))


Overwriting current_reproduction/build_phase22_23_report.py


In [11]:
%%writefile current_reproduction/build_phase24_report.py
"""Document the two requested experiments and their matched control."""
from pathlib import Path
import json,nbformat as nbf,numpy as np
ROOT=Path(__file__).resolve().parent;E=ROOT/'experiments'
baseline=json.loads((E/'phase24-baseline.json').read_text())
current=json.loads((E/'current-best.json').read_text())
results=json.loads((E/'phase24-validation-results.json').read_text())+json.loads((E/'phase24b-validation-results.json').read_text())
frozen=json.loads((E/'phase24-validation-frozen.json').read_text())
configs=json.loads((E/'phase24-configs.json').read_text())
by={r['name']:r for r in results}
summary=dict(raw_configurations=len(configs),co_training_configurations=sum(c['model']=='co_training' for c in configs),
    no_exchange_screen_controls=sum(bool(c.get('no_exchange')) for c in configs),
    metric_configurations=sum(c['model']=='metric_ssl' for c in configs),
    decision_policies_per_configuration=12,repeat_seeds=[20261115,20261116,20261117],
    validation_candidates=len(results),baseline_validation_f1=baseline['best_validation_f1'],
    baseline_matched_cv=float(np.mean([x['ordinary'] for x in frozen['baseline_cv']])),
    co_training_validation_f1=by['phase24_co_training']['validation']['sample_f1'],
    metric_validation_f1=by['phase24_metric_ssl']['validation']['sample_f1'],
    no_exchange_validation_f1=by['phase24_exact_no_exchange']['validation']['sample_f1'],
    best_retained=current['candidate']['config']==baseline['candidate']['config'],
    current_validation_f1=current['best_validation_f1'],target_met=current['best_validation_f1']>.92735,
    validation_fit=False,final_train_plus_validation_refit_performed=False)
(E/'phase24-summary.json').write_text(json.dumps(summary,indent=2))
refs=json.loads((E/'references.json').read_text())
for title,url,use in [
 ('Blum and Mitchell: Combining Labeled and Unlabeled Data with Co-Training','https://www.cs.cmu.edu/~avrim/Papers/cotrain.pdf','Motivation for separate embedding/metadata views exchanging pseudo-labels. Our quota, agreement, weighting and cluster safeguards are adaptations; the original assumptions are not established here.'),
 ('scikit-learn: NeighborhoodComponentsAnalysis','https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.NeighborhoodComponentsAnalysis.html','Gold-only per-reason linear metric learning, combined with original distances in an SVM kernel.')]:
 if not any(r['url']==url for r in refs):refs.append(dict(title=title,url=url,use=use))
(E/'references.json').write_text(json.dumps(refs,indent=2))
nb=nbf.v4.new_notebook();nb.metadata.kernelspec=dict(display_name='Python (AML Midterm)',language='python',name='aml-midterm')
nb.cells=[nbf.v4.new_markdown_cell(f'''# Separate-view co-training and reason-specific distance learning

Both requested approaches were implemented and evaluated. Neither improved the saved **{current['best_validation_f1']:.6f} validation sample-wise F1**. Co-training scored **{summary['co_training_validation_f1']:.6f}**; distance learning scored **{summary['metric_validation_f1']:.6f}**. The target of strictly exceeding 0.92735 remains unmet. Kaggle performance has not been measured.

All 157 original gold labels and 1,014 originally masked training rows stayed in their original roles. The 78 validation rows remain outside fitting. No external trial data, extra labels, ID predictors, deep learning or train-plus-validation refit were used.'''),
nbf.v4.new_code_cell("from pathlib import Path\nimport json,pandas as pd,numpy as np\nimport matplotlib.pyplot as plt\nroot=Path.cwd()\nsummary=json.loads((root/'experiments/phase24-summary.json').read_text())\nsummary"),
nbf.v4.new_markdown_cell('''## Separate-view co-training

The first learner is an RBF SVM over standardized reason embeddings. The second is a regularized logistic classifier over a separate metadata view. Three views were tested: eligibility criteria; drugs, targets, actions and diseases; and their combination. TF-IDF vocabularies and feature normalization are learned only from permitted training rows.

Each binary reason task starts from gold labels. For up to eight rounds, each learner proposes confident positive and negative masked rows for its peer, with at most three additions per class per round and smaller quotas for rare classes. Gold labels never change. Comparisons test mutual agreement versus unrestricted cross-teaching, pseudo-label weights 0.1/0.3, and prediction from the embedding learner alone versus a small metadata vote. These confidence values are margin-based heuristics, not calibrated probabilities.

KMeans provides an unsupervised safeguard: clusters with at least two gold examples and 80% agreement downweight conflicting pseudo-labels by a factor of ten. This uses only training labels and features. The original co-training theory's view-sufficiency/independence assumptions are not established here; weak metadata controls are a reason for caution.

There were 24 exchange configurations and six no-exchange screen controls. The selected policy uses the drug/target/action/disease view, pseudo-label weight 0.3, unrestricted cross-teaching and embedding-only final scores. Its threshold is 0.65, and training CV selected it to replace Recruitment/Other decisions while retaining the incumbent for the other reasons. Empty combined outputs fall back to the incumbent.'''),
nbf.v4.new_markdown_cell('''## Reason-specific distance learning

Each binary reason learns a separate geometry from the gold rows of its current fold. Two families were evaluated: Neighborhood Components Analysis (NCA), and regularized Fisher-style contrasts between positive subtypes and negative examples. The latter uses Ledoit-Wolf covariance shrinkage and one, two or three positive subtype centers.

PCA to 25 or 50 dimensions is fitted using permitted gold and masked training embeddings. The supervised projection uses only gold labels. Learned squared distances are normalized using fitting rows, then blended with original squared distances at weights 0.15, 0.4 or 0.7. The resulting RBF kernel enters the earlier cluster-guided semi-supervised SVM pipeline, retaining its gold labels and low-weight pseudo-label schedule.

Eighteen NCA and eighteen Fisher-style configurations were screened. The selected metric uses 50 PCA components, two positive subtypes, 15% learned distance and threshold 0.55 for all five reasons. No per-reason changes were made after examining validation scores.'''),
nbf.v4.new_markdown_cell('''## Evaluation protocol

The initial screen used grouped five-fold CV with seed 20261114. Identical embeddings stay in the same fold, and matching masked duplicates of held-out groups are excluded from fitting. For each raw configuration, a training-only screen compared four thresholds and three scopes: all reasons, Other alone, or Recruitment/Other. These 792 policy comparisons reuse the 66 raw model configurations; they are not 792 independent model fits.

Ordinary and robust weighted training-CV winners from each family were frozen and reevaluated on seeds 20261115–20261117. The better repeated-CV policy in each family became its validation finalist. The exact no-exchange follow-up copies the selected co-training settings, scope and decoder, disabling only label exchange. It does not tune a new control decoder.

All validation predictions were saved before loading validation labels for scoring. This is controlled feature/model development, but the full historical search is adaptive and validation has been consulted repeatedly. Neither CV nor validation is an unbiased final estimate after that selection.'''),
nbf.v4.new_code_cell("repeat=json.loads((root/'experiments/phase24-repeat-results.json').read_text())\nrows=[dict(learner=r['config']['learner']['model'],metric=r['config']['learner'].get('metric'),scope=r['config']['scope'],threshold=r['config']['decision_rule'],mean_cv=r['ordinary_cv'],split_std=r['cv_std'],weighted_robust_cv=r['robust_cv']) for r in repeat]\npd.DataFrame(rows)"),
nbf.v4.new_code_cell("results=json.loads((root/'experiments/phase24-validation-results.json').read_text())+json.loads((root/'experiments/phase24b-validation-results.json').read_text())\nrows=[dict(candidate=r['name'],cv=r['ordinary_cv'],validation_f1=r['validation']['sample_f1'],macro_f1=r['validation']['macro_f1']) for r in results]\ntable=pd.DataFrame(rows)\ntable"),
nbf.v4.new_markdown_cell(f'''## Interpretation

Label exchange helped its matched control: validation F1 increased from **{summary['no_exchange_validation_f1']:.6f} to {summary['co_training_validation_f1']:.6f}**. That improvement did not exceed the existing hybrid's **{summary['baseline_validation_f1']:.6f}**. The distance-learning finalist also failed to transfer its training-CV improvement to validation. The saved incumbent and its submission therefore remain unchanged.

The incumbent's CV on these same repeat seeds was **{summary['baseline_matched_cv']:.6f}**. Comparisons should use matched seeds; estimates from the earlier split sets differ because the gold dataset is small.'''),
nbf.v4.new_code_cell("names=['Saved best','Co-training','Exact no-exchange control','Distance learning']\nscores=[summary[k] for k in ['baseline_validation_f1','co_training_validation_f1','no_exchange_validation_f1','metric_validation_f1']]\nfig,ax=plt.subplots(figsize=(7.4,3.6))\ny=np.arange(len(names))\nax.scatter(scores,y,s=65,color=['#197144']+['#2563a0']*3)\nfor i,s in enumerate(scores):ax.annotate(f'{s:.5f}',(s,i),xytext=(5,7),textcoords='offset points')\nax.axvline(.92735,color='#9b3e27',linestyle='--',label='Requested validation target')\nax.set(yticks=y,yticklabels=names,xlim=(.82,.98),xlabel='Validation sample-wise F1')\nax.margins(y=.18);ax.invert_yaxis();ax.legend(loc='lower right',fontsize=8)\nfig.tight_layout();fig.savefig(root/'figures/current_phase24_comparison.png',dpi=180)\nplt.show()"),
nbf.v4.new_markdown_cell('''## Integrity and reproducibility

Both approaches passed held-out-label perturbation, query-label/batch invariance, masked-duplicate exclusion and query-versus-masked-output consistency checks. Caches were cleared before perturbation to avoid masking leakage. With learned-distance weight zero, the metric model reproduced the original SVM scores to numerical precision.

Source files: `phase24.py`, `phase24_run.py`, `phase24_control.py` and `phase24_integrity.py`, all embedded in notebook 04. Configurations, policies, update diagnostics, predictions and results are under `experiments/phase24*`. The archive includes every unsuccessful trial.

References: [Blum and Mitchell's co-training paper](https://www.cs.cmu.edu/~avrim/Papers/cotrain.pdf) and [NCA documentation](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.NeighborhoodComponentsAnalysis.html).'''),
nbf.v4.new_code_cell("json.loads((root/'experiments/phase24-integrity-checks.json').read_text())")]
nbf.write(nb,ROOT/'midterm/07_cotraining_and_distances.ipynb')
print(json.dumps(summary,indent=2))


Overwriting current_reproduction/build_phase24_report.py


In [12]:
%%writefile current_reproduction/build_phase25_report.py
"""Document the three requested SVM refinements and matched ablations."""
from pathlib import Path
import json,nbformat as nbf,numpy as np
ROOT=Path(__file__).resolve().parent;E=ROOT/'experiments'
baseline=json.loads((E/'phase25-baseline.json').read_text());current=json.loads((E/'current-best.json').read_text())
results=json.loads((E/'phase25-validation-results.json').read_text())+json.loads((E/'phase25b-validation-results.json').read_text())
frozen=json.loads((E/'phase25-validation-frozen.json').read_text());configs=json.loads((E/'phase25-configs.json').read_text())
summary=dict(raw_configurations=len(configs),families={family:sum(c['model']==family for c in configs) for family in ['label_tuned_ssl','reliable_ssl','local_ssl']},
    uniform_reliability_controls=sum(c.get('reliability')=='uniform' for c in configs),
    inner_parameter_recipes=9,inner_folds=5,decision_policies_per_raw_configuration=16,
    repeat_seeds=[20261119,20261120,20261121],validation_candidates=len(results),
    baseline_validation_f1=baseline['best_validation_f1'],
    baseline_matched_cv=float(np.mean([s['ordinary'] for s in frozen['baseline_cv']])),
    results=[dict(name=r['name'],cv=r['ordinary_cv'],validation=r['validation']['sample_f1'],macro=r['validation']['macro_f1']) for r in results],
    best_retained=current['candidate']['config']==baseline['candidate']['config'],current_validation_f1=current['best_validation_f1'],
    reported_leader='0.93xx; exact score not provided',kaggle_score=None,validation_fit=False,final_train_plus_validation_refit_performed=False)
(E/'phase25-summary.json').write_text(json.dumps(summary,indent=2))
refs=json.loads((E/'references.json').read_text())
for title,url,use in [
 ('scikit-learn: Probability calibration','https://scikit-learn.org/stable/modules/calibration.html','OOF score calibration motivates reliability estimates for pseudo-label weighting. Our target distribution is shifted, so calibrated target-domain accuracy is not guaranteed.'),
 ('Segata and Blanzieri: Fast and Scalable Local Kernel Machines','https://jmlr.org/papers/v11/segata10a.html','Motivation for local SVM specialists; our overlapping weighted experts and global blending are a separate implementation adapted to this small dataset.'),
 ('scikit-learn: SVC','https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html','Regularization, RBF bandwidth and sample weights in the classical SVM classifiers.')]:
 if not any(r['url']==url for r in refs):refs.append(dict(title=title,url=url,use=use))
(E/'references.json').write_text(json.dumps(refs,indent=2))
nb=nbf.v4.new_notebook();nb.metadata.kernelspec=dict(display_name='Python (AML Midterm)',language='python',name='aml-midterm')
nb.cells=[nbf.v4.new_markdown_cell(f'''# Per-reason SVM tuning, pseudo-label reliability and local specialists

The user requested all three remaining refinements and reported a new Kaggle leader in the **0.93xx** range. The exact leaderboard score was not supplied. Kaggle and local validation remain separate evaluations; our model's Kaggle score is unmeasured.

The starting model scored **{baseline['best_validation_f1']:.6f} validation sample-wise F1**. After these tests, the current saved best scores **{current['best_validation_f1']:.6f}**. All 78 validation rows remain excluded from fitting. The final train-plus-validation refit remains deferred.

The raw screen contained 60 configurations: six nested per-reason tuning policies, 24 reliability-weighted variants plus six uniform-weight controls, and 24 local specialist variants. Predictions from each were evaluated under four thresholds and four label scopes, using training data only.'''),
nbf.v4.new_code_cell("from pathlib import Path\nimport json,pandas as pd,numpy as np\nimport matplotlib.pyplot as plt\nroot=Path.cwd()\nsummary=json.loads((root/'experiments/phase25-summary.json').read_text())\nsummary"),
nbf.v4.new_markdown_cell('''## Per-reason SVM settings

Every outer fold runs its own five-fold grouped inner CV over nine SVM recipes. The recipe grid varies regularization C, RBF gamma and pseudo-label weight near the earlier best configuration. Each reason can select a different recipe. The screen compares independent binary-F1 selection and coordinate selection targeting sample-wise F1, with penalties of 0, 0.005 or 0.02 for departing from the common baseline.

Inner preprocessing and pseudo-label updates exclude inner held-out rows and their masked embedding duplicates. The selected recipes then fit the outer training rows. This tests a complete tuning procedure; a single set of globally optimized label parameters is not reused across outer validation folds. Original gold labels are never changed.

The final label scopes considered are all reasons, Other alone, Recruitment/Other, and Safety/Funding/Efficacy. Scoped policies preserve the previous model's decisions for the remaining reasons. Empty combined predictions fall back to that model.'''),
nbf.v4.new_markdown_cell('''## Reliability-weighted pseudo-labels

The teacher is either a supervised gold-only SVM or the earlier cluster-guided semi-supervised SVM. Five-fold inner OOF predictions train the reliability estimator. The per-reason variant calibrates each reason's signed margin to a class probability. The pooled variant predicts whether a margin's binary decision is correct, using margin magnitude, signed margin and reason indicators across all five tasks.

The full-fold teacher supplies fixed provisional labels for the masked rows. Reliability controls their training weights: max(2r−1, 0) squared, with a minimum reliability cutoff of 0.5 or 0.8. Overall pseudo-label weights are 0.03, 0.1 or 0.3. KMeans cluster agreement further downweights conflicts. Zero-weight pseudo-label rows are removed so they do not influence automatic class-frequency balancing.

The student fits gold labels plus these weighted provisional labels. The calibrator affects training weights; it is not merely a final prediction threshold adjustment. Teacher labels remain fixed during this student fit. Reliability is estimated on the labeled training distribution, so it is not guaranteed to stay calibrated under the later-trial distribution shift.'''),
nbf.v4.new_markdown_cell('''## Local SVM specialists

KMeans identifies three or six regions using only permitted training embeddings. Each expert retains all gold examples with distance-based weights and uses the 250 masked examples nearest its region center, labeled by the earlier global semi-supervised SVM. Gold weights have a floor to preserve shared information across regions.

Each expert fits five binary SVMs. Its output is blended with global scores at a maximum weight of 0.25 or 0.5, reduced further for reasons with little weighted positive or negative support. Smooth routing averages nearby experts. Regularization C of 0.3/1/3 and gamma of 0.001/0.003 are screened. No query features affect cluster fitting, neighborhood selection or model training.

This tests local decision functions and global sharing. It is distinct from the earlier experiment that changed the distance metric of one global SVM.'''),
nbf.v4.new_markdown_cell('''## Selection and controls

The initial outer CV uses seed 20261118. Ordinary and robust weighted CV nominate up to two policies from each family. Those policies are frozen and reevaluated on seeds 20261119–20261121. One repeated-CV winner per family is frozen before validation scoring.

Two matched controls keep their selected policy's scope and threshold: one removes per-reason parameter tuning, and one replaces learned reliability weights with uniform pseudo-label weights. No control-specific retuning is performed. The local expert implementation also has an exact numerical zero-blend check against its global model.

All splits group identical reason embeddings. Matching masked duplicates of held-out groups are removed at both inner and outer levels. All validation predictions are saved before labels are read for reporting. The entire historical search remains adaptive; nested tuning and leakage checks do not erase selection bias from repeated validation comparisons.'''),
nbf.v4.new_code_cell("repeats=json.loads((root/'experiments/phase25-repeat-results.json').read_text())\nrows=[dict(family=r['config']['learner']['model'],scope=r['config']['scope'],threshold=r['config']['decision_rule'],mean_cv=r['ordinary_cv'],split_std=r['cv_std'],robust_cv=r['robust_cv'],settings=json.dumps(r['config']['learner'],sort_keys=True)) for r in repeats]\npd.DataFrame(rows)"),
nbf.v4.new_code_cell("results=json.loads((root/'experiments/phase25-validation-results.json').read_text())+json.loads((root/'experiments/phase25b-validation-results.json').read_text())\ntable=pd.DataFrame([dict(candidate=r['name'],cv=r['ordinary_cv'],validation_f1=r['validation']['sample_f1'],macro_f1=r['validation']['macro_f1']) for r in results])\ntable"),
nbf.v4.new_code_cell("names=['Starting best']+table.candidate.tolist()\nscores=[summary['baseline_validation_f1']]+table.validation_f1.tolist()\nfig,ax=plt.subplots(figsize=(8,4.3))\ny=np.arange(len(names));ax.scatter(scores,y,s=60)\nfor i,s in enumerate(scores):ax.annotate(f'{s:.5f}',(s,i),xytext=(5,6),textcoords='offset points')\nax.axvline(.93,color='#9b3e27',linestyle='--',label='0.93 reference; reported leader is 0.93xx')\nax.set(yticks=y,yticklabels=names,xlim=(min(.80,min(scores)-.02),max(.98,max(scores)+.02)),xlabel='Validation sample-wise F1')\nax.margins(y=.15);ax.invert_yaxis();ax.legend(loc='lower right',fontsize=7)\nfig.tight_layout();fig.savefig(root/'figures/current_phase25_comparison.png',dpi=180)\nplt.show()"),
nbf.v4.new_markdown_cell('''## Reproduction and integrity

All three approaches passed uncached held-out-label perturbation tests, query-label/batch invariance, masked-duplicate exclusion and query/masked-output consistency checks. A zero local-expert blend reproduces the global model's scores. Source is embedded in notebook 04; protocols, exact model settings, OOF predictions, diagnostics and unsuccessful outcomes are under `experiments/phase25*`.

No external trial data, extra labels, ID predictors or deep learning were used. The frozen current-best configuration and its predictions are reproduced in notebook 04; raw file hashes and gold-label preservation are checked.

References: [calibration](https://scikit-learn.org/stable/modules/calibration.html), [local SVMs](https://jmlr.org/papers/v11/segata10a.html), and [SVC settings](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html).'''),
nbf.v4.new_code_cell("json.loads((root/'experiments/phase25-integrity-checks.json').read_text())")]
nbf.write(nb,ROOT/'midterm/08_svm_refinements.ipynb')
print(json.dumps(summary,indent=2))


Overwriting current_reproduction/build_phase25_report.py


In [13]:
%%writefile current_reproduction/build_phase26_report.py
"""Build the record of three new classical methods and their matched controls."""
from pathlib import Path
import json
import numpy as np
import nbformat as nbf

ROOT=Path(__file__).resolve().parent;E=ROOT/'experiments'
baseline=json.loads((E/'phase26-baseline.json').read_text())
current=json.loads((E/'current-best.json').read_text())
results=json.loads((E/'phase26-validation-results.json').read_text())
frozen=json.loads((E/'phase26-validation-frozen.json').read_text())
configs=json.loads((E/'phase26-configs.json').read_text())
summary=dict(raw_configurations=len(configs),family_counts={family:sum(c['model']==family for c in configs)
    for family in ['label_shift26','lap_svm26','other_subclasses26']},
    baseline_validation_f1=baseline['best_validation_f1'],baseline_matched_cv=float(np.mean([s['ordinary'] for s in frozen['baseline_cv']])),
    repeat_seeds=[20261123,20261124,20261125],validation_comparisons=len(results),
    results=[dict(name=r['name'],matched_control=r['matched_control'],cv=r['ordinary_cv'],cv_std=r['cv_std'],
        validation=r['validation']['sample_f1'],macro=r['validation']['macro_f1']) for r in results],
    baseline_retained=current['candidate']['config']==baseline['candidate']['config'],
    current_validation_f1=current['best_validation_f1'],reported_leader='0.93xx; exact score unknown',
    kaggle_score=None,validation_fit=False,final_train_plus_validation_refit=False)
summary['matched_effects']={}
for r in results:
 if r['matched_control']:continue
 control=next(x for x in results if x['matched_control'] and x['config']['learner']['model']==r['config']['learner']['model'])
 summary['matched_effects'][r['config']['learner']['model']]=dict(
    cv_difference=r['ordinary_cv']-control['ordinary_cv'],
    validation_difference=r['validation']['sample_f1']-control['validation']['sample_f1'])
(E/'phase26-summary.json').write_text(json.dumps(summary,indent=2))
refs=json.loads((E/'references.json').read_text())
for title,url,use in [
 ('Lipton et al. (2018): Detecting and Correcting for Label Shift with Black Box Predictors','https://proceedings.mlr.press/v80/lipton18a.html','Motivates estimating changed reason frequencies from unlabeled examples. Our binary soft-confusion estimator uses ridge shrinkage; the stable conditional-distribution assumption may fail here.'),
 ('Belkin et al. (2006): Manifold Regularization','https://www.jmlr.org/papers/v7/belkin06a.html','Motivates Laplacian SVM penalties. Our experiment uses a finite 192-component Nystroem kernel approximation and fold-local graph fitting.')]:
 if not any(r['url']==url for r in refs):refs.append(dict(title=title,url=url,use=use))
(E/'references.json').write_text(json.dumps(refs,indent=2))
nb=nbf.v4.new_notebook();nb.metadata.kernelspec=dict(display_name='Python (AML Midterm)',language='python',name='aml-midterm')
nb.cells=[nbf.v4.new_markdown_cell(f'''# Label-frequency correction, graph SVMs and Other subgroups

The requested follow-up tested **84 configurations**: 24 label-shift corrections, 36 graph-regularized SVMs and 24 Other-subclass models. The starting candidate had validation sample-wise F1 **{baseline['best_validation_f1']:.6f}**. The current saved candidate has **{current['best_validation_f1']:.6f}**. Validation remains excluded from fitting; the final train-plus-validation refit is deferred.

The reported Kaggle leader is in the 0.93xx range; its exact score is unknown. Our actual Kaggle score is unmeasured. Local CV and validation are separate evaluations.'''),
nbf.v4.new_code_cell("from pathlib import Path\nimport json,numpy as np,pandas as pd\nimport matplotlib.pyplot as plt\nroot=Path.cwd()\nsummary=json.loads((root/'experiments/phase26-summary.json').read_text())\nsummary"),
nbf.v4.new_markdown_cell('''## Label-frequency correction

The teacher combines the incumbent's SVM scores for Safety/Funding/Efficacy and tri-training scores for Recruitment/Other. SVM scores are aligned so their previous 0.55 threshold corresponds to 0.5. A five-fold grouped inner CV supplies teacher scores for every outer-training gold row. Logistic calibrators (C=0.3 or 3) fit those OOF scores to the original binary labels.

The full-fold teacher predicts the allowed masked training examples. Two estimators infer each reason's prevalence in those examples: regularized soft-confusion inversion (a binary BBSE adaptation) and an EM likelihood update with 20 source-prior pseudo-observations. BBSE uses ridge 0.01 toward the labeled source frequency. Estimated prevalences are clipped to [0.01,0.99]. The log-odds adjustment is capped at ±1.5 and multiplied by 0.25, 0.5 or 1.

The shift is applied either to calibrated scores or the original aligned teacher scores. The uncalibrated-score variant is a heuristic comparison. Calibration summaries and prevalence estimates are training estimates, not independent accuracy guarantees. The method assumes stable score distributions conditional on each reason; deliberate rare-label sampling, changing trial years and correlated labels can violate this assumption.

Only the 1,014 originally masked training rows, with held-out embedding duplicates removed, inform target prevalence. Validation and test features do not enter prior estimation. The matched control sets the shift strength to zero and keeps the same calibration, threshold and scope.'''),
nbf.v4.new_markdown_cell(r'''## Graph-regularized SVM

Each training fold fits StandardScaler and a 192-component Nystroem approximation of the RBF kernel. A graph joins the 10 or 25 nearest neighbors among allowed gold and masked training embeddings. Edge weights use local distance scales; the graph is symmetrized. Its combinatorial Laplacian is $L=D-W$.

Writing the finite feature matrix as $\Phi$, compute $G=\Phi^T L\Phi/N$ and scale it to mean diagonal one. For each reason, minimize

$$\tfrac12 w^T(I+\lambda G)w+C\sum_{i\in\mathrm{gold}} a_i\max(0,1-y_i(\phi_i^Tw+b)).$$

Cholesky whitening converts this objective to an ordinary balanced SVM. The graph weights tested are 0.1, 1 and 10; C is 0.3, 1 or 3; RBF gamma is 0.001 or 0.003. Only gold labels enter the hinge loss. The unsupervised graph changes the fitted classifier, whose predictions supply the missing labels. This is a finite-kernel approximation of a Laplacian SVM, not an exact infinite-dimensional solver.

The matched control uses the identical kernel approximation and settings with graph weight zero. Its purpose is to isolate the graph penalty. Query features only undergo the fitted transformations; they never alter the graph.'''),
nbf.v4.new_markdown_cell('''## Other subgroups

PCA to 25 components is fitted on permitted training embeddings. KMeans then clusters the gold Other-positive rows into two or three subgroups. If a subgroup contains fewer than three gold examples, the number of subgroups is reduced. All Other-negative gold examples remain class zero.

A regularized multiclass logistic classifier learns these latent subclass targets. Its Other score is the sum of probabilities for the positive subclasses. The representation is either standardized embeddings or a 192-component RBF kernel map. C is 0.3, 1 or 3. Aggregate Other versus non-Other receives balanced gold weights; each positive subclass is not independently rebalanced to the size of the negative class.

Masked rows whose teacher Other score is at least 0.8 or at most 0.2 receive fixed provisional targets with weight 0.03. Positive provisional rows are assigned to the nearest learned Other center. The resulting score replaces or blends equally with the incumbent's Other score. Other reason decisions are preserved by the scoped policy. Original gold labels are never overwritten.

The matched control uses one Other subgroup with identical remaining settings. This isolates the benefit of splitting the positive class.'''),
nbf.v4.new_markdown_cell('''## Selection and validation discipline

The first screen uses grouped five-fold CV seed 20261122. Four thresholds (0.4/0.5/0.6/0.7) and four label scopes are considered for shift and graph models; subclass models alter only Other. Scope choices for the first two families are all labels, Safety/Funding/Efficacy, Efficacy/Other, or Recruitment/Other. Empty scoped predictions fall back to the incumbent's decisions.

Ordinary and robust weighted CV each nominate a policy per family. Up to six policies are frozen and evaluated on three new grouped split seeds (20261123–20261125). Selection within each family maximizes mean sample-F1 minus 0.2 times its split standard deviation, with robust weighted performance breaking ties. Matched controls inherit the chosen settings, scope and threshold without retuning.

The three finalists and three controls are frozen before validation evaluation. Their validation predictions are saved before validation labels are read for reporting. The original engine also logs its best decoding rule for diagnostic purposes; phase 26 selection uses only the explicit frozen policies described here.

The repeated full history of CV and validation comparisons creates selection optimism. A nested inner calibrator does not make the entire historical search an unbiased nested-CV estimate. All duplicate reason embeddings are grouped, and matching masked duplicates of held-out groups are excluded at every fitting level.'''),
nbf.v4.new_code_cell("repeats=json.loads((root/'experiments/phase26-repeat-results.json').read_text())\npd.DataFrame([dict(family=r['config']['learner']['model'],scope=r['config']['scope'],threshold=r['config']['decision_rule'],mean_cv=r['ordinary_cv'],split_std=r['cv_std'],settings=json.dumps(r['config']['learner'],sort_keys=True)) for r in repeats])"),
nbf.v4.new_code_cell("results=json.loads((root/'experiments/phase26-validation-results.json').read_text())\ntable=pd.DataFrame([dict(candidate=r['name'],control=r['matched_control'],cv=r['ordinary_cv'],validation_f1=r['validation']['sample_f1'],macro_f1=r['validation']['macro_f1']) for r in results])\ntable"),
nbf.v4.new_markdown_cell('''## What the controls establish

None of the three finalists or controls exceeds the retained 0.912393 validation model. Label-frequency correction scores 0.871795 versus 0.876068 without the shift, so correction itself does not explain a gain. The graph penalty scores 0.880342 versus 0.858974 without that penalty, an improvement within this matched pair that still trails the incumbent. Three Other subgroups score 0.897436, exactly the same aggregate validation F1 as one subgroup.

The selected frequency-correction policy reaches 0.930644 on repeated training CV but loses substantially on validation. Its zero-shift control also has high CV and lower validation. This supports caution about transferring training-CV rankings; it does not by itself prove a particular cause of the mismatch or a Kaggle score. The incumbent is preserved rather than replacing it based on CV alone.'''),
nbf.v4.new_code_cell("names=['Starting best']+table.candidate.str.replace('phase26_','',regex=False).tolist()\nscores=[summary['baseline_validation_f1']]+table.validation_f1.tolist()\nfig,ax=plt.subplots(figsize=(9,4.8))\ny=np.arange(len(names));ax.scatter(scores,y,s=55)\nfor i,s in enumerate(scores):ax.annotate(f'{s:.5f}',(s,i),xytext=(5,5),textcoords='offset points')\nax.axvline(.93,color='#9b3e27',linestyle='--',label='0.93 reference; reported Kaggle leader is 0.93xx')\nax.set(yticks=y,yticklabels=names,xlim=(min(.80,min(scores)-.02),max(.98,max(scores)+.02)),xlabel='Validation sample-wise F1')\nax.margins(y=.13);ax.invert_yaxis();ax.legend(loc='lower right',fontsize=7)\nfig.tight_layout();fig.savefig(root/'figures/current_phase26_comparison.png',dpi=180)\nplt.show()"),
nbf.v4.new_markdown_cell('''## Integrity and reproduction

All three methods pass held-out-label perturbation, query-label invariance, query-batch invariance, masked-duplicate exclusion and consistent query/masked output tests. Additional checks confirm the graph objective's algebraic transformation, equality with an ordinary finite-kernel SVM when the graph penalty is zero, and zero-shift equality with the aligned teacher scores. Synthetic label-shift inputs verify adjustment direction.

No external trial data, added labels, ID predictors or deep learning are used. Source is embedded in notebook 04. Exact configurations, OOF outputs, selection records and diagnostics are stored under `experiments/phase26*`. Raw input hashes and preservation of all 157 original gold labels are checked when reproducing the retained candidate.

References: [label-shift estimation](https://proceedings.mlr.press/v80/lipton18a.html) and [manifold regularization](https://www.jmlr.org/papers/v7/belkin06a.html).'''),
nbf.v4.new_code_cell("json.loads((root/'experiments/phase26-integrity-checks.json').read_text())")]
nbf.write(nb,ROOT/'midterm/09_shift_graph_and_subgroups.ipynb')
print(json.dumps(summary,indent=2))


Overwriting current_reproduction/build_phase26_report.py


In [14]:
%%writefile current_reproduction/build_pooled_benchmark_report.py
"""Report fixed-recipe training-only CV, pooled CV, and original validation."""
from pathlib import Path
import json,numpy as np,pandas as pd,nbformat as nbf
ROOT=Path(__file__).resolve().parent;P=ROOT/'experiments/pooled_benchmark_v1'
s=json.loads((P/'summary.json').read_text());p=json.loads((P/'protocol.json').read_text())
t=pd.read_csv(P/'comparison.csv');current=t.loc[t.incumbent].iloc[0]
columns=['training_cv_mean','pooled_cv_mean','original_validation_f1']
leaders={c:t.loc[np.isclose(t[c],t[c].max(),atol=1e-12,rtol=0),'candidate'].tolist() for c in columns}
conclusions=dict(model_count=len(t),leaders=leaders,current={c:float(current[c]) for c in columns},
    current_ranks={c:int(current[c+'_rank']) for c in columns},
    current_leads_all_three=all(current[c]>=t[c].max()-1e-12 for c in columns),
    current_submission_changed=False,full_train_plus_validation_refit=False,
    interpretation='Point-estimate ranks on overlapping development data, not significance tests or three independent tests.')
raw_train=pd.read_csv(ROOT/'data/part1_train.csv');raw_val=pd.read_csv(ROOT/'data/part1_val.csv')
labels=['Safety','Funding','Efficacy','Recruitment','Other'];emb=[f'emb_{i}' for i in range(150)]
saved=np.load(P/(current.pooled_result_key+'.npz'),allow_pickle=True)
truth=saved['truth'];pred=saved['predictions'];ids=saved['ids']
row_f1=2*(pred*truth[None,:,:]).sum(2)/(pred.sum(2)+truth.sum(1)[None,:])
from_val=np.isin(ids,raw_val.ID.to_numpy())
gt=pd.util.hash_pandas_object(raw_train[emb].round(10),index=False)
gv=pd.util.hash_pandas_object(raw_val[emb].round(10),index=False)
masked=raw_train[labels].eq(-100).all(axis=1)
diagnostics=dict(current_pooled_oof_original_training_rows=float(row_f1[:,~from_val].mean()),
    current_pooled_oof_original_validation_rows=float(row_f1[:,from_val].mean()),
    original_validation_rows_matching_masked_training_embeddings=int(gv.isin(gt[masked]).sum()),
    original_validation_rows_matching_gold_training_embeddings=int(gv.isin(gt[~masked]).sum()),
    interpretation='Subset OOF diagnostics, not separate holdout scores. Group CV excludes matching masked rows; the original training-only full fit legitimately used all provided training rows. This difference is not established as the cause of a score change.')
(P/'population-diagnostics.json').write_text(json.dumps(diagnostics,indent=2))
(P/'conclusions.json').write_text(json.dumps(conclusions,indent=2))
nb=nbf.v4.new_notebook();nb.metadata.kernelspec=dict(display_name='Python (AML Midterm)',language='python',name='aml-midterm')
nb.cells=[nbf.v4.new_markdown_cell(f'''# Frozen-recipe comparison with pooled cross-validation

We compared **{len(t)} recipes** using three separate measurements: the primary training-only CV, supplementary pooled train-plus-validation CV, and each original training-only fit's recorded validation score. No hyperparameters, feature choices, thresholds or model combinations were retuned during this comparison.

The retained hybrid scores **{current.training_cv_mean:.6f} training-only CV**, **{current.pooled_cv_mean:.6f} pooled CV**, and **{current.original_validation_f1:.6f} original validation F1**. Its ranks by these point estimates are **{int(current.training_cv_mean_rank)}, {int(current.pooled_cv_mean_rank)}, and {int(current.original_validation_f1_rank)}**, respectively. Tied scores share a rank.

The saved model and submission remain unchanged. The original validation score describes the original model trained without validation labels. It is not a holdout score for any pooled-fold model. No final model was fitted on all 235 labels and no Kaggle score was measured.'''),
nbf.v4.new_code_cell("from pathlib import Path\nimport json,pandas as pd,numpy as np\nimport matplotlib.pyplot as plt\nroot=Path.cwd();pooled=root/'experiments/pooled_benchmark_v1'\ntable=pd.read_csv(pooled/'comparison.csv')\ntable[['candidate','training_cv_mean','training_cv_std','pooled_cv_mean','pooled_cv_std','original_validation_f1']].sort_values('pooled_cv_mean',ascending=False)"),
nbf.v4.new_markdown_cell('''## Cohort and fixed settings

The comparison cohort was frozen before the pooled results were computed. It contains the five existing common-benchmark recipes plus every distinct inductive recipe with previously recorded validation sample-wise F1 of at least 0.900. Exact configuration/decision-rule duplicates were removed. Twenty recipes remain; some similar recipes can still make identical predictions.

Clustering and classification families, feature selections, regularization settings, pseudo-labeling rules, random seeds and decoding thresholds remain unchanged. Each fold trains a fresh instance. Learned coefficients, scaler statistics, cluster locations and provisional labels can change because the training rows change. Thus CV tests the fixed training recipe, not a single saved fitted object.

Existing internal calibration steps still fit only within their outer training fold, using their frozen settings. The Other subclass algorithm can reduce its requested cluster count if its pre-existing minimum-support rule requires it; that rule is unchanged. There is no new hyperparameter search.'''),
nbf.v4.new_markdown_cell('''## The three measurements

| Measurement | Gold labels available for splitting | Fitting and scoring |
|---|---:|---|
| Primary training-only CV | 157 | Same saved grouped five-fold splits and three seeds for every recipe; score complete out-of-fold predictions. |
| Supplementary pooled CV | 235 | Combine the 157 original training labels and 78 original validation labels; create a separate saved grouped five-fold partition for each of the same three seeds. |
| Original validation F1 | 157 for fitting, 78 held out | Reuse each unchanged recipe's already recorded score from its training-only fit. |

Both CV protocols use seeds 20261126–20261128. The actual fold memberships differ because the sample pools differ. Pair model differences within a protocol; matching seed numbers do not make the training-only and pooled estimates paired evaluations of the same held-out rows.

For each CV repeat, sample-wise F1 is calculated over the complete OOF matrix, weighting trials equally. The headline is the mean of those three repeat scores. Their standard deviation describes sensitivity to the partition; it is not a confidence interval.'''),
nbf.v4.new_code_cell("protocol=json.loads((pooled/'protocol.json').read_text())\nfolds=pd.DataFrame([{k:f[k] for k in ['seed','fold','training_gold_rows','heldout_gold_rows','original_validation_rows_in_training','original_validation_rows_heldout','masked_duplicates_excluded']} for f in protocol['folds']])\nfolds"),
nbf.v4.new_markdown_cell('''## Grouping, fitting boundaries and input preservation

The combined pool contains 235 gold rows in 222 unique embedding groups, plus the same 1,014 originally masked training examples. Identical embedding groups never cross an outer fold boundary. Masked examples matching a held-out group's embedding are excluded from that training fold. The observed gold duplicates have no conflicting reason labels.

Scaling, PCA, clusters, classifiers and pseudo-labeling fit only the current outer training gold and permitted masked rows. Rows originally belonging to validation can supply labels only when assigned to the training portion of that pooled fold. Held-out inputs do not participate in fitting. Kaggle test features are not used.

The pooled script creates an isolated in-memory dataset context and restores the original engine afterward. It does not modify either original CSV, the saved current-best configuration, any current prediction file or the primary CV fold assignments. Hash checks verify those boundaries. The current-model and nested-calibration paths also pass held-out-label perturbation and query-batch invariance tests in the enlarged data context.'''),
nbf.v4.new_code_cell("conclusions=json.loads((pooled/'conclusions.json').read_text())\nconclusions"),
nbf.v4.new_code_cell("summary=json.loads((pooled/'summary.json').read_text())\nrows=[]\nfor r in summary['models']:\n    result=json.loads((pooled/(r['pooled_result_key']+'.json')).read_text())\n    for seed,m in zip(summary['seeds'],result['repeat_scores']):\n        rows.append(dict(candidate=r['candidate'],seed=seed,F1=m['sample_f1']))\npd.DataFrame(rows).pivot(index='candidate',columns='seed',values='F1')"),
nbf.v4.new_code_cell("plot=table.sort_values('pooled_cv_mean')\ny=np.arange(len(plot));fig,ax=plt.subplots(figsize=(10,8))\nfor col,label,color in [('training_cv_mean','Training-only CV','#2864aa'),('pooled_cv_mean','Pooled CV','#188977'),('original_validation_f1','Original validation','#c46b23')]:\n    ax.scatter(plot[col],y,label=label,color=color,s=35)\nax.set(yticks=y,yticklabels=plot.candidate,xlabel='Sample-wise F1',xlim=(min(.84,float(plot[['training_cv_mean','pooled_cv_mean','original_validation_f1']].min().min())-.01),1.0))\nax.grid(axis='x',alpha=.2);ax.legend(loc='lower right',fontsize=8)\nfig.tight_layout();fig.savefig(root/'figures/current_pooled_benchmark.png',dpi=180)\nplt.show()"),
nbf.v4.new_markdown_cell('''## Interpreting the rankings

These are three complementary measurements on overlapping data, not three independent tests. Pooled CV uses a larger labeled training sample in each fold and a different mixture of labeled examples. A higher pooled score therefore cannot be attributed solely to larger held-out folds, and it is not a remeasurement of the exact saved model.

Prior development already used training labels and repeatedly consulted validation. Freezing recipes and common folds improves comparability but does not erase that selection history. The later-year Kaggle population can still differ from both labeled pools. Small rank differences should not be treated as established superiority, especially with only three dependent repeats.

The cohort and all settings were frozen before this comparison. Point-estimate leaders are reported explicitly; no recipe is automatically promoted just for topping one column. The primary training-only protocol remains the standard for future comparisons. The final all-label refit remains deferred.'''),
nbf.v4.new_markdown_cell('''## Why pooled CV need not score higher

More training labels do not guarantee a higher measured score. Pooled CV changes both the training sample and the population being scored, while keeping the algorithm settings fixed. It also applies the same strict duplicate-group exclusion rule to the original validation rows: matching masked training rows are removed when their group is held out. The original standalone validation fit used all supplied training rows, which included some unlabeled examples with identical embeddings.

The diagnostics below quantify those differences for the retained recipe. The source-specific pooled OOF scores are descriptive subsets of the same CV run; neither is a new independent validation result. These observations do not establish which factor caused the score change.'''),
nbf.v4.new_code_cell("json.loads((pooled/'population-diagnostics.json').read_text())"),
nbf.v4.new_code_cell("json.loads((pooled/'integrity.json').read_text())"),
nbf.v4.new_code_cell("preservation=json.loads((pooled/'preservation.json').read_text())\n{k:v for k,v in preservation.items() if k!='protected_sha256'}"),
nbf.v4.new_markdown_cell('''## Reproduction

Run `.venv/bin/python benchmark_cv.py` for the primary comparison and `.venv/bin/python benchmark_pooled_cv.py` for the separate pooled comparison. Saved results are reused only when the protocol, recipe, decoder and source hashes match. `--verify-only` checks pooled fold and recipe consistency; `--integrity-only` checks fitting boundaries without running the full cohort.

The cohort snapshot, exact configurations, source hashes, complete OOF predictions, fold memberships and per-fold metrics are included under `experiments/benchmark_v1` and `experiments/pooled_benchmark_v1`. Notebook 04 embeds the source files. Historical validation measurements retain their source filenames in the frozen registries.''')]
nbf.write(nb,ROOT/'midterm/11_pooled_cv_comparison.ipynb')
print(json.dumps(conclusions,indent=2))


Writing current_reproduction/build_pooled_benchmark_report.py


In [15]:
%%writefile current_reproduction/continued_select.py
"""Repeated grouped CV selection and frozen validation for a specified search phase."""
import phase20,experiment as e
import json,datetime,numpy as np,pandas as pd,argparse
from pathlib import Path
from threadpoolctl import threadpool_limits
parser=argparse.ArgumentParser();parser.add_argument('tag');args=parser.parse_args();tag=args.tag
rs=[json.loads(s) for s in Path('experiments/trials.jsonl').read_text().splitlines()];rs=[r for r in rs if r['tag']==tag and r['status']=='ok']
def family(c):
 if tag=='phase8':return c['model']
 if tag=='phase9':
  if c.get('white'):return 'whitening'
  if c.get('kernel','rbf')!='rbf':return 'non_rbf_kernel'
  return 'embedding_geometry'
 if tag=='phase10':return 'structured_cluster' if c.get('cluster_gate') else 'structured_joint'
 if tag=='phase11':return 'random_subspace' if c.get('members',1)>1 else ('prior_constraint' if c.get('constrain_prior') else 'confidence_filter')
 if tag=='phase14':return 'label_thresholds' if c['calibration']=='thresholds' else ('label_patterns' if c['calibration']=='patterns' else 'margin_calibration')
 if tag=='phase16':return 'alternative_loss'
 if tag=='phase17':return c['student']
 if tag=='phase18':return 'time_weights' if c['weighting'] in ['year','late'] else 'distribution_weights'
 if tag=='phase19':return 'size_matched_thresholds' if c['calibration']=='thresholds' else 'size_matched_calibration'
 if tag=='phase20':return 'separate_feature_kernels' if c.get('aux_blocks') else ('repeated_vector_weighting' if c.get('deduplicate') else 'effective_class_balance')
 raise ValueError(tag)
chosen={}
for fam in sorted({family(r['config']) for r in rs}):
 rows=[r for r in rs if family(r['config'])==fam]
 for r in [max(rows,key=lambda r:r['score']),max(rows,key=lambda r:r['macro_f1'])]:chosen[r['id']]=r
summary=[]
with threadpool_limits(limits=1):
 for r in chosen.values():
  ps=[]
  for seed in [20261104,20261105,20261106]:
   t=tag if seed==20261104 else tag+'_repeat';e.evaluate(r['config'],seed=seed,tag=t)
   ps.append(np.load(f"experiments/{t}-{r['id']}-{seed}.npz",allow_pickle=True)['p'])
  p=np.mean(ps,axis=0);rule,score=e.choose_rule(e.Y,p)
  rr=dict(name=family(r['config']),config=r['config'],rule=rule,cv=score,cv_mean=float(np.mean([e.metrics(e.Y,z,rule)['sample_f1'] for z in ps])),cv_macro=float(np.mean([e.metrics(e.Y,z,rule)['macro_f1'] for z in ps])))
  summary.append(rr);print('SUMMARY',json.dumps(rr),flush=True)
e.save_json(f'experiments/{tag}-selection.json',summary)
candidates=[max([r for r in summary if r['name']==fam],key=lambda r:.8*r['cv_mean']+.2*r['cv_macro']) for fam in sorted({r['name'] for r in summary})]
e.save_json(f'experiments/{tag}-validation-frozen.json',dict(frozen_at=datetime.datetime.now(datetime.timezone.utc).isoformat(),candidates=candidates))
V=pd.read_csv('data/part1_val.csv',usecols=lambda c:c not in e.LABELS)
with threadpool_limits(limits=1):preds=[e.predict(np.arange(len(e.Y)),V,c['config']) for c in candidates]
np.savez_compressed(f'experiments/{tag}-validation-predictions.npz',p=preds)
y=pd.read_csv('data/part1_val.csv')[e.LABELS].to_numpy();results=[]
for c,p in zip(candidates,preds):
 r={**c,'validation':e.metrics(y,p,c['rule'])};results.append(r);print('VALIDATION',json.dumps(r),flush=True)
e.save_json(f'experiments/{tag}-validation-results.json',results)


Overwriting current_reproduction/continued_select.py


In [16]:
%%writefile current_reproduction/evaluate_final_validation.py
"""Evaluate the already-selected full-fit candidate, without retuning or validation fitting."""
from pathlib import Path
import json, hashlib, datetime
import numpy as np
import pandas as pd
from threadpoolctl import threadpool_limits
import experiment as e

manifest_path=Path('experiments/final-candidate.json')
manifest=json.loads(manifest_path.read_text())
assert hashlib.sha256(Path('experiment.py').read_bytes()).hexdigest()==manifest['engine_sha256']
assert hashlib.sha256(Path(manifest['submission']).read_bytes()).hexdigest()==manifest['submission_sha256']
selected=manifest['selected']
val=pd.read_csv('data/part1_val.csv',usecols=lambda c:c not in e.LABELS)
test=pd.read_csv('data/part1_test.csv')
unlabeled=e.DATA.iloc[e.UNLAB].drop(columns=e.LABELS)
query=pd.concat([val,test,unlabeled],ignore_index=True)
with threadpool_limits(limits=1):
    p=e.predict(np.arange(len(e.Y)),query,selected['config'])
# Check this is the same candidate that generated the delivered Kaggle predictions.
saved=pd.read_csv('predictions/fullfit_test_probabilities.csv')
np.testing.assert_allclose(p[len(val):len(val)+len(test)],saved[['p_'+l for l in e.LABELS]].to_numpy(),rtol=1e-10,atol=1e-12)
test_predictions=e.decisions(p[len(val):len(val)+len(test)],selected['rule'])
strings=[' '.join(l for l,v in zip(e.LABELS,row) if v) for row in test_predictions]
assert strings==pd.read_csv(manifest['submission']).Predicted.tolist()
# Validation labels are accessed only after fitting and prediction.
y=pd.read_csv('data/part1_val.csv',usecols=e.LABELS)[e.LABELS].to_numpy(int)
vp=p[:len(val)];hard=e.decisions(vp,selected['rule'])
result={
    'evaluated_at':datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'candidate':selected['name'],'n_validation':len(val),
    'rule':selected['rule'],'config':selected['config'],
    **e.metrics(y,vp,selected['rule']),
    'training_gold_rows':len(e.Y),'validation_used_for_fitting':False,
    'settings_or_threshold_changed':False,'same_test_probabilities_verified':True,
    'engine_sha256':manifest['engine_sha256'],'submission_sha256':manifest['submission_sha256'],
    'context':'User requested direct evaluation of the frozen full-fit candidate. This is a second validation evaluation occasion, with no retuning. Earlier bagged-model validation results remain separate.'
}
e.save_json('experiments/final-validation.json',result)
out=pd.DataFrame(vp,columns=['p_'+l for l in e.LABELS]);out.insert(0,'ID',val.ID.to_numpy())
for j,l in enumerate(e.LABELS):out['pred_'+l]=hard[:,j];out['true_'+l]=y[:,j]
out['sample_f1']=2*(y*hard).sum(1)/(y.sum(1)+hard.sum(1))
out.to_csv('predictions/fullfit_validation_audit.csv',index=False)
manifest['validation_status']='Evaluated on all 78 validation rows at user request, with model settings and threshold unchanged.'
manifest['validation_metrics']={k:result[k] for k in ['sample_f1','macro_f1']+['f1_'+l for l in e.LABELS]}
manifest['validation_result']='experiments/final-validation.json'
e.save_json(manifest_path,manifest)
print(json.dumps(result,indent=2))


Overwriting current_reproduction/evaluate_final_validation.py


In [17]:
%%writefile current_reproduction/experiment.py
"""Reproducible, train-only Part 1 development engine. See experiments/protocol.txt."""
import os
for var in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','VECLIB_MAXIMUM_THREADS','NUMEXPR_NUM_THREADS']:
    os.environ.setdefault(var,'1')
import argparse, hashlib, json, time, traceback, warnings, re, ast
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.special import expit, softmax
from scipy.special import logit
from scipy.sparse.linalg import spsolve
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.kernel_ridge import KernelRidge
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF, ConstantKernel
from sklearn.neighbors import NeighborhoodComponentsAnalysis
from sklearn.svm import SVC, LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.neighbors import KNeighborsRegressor, NearestNeighbors
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.cluster import KMeans, BisectingKMeans, AgglomerativeClustering, DBSCAN, HDBSCAN, SpectralClustering, Birch
from sklearn.mixture import GaussianMixture
from sklearn.metrics import f1_score, pairwise_distances
from sklearn.multioutput import ClassifierChain
from sklearn.cross_decomposition import PLSRegression
from scipy.special import logsumexp
from scipy.linalg import cho_factor, cho_solve
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from threadpoolctl import threadpool_limits
warnings.filterwarnings('ignore',category=FutureWarning)
ROOT=Path(__file__).resolve().parent
LABELS=['Safety','Funding','Efficacy','Recruitment','Other']; EMB=[f'emb_{i}' for i in range(150)]
DATA=pd.read_csv(ROOT/'data/part1_train.csv')
LAB=np.flatnonzero(DATA[LABELS].ge(0).all(axis=1)); UNLAB=np.flatnonzero(DATA[LABELS].eq(-100).all(axis=1))
Y=DATA.iloc[LAB][LABELS].to_numpy(int)
GROUP=pd.util.hash_pandas_object(DATA[EMB].round(10),index=False).to_numpy()
assert len(LAB)+len(UNLAB)==len(DATA)
CACHE={}

def config_id(c): return hashlib.sha256(json.dumps(c,sort_keys=True).encode()).hexdigest()[:14]
def save_json(path,obj): Path(path).write_text(json.dumps(obj,indent=2,default=lambda x:x.item() if isinstance(x,np.generic) else str(x)))
def make_splits(y=Y,groups=None,seed=20261104,n_splits=5):
    if groups is None: groups=GROUP[LAB]
    unique,inv=np.unique(groups,return_inverse=True)
    gy=np.array([y[inv==i].max(axis=0) for i in range(len(unique))])
    return [(np.flatnonzero(np.isin(inv,a)),np.flatnonzero(np.isin(inv,b))) for a,b in MultilabelStratifiedKFold(n_splits=n_splits,shuffle=True,random_state=seed).split(gy,gy)]

def decisions(p,rule='t0.5'):
    p=np.clip(np.asarray(p),0,1)
    if rule=='expected_f1':
        patterns=((np.arange(32)[:,None]>>np.arange(5))&1)
        prob=np.prod(np.where(patterns[None,:,:],p[:,None,:],1-p[:,None,:]),axis=2)
        choices=patterns[1:]
        f1=2*(choices@patterns.T)/(choices.sum(1)[:,None]+patterns.sum(1)[None,:])
        return choices[np.argmax(prob@f1.T,axis=1)]
    if rule=='argmax':
        out=np.zeros_like(p,dtype=int);out[np.arange(len(p)),p.argmax(1)]=1;return out
    out=(p>=float(rule[1:])).astype(int)
    empty=out.sum(1)==0
    out[np.flatnonzero(empty),p[empty].argmax(1)]=1
    return out
RULES=['t0.25','t0.35','t0.45','t0.5','t0.55','expected_f1','argmax']
def metrics(y,p,rule='t0.5'):
    pred=decisions(p,rule)
    return {'sample_f1':float(f1_score(y,pred,average='samples',zero_division=0)), 'macro_f1':float(f1_score(y,pred,average='macro',zero_division=0)),**{f'f1_{l}':float(v) for l,v in zip(LABELS,f1_score(y,pred,average=None,zero_division=0))}}
def choose_rule(y,p):
    scores=[metrics(y,p,r)['sample_f1'] for r in RULES]
    # Prefer default threshold when differences are negligible.
    best=max(scores); idx=min([i for i,s in enumerate(scores) if s>=best-1e-12],key=lambda i:abs(i-3))
    return RULES[idx],best

def preprocess(A,B,c):
    A=np.asarray(A,float);B=np.asarray(B,float)
    mode=c.get('scale','standard')
    if mode=='standard':
        sc=StandardScaler();A=sc.fit_transform(A);B=sc.transform(B)
    elif mode=='global':
        mean=A.mean(0);std=np.sqrt(np.mean(np.var(A,axis=0)));A=(A-mean)/std;B=(B-mean)/std
    elif mode=='raw':
        # Global scale without centering preserves cosine and original relative geometry.
        std=np.sqrt(np.mean(np.var(A,axis=0)));A=A/std;B=B/std
    n=c.get('pca',0)
    if n:
        pc=PCA(n_components=min(n,A.shape[1],A.shape[0]-1),svd_solver='full');A=pc.fit_transform(A);B=pc.transform(B)
        power=c.get('white',0)
        if power:
            scale=np.maximum(pc.explained_variance_,1e-8)**(power/2);A/=scale;B/=scale
    if c.get('norm',False):A=normalize(A);B=normalize(B)
    return A,B

def chemical(frame,kind):
    from rdkit import Chem, RDLogger
    from rdkit.Chem import Descriptors, rdFingerprintGenerator
    RDLogger.DisableLog('rdApp.*')
    gen=rdFingerprintGenerator.GetMorganGenerator(radius=2,fpSize=1024)
    desc=[Descriptors.MolWt,Descriptors.MolLogP,Descriptors.TPSA,Descriptors.NumHDonors,Descriptors.NumHAcceptors,Descriptors.NumRotatableBonds,Descriptors.RingCount,Descriptors.FractionCSP3,Descriptors.HeavyAtomCount]
    out=[]
    for field in frame.drug_smiles.fillna(''):
        mols=[m for s in field.split(';') if s.strip() and (m:=Chem.MolFromSmiles(s.strip())) is not None]
        if kind=='fingerprint':v=np.max([gen.GetFingerprintAsNumPy(m) for m in mols],axis=0) if mols else np.zeros(1024)
        else:
            z=np.array([[fn(m) for fn in desc] for m in mols])
            v=np.r_[len(mols),z.mean(0),z.max(0)] if mols else np.zeros(19)
        out.append(v)
    return np.array(out,float)

def block_features(A,B,block):
    if block in ('fingerprint','descriptors'):
        a=chemical(A,block);b=chemical(B,block)
    elif block=='year':a=A[['start_year']].to_numpy();b=B[['start_year']].to_numpy()
    elif block=='missingness':
        cols=['criteria','diseases_cleaned','drugs','icdcodes','matched_drugs','drug_targets','drug_actions','drug_pathways','drug_smiles']
        a=A[cols].isna().astype(float).to_numpy();b=B[cols].isna().astype(float).to_numpy()
    elif block=='counts':
        cols=['criteria','diseases_cleaned','drugs','icdcodes','matched_drugs','drug_targets','drug_actions','drug_pathways','drug_smiles']
        def counts(D):return np.column_stack([np.log1p(D[c].fillna('').str.len()) for c in cols]+[D[c].fillna('').str.count(';')+D[c].notna() for c in cols[4:]])
        a=counts(A);b=counts(B)
    elif block.startswith('pathway_level'):
        catalog=pd.read_csv(ROOT/'data/pathway_catalog.csv');levels=dict(zip(catalog.reactome_id,catalog.pathway_level))
        if block=='pathway_levels':
            def calc(D):return np.array([[sum(levels.get(s.strip(),-1)==i for s in str(v).split(';')) for i in range(10)] for v in D.drug_pathways.fillna('')],float)
            a=np.log1p(calc(A));b=np.log1p(calc(B))
        else:
            lev=int(block[-1]); allowed=set(catalog.loc[catalog.pathway_level<=lev,'reactome_id'])
            def clean(D):return D.drug_pathways.fillna('').map(lambda x:' '.join(t.strip() for t in x.split(';') if t.strip() in allowed))
            v=TfidfVectorizer(token_pattern=r'[^\s]+',min_df=2,max_features=1000);a=v.fit_transform(clean(A)).toarray();b=v.transform(clean(B)).toarray();return a,b
    else:
        col=block.replace('_char','');char=block.endswith('_char')
        v=TfidfVectorizer(lowercase=True,analyzer='char_wb' if char else 'word',ngram_range=(3,5) if char else (1,2),min_df=2,max_df=0.99,max_features=3000,sublinear_tf=True,token_pattern=r'(?u)\b[\w-]+\b')
        a=v.fit_transform(A[col].fillna(''));b=v.transform(B[col].fillna(''))
        # Keep exact sparse-text geometry; at this dataset size dense blocks fit in memory.
        return a.toarray(),b.toarray()
    if block!='fingerprint':
        sc=StandardScaler();a=sc.fit_transform(a);b=sc.transform(b)
    a=np.nan_to_num(a);b=np.nan_to_num(b)
    norm=np.sqrt(np.mean(np.sum(a*a,axis=1)))
    return a/max(norm,1e-8),b/max(norm,1e-8)

def features(train_idx,query_frame,c):
    fitframe=DATA.iloc[train_idx]
    A,B=preprocess(fitframe[EMB],query_frame[EMB],c)
    blocks=c.get('features',[])
    if blocks:
        base_norm=np.sqrt(np.mean(np.sum(A*A,axis=1)))
        weight=c.get('feature_weight',0.3)
        for block in blocks:
            a,b=block_features(fitframe,query_frame,block)
            A=np.column_stack([A,a*base_norm*weight]);B=np.column_stack([B,b*base_norm*weight])
    return A,B

class Graph:
    """Normalized graph diffusion of all five binary labels; independent negative mass implicit."""
    def __init__(self,c):self.c=c
    def fit(self,X,Y):
        self.X=X; n=len(X);c=self.c;k=min(c.get('k',15),n-1);self.k=k
        self.nn=NearestNeighbors(n_neighbors=k+1).fit(X);dist,ix=self.nn.kneighbors(X)
        dist=dist[:,1:];ix=ix[:,1:];self.sigma=np.maximum(dist[:,-1],1e-8)
        self.band=float(np.median(dist[:,-1]))*c.get('band',1.)
        if c.get('graph','adaptive')=='adaptive':w=np.exp(-dist**2/(self.sigma[:,None]*self.sigma[ix]+1e-12))
        elif c.get('graph')=='binary':w=np.ones_like(dist)
        else:w=np.exp(-dist**2/max(self.band**2,1e-12))
        W=sparse.csr_matrix((w.ravel(),(np.repeat(np.arange(n),k),ix.ravel())),shape=(n,n))
        W=(W+W.T)*0.5
        degree=np.maximum(np.asarray(W.sum(1)).ravel(),1e-12)
        known=Y[:,0]>=0
        if c.get('hard',False):
            P=sparse.diags(1/degree)@W;u=np.flatnonzero(~known);l=np.flatnonzero(known)
            alpha=c.get('alpha',0.99)
            U=spsolve(sparse.eye(len(u),format='csc')-alpha*P[u][:,u].tocsc(),alpha*P[u][:,l]@Y[l]+(1-alpha)*Y[l].mean(0))
            self.F=Y.copy().astype(float);self.F[u]=U
        else:
            S=sparse.diags(degree**-0.5)@W@sparse.diags(degree**-0.5)
            Z=np.zeros((n,6));Z[known,:5]=Y[known];Z[known,5]=1
            sol=spsolve(sparse.eye(n,format='csc')-c.get('alpha',0.95)*S.tocsc(),Z)
            self.F=np.divide(sol[:,:5],sol[:,5,None],out=np.tile(Y[known].mean(0),(n,1)),where=sol[:,5,None]>1e-15)
        self.F=np.clip(self.F,0,1);return self
    def predict(self,Q):
        d,ix=self.nn.kneighbors(Q,n_neighbors=self.k)
        if self.c.get('graph','adaptive')=='adaptive':w=np.exp(-d*d/(np.maximum(d[:,-1,None],1e-8)*self.sigma[ix]+1e-12))
        elif self.c.get('graph')=='binary':w=np.ones_like(d)
        else:w=np.exp(-d*d/max(self.band**2,1e-12))
        exact=d[:,0]<1e-8
        p=np.einsum('ij,ijk->ik',w,self.F[ix])/np.maximum(w.sum(1)[:,None],1e-15)
        p[exact]=self.F[ix[exact,0]]
        return p

class Cluster:
    def __init__(self,c):self.c=c
    def fit(self,X,Y):
        c=self.c;typ=c['cluster'];n=c.get('clusters',12)
        if typ=='kmeans':model=KMeans(n_clusters=n,n_init=10,random_state=42)
        elif typ=='bisect':model=BisectingKMeans(n_clusters=n,n_init=3,random_state=42)
        elif typ=='gmm':model=GaussianMixture(n_components=n,covariance_type=c.get('cov','diag'),reg_covar=0.02,n_init=2,max_iter=200,random_state=42)
        elif typ=='agglomerative':model=AgglomerativeClustering(n_clusters=n,linkage=c.get('linkage','ward'))
        elif typ=='spectral':model=SpectralClustering(n_clusters=n,affinity='nearest_neighbors',n_neighbors=c.get('k',15),assign_labels='cluster_qr',random_state=42)
        elif typ=='birch':model=Birch(n_clusters=n,threshold=0.5)
        elif typ=='hdbscan':model=HDBSCAN(min_cluster_size=c.get('min_cluster',8),min_samples=c.get('min_samples',3))
        elif typ=='dbscan':
            ds=NearestNeighbors(n_neighbors=6).fit(X).kneighbors()[0][:,-1]
            model=DBSCAN(eps=float(np.quantile(ds,c.get('eps_q',0.5))),min_samples=4)
        else:raise ValueError(typ)
        z=model.fit_predict(X);self.model=model;self.X=X;self.z=z
        known=Y[:,0]>=0;self.prior=Y[known].mean(0)
        ids=sorted(set(z)-{-1});self.ids=ids;self.centers=np.array([X[z==i].mean(0) for i in ids])
        self.mapping={};smooth=c.get('smooth',0.2)
        for i in ids:
            mask=known&(z==i);self.mapping[i]=(Y[mask].sum(0)+smooth*self.prior)/(mask.sum()+smooth)
        self.fallback=KNeighborsRegressor(n_neighbors=min(5,known.sum()),weights='distance').fit(X[known],Y[known])
        self.known=known;self.Y=Y
        if typ=='gmm':
            R=model.predict_proba(X[known]);self.softmap=(R.T@Y[known]+smooth*self.prior)/(R.sum(0)[:,None]+smooth)
        return self
    def predict(self,Q):
        typ=self.c['cluster']
        if typ=='gmm':return self.model.predict_proba(Q)@self.softmap
        if hasattr(self.model,'predict'):z=self.model.predict(Q)
        else:
            # Inductive extension for algorithms without predict: nearest training member.
            _,ix=NearestNeighbors(n_neighbors=1).fit(self.X).kneighbors(Q);z=self.z[ix[:,0]]
        p=self.fallback.predict(Q)
        for i in self.ids:
            if np.sum(self.known&(self.z==i))>0:p[z==i]=self.mapping[i]
        return p

def fit_supervised(X,Y,Q,c,weights=None):
    typ=c['model'];param=c.get('C',1.);out=[]
    if typ=='knn':return KNeighborsRegressor(n_neighbors=min(c.get('k',5),len(X)),weights=c.get('weights','distance'),metric=c.get('metric','euclidean')).fit(X,Y).predict(Q)
    if typ=='ridge':return np.clip(Ridge(alpha=c.get('alpha',10)).fit(X,Y,sample_weight=weights).predict(Q),0,1)
    if typ=='krr':return np.clip(KernelRidge(alpha=c.get('alpha',.1),kernel='rbf',gamma=c.get('gamma',.003)).fit(X,Y,sample_weight=weights).predict(Q),0,1)
    if typ=='pls':return np.clip(PLSRegression(n_components=c.get('components',5)).fit(X,Y).predict(Q),0,1)
    if typ in ('softmax','softsvc'):
        rows,classes=np.where(Y>0);ww=1/Y.sum(1)[rows]
        if weights is not None:ww*=weights[rows]
        if typ=='softmax':m=LogisticRegression(C=param,max_iter=2000)
        else:m=SVC(C=param,gamma=c.get('gamma','scale'),probability=True,random_state=42)
        m.fit(X[rows],classes,sample_weight=ww)
        p=np.zeros((len(Q),Y.shape[1]));p[:,m.classes_]=m.predict_proba(Q);return p
    if typ=='nca':
        nca=NeighborhoodComponentsAnalysis(n_components=c.get('components',5),max_iter=100,random_state=42)
        xx=nca.fit_transform(X,Y@2**np.arange(5));qq=nca.transform(Q)
        return KNeighborsRegressor(n_neighbors=c.get('k',5),weights='distance').fit(xx,Y).predict(qq)
    if typ=='chain':
        model=ClassifierChain(LogisticRegression(C=param,max_iter=1000),order='random',random_state=c.get('seed',42),cv=3,chain_method='predict_proba')
        return model.fit(X,Y).predict_proba(Q)
    if typ=='powerset':
        # Preserve multilabel patterns instead of imposing mutual exclusivity.
        code=Y@2**np.arange(5)
        if c.get('kernel','rbf')=='linear':model=LogisticRegression(C=param,max_iter=1000)
        else:model=SVC(C=param,gamma=c.get('gamma','scale'),probability=True,random_state=42)
        model.fit(X,code,sample_weight=weights);patterns=((model.classes_[:,None]>>np.arange(5))&1)
        return model.predict_proba(Q)@patterns
    for j in range(Y.shape[1]):
        y=Y[:,j]
        if len(np.unique(y))<2:out.append(np.full(len(Q),y[0]));continue
        if typ=='logistic':m=LogisticRegression(C=param,max_iter=2000,class_weight=c.get('balanced'),solver='lbfgs')
        elif typ=='svc':m=SVC(C=param,kernel=c.get('kernel','rbf'),gamma=c.get('gamma','scale'),degree=c.get('degree',2),class_weight=c.get('balanced'),probability=True,random_state=42)
        elif typ=='linear_svc':m=CalibratedClassifierCV(LinearSVC(C=param,class_weight=c.get('balanced'),max_iter=5000),cv=3)
        elif typ=='lda':m=LinearDiscriminantAnalysis(solver='lsqr',shrinkage=c.get('shrinkage','auto'))
        elif typ=='qda':m=QuadraticDiscriminantAnalysis(reg_param=c.get('reg',0.9))
        elif typ=='gnb':m=GaussianNB(var_smoothing=c.get('smooth',1e-2))
        elif typ=='gp':m=GaussianProcessClassifier(kernel=ConstantKernel(c.get('amplitude',1))*RBF(c.get('length',10)),optimizer=None,random_state=42)
        elif typ in ('rf','extra'):
            cls=RandomForestClassifier if typ=='rf' else ExtraTreesClassifier
            m=cls(n_estimators=c.get('trees',200),max_depth=c.get('depth'),min_samples_leaf=c.get('leaf',2),max_features=c.get('max_features','sqrt'),class_weight=c.get('balanced'),random_state=42,n_jobs=1)
        elif typ=='hist':m=HistGradientBoostingClassifier(max_iter=150,max_leaf_nodes=7,l2_regularization=5,random_state=42)
        elif typ=='catboost':
            from catboost import CatBoostClassifier
            m=CatBoostClassifier(iterations=c.get('trees',250),depth=c.get('depth',3),learning_rate=0.04,l2_leaf_reg=5,verbose=False,thread_count=1,allow_writing_files=False,random_seed=42)
        elif typ=='xgb':
            from xgboost import XGBClassifier
            m=XGBClassifier(n_estimators=c.get('trees',150),max_depth=c.get('depth',2),learning_rate=.04,reg_lambda=5,min_child_weight=2,subsample=.9,colsample_bytree=.8,n_jobs=1,random_state=42)
        elif typ=='lightgbm':
            from lightgbm import LGBMClassifier
            m=LGBMClassifier(n_estimators=150,num_leaves=7,min_child_samples=10,learning_rate=.04,reg_lambda=5,verbosity=-1,n_jobs=1,random_state=42)
        else:raise ValueError(typ)
        if weights is not None and typ not in ('lda','qda'):m.fit(X,y,sample_weight=weights)
        else:m.fit(X,y)
        out.append(m.predict_proba(Q)[:,1])
    return np.array(out).T

def predict(train_pos,query_frame,c,excluded_groups=None,return_unlabeled=False):
    if c['model']=='ensemble':
        weights=np.asarray(c.get('weights',np.ones(len(c['members']))),float);weights/=weights.sum()
        outputs=[predict(train_pos,query_frame,m,excluded_groups,return_unlabeled) for m in c['members']]
        if return_unlabeled:
            return sum(w*x[0] for w,x in zip(weights,outputs)),sum(w*x[1] for w,x in zip(weights,outputs)),outputs[0][2]
        return sum(w*x for w,x in zip(weights,outputs))
    # Fit unsupervised transforms only on fold-training labels and legitimate masked rows.
    actual=LAB[np.asarray(train_pos)]
    u=UNLAB if excluded_groups is None else UNLAB[~np.isin(GROUP[UNLAB],excluded_groups)]
    idx=np.r_[actual,u];n=len(actual)
    A,B=features(idx,query_frame,c);y=Y[train_pos];ys=np.full((len(A),5),-1);ys[:n]=y
    typ=c['model']
    if typ=='semigmm':
        # Fractional membership handles multilabel gold rows; masked rows participate in EM.
        # Labeled responsibilities remain fixed, and receive an explicit reliability weight.
        z=y/np.maximum(y.sum(1)[:,None],1)
        means=np.array([np.average(A[:n],axis=0,weights=z[:,j]+1e-8) for j in range(5)])
        covariance=np.cov(A.T)+np.eye(A.shape[1])*.1
        prior=(z.sum(0)+1)/(z.sum()+5)
        labelweight=c.get('label_weight',5.)
        def posterior(Q):
            ll=[]
            for j in range(5):
                cov=covs[j];factor=cho_factor(cov,lower=True);diff=Q-means[j]
                ll.append(-.5*np.sum(diff*cho_solve(factor,diff.T).T,axis=1)-np.log(np.diag(factor[0])).sum()+np.log(prior[j]))
            return softmax(np.array(ll).T,axis=1)
        covs=np.array([covariance.copy() for _ in range(5)])
        for iteration in range(80):
            R=posterior(A);R[:n]=z;R[:n]*=labelweight
            counts=R.sum(0)+1e-8;newmeans=R.T@A/counts[:,None]
            delta=np.max(np.abs(newmeans-means));means=newmeans;prior=(counts+1)/(counts.sum()+5)
            covlist=[]
            for j in range(5):
                diff=A-means[j];cov=(diff.T*R[:,j])@diff/counts[j]
                shrink=c.get('shrink',.8);cov=(1-shrink)*cov+shrink*covariance+np.eye(A.shape[1])*.05;covlist.append(cov)
            if c.get('cov','shared')=='shared':
                cov=np.average(covlist,axis=0,weights=counts);covs=np.array([cov]*5)
            else:covs=np.array(covlist)
            if delta<1e-5:break
        p=posterior(B);pu=posterior(A[n:])
    elif typ=='laprls':
        # RBF kernel regression with graph-Laplacian regularization from masked data.
        from sklearn.metrics.pairwise import rbf_kernel
        gamma=c.get('gamma',.003);K=rbf_kernel(A,gamma=gamma);KB=rbf_kernel(B,A,gamma=gamma)
        nn=NearestNeighbors(n_neighbors=11).fit(A);dd,ii=nn.kneighbors(A);dd=dd[:,1:];ii=ii[:,1:]
        W=sparse.csr_matrix((np.exp(-dd.ravel()**2/np.median(dd[:,-1])**2),(np.repeat(np.arange(len(A)),10),ii.ravel())),shape=(len(A),len(A)))
        W=(W+W.T)*.5;degree=np.asarray(W.sum(1)).ravel();L=sparse.diags(degree)-W
        # Normal equations in the representer coefficients; ridge makes the system stable.
        mat=K[:n].T@K[:n]+c.get('alpha',1)*K+c.get('mu',1)*(K@(L@K))+np.eye(len(A))*1e-6
        coef=cho_solve(cho_factor(mat,lower=True),K[:n].T@y)
        p=np.clip(KB@coef,0,1);pu=np.clip(K[n:]@coef,0,1)
    elif typ=='cluster_features':
        # Unsupervised prototype dictionary from genuine training + masked rows only.
        km=KMeans(n_clusters=c.get('clusters',40),n_init=5,random_state=42).fit(A)
        da=km.transform(A);db=km.transform(B)
        band=np.median(np.min(da,axis=1))*c.get('band',1.)
        H=np.exp(-da**2/max(2*band**2,1e-8));J=np.exp(-db**2/max(2*band**2,1e-8))
        sc=StandardScaler();H=sc.fit_transform(H);J=sc.transform(J)
        mix=c.get('original_weight',.3)
        H=np.column_stack([H,A*mix]);J=np.column_stack([J,B*mix])
        out=fit_supervised(H[:n],y,np.r_[J,H[n:]],{**c,'model':'logistic'})
        p=out[:len(B)];pu=out[len(B):]
    elif typ=='metric_graph':
        # Label-informed metric fitted only to fold-training labels, followed by unsupervised graph inference.
        base={**c,'model':c.get('metric_model','logistic')}
        scores=fit_supervised(A[:n],y,np.r_[A,B],base)
        scores=logit(np.clip(scores,.005,.995))
        sc=StandardScaler();s=sc.fit_transform(scores[:len(A)]);q=sc.transform(scores[len(A):])
        residual=c.get('residual',.3)
        norm=np.sqrt(np.mean(np.sum(A*A,axis=1)))
        U=np.column_stack([s,A*residual/max(norm,1e-8)])
        V=np.column_stack([q,B*residual/max(norm,1e-8)])
        model=Graph(c).fit(U,ys);p=model.predict(V);pu=model.F[n:]
    elif typ=='graph':
        model=Graph(c).fit(A,ys);p=model.predict(B);pu=model.F[n:]
    elif typ=='cluster':
        model=Cluster(c).fit(A,ys);p=model.predict(B);pu=model.predict(A[n:])
    elif typ=='pseudo':
        teacher=c['teacher'];T,U=preprocess(DATA.iloc[idx][EMB],query_frame[EMB],teacher)
        if teacher['model']=='graph':m=Graph(teacher).fit(T,ys);pt=m.F[n:]
        else:m=Cluster(teacher).fit(T,ys);pt=m.predict(T[n:])
        student={**c,'model':c.get('student','svc')}
        conf=c.get('confidence',.9);w=c.get('pseudo_weight',.2)
        base_out=None
        if c.get('consensus',False):
            base_out=fit_supervised(A[:n],y,np.r_[B,A[n:]],student)
            base_u=base_out[len(B):]
            agreement=(pt>=.5)==(base_u>=.5)
            # Confidence must be supported by both independently constructed views.
            pt=np.where(agreement,(pt+base_u)*.5,.5)
        yp=(pt>=.5).astype(int)
        rows=np.min(np.maximum(pt,1-pt),axis=1)>=conf
        if c.get('labelwise',False):
            p=np.zeros((len(B),5));pu=np.zeros((len(u),5))
            for j in range(5):
                sel=np.maximum(pt[:,j],1-pt[:,j])>=conf
                xx=np.r_[A[:n],A[n:][sel]];yy=np.r_[y[:,j],yp[sel,j]][:,None];ww=np.r_[np.ones(n),np.full(sel.sum(),w)]
                out=fit_supervised(xx,yy,np.r_[B,A[n:]],student,ww);p[:,j]=out[:len(B),0];pu[:,j]=out[len(B):,0]
        else:
            xx=np.r_[A[:n],A[n:][rows]];yy=np.r_[y,yp[rows]];ww=np.r_[np.ones(n),np.full(rows.sum(),w)]
            out=fit_supervised(xx,yy,np.r_[B,A[n:]],student,ww);p=out[:len(B)];pu=out[len(B):]
        if c.get('base_mix',0):
            if base_out is None:base_out=fit_supervised(A[:n],y,np.r_[B,A[n:]],student)
            mix=c['base_mix'];p=(1-mix)*p+mix*base_out[:len(B)];pu=(1-mix)*pu+mix*base_out[len(B):]
    else:
        out=fit_supervised(A[:n],y,np.r_[B,A[n:]] if return_unlabeled else B,c)
        p=out[:len(B)];pu=out[len(B):] if return_unlabeled else None
    if return_unlabeled:return p,pu,u
    return np.clip(p,0,1)

def evaluate(c,splits=None,seed=20261104,tag='screen',force=False):
    if splits is None:splits=make_splits(seed=seed)
    cid=config_id(c);path=ROOT/'experiments'/f'{tag}-{cid}-{seed}.npz'
    if path.exists() and not force:return dict(np.load(path,allow_pickle=True)['record'].item())
    started=time.time();p=np.full(Y.shape,np.nan);foldids=np.full(len(Y),-1)
    try:
        with threadpool_limits(limits=1):
            for fold,(a,b) in enumerate(splits):
                p[b]=predict(a,DATA.iloc[LAB[b]],c,excluded_groups=GROUP[LAB[b]])
                foldids[b]=fold
        use=foldids>=0;rule,score=choose_rule(Y[use],p[use]);base=metrics(Y[use],p[use]);tuned=metrics(Y[use],p[use],rule)
        fs=[metrics(Y[b],p[b],rule)['sample_f1'] for a,b in splits]
        record={'id':cid,'tag':tag,'seed':seed,'model':c['model'],'config':c,'n_scored':int(use.sum()),'score':score,'rule':rule,'default_score':base['sample_f1'],'fold_scores':fs,'fold_std':float(np.std(fs)),**tuned,'seconds':round(time.time()-started,3),'status':'ok'}
        np.savez_compressed(path,p=p,fold=foldids,record=record)
    except Exception as e:
        record={'id':cid,'tag':tag,'seed':seed,'config':c,'status':'error','error':str(e),'traceback':traceback.format_exc(),'seconds':round(time.time()-started,3)}
    with open(ROOT/'experiments/trials.jsonl','a') as f:f.write(json.dumps(record)+'\n')
    print(json.dumps({k:record.get(k) for k in ['id','model','score','rule','default_score','seconds','status','error']}),flush=True)
    return record

def screen_configs():
    out=[]
    for scale in ['standard','global']:
        for C in [.01,.1,1,10,100]:out.append(dict(model='logistic',C=C,scale=scale))
        for C in [.1,1,10,100]:
            for gamma in [.001,.003,.01,.03]:out.append(dict(model='svc',C=C,gamma=gamma,scale=scale))
        for k in [1,3,5,9,15]:
            for weights in ['uniform','distance']:out.append(dict(model='knn',k=k,weights=weights,scale=scale))
        for a in [1,10,100,1000]:out.append(dict(model='ridge',alpha=a,scale=scale))
        for C in [.01,.1,1,10]:out.append(dict(model='linear_svc',C=C,scale=scale))
    for typ in ['logistic','svc']:
        for C in [.1,1,10]:out.append(dict(model=typ,C=C,balanced='balanced'))
    for pc in [10,25,50,100]:
        for white in [0,.5,1]:
            for typ in ['svc','logistic','knn']:
                out.append(dict(model=typ,pca=pc,white=white,C=10,k=3))
    for typ in ['logistic','svc','knn']:
        for scale in ['standard','global','raw']:out.append(dict(model=typ,scale=scale,norm=True,C=10,k=3,gamma='scale'))
    for typ in ['lda','qda','gnb','rf','extra','hist','catboost','xgb','lightgbm','chain','powerset']:out.append(dict(model=typ))
    for typ in ['rf','extra','catboost','xgb']:
        for pc in [15,40]:out.append(dict(model=typ,pca=pc))
    for typ in ['kmeans','bisect','gmm','agglomerative','spectral','birch']:
        for n in [5,12,25,50]:out.append(dict(model='cluster',cluster=typ,clusters=n,pca=25))
    for n in [5,12,25]:out.append(dict(model='cluster',cluster='gmm',clusters=n,pca=15,cov='full'))
    for mc in [5,10,20]:out.append(dict(model='cluster',cluster='hdbscan',min_cluster=mc,pca=25))
    for q in [.25,.5,.75]:out.append(dict(model='cluster',cluster='dbscan',eps_q=q,pca=25))
    for pc in [0,25,50]:
        for k in [5,10,20,40]:
            for alpha in [.8,.95,.99]:out.append(dict(model='graph',pca=pc,k=k,alpha=alpha))
    for k in [5,15,40]:
        for alpha in [.95,.995]:out.append(dict(model='graph',k=k,alpha=alpha,hard=True))
    return out

def main():
    parser=argparse.ArgumentParser();parser.add_argument('--stage',default='screen');parser.add_argument('--configs');parser.add_argument('--seed',type=int,default=20261104);args=parser.parse_args()
    configs=json.loads(Path(args.configs).read_text()) if args.configs else screen_configs()
    save_json(ROOT/'experiments'/f'{args.stage}-configs.json',configs)
    for i,c in enumerate(configs):
        print(f'RUN {i+1}/{len(configs)} {c}',flush=True);evaluate(c,seed=args.seed,tag=args.stage)
    records=[json.loads(s) for s in (ROOT/'experiments/trials.jsonl').read_text().splitlines()]
    pd.DataFrame([{**r,'config':json.dumps(r['config'],sort_keys=True)} for r in records]).to_csv(ROOT/'experiments/trials.csv',index=False)

if __name__=='__main__':main()


Overwriting current_reproduction/experiment.py


In [18]:
%%writefile current_reproduction/experiment_v1.py
"""Reproducible, train-only Part 1 development engine. See experiments/protocol.txt."""
import os
for var in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','VECLIB_MAXIMUM_THREADS','NUMEXPR_NUM_THREADS']:
    os.environ.setdefault(var,'1')
import argparse, hashlib, json, time, traceback, warnings, re, ast
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.special import expit, softmax
from scipy.special import logit
from scipy.sparse.linalg import spsolve
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.kernel_ridge import KernelRidge
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF, ConstantKernel
from sklearn.neighbors import NeighborhoodComponentsAnalysis
from sklearn.svm import SVC, LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.neighbors import KNeighborsRegressor, NearestNeighbors
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.cluster import KMeans, BisectingKMeans, AgglomerativeClustering, DBSCAN, HDBSCAN, SpectralClustering, Birch
from sklearn.mixture import GaussianMixture
from sklearn.metrics import f1_score, pairwise_distances
from sklearn.multioutput import ClassifierChain
from sklearn.cross_decomposition import PLSRegression
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from threadpoolctl import threadpool_limits
warnings.filterwarnings('ignore',category=FutureWarning)
ROOT=Path(__file__).resolve().parent
LABELS=['Safety','Funding','Efficacy','Recruitment','Other']; EMB=[f'emb_{i}' for i in range(150)]
DATA=pd.read_csv(ROOT/'data/part1_train.csv')
LAB=np.flatnonzero(DATA[LABELS].ge(0).all(axis=1)); UNLAB=np.flatnonzero(DATA[LABELS].eq(-100).all(axis=1))
Y=DATA.iloc[LAB][LABELS].to_numpy(int)
GROUP=pd.util.hash_pandas_object(DATA[EMB].round(10),index=False).to_numpy()
assert len(LAB)+len(UNLAB)==len(DATA)
CACHE={}

def config_id(c): return hashlib.sha256(json.dumps(c,sort_keys=True).encode()).hexdigest()[:14]
def save_json(path,obj): Path(path).write_text(json.dumps(obj,indent=2,default=lambda x:x.item() if isinstance(x,np.generic) else str(x)))
def make_splits(y=Y,groups=None,seed=20261104,n_splits=5):
    if groups is None: groups=GROUP[LAB]
    unique,inv=np.unique(groups,return_inverse=True)
    gy=np.array([y[inv==i].max(axis=0) for i in range(len(unique))])
    return [(np.flatnonzero(np.isin(inv,a)),np.flatnonzero(np.isin(inv,b))) for a,b in MultilabelStratifiedKFold(n_splits=n_splits,shuffle=True,random_state=seed).split(gy,gy)]

def decisions(p,rule='t0.5'):
    p=np.clip(np.asarray(p),0,1)
    if rule=='expected_f1':
        patterns=((np.arange(32)[:,None]>>np.arange(5))&1)
        prob=np.prod(np.where(patterns[None,:,:],p[:,None,:],1-p[:,None,:]),axis=2)
        choices=patterns[1:]
        f1=2*(choices@patterns.T)/(choices.sum(1)[:,None]+patterns.sum(1)[None,:])
        return choices[np.argmax(prob@f1.T,axis=1)]
    if rule=='argmax':
        out=np.zeros_like(p,dtype=int);out[np.arange(len(p)),p.argmax(1)]=1;return out
    out=(p>=float(rule[1:])).astype(int)
    empty=out.sum(1)==0
    out[np.flatnonzero(empty),p[empty].argmax(1)]=1
    return out
RULES=['t0.25','t0.35','t0.45','t0.5','t0.55','expected_f1','argmax']
def metrics(y,p,rule='t0.5'):
    pred=decisions(p,rule)
    return {'sample_f1':float(f1_score(y,pred,average='samples',zero_division=0)), 'macro_f1':float(f1_score(y,pred,average='macro',zero_division=0)),**{f'f1_{l}':float(v) for l,v in zip(LABELS,f1_score(y,pred,average=None,zero_division=0))}}
def choose_rule(y,p):
    scores=[metrics(y,p,r)['sample_f1'] for r in RULES]
    # Prefer default threshold when differences are negligible.
    best=max(scores); idx=min([i for i,s in enumerate(scores) if s>=best-1e-12],key=lambda i:abs(i-3))
    return RULES[idx],best

def preprocess(A,B,c):
    A=np.asarray(A,float);B=np.asarray(B,float)
    mode=c.get('scale','standard')
    if mode=='standard':
        sc=StandardScaler();A=sc.fit_transform(A);B=sc.transform(B)
    elif mode=='global':
        mean=A.mean(0);std=np.sqrt(np.mean(np.var(A,axis=0)));A=(A-mean)/std;B=(B-mean)/std
    elif mode=='raw':
        # Global scale without centering preserves cosine and original relative geometry.
        std=np.sqrt(np.mean(np.var(A,axis=0)));A=A/std;B=B/std
    n=c.get('pca',0)
    if n:
        pc=PCA(n_components=min(n,A.shape[1],A.shape[0]-1),svd_solver='full');A=pc.fit_transform(A);B=pc.transform(B)
        power=c.get('white',0)
        if power:
            scale=np.maximum(pc.explained_variance_,1e-8)**(power/2);A/=scale;B/=scale
    if c.get('norm',False):A=normalize(A);B=normalize(B)
    return A,B

def chemical(frame,kind):
    from rdkit import Chem, RDLogger
    from rdkit.Chem import Descriptors, rdFingerprintGenerator
    RDLogger.DisableLog('rdApp.*')
    gen=rdFingerprintGenerator.GetMorganGenerator(radius=2,fpSize=1024)
    desc=[Descriptors.MolWt,Descriptors.MolLogP,Descriptors.TPSA,Descriptors.NumHDonors,Descriptors.NumHAcceptors,Descriptors.NumRotatableBonds,Descriptors.RingCount,Descriptors.FractionCSP3,Descriptors.HeavyAtomCount]
    out=[]
    for field in frame.drug_smiles.fillna(''):
        mols=[m for s in field.split(';') if s.strip() and (m:=Chem.MolFromSmiles(s.strip())) is not None]
        if kind=='fingerprint':v=np.max([gen.GetFingerprintAsNumPy(m) for m in mols],axis=0) if mols else np.zeros(1024)
        else:
            z=np.array([[fn(m) for fn in desc] for m in mols])
            v=np.r_[len(mols),z.mean(0),z.max(0)] if mols else np.zeros(19)
        out.append(v)
    return np.array(out,float)

def block_features(A,B,block):
    if block in ('fingerprint','descriptors'):
        a=chemical(A,block);b=chemical(B,block)
    elif block=='year':a=A[['start_year']].to_numpy();b=B[['start_year']].to_numpy()
    elif block=='missingness':
        cols=['criteria','diseases_cleaned','drugs','icdcodes','matched_drugs','drug_targets','drug_actions','drug_pathways','drug_smiles']
        a=A[cols].isna().astype(float).to_numpy();b=B[cols].isna().astype(float).to_numpy()
    elif block=='counts':
        cols=['criteria','diseases_cleaned','drugs','icdcodes','matched_drugs','drug_targets','drug_actions','drug_pathways','drug_smiles']
        def counts(D):return np.column_stack([np.log1p(D[c].fillna('').str.len()) for c in cols]+[D[c].fillna('').str.count(';')+D[c].notna() for c in cols[4:]])
        a=counts(A);b=counts(B)
    elif block.startswith('pathway_level'):
        catalog=pd.read_csv(ROOT/'data/pathway_catalog.csv');levels=dict(zip(catalog.reactome_id,catalog.pathway_level))
        if block=='pathway_levels':
            def calc(D):return np.array([[sum(levels.get(s.strip(),-1)==i for s in str(v).split(';')) for i in range(10)] for v in D.drug_pathways.fillna('')],float)
            a=np.log1p(calc(A));b=np.log1p(calc(B))
        else:
            lev=int(block[-1]); allowed=set(catalog.loc[catalog.pathway_level<=lev,'reactome_id'])
            def clean(D):return D.drug_pathways.fillna('').map(lambda x:' '.join(t.strip() for t in x.split(';') if t.strip() in allowed))
            v=TfidfVectorizer(token_pattern=r'[^\s]+',min_df=2,max_features=1000);a=v.fit_transform(clean(A)).toarray();b=v.transform(clean(B)).toarray();return a,b
    else:
        col=block.replace('_char','');char=block.endswith('_char')
        v=TfidfVectorizer(lowercase=True,analyzer='char_wb' if char else 'word',ngram_range=(3,5) if char else (1,2),min_df=2,max_df=0.99,max_features=3000,sublinear_tf=True,token_pattern=r'(?u)\b[\w-]+\b')
        a=v.fit_transform(A[col].fillna(''));b=v.transform(B[col].fillna(''))
        # Keep exact sparse-text geometry; at this dataset size dense blocks fit in memory.
        return a.toarray(),b.toarray()
    if block!='fingerprint':
        sc=StandardScaler();a=sc.fit_transform(a);b=sc.transform(b)
    a=np.nan_to_num(a);b=np.nan_to_num(b)
    norm=np.sqrt(np.mean(np.sum(a*a,axis=1)))
    return a/max(norm,1e-8),b/max(norm,1e-8)

def features(train_idx,query_frame,c):
    fitframe=DATA.iloc[train_idx]
    A,B=preprocess(fitframe[EMB],query_frame[EMB],c)
    blocks=c.get('features',[])
    if blocks:
        base_norm=np.sqrt(np.mean(np.sum(A*A,axis=1)))
        weight=c.get('feature_weight',0.3)
        for block in blocks:
            a,b=block_features(fitframe,query_frame,block)
            A=np.column_stack([A,a*base_norm*weight]);B=np.column_stack([B,b*base_norm*weight])
    return A,B

class Graph:
    """Normalized graph diffusion of all five binary labels; independent negative mass implicit."""
    def __init__(self,c):self.c=c
    def fit(self,X,Y):
        self.X=X; n=len(X);c=self.c;k=min(c.get('k',15),n-1);self.k=k
        self.nn=NearestNeighbors(n_neighbors=k+1).fit(X);dist,ix=self.nn.kneighbors(X)
        dist=dist[:,1:];ix=ix[:,1:];self.sigma=np.maximum(dist[:,-1],1e-8)
        self.band=float(np.median(dist[:,-1]))*c.get('band',1.)
        if c.get('graph','adaptive')=='adaptive':w=np.exp(-dist**2/(self.sigma[:,None]*self.sigma[ix]+1e-12))
        elif c.get('graph')=='binary':w=np.ones_like(dist)
        else:w=np.exp(-dist**2/max(self.band**2,1e-12))
        W=sparse.csr_matrix((w.ravel(),(np.repeat(np.arange(n),k),ix.ravel())),shape=(n,n))
        W=(W+W.T)*0.5
        degree=np.maximum(np.asarray(W.sum(1)).ravel(),1e-12)
        known=Y[:,0]>=0
        if c.get('hard',False):
            P=sparse.diags(1/degree)@W;u=np.flatnonzero(~known);l=np.flatnonzero(known)
            alpha=c.get('alpha',0.99)
            U=spsolve(sparse.eye(len(u),format='csc')-alpha*P[u][:,u].tocsc(),alpha*P[u][:,l]@Y[l]+(1-alpha)*Y[l].mean(0))
            self.F=Y.copy().astype(float);self.F[u]=U
        else:
            S=sparse.diags(degree**-0.5)@W@sparse.diags(degree**-0.5)
            Z=np.zeros((n,6));Z[known,:5]=Y[known];Z[known,5]=1
            sol=spsolve(sparse.eye(n,format='csc')-c.get('alpha',0.95)*S.tocsc(),Z)
            self.F=np.divide(sol[:,:5],sol[:,5,None],out=np.tile(Y[known].mean(0),(n,1)),where=sol[:,5,None]>1e-15)
        self.F=np.clip(self.F,0,1);return self
    def predict(self,Q):
        d,ix=self.nn.kneighbors(Q,n_neighbors=self.k)
        if self.c.get('graph','adaptive')=='adaptive':w=np.exp(-d*d/(np.maximum(d[:,-1,None],1e-8)*self.sigma[ix]+1e-12))
        elif self.c.get('graph')=='binary':w=np.ones_like(d)
        else:w=np.exp(-d*d/max(self.band**2,1e-12))
        exact=d[:,0]<1e-8
        p=np.einsum('ij,ijk->ik',w,self.F[ix])/np.maximum(w.sum(1)[:,None],1e-15)
        p[exact]=self.F[ix[exact,0]]
        return p

class Cluster:
    def __init__(self,c):self.c=c
    def fit(self,X,Y):
        c=self.c;typ=c['cluster'];n=c.get('clusters',12)
        if typ=='kmeans':model=KMeans(n_clusters=n,n_init=10,random_state=42)
        elif typ=='bisect':model=BisectingKMeans(n_clusters=n,n_init=3,random_state=42)
        elif typ=='gmm':model=GaussianMixture(n_components=n,covariance_type=c.get('cov','diag'),reg_covar=0.02,n_init=2,max_iter=200,random_state=42)
        elif typ=='agglomerative':model=AgglomerativeClustering(n_clusters=n,linkage=c.get('linkage','ward'))
        elif typ=='spectral':model=SpectralClustering(n_clusters=n,affinity='nearest_neighbors',n_neighbors=c.get('k',15),assign_labels='cluster_qr',random_state=42)
        elif typ=='birch':model=Birch(n_clusters=n,threshold=0.5)
        elif typ=='hdbscan':model=HDBSCAN(min_cluster_size=c.get('min_cluster',8),min_samples=c.get('min_samples',3))
        elif typ=='dbscan':
            ds=NearestNeighbors(n_neighbors=6).fit(X).kneighbors()[0][:,-1]
            model=DBSCAN(eps=float(np.quantile(ds,c.get('eps_q',0.5))),min_samples=4)
        else:raise ValueError(typ)
        z=model.fit_predict(X);self.model=model;self.X=X;self.z=z
        known=Y[:,0]>=0;self.prior=Y[known].mean(0)
        ids=sorted(set(z)-{-1});self.ids=ids;self.centers=np.array([X[z==i].mean(0) for i in ids])
        self.mapping={};smooth=c.get('smooth',0.2)
        for i in ids:
            mask=known&(z==i);self.mapping[i]=(Y[mask].sum(0)+smooth*self.prior)/(mask.sum()+smooth)
        self.fallback=KNeighborsRegressor(n_neighbors=min(5,known.sum()),weights='distance').fit(X[known],Y[known])
        self.known=known;self.Y=Y
        if typ=='gmm':
            R=model.predict_proba(X[known]);self.softmap=(R.T@Y[known]+smooth*self.prior)/(R.sum(0)[:,None]+smooth)
        return self
    def predict(self,Q):
        typ=self.c['cluster']
        if typ=='gmm':return self.model.predict_proba(Q)@self.softmap
        if hasattr(self.model,'predict'):z=self.model.predict(Q)
        else:
            # Inductive extension for algorithms without predict: nearest training member.
            _,ix=NearestNeighbors(n_neighbors=1).fit(self.X).kneighbors(Q);z=self.z[ix[:,0]]
        p=self.fallback.predict(Q)
        for i in self.ids:
            if np.sum(self.known&(self.z==i))>0:p[z==i]=self.mapping[i]
        return p

def fit_supervised(X,Y,Q,c,weights=None):
    typ=c['model'];param=c.get('C',1.);out=[]
    if typ=='knn':return KNeighborsRegressor(n_neighbors=min(c.get('k',5),len(X)),weights=c.get('weights','distance'),metric=c.get('metric','euclidean')).fit(X,Y).predict(Q)
    if typ=='ridge':return np.clip(Ridge(alpha=c.get('alpha',10)).fit(X,Y,sample_weight=weights).predict(Q),0,1)
    if typ=='krr':return np.clip(KernelRidge(alpha=c.get('alpha',.1),kernel='rbf',gamma=c.get('gamma',.003)).fit(X,Y,sample_weight=weights).predict(Q),0,1)
    if typ=='pls':return np.clip(PLSRegression(n_components=c.get('components',5)).fit(X,Y).predict(Q),0,1)
    if typ=='nca':
        nca=NeighborhoodComponentsAnalysis(n_components=c.get('components',5),max_iter=100,random_state=42)
        xx=nca.fit_transform(X,Y@2**np.arange(5));qq=nca.transform(Q)
        return KNeighborsRegressor(n_neighbors=c.get('k',5),weights='distance').fit(xx,Y).predict(qq)
    if typ=='chain':
        model=ClassifierChain(LogisticRegression(C=param,max_iter=1000),order='random',random_state=c.get('seed',42),cv=3,chain_method='predict_proba')
        return model.fit(X,Y).predict_proba(Q)
    if typ=='powerset':
        # Preserve multilabel patterns instead of imposing mutual exclusivity.
        code=Y@2**np.arange(5)
        if c.get('kernel','rbf')=='linear':model=LogisticRegression(C=param,max_iter=1000)
        else:model=SVC(C=param,gamma=c.get('gamma','scale'),probability=True,random_state=42)
        model.fit(X,code,sample_weight=weights);patterns=((model.classes_[:,None]>>np.arange(5))&1)
        return model.predict_proba(Q)@patterns
    for j in range(Y.shape[1]):
        y=Y[:,j]
        if len(np.unique(y))<2:out.append(np.full(len(Q),y[0]));continue
        if typ=='logistic':m=LogisticRegression(C=param,max_iter=2000,class_weight=c.get('balanced'),solver='lbfgs')
        elif typ=='svc':m=SVC(C=param,kernel=c.get('kernel','rbf'),gamma=c.get('gamma','scale'),degree=c.get('degree',2),class_weight=c.get('balanced'),probability=True,random_state=42)
        elif typ=='linear_svc':m=CalibratedClassifierCV(LinearSVC(C=param,class_weight=c.get('balanced'),max_iter=5000),cv=3)
        elif typ=='lda':m=LinearDiscriminantAnalysis(solver='lsqr',shrinkage=c.get('shrinkage','auto'))
        elif typ=='qda':m=QuadraticDiscriminantAnalysis(reg_param=c.get('reg',0.9))
        elif typ=='gnb':m=GaussianNB(var_smoothing=c.get('smooth',1e-2))
        elif typ=='gp':m=GaussianProcessClassifier(kernel=ConstantKernel(c.get('amplitude',1))*RBF(c.get('length',10)),optimizer=None,random_state=42)
        elif typ in ('rf','extra'):
            cls=RandomForestClassifier if typ=='rf' else ExtraTreesClassifier
            m=cls(n_estimators=c.get('trees',200),max_depth=c.get('depth'),min_samples_leaf=c.get('leaf',2),max_features=c.get('max_features','sqrt'),class_weight=c.get('balanced'),random_state=42,n_jobs=1)
        elif typ=='hist':m=HistGradientBoostingClassifier(max_iter=150,max_leaf_nodes=7,l2_regularization=5,random_state=42)
        elif typ=='catboost':
            from catboost import CatBoostClassifier
            m=CatBoostClassifier(iterations=c.get('trees',250),depth=c.get('depth',3),learning_rate=0.04,l2_leaf_reg=5,verbose=False,thread_count=1,allow_writing_files=False,random_seed=42)
        elif typ=='xgb':
            from xgboost import XGBClassifier
            m=XGBClassifier(n_estimators=c.get('trees',150),max_depth=c.get('depth',2),learning_rate=.04,reg_lambda=5,min_child_weight=2,subsample=.9,colsample_bytree=.8,n_jobs=1,random_state=42)
        elif typ=='lightgbm':
            from lightgbm import LGBMClassifier
            m=LGBMClassifier(n_estimators=150,num_leaves=7,min_child_samples=10,learning_rate=.04,reg_lambda=5,verbosity=-1,n_jobs=1,random_state=42)
        else:raise ValueError(typ)
        if weights is not None and typ not in ('lda','qda'):m.fit(X,y,sample_weight=weights)
        else:m.fit(X,y)
        out.append(m.predict_proba(Q)[:,1])
    return np.array(out).T

def predict(train_pos,query_frame,c,excluded_groups=None,return_unlabeled=False):
    if c['model']=='ensemble':
        weights=np.asarray(c.get('weights',np.ones(len(c['members']))),float);weights/=weights.sum()
        outputs=[predict(train_pos,query_frame,m,excluded_groups,return_unlabeled) for m in c['members']]
        if return_unlabeled:
            return sum(w*x[0] for w,x in zip(weights,outputs)),sum(w*x[1] for w,x in zip(weights,outputs)),outputs[0][2]
        return sum(w*x for w,x in zip(weights,outputs))
    # Fit unsupervised transforms only on fold-training labels and legitimate masked rows.
    actual=LAB[np.asarray(train_pos)]
    u=UNLAB if excluded_groups is None else UNLAB[~np.isin(GROUP[UNLAB],excluded_groups)]
    idx=np.r_[actual,u];n=len(actual)
    A,B=features(idx,query_frame,c);y=Y[train_pos];ys=np.full((len(A),5),-1);ys[:n]=y
    typ=c['model']
    if typ=='cluster_features':
        # Unsupervised prototype dictionary from genuine training + masked rows only.
        km=KMeans(n_clusters=c.get('clusters',40),n_init=5,random_state=42).fit(A)
        da=km.transform(A);db=km.transform(B)
        band=np.median(np.min(da,axis=1))*c.get('band',1.)
        H=np.exp(-da**2/max(2*band**2,1e-8));J=np.exp(-db**2/max(2*band**2,1e-8))
        sc=StandardScaler();H=sc.fit_transform(H);J=sc.transform(J)
        mix=c.get('original_weight',.3)
        H=np.column_stack([H,A*mix]);J=np.column_stack([J,B*mix])
        out=fit_supervised(H[:n],y,np.r_[J,H[n:]],{**c,'model':'logistic'})
        p=out[:len(B)];pu=out[len(B):]
    elif typ=='metric_graph':
        # Label-informed metric fitted only to fold-training labels, followed by unsupervised graph inference.
        base={**c,'model':c.get('metric_model','logistic')}
        scores=fit_supervised(A[:n],y,np.r_[A,B],base)
        scores=logit(np.clip(scores,.005,.995))
        sc=StandardScaler();s=sc.fit_transform(scores[:len(A)]);q=sc.transform(scores[len(A):])
        residual=c.get('residual',.3)
        norm=np.sqrt(np.mean(np.sum(A*A,axis=1)))
        U=np.column_stack([s,A*residual/max(norm,1e-8)])
        V=np.column_stack([q,B*residual/max(norm,1e-8)])
        model=Graph(c).fit(U,ys);p=model.predict(V);pu=model.F[n:]
    elif typ=='graph':
        model=Graph(c).fit(A,ys);p=model.predict(B);pu=model.F[n:]
    elif typ=='cluster':
        model=Cluster(c).fit(A,ys);p=model.predict(B);pu=model.predict(A[n:])
    elif typ=='pseudo':
        teacher=c['teacher'];T,U=preprocess(DATA.iloc[idx][EMB],query_frame[EMB],teacher)
        if teacher['model']=='graph':m=Graph(teacher).fit(T,ys);pt=m.F[n:]
        else:m=Cluster(teacher).fit(T,ys);pt=m.predict(T[n:])
        student={**c,'model':c.get('student','svc')}
        conf=c.get('confidence',.9);w=c.get('pseudo_weight',.2)
        base_out=None
        if c.get('consensus',False):
            base_out=fit_supervised(A[:n],y,np.r_[B,A[n:]],student)
            base_u=base_out[len(B):]
            agreement=(pt>=.5)==(base_u>=.5)
            # Confidence must be supported by both independently constructed views.
            pt=np.where(agreement,(pt+base_u)*.5,.5)
        yp=(pt>=.5).astype(int)
        rows=np.min(np.maximum(pt,1-pt),axis=1)>=conf
        if c.get('labelwise',False):
            p=np.zeros((len(B),5));pu=np.zeros((len(u),5))
            for j in range(5):
                sel=np.maximum(pt[:,j],1-pt[:,j])>=conf
                xx=np.r_[A[:n],A[n:][sel]];yy=np.r_[y[:,j],yp[sel,j]][:,None];ww=np.r_[np.ones(n),np.full(sel.sum(),w)]
                out=fit_supervised(xx,yy,np.r_[B,A[n:]],student,ww);p[:,j]=out[:len(B),0];pu[:,j]=out[len(B):,0]
        else:
            xx=np.r_[A[:n],A[n:][rows]];yy=np.r_[y,yp[rows]];ww=np.r_[np.ones(n),np.full(rows.sum(),w)]
            out=fit_supervised(xx,yy,np.r_[B,A[n:]],student,ww);p=out[:len(B)];pu=out[len(B):]
        if c.get('base_mix',0):
            if base_out is None:base_out=fit_supervised(A[:n],y,np.r_[B,A[n:]],student)
            mix=c['base_mix'];p=(1-mix)*p+mix*base_out[:len(B)];pu=(1-mix)*pu+mix*base_out[len(B):]
    else:
        out=fit_supervised(A[:n],y,np.r_[B,A[n:]] if return_unlabeled else B,c)
        p=out[:len(B)];pu=out[len(B):] if return_unlabeled else None
    if return_unlabeled:return p,pu,u
    return np.clip(p,0,1)

def evaluate(c,splits=None,seed=20261104,tag='screen',force=False):
    if splits is None:splits=make_splits(seed=seed)
    cid=config_id(c);path=ROOT/'experiments'/f'{tag}-{cid}-{seed}.npz'
    if path.exists() and not force:return dict(np.load(path,allow_pickle=True)['record'].item())
    started=time.time();p=np.full(Y.shape,np.nan);foldids=np.full(len(Y),-1)
    try:
        with threadpool_limits(limits=1):
            for fold,(a,b) in enumerate(splits):
                p[b]=predict(a,DATA.iloc[LAB[b]],c,excluded_groups=GROUP[LAB[b]])
                foldids[b]=fold
        use=foldids>=0;rule,score=choose_rule(Y[use],p[use]);base=metrics(Y[use],p[use]);tuned=metrics(Y[use],p[use],rule)
        fs=[metrics(Y[b],p[b],rule)['sample_f1'] for a,b in splits]
        record={'id':cid,'tag':tag,'seed':seed,'model':c['model'],'config':c,'n_scored':int(use.sum()),'score':score,'rule':rule,'default_score':base['sample_f1'],'fold_scores':fs,'fold_std':float(np.std(fs)),**tuned,'seconds':round(time.time()-started,3),'status':'ok'}
        np.savez_compressed(path,p=p,fold=foldids,record=record)
    except Exception as e:
        record={'id':cid,'tag':tag,'seed':seed,'config':c,'status':'error','error':str(e),'traceback':traceback.format_exc(),'seconds':round(time.time()-started,3)}
    with open(ROOT/'experiments/trials.jsonl','a') as f:f.write(json.dumps(record)+'\n')
    print(json.dumps({k:record.get(k) for k in ['id','model','score','rule','default_score','seconds','status','error']}),flush=True)
    return record

def screen_configs():
    out=[]
    for scale in ['standard','global']:
        for C in [.01,.1,1,10,100]:out.append(dict(model='logistic',C=C,scale=scale))
        for C in [.1,1,10,100]:
            for gamma in [.001,.003,.01,.03]:out.append(dict(model='svc',C=C,gamma=gamma,scale=scale))
        for k in [1,3,5,9,15]:
            for weights in ['uniform','distance']:out.append(dict(model='knn',k=k,weights=weights,scale=scale))
        for a in [1,10,100,1000]:out.append(dict(model='ridge',alpha=a,scale=scale))
        for C in [.01,.1,1,10]:out.append(dict(model='linear_svc',C=C,scale=scale))
    for typ in ['logistic','svc']:
        for C in [.1,1,10]:out.append(dict(model=typ,C=C,balanced='balanced'))
    for pc in [10,25,50,100]:
        for white in [0,.5,1]:
            for typ in ['svc','logistic','knn']:
                out.append(dict(model=typ,pca=pc,white=white,C=10,k=3))
    for typ in ['logistic','svc','knn']:
        for scale in ['standard','global','raw']:out.append(dict(model=typ,scale=scale,norm=True,C=10,k=3,gamma='scale'))
    for typ in ['lda','qda','gnb','rf','extra','hist','catboost','xgb','lightgbm','chain','powerset']:out.append(dict(model=typ))
    for typ in ['rf','extra','catboost','xgb']:
        for pc in [15,40]:out.append(dict(model=typ,pca=pc))
    for typ in ['kmeans','bisect','gmm','agglomerative','spectral','birch']:
        for n in [5,12,25,50]:out.append(dict(model='cluster',cluster=typ,clusters=n,pca=25))
    for n in [5,12,25]:out.append(dict(model='cluster',cluster='gmm',clusters=n,pca=15,cov='full'))
    for mc in [5,10,20]:out.append(dict(model='cluster',cluster='hdbscan',min_cluster=mc,pca=25))
    for q in [.25,.5,.75]:out.append(dict(model='cluster',cluster='dbscan',eps_q=q,pca=25))
    for pc in [0,25,50]:
        for k in [5,10,20,40]:
            for alpha in [.8,.95,.99]:out.append(dict(model='graph',pca=pc,k=k,alpha=alpha))
    for k in [5,15,40]:
        for alpha in [.95,.995]:out.append(dict(model='graph',k=k,alpha=alpha,hard=True))
    return out

def main():
    parser=argparse.ArgumentParser();parser.add_argument('--stage',default='screen');parser.add_argument('--configs');parser.add_argument('--seed',type=int,default=20261104);args=parser.parse_args()
    configs=json.loads(Path(args.configs).read_text()) if args.configs else screen_configs()
    save_json(ROOT/'experiments'/f'{args.stage}-configs.json',configs)
    for i,c in enumerate(configs):
        print(f'RUN {i+1}/{len(configs)} {c}',flush=True);evaluate(c,seed=args.seed,tag=args.stage)
    records=[json.loads(s) for s in (ROOT/'experiments/trials.jsonl').read_text().splitlines()]
    pd.DataFrame([{**r,'config':json.dumps(r['config'],sort_keys=True)} for r in records]).to_csv(ROOT/'experiments/trials.csv',index=False)

if __name__=='__main__':main()


Overwriting current_reproduction/experiment_v1.py


In [19]:
%%writefile current_reproduction/extend_benchmark_leaders.py
"""Freeze the comparison cohort before running the supplementary pooled CV."""
import json
from pathlib import Path
import benchmark_cv as b

def main():
    b.protocol();rs=b.registry();seen={b.digest(dict(config=r['config'],rule=r['rule'])) for r in rs}
    selection=b.OUT/'leader_cohort.json'
    if selection.exists():
        print('Leader cohort already frozen:',len(rs));return
    added=[]
    names={('phase10','structured_cluster'):'Structured-label SVM',('phase11','random_subspace'):'Unlabeled-subsample SVM ensemble',
        ('phase12',None):'Low-density SVM',('phase13',None):'Cluster SVM at threshold 0.60',('phase18','time_weights'):'Year-weighted SVM',
        ('phase21b','training_selected_reason_hybrid'):'Kernel-committee reason hybrid',('phase21c','classical_tri_majority'):'Classical tri-training',
        ('phase24','phase24_co_training'):'Metadata co-training hybrid',('phase25','phase25_local_ssl'):'Local SVM specialist hybrid',
        ('phase25b','phase25_control_label_tuned_ssl'):'Other SVM override hybrid',('phase25b','phase25_control_reliable_ssl'):'Uniform pseudo-weight hybrid',
        ('phase45',None):'Earlier low-density SVM',('phase6','pca'):'PCA-125 low-density SVM',
        ('phase6','auxiliary_features'):'Drug-target augmented SVM',('phase9','embedding_geometry'):'Embedding-kernel SVM'}
    for path in sorted((b.ROOT/'experiments').glob('phase*-validation-results.json')):
        obj=json.loads(path.read_text());items=obj if isinstance(obj,list) else [obj]
        for r in items:
            v=r.get('validation',r);score=v.get('sample_f1',r.get('validation_sample_f1',0))
            if not score or score<.9 or not r.get('config') or not r.get('rule'):continue
            if b.transductive(r['config']):continue
            key=b.digest(dict(config=r['config'],rule=r['rule']))
            if key in seen:continue
            phase=path.stem.split('-')[0];name=names.get((phase,r.get('name')),phase+': '+r.get('name',r['config']['model']))
            item=b.entry({**r,'validation':v},name,str(path.relative_to(b.ROOT)))
            rs.append(item);added.append(item);seen.add(key)
    b.save(selection,dict(frozen_at=b.now(),criterion='Existing five common-benchmark recipes plus every distinct inductive recipe with recorded validation sample-F1 >= 0.900. Deduplicate exact config and rule; no tuning.',
        model_count=len(rs),added_names=[r['name'] for r in added],recipes_sha256=b.digest([dict(config=r['config'],rule=r['rule']) for r in rs])))
    b.save(b.OUT/'registry.json',rs);print('Frozen comparison cohort:',len(rs),'recipes;',len(added),'added.')

if __name__=='__main__':main()


Writing current_reproduction/extend_benchmark_leaders.py


In [20]:
%%writefile current_reproduction/finalize.py
"""Fit frozen candidates, evaluate validation once, and generate Kaggle artifacts."""
import experiment_v1 as e
import pandas as pd,numpy as np,json,datetime,hashlib
from pathlib import Path
from threadpoolctl import threadpool_limits
frozen=json.loads(Path('experiments/frozen-shortlist.json').read_text())
assert hashlib.sha256(Path('experiment_v1.py').read_bytes()).hexdigest()==frozen['engine_sha256']
# Read only predictor columns until all candidate predictions are committed to disk.
val=pd.read_csv('data/part1_val.csv',usecols=lambda c:c not in e.LABELS)
test=pd.read_csv('data/part1_test.csv');unlab=e.DATA.iloc[e.UNLAB].drop(columns=e.LABELS)
query=pd.concat([val,test,unlab],ignore_index=True)
seeds=[20261104,20261105,20261106];member_predictions={}
with threadpool_limits(limits=1):
 for member in frozen['members']:
  cid=e.config_id(member);out=Path('predictions')/f'locked-member-{cid}.npz'
  if out.exists():
   z=np.load(out);assert np.array_equal(z['ids'],query.ID.to_numpy(str));member_predictions[cid]=z['p'];continue
  preds=[]
  for seed in seeds:
   for fold,(a,b) in enumerate(e.make_splits(seed=seed)):
    print('FIT',cid,seed,fold,flush=True)
    preds.append(e.predict(a,query,member))
  p=np.mean(preds,axis=0);np.savez_compressed(out,p=p,ids=query.ID.to_numpy(str),seed_predictions=np.array(preds));member_predictions[cid]=p
 for candidate in frozen['shortlist']:
  c=candidate['config'];weights=np.array(c['weights'],float);weights/=weights.sum()
  p=sum(w*member_predictions[e.config_id(m)] for w,m in zip(weights,c['members']))
  np.savez_compressed('predictions/locked-'+candidate['name']+'.npz',p=p,ids=query.ID.to_numpy(str))
# Now and only now use validation labels for the frozen comparison.
result_path=Path('experiments/validation-once.json')
if result_path.exists():raise SystemExit('Validation comparison already exists. Refusing to run another comparison.')
y=pd.read_csv('data/part1_val.csv',usecols=e.LABELS)[e.LABELS].to_numpy(int)
records=[]
for candidate in frozen['shortlist']:
 p=np.load('predictions/locked-'+candidate['name']+'.npz')['p'][:len(val)]
 r={**candidate,**e.metrics(y,p,candidate['rule'])};records.append(r);print('VALIDATION',candidate['name'],r['sample_f1'],r['macro_f1'],flush=True)
maximum=max(r['sample_f1'] for r in records)
priority={'fingerprint_ssl':0,'embedding_ssl':1,'balanced_ensemble':2}
winner=min([r for r in records if r['sample_f1']>=maximum-.005],key=lambda r:priority[r['name']])
p=np.load('predictions/locked-'+winner['name']+'.npz')['p'];hard=e.decisions(p,winner['rule'])
# Cluster bootstrap by exact embedding group, keeping duplicate reasons together.
vg=pd.util.hash_pandas_object(val[e.EMB].round(10),index=False).to_numpy();groups=np.unique(vg)
row_f1=2*(y*hard[:len(val)]).sum(1)/(y.sum(1)+hard[:len(val)].sum(1))
rng=np.random.default_rng(8102026);boot=[]
for _ in range(10000):
 sampled=rng.choice(groups,len(groups),replace=True);idx=np.concatenate([np.flatnonzero(vg==g) for g in sampled]);boot.append(float(row_f1[idx].mean()))
ci=np.quantile(boot,[.025,.975]).tolist()
result=dict(evaluated_at=datetime.datetime.now(datetime.timezone.utc).isoformat(),validation_rows=len(val),candidates=records,selected=winner['name'],selected_sample_f1=winner['sample_f1'],selected_macro_f1=winner['macro_f1'],bootstrap_95_interval=ci,limitation='Single, frozen 3-candidate selection on 78 validation trials; the selected validation score has selection optimism. This is not a Kaggle score. No validation refit or error-specific tuning.',user_kaggle_baseline=.89423,user_kaggle_leader=.92735)
e.save_json(result_path,result)
# Save every frozen submission for reproducibility; main filename always points to selected candidate.
for candidate in frozen['shortlist']:
 pp=np.load('predictions/locked-'+candidate['name']+'.npz')['p'];hh=e.decisions(pp,candidate['rule'])
 sub=pd.DataFrame({'ID':test.ID,'Predicted':[' '.join(l for l,x in zip(e.LABELS,row) if x) for row in hh[len(val):len(val)+len(test)]]})
 sub.to_csv('predictions/submission_'+candidate['name']+'.csv',index=False)
 if candidate['name']==winner['name']:sub.to_csv('predictions/part1_submission.csv',index=False)
for name,frame,sl in [('validation',val,slice(0,len(val))),('test',test,slice(len(val),len(val)+len(test))),('unlabeled_train',unlab,slice(len(val)+len(test),None))]:
 out=pd.DataFrame(p[sl],columns=['p_'+l for l in e.LABELS]);out.insert(0,'ID',frame.ID.to_numpy());out.to_csv('predictions/'+name+'_probabilities.csv',index=False)
aug=e.DATA.copy();aug.loc[e.UNLAB,e.LABELS]=hard[len(val)+len(test):];aug['label_source']='original';aug.loc[e.UNLAB,'label_source']='pseudo_label_'+winner['name'];aug.to_csv('predictions/part1_train_augmented.csv',index=False)
# Validation outputs for report only; do not feed these back into model selection.
v=pd.DataFrame(hard[:len(val)],columns=['pred_'+l for l in e.LABELS]);v.insert(0,'ID',val.ID.to_numpy());v['sample_f1']=row_f1
for j,l in enumerate(e.LABELS):v['true_'+l]=y[:,j]
v.to_csv('predictions/validation_audit.csv',index=False)
sample=pd.read_csv('data/part1_sample_submission.csv');submission=pd.read_csv('predictions/part1_submission.csv')
assert list(submission.columns)==['ID','Predicted'] and len(submission)==312
assert submission.ID.tolist()==sample.ID.tolist() and submission.ID.is_unique
assert submission.Predicted.notna().all() and submission.Predicted.map(lambda s:bool(s) and set(s.split())<=set(e.LABELS)).all()
assert aug[e.LABELS].isin([0,1]).all().all() and np.array_equal(aug.loc[e.LAB,e.LABELS],e.Y)
e.save_json('experiments/final-checks.json',dict(submission_rows=312,submission_id_order_matches_sample=True,labels_valid=True,augmented_rows=len(aug),preserved_ground_truth_rows=len(e.LAB),generated_label_rows=len(e.UNLAB),submission_sha256=hashlib.sha256(Path('predictions/part1_submission.csv').read_bytes()).hexdigest()))
print('SELECTED',winner['name'],'SAMPLE F1',winner['sample_f1'],'95% interval',ci,flush=True)


Overwriting current_reproduction/finalize.py


In [21]:
%%writefile current_reproduction/finish_current_package.py
"""Execute current report/reproduction notebooks and refresh the portable archive."""
from pathlib import Path
import os,json,shutil,zipfile,hashlib
import nbformat
from nbclient import NotebookClient
from nbconvert import HTMLExporter
ROOT=Path(__file__).resolve().parent;mid=ROOT/'midterm'
os.environ['JUPYTER_PATH']=str(ROOT/'.venv/share/jupyter')
notebooks=['03_current_report','04_reproduce_current']
if (mid/'05_tri_training_and_shift.ipynb').exists():notebooks.append('05_tri_training_and_shift')
if (mid/'06_stability_and_specialists.ipynb').exists():notebooks.append('06_stability_and_specialists')
if (mid/'07_cotraining_and_distances.ipynb').exists():notebooks.append('07_cotraining_and_distances')
if (mid/'08_svm_refinements.ipynb').exists():notebooks.append('08_svm_refinements')
if (mid/'09_shift_graph_and_subgroups.ipynb').exists():notebooks.append('09_shift_graph_and_subgroups')
if (mid/'10_common_cv_benchmark.ipynb').exists():notebooks.append('10_common_cv_benchmark')
if (mid/'11_pooled_cv_comparison.ipynb').exists():notebooks.append('11_pooled_cv_comparison')
for name in notebooks:
 p=mid/(name+'.ipynb');nb=nbformat.read(p,as_version=4)
 NotebookClient(nb,timeout=600,kernel_name='aml-midterm',resources={'metadata':{'path':str(mid)}}).execute()
 nbformat.write(nb,p);html,_=HTMLExporter().from_notebook_node(nb);(mid/(name+'.html')).write_text(html)
 print('Executed',name,flush=True)
manifest=json.loads((ROOT/'predictions/current_best/manifest.json').read_text());summary=json.loads((ROOT/'experiments/current-summary.json').read_text())
checks=dict(current_submission_rows=312,gold_rows_preserved=157,missing_rows_predicted=1014,raw_hashes_verified=True,notebooks_executed=notebooks,exact_reproduction_sha256=manifest['submission_sha256'],validation_sample_f1=manifest['validation']['sample_f1'],validation_macro_f1=manifest['validation']['macro_f1'],validation_candidate_evaluations=summary['validation_candidate_evaluations'],target_met=manifest['target_met'],kaggle_score=None,integrity=json.loads((ROOT/'experiments/final-integrity-checks.json').read_text()))
if (ROOT/'experiments/phase21-integrity-checks.json').exists():checks['tri_training_integrity']=json.loads((ROOT/'experiments/phase21-integrity-checks.json').read_text())
if (ROOT/'experiments/phase23-integrity-checks.json').exists():checks['nested_specialist_integrity']=json.loads((ROOT/'experiments/phase23-integrity-checks.json').read_text())
if (ROOT/'experiments/phase24-integrity-checks.json').exists():checks['cotraining_metric_integrity']=json.loads((ROOT/'experiments/phase24-integrity-checks.json').read_text())
if (ROOT/'experiments/phase25-integrity-checks.json').exists():checks['svm_refinement_integrity']=json.loads((ROOT/'experiments/phase25-integrity-checks.json').read_text())
if (ROOT/'experiments/phase26-integrity-checks.json').exists():checks['shift_graph_subgroup_integrity']=json.loads((ROOT/'experiments/phase26-integrity-checks.json').read_text())
if (ROOT/'experiments/benchmark_v1/summary.json').exists():checks['common_cv_benchmark']=json.loads((ROOT/'experiments/benchmark_v1/summary.json').read_text())
if (ROOT/'experiments/pooled_benchmark_v1/conclusions.json').exists():checks['pooled_cv_comparison']=json.loads((ROOT/'experiments/pooled_benchmark_v1/conclusions.json').read_text())
(ROOT/'experiments/current-deliverable-checks.json').write_text(json.dumps(checks,indent=2));shutil.copy2(ROOT/'experiments/current-deliverable-checks.json',mid/'experiments/current-deliverable-checks.json')
# Notebook-generated figure is also available at the project top level.
(ROOT/'figures').mkdir(exist_ok=True)
for p in (mid/'figures').glob('current_*.png'):shutil.copy2(p,ROOT/'figures'/p.name)
archive=ROOT/'AMLMidterm_Part1.zip';temporary=ROOT/'AMLMidterm_Part1.building.zip'
with zipfile.ZipFile(temporary,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
 for p in sorted(mid.rglob('*')):
  if not p.is_file() or any(part in ['__pycache__','.venv','reproduction','current_reproduction'] for part in p.relative_to(mid).parts):continue
  z.write(p,Path('midterm')/p.relative_to(mid))
with zipfile.ZipFile(temporary) as z:
 assert z.testzip() is None
 assert hashlib.sha256(z.read('midterm/predictions/current_best/part1_submission.csv')).hexdigest()==manifest['submission_sha256']
 assert 'midterm/04_reproduce_current.ipynb' in z.namelist()
os.replace(temporary,archive)
print(json.dumps({'archive':str(archive),'bytes':archive.stat().st_size,'checks':checks},indent=2))


Overwriting current_reproduction/finish_current_package.py


In [22]:
%%writefile current_reproduction/freeze_shortlist.py
import json, numpy as np, hashlib, datetime
from pathlib import Path
import experiment as e
path=Path('experiments/frozen-shortlist.json')
if path.exists():raise SystemExit('Shortlist already frozen; refusing overwrite.')
fp=json.loads(Path('experiments/ssl_features-ranking.json').read_text())[0]['config']
combo=json.loads(Path('experiments/combinations-ranking.json').read_text())[0]['config']
target=dict(model='svc',C=10,gamma=.001,scale='standard',features=['drug_targets'],feature_weight=1.)
plain=next(x['config'] for x in json.loads(Path('experiments/robust-ranking.json').read_text()) if x['config']['model']=='pseudo')
models=[fp,combo,target,plain];cache={}
for f in Path('experiments').glob('*.npz'):
 z=np.load(f,allow_pickle=True)
 if 'record' not in z:continue
 r=z['record'].item()
 if 'seed' in r:cache[(r['id'],r['seed'])]=z['p']
shortlist=[]
for name,weights in [('fingerprint_ssl',[1,0,0,0]),('balanced_ensemble',[.5,.25,.25,0]),('embedding_ssl',[0,0,0,1])]:
 ps=[]
 for seed in [20261104,20261105,20261106]:ps.append(sum(w*cache[(e.config_id(m),seed)] for w,m in zip(weights,models) if w))
 p=np.mean(ps,axis=0);rule,score=e.choose_rule(e.Y,p)
 c=dict(model='ensemble',members=[m for m,w in zip(models,weights) if w],weights=[w for w in weights if w])
 rec=dict(name=name,config=c,rule=rule,repeated_oof_f1=score,mean_split_f1=float(np.mean([e.metrics(e.Y,z,rule)['sample_f1'] for z in ps])),prediction_strategy='Mean of 15 models: 5 grouped folds x seeds 20261104,20261105,20261106. No validation labels used in fits.')
 shortlist.append(rec);np.savez_compressed('experiments/frozen-oof-'+name+'.npz',p=p,y=e.Y);print(rec,flush=True)
obj=dict(frozen_at=datetime.datetime.now(datetime.timezone.utc).isoformat(),members=models,shortlist=shortlist,validation_selection='Choose the highest sample-F1 among these three frozen models; if tied within 0.005, choose fingerprint_ssl, then embedding_ssl, then balanced_ensemble. No subsequent validation-error-driven tuning.',training_data='Official training file only; validation remains held out even for final submission.',engine_sha256=hashlib.sha256(Path('experiment.py').read_bytes()).hexdigest())
e.save_json(path,obj)


Overwriting current_reproduction/freeze_shortlist.py


In [23]:
%%writefile current_reproduction/fullsize_cv.py
"""Nearly-full-training-size audit using leave-one-embedding-group-out CV. No validation access."""
import experiment as e,numpy as np,json
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from threadpoolctl import threadpool_limits
frozen=json.loads(Path('experiments/frozen-shortlist.json').read_text());members=frozen['members']
# The selection candidates are unchanged in structure; this audit addresses training-size and decoding.
unique=np.unique(e.GROUP[e.LAB]);allpos=np.arange(len(e.Y))
def run_member(c):
 cid=e.config_id(c);out=Path('experiments')/('fullsize-'+cid+'.npz')
 if out.exists():return
 p=np.zeros_like(e.Y,float)
 for gi,g in enumerate(unique):
  b=np.flatnonzero(e.GROUP[e.LAB]==g);a=allpos[~np.isin(allpos,b)]
  p[b]=e.predict(a,e.DATA.iloc[e.LAB[b]],c,excluded_groups=[g])
  if gi%25==0:print('LEAVE-GROUP-OUT',cid,gi,len(unique),flush=True)
 rule,score=e.choose_rule(e.Y,p)
 e.save_json('experiments/fullsize-'+cid+'.json',dict(config=c,rule=rule,score=score,n_groups=len(unique),mean_training_size=float(np.mean([len(e.Y)-np.sum(e.GROUP[e.LAB]==g) for g in unique]))))
 np.savez_compressed(out,p=p);print('DONE',cid,score,rule,flush=True)
with threadpool_limits(limits=1):
 with ThreadPoolExecutor(max_workers=2) as pool:list(pool.map(run_member,members))
results=[]
for candidate in frozen['shortlist']:
 c=candidate['config'];w=np.array(c['weights'],float);w/=w.sum()
 p=sum(wi*np.load('experiments/fullsize-'+e.config_id(m)+'.npz')['p'] for wi,m in zip(w,c['members']))
 rule,score=e.choose_rule(e.Y,p);r=dict(name=candidate['name'],config=c,rule=rule,fullsize_oof_f1=score,metrics=e.metrics(e.Y,p,rule));results.append(r)
 np.savez_compressed('experiments/fullsize-oof-'+candidate['name']+'.npz',p=p,y=e.Y)
e.save_json('experiments/fullsize-ranking.json',sorted(results,key=lambda r:r['fullsize_oof_f1'],reverse=True))
print('RESULTS',results,flush=True)


Overwriting current_reproduction/fullsize_cv.py


In [24]:
%%writefile current_reproduction/make_candidate.py
"""Regenerate the final train-only full-fit candidate. Never reads validation labels."""
from pathlib import Path
import json,hashlib,datetime
import numpy as np,pandas as pd
import experiment as e
from threadpoolctl import threadpool_limits
rank=json.loads(Path('experiments/fullsize-ranking.json').read_text())
selected=rank[0]
test=pd.read_csv('data/part1_test.csv');u=e.DATA.iloc[e.UNLAB].drop(columns=e.LABELS)
query=pd.concat([test,u],ignore_index=True)
with threadpool_limits(limits=1):p=e.predict(np.arange(len(e.Y)),query,selected['config'])
h=e.decisions(p,selected['rule'])
sub=pd.DataFrame({'ID':test.ID,'Predicted':[' '.join(l for l,v in zip(e.LABELS,row) if v) for row in h[:len(test)]]})
path=Path('predictions/part1_fullfit_candidate.csv');sub.to_csv(path,index=False)
probs=pd.DataFrame(p[:len(test)],columns=['p_'+l for l in e.LABELS]);probs.insert(0,'ID',test.ID);probs.to_csv('predictions/fullfit_test_probabilities.csv',index=False)
aug=e.DATA.copy();aug.loc[e.UNLAB,e.LABELS]=h[len(test):];aug['label_source']='original';aug.loc[e.UNLAB,'label_source']='fullfit_pseudo_label';aug.to_csv('predictions/part1_train_augmented_fullfit.csv',index=False)
up=pd.DataFrame(p[len(test):],columns=['p_'+l for l in e.LABELS]);up.insert(0,'ID',u.ID.to_numpy());up.to_csv('predictions/fullfit_unlabeled_probabilities.csv',index=False)
assert sub.ID.tolist()==pd.read_csv('data/part1_sample_submission.csv').ID.tolist()
assert sub.Predicted.map(lambda s:bool(s) and set(s.split())<=set(e.LABELS)).all()
assert aug[e.LABELS].isin([0,1]).all().all() and np.array_equal(aug.loc[e.LAB,e.LABELS],e.Y)
manifest=dict(created_at=datetime.datetime.now(datetime.timezone.utc).isoformat(),selected=selected,training_gold_rows=len(e.Y),unlabeled_rows_used=len(e.UNLAB),test_rows=len(test),prediction_strategy='Fit each member on all 157 labeled rows; unsupervised transformations and teacher clustering use the permitted training unlabeled pool. Average member probabilities with weights 0.50 / 0.25 / 0.25; threshold 0.35, fallback to highest probability if empty.',validation_status='Not evaluated again on the used external validation set. Earlier 15-fold-bagged models achieved best 0.850427; their scores do not measure this full-fit candidate.',kaggle_score=None,submission=str(path),submission_sha256=hashlib.sha256(path.read_bytes()).hexdigest(),engine_sha256=hashlib.sha256(Path('experiment.py').read_bytes()).hexdigest(),claims='0.936306 is training-only leave-one-embedding-group-out development F1, not Kaggle F1. The winning-score objective remains unverified.')
e.save_json('experiments/final-candidate.json',manifest)
print(json.dumps(manifest,indent=2))


Overwriting current_reproduction/make_candidate.py


In [25]:
%%writefile current_reproduction/phase10.py
"""Joint multilabel pseudo-label updates with training-derived structural constraints."""
import phase9,experiment as e
import numpy as np,json
from scipy.special import expit
from sklearn.svm import SVC
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.cluster import KMeans
legacy_predict=e.predict

def assign(d,Y,c):
 mode=c.get('assignment','nonempty');p=expit(2*d)
 if mode=='nonempty':return e.decisions(p,'t0.5')
 if mode=='expected_f1':return e.decisions(p,'expected_f1')
 if mode=='exclusive_rare':
  out=(d>=0).astype(int);rare=[0,1,2,4];r=d[:,rare].argmax(1);out[:,rare]=0
  for i,j in enumerate(r):out[i,rare[j]]=int(d[i,rare[j]]>=0)
  empty=out.sum(1)==0;out[np.flatnonzero(empty),d[empty].argmax(1)]=1
  return out
 if mode=='patterns':
  patterns,counts=np.unique(Y,axis=0,return_counts=True)
  score=d@patterns.T*2+c.get('prior_weight',0.)*np.log(counts/counts.sum())[None,:]
  return patterns[score.argmax(1)]
 raise ValueError(mode)

def predict10(train_pos,query_frame,c,excluded_groups=None,return_unlabeled=False):
 if c['model']!='structured_ssl':return legacy_predict(train_pos,query_frame,c,excluded_groups,return_unlabeled)
 pos=np.asarray(train_pos);n=len(pos);u=e.UNLAB if excluded_groups is None else e.UNLAB[~np.isin(e.GROUP[e.UNLAB],excluded_groups)]
 A,B=e.features(np.r_[e.LAB[pos],u],query_frame,c);Y=e.Y[pos];Q=np.r_[B,A[n:]] if return_unlabeled else B
 K=rbf_kernel(A,gamma=c.get('gamma',.001));Kq=rbf_kernel(Q,A,gamma=c.get('gamma',.001));cluster=None
 if c.get('cluster_gate'):
  z=KMeans(n_clusters=c.get('clusters',10),n_init=10,random_state=42).fit_predict(A);cluster=np.full((len(u),5),.5)
  for k in np.unique(z):
   known=z[:n]==k
   if known.sum()>=2:cluster[z[n:]==k]=Y[known].mean(0)
 def fit(y,w,kernel):
  return [SVC(C=c.get('C',1),kernel='precomputed',class_weight='balanced').fit(kernel,y[:,j],sample_weight=np.ascontiguousarray(w[:,j])) if len(np.unique(y[:,j]))>1 else float(y[0,j]*2-1) for j in range(5)]
 def decision(models,kernel):return np.column_stack([m.decision_function(kernel) if hasattr(m,'predict') else np.full(len(kernel),m) for m in models])
 models=fit(Y,np.ones(Y.shape),K[:n,:n]);cols=n
 for fraction in [.1,1/3,1]:
  for _ in range(c.get('iterations',3)):
   d=decision(models,K[n:,:cols]);pseudo=assign(d,Y,c);w=np.full((len(u),5),c.get('pseudo_weight',.03)*fraction)
   if cluster is not None:w[((cluster>=.5)!=pseudo)&(np.maximum(cluster,1-cluster)>=.8)]*=.1
   models=fit(np.r_[Y,pseudo],np.r_[np.ones(Y.shape),w],K);cols=len(A)
 p=expit(2*decision(models,Kq))
 if c.get('decode_patterns'):
  patterns=np.unique(Y,axis=0);d=np.log(p.clip(1e-8,1-1e-8)/(1-p).clip(1e-8,1));weights=np.exp(2*(d@patterns.T-np.max(d@patterns.T,axis=1)[:,None]));weights/=weights.sum(1)[:,None];p=weights@patterns
 if return_unlabeled:return p[:len(B)],p[len(B):],u
 return p

e.predict=predict10

def configs():
 cs=[]
 for mode in ['nonempty','exclusive_rare','patterns','expected_f1']:
  for C in [.3,1,3]:
   for w in [.01,.03,.1]:cs.append(dict(model='structured_ssl',assignment=mode,C=C,gamma=.001,pseudo_weight=w))
 for prior in [.25,.5,1]:
  for C in [1,3]:cs.append(dict(model='structured_ssl',assignment='patterns',prior_weight=prior,C=C,gamma=.001,pseudo_weight=.03))
 for mode in ['nonempty','exclusive_rare','patterns']:
  for C in [1,3]:cs.append(dict(model='structured_ssl',assignment=mode,C=C,gamma=.001,pseudo_weight=.03,cluster_gate=True))
 return cs
if __name__=='__main__':
 cs=configs();e.save_json('experiments/phase10-configs.json',cs)
 for i,c in enumerate(cs):print('RUN',i+1,len(cs),c,flush=True);e.evaluate(c,tag='phase10')


Overwriting current_reproduction/phase10.py


In [26]:
%%writefile current_reproduction/phase11.py
"""Classical random-subspace SSL ensembles and constrained pseudo-label proportions."""
import phase10,experiment as e
import numpy as np,json
from scipy.special import expit
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.svm import SVC
legacy_predict=e.predict

def one(A,Q,Y,c,seed):
 rng=np.random.default_rng(seed);n=len(Y);U=np.arange(n,len(A));features=np.arange(A.shape[1])
 if c.get('feature_fraction',1)<1:features=np.sort(rng.choice(features,size=max(5,int(len(features)*c['feature_fraction'])),replace=False))
 if c.get('unlabeled_fraction',1)<1:U=np.sort(rng.choice(U,size=max(10,int(len(U)*c['unlabeled_fraction'])),replace=False))
 ix=np.r_[np.arange(n),U];A=A[ix][:,features];Q=Q[:,features];gamma=c.get('gamma',.001)*150/len(features)
 K=rbf_kernel(A,gamma=gamma);Kq=rbf_kernel(Q,A,gamma=gamma);outputs=[]
 for j in range(5):
  y=Y[:,j]
  if len(np.unique(y))<2:outputs.append(np.full(len(Q),y[0]));continue
  m=SVC(C=c.get('C',1),kernel='precomputed',class_weight='balanced').fit(K[:n,:n],y)
  initial=m.decision_function(K[n:,:n]);initial_prior=np.mean(initial>=0)
  prior=(1-c.get('prior_mix',0))*initial_prior+c.get('prior_mix',0)*y.mean()
  for fraction in [.1,1/3,1]:
   for _ in range(c.get('iterations',3)):
    d=m.decision_function(K[n:,:m.n_features_in_]);labels=(d>=0).astype(int);w=np.full(len(d),c.get('pseudo_weight',.03)*fraction)
    if c.get('constrain_prior'):
     labels[:]=0;k=max(1,min(len(d)-1,round(prior*len(d))));labels[np.argsort(d)[-k:]]=1
    if c.get('margin_min'):w[np.abs(d)<c['margin_min']]=0
    m=SVC(C=c.get('C',1),kernel='precomputed',class_weight='balanced').fit(K,np.r_[y,labels],sample_weight=np.r_[np.ones(n),w])
  outputs.append(expit(2*m.decision_function(Kq)))
 return np.array(outputs).T

def predict11(train_pos,query_frame,c,excluded_groups=None,return_unlabeled=False):
 if c['model']!='subspace_ssl':return legacy_predict(train_pos,query_frame,c,excluded_groups,return_unlabeled)
 pos=np.asarray(train_pos);n=len(pos);u=e.UNLAB if excluded_groups is None else e.UNLAB[~np.isin(e.GROUP[e.UNLAB],excluded_groups)]
 A,B=e.features(np.r_[e.LAB[pos],u],query_frame,c);Y=e.Y[pos];Q=np.r_[B,A[n:]] if return_unlabeled else B
 p=np.mean([one(A,Q,Y,c,1000+i) for i in range(c.get('members',1))],axis=0)
 if return_unlabeled:return p[:len(B)],p[len(B):],u
 return p

e.predict=predict11

def configs():
 cs=[]
 for ff in [.5,.75,.9,1.]:
  for uf in [.5,.8,1.]:
   if ff==1. and uf==1.:continue
   cs.append(dict(model='subspace_ssl',C=1,gamma=.001,feature_fraction=ff,unlabeled_fraction=uf,members=9))
 for mix in [0,.25,.5,.75,1.]:
  for w in [.01,.03,.1]:cs.append(dict(model='subspace_ssl',C=1,gamma=.001,constrain_prior=True,prior_mix=mix,pseudo_weight=w))
 for margin in [.1,.3,.6,.9,1.2]:
  for w in [.03,.1]:cs.append(dict(model='subspace_ssl',C=1,gamma=.001,margin_min=margin,pseudo_weight=w))
 return cs
if __name__=='__main__':
 cs=configs();e.save_json('experiments/phase11-configs.json',cs)
 for i,c in enumerate(cs):print('RUN',i+1,len(cs),c,flush=True);e.evaluate(c,tag='phase11')


Overwriting current_reproduction/phase11.py


In [27]:
%%writefile current_reproduction/phase12_fullsize.py
"""Leave-one-embedding-group-out selection for training-size-sensitive SSL.
No validation access until every full-size CV prediction and the shortlist are frozen.
"""
import phase11,experiment as e
import json,datetime,numpy as np,pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from threadpoolctl import threadpool_limits
configs=[dict(model='lowdensity_v2',C=C,gamma=.001,pseudo_weight=.03) for C in [.4,.6,.8,1,1.5,2]]
configs += [dict(model='lowdensity_v2',C=1,gamma=g,pseudo_weight=.03) for g in [.0006,.002]]
configs += [dict(model='lowdensity_v2',C=1,gamma=.001,pseudo_weight=.03,cluster_gate=True,clusters=10)]
e.save_json('experiments/phase12-configs.json',configs)
groups=np.unique(e.GROUP[e.LAB]);positions=np.arange(len(e.Y))
def run(c):
 cid=e.config_id(c);path=Path(f'experiments/phase12-loo-{cid}.npz')
 if path.exists():return np.load(path,allow_pickle=True)['record'].item()
 p=np.zeros(e.Y.shape,float)
 for gi,g in enumerate(groups):
  b=np.flatnonzero(e.GROUP[e.LAB]==g);a=positions[~np.isin(positions,b)]
  p[b]=e.predict(a,e.DATA.iloc[e.LAB[b]],c,excluded_groups=[g])
  if gi%30==0:print('LEAVE-GROUP-OUT',cid,gi,len(groups),flush=True)
 rule,score=e.choose_rule(e.Y,p);record=dict(config=c,id=cid,rule=rule,cv=score,cv_metrics=e.metrics(e.Y,p,rule),n_groups=len(groups))
 np.savez_compressed(path,p=p,record=record);print('DONE',json.dumps(record),flush=True);return record
with threadpool_limits(limits=1):
 with ThreadPoolExecutor(max_workers=2) as pool:summary=list(pool.map(run,configs))
e.save_json('experiments/phase12-ranking.json',sorted(summary,key=lambda r:r['cv'],reverse=True))
# One sample-F1 champion and one macro-F1 champion. All decisions training-only.
cs={}
for r in [max(summary,key=lambda r:r['cv']),max(summary,key=lambda r:.7*r['cv']+.3*r['cv_metrics']['macro_f1'])]:cs[r['id']]=r
candidates=list(cs.values());e.save_json('experiments/phase12-validation-frozen.json',dict(frozen_at=datetime.datetime.now(datetime.timezone.utc).isoformat(),candidates=candidates))
V=pd.read_csv('data/part1_val.csv',usecols=lambda col:col not in e.LABELS)
with threadpool_limits(limits=1):preds=[e.predict(positions,V,r['config']) for r in candidates]
np.savez_compressed('experiments/phase12-validation-predictions.npz',p=preds)
y=pd.read_csv('data/part1_val.csv')[e.LABELS].to_numpy();results=[]
for r,p in zip(candidates,preds):
 out={**r,'validation':e.metrics(y,p,r['rule'])};results.append(out);print('VALIDATION',json.dumps(out),flush=True)
e.save_json('experiments/phase12-validation-results.json',results)


Overwriting current_reproduction/phase12_fullsize.py


In [28]:
%%writefile current_reproduction/phase13_global.py
"""Extend the global decision threshold range using training OOF predictions only."""
import phase11,experiment as e
import numpy as np,pandas as pd,json,datetime
from pathlib import Path
from threadpoolctl import threadpool_limits
# Prefer the tied validation winner with an explicit unsupervised clustering component.
c={'model':'lowdensity_v2','cluster_gate':True,'clusters':10,'C':1,'gamma':.001,'pseudo_weight':.03};cid=e.config_id(c)
ps=[]
for seed in [20261104,20261105,20261106]:
 tag='phase6' if seed==20261104 else 'phase6_repeat'
 path=Path(f'experiments/{tag}-{cid}-{seed}.npz')
 if not path.exists():e.evaluate(c,tag=tag,seed=seed)
 ps.append(np.load(path,allow_pickle=True)['p'])
# Same global threshold on all five labels; tie-breaking favors the default 0.5.
rules=['t'+str(t) for t in [.2,.25,.3,.35,.4,.45,.5,.55,.6,.65,.7,.75,.8]]+['argmax','expected_f1']
p=np.mean(ps,axis=0);rows=[]
for rule in rules:
 ms=[e.metrics(e.Y,z,rule) for z in ps];rows.append(dict(rule=rule,cv_mean=float(np.mean([m['sample_f1'] for m in ms])),cv_macro=float(np.mean([m['macro_f1'] for m in ms])),averaged_oof=e.metrics(e.Y,p,rule)['sample_f1']))
r=max(rows,key=lambda r:(r['cv_mean'],r['averaged_oof'],-abs(float(r['rule'][1:])-.5) if r['rule'].startswith('t') else -1));candidate=dict(config=c,**r)
e.save_json('experiments/phase13-threshold-cv.json',rows)
e.save_json('experiments/phase13-validation-frozen.json',dict(frozen_at=datetime.datetime.now(datetime.timezone.utc).isoformat(),candidates=[candidate]))
V=pd.read_csv('data/part1_val.csv',usecols=lambda col:col not in e.LABELS)
with threadpool_limits(limits=1):pred=e.predict(np.arange(len(e.Y)),V,c)
np.savez_compressed('experiments/phase13-validation-predictions.npz',p=pred)
y=pd.read_csv('data/part1_val.csv')[e.LABELS].to_numpy();result={**candidate,'validation':e.metrics(y,pred,r['rule'])}
e.save_json('experiments/phase13-validation-results.json',[result]);print(json.dumps(result,indent=2))


Overwriting current_reproduction/phase13_global.py


In [29]:
%%writefile current_reproduction/phase14.py
"""Nested training-only calibration of SSL margins and multilabel dependencies."""
import phase11,experiment as e
import numpy as np,json
from pathlib import Path
from scipy.special import expit,logit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
legacy_predict=e.predict
CACHE={}
BASE=dict(model='lowdensity_v2',cluster_gate=True,clusters=10,C=1,gamma=.001,pseudo_weight=.03)

def shifted(p,t):return expit(logit(np.clip(p,1e-7,1-1e-7))-logit(np.clip(t,1e-7,1-1e-7)))

def fit_thresholds(p,y,c):
 grid=np.array([.25,.3,.35,.4,.45,.5,.55,.6,.65,.7,.75]);t=np.full(5,.5)
 def objective(t):
  h=e.decisions(shifted(p,t),'t0.5');s=f1_score(y,h,average='samples',zero_division=0);m=f1_score(y,h,average='macro',zero_division=0)
  return (1-c.get('macro_weight',0))*s+c.get('macro_weight',0)*m-c.get('penalty',.02)*np.mean((t-.5)**2)
 for _ in range(4):
  changed=False
  for j in range(5):
   opts=[]
   for v in grid:
    tt=t.copy();tt[j]=v;opts.append((objective(tt),-abs(v-.5),v))
   v=max(opts)[2]
   if t[j]!=v:changed=True;t[j]=v
  if not changed:break
 return t

def calibrated(p,y,q,c):
 kind=c['calibration'];X=logit(np.clip(p,1e-6,1-1e-6));Q=logit(np.clip(q,1e-6,1-1e-6))
 if kind=='thresholds':return shifted(q,fit_thresholds(p,y,c))
 if kind=='patterns':
  patterns,yy=np.unique(y,axis=0,return_inverse=True)
  model=LogisticRegression(C=c.get('meta_C',1),max_iter=2000).fit(X,yy);pp=model.predict_proba(Q)
  if c.get('gfm'):
   patterns=patterns[model.classes_];choices=((np.arange(1,32)[:,None]>>np.arange(5))&1)
   utility=2*(choices@patterns.T)/(choices.sum(1)[:,None]+patterns.sum(1)[None,:])
   return choices[np.argmax(pp@utility.T,axis=1)].astype(float)
  return pp@patterns[model.classes_]
 out=[]
 for j in range(5):
  xx=X if kind=='joint' else X[:,j:j+1];qq=Q if kind=='joint' else Q[:,j:j+1]
  if len(np.unique(y[:,j]))<2:out.append(np.full(len(q),y[0,j]));continue
  m=LogisticRegression(C=c.get('meta_C',1),class_weight=c.get('meta_balance'),max_iter=1000).fit(xx,y[:,j]);out.append(m.predict_proba(qq)[:,1])
 return np.array(out).T

def predict14(train_pos,query_frame,c,excluded_groups=None,return_unlabeled=False):
 if c['model']!='calibrated_ssl':return legacy_predict(train_pos,query_frame,c,excluded_groups,return_unlabeled)
 pos=np.asarray(train_pos);y=e.Y[pos];base=c.get('base',BASE);excluded=[] if excluded_groups is None else list(excluded_groups)
 # Cache is indexed by all label-bearing positions, exclusion groups and base config.
 inner_splits=c.get('inner_splits',3)
 key=(tuple(pos),tuple(sorted(excluded)),e.config_id(base),inner_splits)
 if key not in CACHE:
  p=np.zeros_like(y,float)
  for a,b in e.make_splits(y,groups=e.GROUP[e.LAB[pos]],seed=130713,n_splits=inner_splits):
   p[b]=legacy_predict(pos[a],e.DATA.iloc[e.LAB[pos[b]]],base,excluded_groups=np.r_[excluded,e.GROUP[e.LAB[pos[b]]]])
  CACHE[key]=p
 p=CACHE[key]
 pred=legacy_predict(pos,query_frame,base,excluded_groups=excluded_groups,return_unlabeled=return_unlabeled)
 if return_unlabeled:
  q,u,ix=pred;qq=calibrated(p,y,np.r_[q,u],c);return qq[:len(q)],qq[len(q):],ix
 return calibrated(p,y,pred,c)

e.predict=predict14

def configs():
 cs=[]
 for kind in ['independent','joint','patterns']:
  for C in [.1,1,10]:cs.append(dict(model='calibrated_ssl',calibration=kind,meta_C=C))
 for weight in [0,.25,.5,1]:cs.append(dict(model='calibrated_ssl',calibration='thresholds',macro_weight=weight))
 for C in [.1,1,10]:cs.append(dict(model='calibrated_ssl',calibration='patterns',gfm=True,meta_C=C))
 return cs
if __name__=='__main__':
 cs=configs();e.save_json('experiments/phase14-configs.json',cs)
 for i,c in enumerate(cs):print('RUN',i+1,len(cs),c,flush=True);e.evaluate(c,tag='phase14')


Overwriting current_reproduction/phase14.py


In [30]:
%%writefile current_reproduction/phase15_ensembles.py
"""Small-support convex ensembles, selected solely on repeated training OOF outputs."""
import phase14,experiment as e
import numpy as np,pandas as pd,json,datetime,itertools
from pathlib import Path
from threadpoolctl import threadpool_limits
base=dict(model='lowdensity_v2',cluster_gate=True,clusters=10,C=1,gamma=.001,pseudo_weight=.03)
rows=[json.loads(s) for s in Path('experiments/trials.jsonl').read_text().splitlines()];by={}
for r in rows:
 if r['status']=='ok' and r['seed'] in [20261104,20261105,20261106]:by.setdefault(r['id'],{})[r['seed']]=r
# One high-performing model per broad mechanism. Freeze candidates without reading validation results.
pool=[]
for family in ['logistic','svc','pseudo','margin_svm','lowdensity_svm','structured_ssl','subspace_ssl','kernel_lowdensity','calibrated_ssl','dense_graph']:
 candidates=[]
 for cid,seeds in by.items():
  if len(seeds)<3:continue
  r=seeds[20261104]
  if r['config']['model']!=family:continue
  ps=[]
  for seed,rr in sorted(seeds.items()):
   z=np.load(f"experiments/{rr['tag']}-{cid}-{seed}.npz",allow_pickle=True)['p'];ps.append(z)
  if any(z.shape!=(157,5) or not np.isfinite(z).all() for z in ps):continue
  p=np.mean(ps,axis=0);rule,_=e.choose_rule(e.Y,p);score=float(np.mean([e.metrics(e.Y,z,rule)['sample_f1'] for z in ps]));candidates.append((score,r['config'],ps))
 if candidates:pool.append(max(candidates,key=lambda x:x[0]))
cid=e.config_id(base);ps=[]
for seed in [20261104,20261105,20261106]:
 rr=by[cid][seed];ps.append(np.load(f"experiments/{rr['tag']}-{cid}-{seed}.npz",allow_pickle=True)['p'])
members=[base]+[x[1] for x in pool];preds=[ps]+[x[2] for x in pool]
configs=[]
for j in range(1,len(members)):
 for w in [.25,.5,.75]:configs.append(([0,j],[1-w,w]))
for j,k in itertools.combinations(range(1,len(members)),2):configs.append(([0,j,k],[.5,.25,.25]))
results=[]
for inds,weights in configs:
 pp=[sum(w*preds[i][seed] for i,w in zip(inds,weights)) for seed in range(3)];p=np.mean(pp,axis=0);rule,_=e.choose_rule(e.Y,p)
 scores=[e.metrics(e.Y,z,rule) for z in pp];r=dict(config=dict(model='ensemble',members=[members[i] for i in inds],weights=weights),rule=rule,cv_mean=float(np.mean([m['sample_f1'] for m in scores])),cv_macro=float(np.mean([m['macro_f1'] for m in scores])),cv_std=float(np.std([m['sample_f1'] for m in scores])))
 results.append(r)
e.save_json('experiments/phase15-ensemble-screen.json',results)
cs={}
for metric in ['sample','balanced']:
 r=max(results,key=lambda r:((r['cv_mean'] if metric=='sample' else .7*r['cv_mean']+.3*r['cv_macro'])-.2*r['cv_std']))
 cs[e.config_id(r['config'])]=r
candidates=list(cs.values());e.save_json('experiments/phase15-validation-frozen.json',dict(frozen_at=datetime.datetime.now(datetime.timezone.utc).isoformat(),candidates=candidates))
V=pd.read_csv('data/part1_val.csv',usecols=lambda col:col not in e.LABELS)
with threadpool_limits(limits=1):p=[e.predict(np.arange(len(e.Y)),V,r['config']) for r in candidates]
np.savez_compressed('experiments/phase15-validation-predictions.npz',p=p)
y=pd.read_csv('data/part1_val.csv')[e.LABELS].to_numpy();out=[]
for r,z in zip(candidates,p):
 x={**r,'validation':e.metrics(y,z,r['rule'])};out.append(x);print('VALIDATION',json.dumps(x),flush=True)
e.save_json('experiments/phase15-validation-results.json',out)


Overwriting current_reproduction/phase15_ensembles.py


In [31]:
%%writefile current_reproduction/phase16.py
"""Alternative supervised losses with the same low-density pseudo-label schedule."""
import phase14,experiment as e
import numpy as np,json
from scipy.special import expit
from sklearn.kernel_approximation import Nystroem
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression,RidgeClassifier
legacy_predict=e.predict

def predict16(train_pos,query_frame,c,excluded_groups=None,return_unlabeled=False):
 if c['model']!='loss_ssl':return legacy_predict(train_pos,query_frame,c,excluded_groups,return_unlabeled)
 pos=np.asarray(train_pos);n=len(pos);u=e.UNLAB if excluded_groups is None else e.UNLAB[~np.isin(e.GROUP[e.UNLAB],excluded_groups)]
 A,B=e.features(np.r_[e.LAB[pos],u],query_frame,c);Y=e.Y[pos];Q=np.r_[B,A[n:]] if return_unlabeled else B
 if c.get('kernel','rbf')=='rbf':
  featuremap=Nystroem(kernel='rbf',gamma=c.get('gamma',.001),n_components=min(c.get('components',300),len(A)),random_state=42);A=featuremap.fit_transform(A);Q=featuremap.transform(Q)
 outputs=[]
 def model():
  if c['loss']=='squared_hinge':return LinearSVC(C=c.get('C',1),class_weight='balanced',max_iter=10000,random_state=42,dual='auto')
  if c['loss']=='logistic':return LogisticRegression(C=c.get('C',1),class_weight='balanced',max_iter=1000)
  return RidgeClassifier(alpha=1/c.get('C',1),class_weight='balanced')
 for j in range(5):
  y=Y[:,j]
  if len(np.unique(y))<2:outputs.append(np.full(len(Q),y[0]));continue
  m=model().fit(A[:n],y)
  for fraction in [.1,1/3,1]:
   for _ in range(3):
    labels=(m.decision_function(A[n:])>=0).astype(int)
    m=model().fit(A,np.r_[y,labels],sample_weight=np.r_[np.ones(n),np.full(len(u),c.get('pseudo_weight',.03)*fraction)])
  outputs.append(expit(m.decision_function(Q)*(1 if c['loss']=='logistic' else 2)))
 p=np.array(outputs).T
 if return_unlabeled:return p[:len(B)],p[len(B):],u
 return p

e.predict=predict16

def configs():
 cs=[]
 for loss in ['squared_hinge','logistic','squared_error']:
  for kernel,values in [('rbf',[.3,1,3,10]),('linear',[.01,.1,1])]:
   for C in values:cs.append(dict(model='loss_ssl',loss=loss,kernel=kernel,C=C))
 for loss in ['squared_hinge','logistic']:
  for C in [1,3,10]:cs.append(dict(model='loss_ssl',loss=loss,kernel='rbf',C=C,components=600))
 return cs
if __name__=='__main__':
 cs=configs();e.save_json('experiments/phase16-configs.json',cs)
 for i,c in enumerate(cs):print('RUN',i+1,len(cs),c,flush=True);e.evaluate(c,tag='phase16')


Overwriting current_reproduction/phase16.py


In [32]:
%%writefile current_reproduction/phase17.py
"""Classical tree students trained on fold-local cluster/SVM pseudo-labels."""
import phase16,experiment as e
import numpy as np,json
from sklearn.ensemble import ExtraTreesRegressor
legacy_predict=e.predict
TEACHER=dict(model='lowdensity_v2',cluster_gate=True,clusters=10,C=1,gamma=.001,pseudo_weight=.03)
CACHE={}

def predict17(train_pos,query_frame,c,excluded_groups=None,return_unlabeled=False):
 if c['model']!='tree_student':return legacy_predict(train_pos,query_frame,c,excluded_groups,return_unlabeled)
 pos=np.asarray(train_pos);n=len(pos);u=e.UNLAB if excluded_groups is None else e.UNLAB[~np.isin(e.GROUP[e.UNLAB],excluded_groups)]
 key=(tuple(pos),tuple(u))
 if key not in CACHE:CACHE[key]=legacy_predict(pos,e.DATA.iloc[u].drop(columns=e.LABELS),TEACHER,excluded_groups=excluded_groups)
 pseudo=CACHE[key]
 if c.get('hard',False):pseudo=e.decisions(pseudo,'t0.55').astype(float)
 A,B=e.features(np.r_[e.LAB[pos],u],query_frame,c);Y=e.Y[pos];Q=np.r_[B,A[n:]] if return_unlabeled else B
 yy=np.r_[Y,pseudo];outputs=[]
 for j in range(5):
  gold_count=np.bincount(Y[:,j],minlength=2);goldweights=np.array([(n/(2*max(gold_count[v],1)))**c.get('balance_power',1) for v in Y[:,j]])
  w=np.r_[goldweights,np.full(len(u),c.get('pseudo_weight',.1))]
  if c['student']=='xgb':
   import xgboost as xgb
   m=xgb.train(dict(objective='binary:logistic',max_depth=c.get('depth',3),eta=.03,min_child_weight=2,subsample=.8,colsample_bytree=.8,reg_lambda=c.get('regularization',1),nthread=1,seed=42),xgb.DMatrix(A,label=yy[:,j],weight=w),num_boost_round=300)
   pred=m.predict(xgb.DMatrix(Q))
  elif c['student']=='catboost':
   from catboost import CatBoostClassifier
   m=CatBoostClassifier(loss_function='CrossEntropy',iterations=300,depth=c.get('depth',4),learning_rate=.03,l2_leaf_reg=10,thread_count=1,random_seed=42,verbose=False,allow_writing_files=False)
   m.fit(A,yy[:,j],sample_weight=w);pred=m.predict_proba(Q)[:,1]
  else:
   m=ExtraTreesRegressor(n_estimators=300,min_samples_leaf=c.get('leaf',2),max_features=.7,n_jobs=1,random_state=42).fit(A,yy[:,j],sample_weight=w);pred=m.predict(Q)
  outputs.append(pred)
 p=np.array(outputs).T
 if return_unlabeled:return p[:len(B)],p[len(B):],u
 return p

e.predict=predict17

def configs():
 cs=[]
 for student in ['xgb','catboost','extra']:
  for pca in [0,25,75]:
   for w in [.03,.3]:cs.append(dict(model='tree_student',student=student,pca=pca,pseudo_weight=w))
 for student in ['xgb','catboost','extra']:
  for pca in [0,25]:cs.append(dict(model='tree_student',student=student,pca=pca,pseudo_weight=.1,hard=True))
 return cs
if __name__=='__main__':
 cs=configs();e.save_json('experiments/phase17-configs.json',cs)
 for i,c in enumerate(cs):print('RUN',i+1,len(cs),c,flush=True);e.evaluate(c,tag='phase17')


Overwriting current_reproduction/phase17.py


In [33]:
%%writefile current_reproduction/phase18.py
"""Fold-local covariate-shift weights for gold labels in cluster-guided SSL."""
import phase17,experiment as e
import numpy as np,json
from scipy.special import expit
from scipy.optimize import minimize
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.cluster import KMeans
legacy_predict=e.predict

def predict18(train_pos,query_frame,c,excluded_groups=None,return_unlabeled=False):
 if c['model']!='weighted_ssl':return legacy_predict(train_pos,query_frame,c,excluded_groups,return_unlabeled)
 pos=np.asarray(train_pos);n=len(pos);u=e.UNLAB if excluded_groups is None else e.UNLAB[~np.isin(e.GROUP[e.UNLAB],excluded_groups)]
 A,B=e.features(np.r_[e.LAB[pos],u],query_frame,c);Y=e.Y[pos];Q=np.r_[B,A[n:]] if return_unlabeled else B
 K=rbf_kernel(A,gamma=.001);Kq=rbf_kernel(Q,A,gamma=.001);kind=c['weighting'];years=e.DATA.iloc[e.LAB[pos]].start_year.to_numpy();gold=np.ones(n)
 if kind=='year':gold=np.exp(c['slope']*(years-2002)/8)
 if kind=='late':gold=np.where(years>2002,c['late_weight'],1.)
 if kind=='domain':
  Z=PCA(n_components=25,svd_solver='full').fit_transform(A);yy=np.r_[np.zeros(n),np.ones(len(u))]
  lr=LogisticRegression(C=c['domain_C'],max_iter=1000).fit(Z,yy);pr=lr.predict_proba(Z[:n])[:,1]
  gold=((pr/(1-pr).clip(1e-8))*n/len(u))**c.get('power',1)
 if kind=='kmm':
  KK=rbf_kernel(A[:n],gamma=c['weight_gamma']);kappa=n*rbf_kernel(A[:n],A[n:],gamma=c['weight_gamma']).mean(1)
  solution=minimize(lambda w:.5*w@KK@w-kappa@w,np.ones(n),jac=lambda w:KK@w-kappa,method='SLSQP',bounds=[(.2,5)]*n,constraints=[{'type':'eq','fun':lambda w:w.sum()-n,'jac':lambda w:np.ones(n)}],options={'maxiter':300,'ftol':1e-8})
  if not solution.success:raise RuntimeError(solution.message)
  gold=1-c.get('blend',1)+c.get('blend',1)*solution.x
 gold=np.clip(gold,.2,5);gold/=gold.mean();outputs=[]
 z=KMeans(n_clusters=10,n_init=10,random_state=42).fit_predict(A);cluster=np.full((len(u),5),.5)
 for k in np.unique(z):
  known=z[:n]==k
  if known.sum()>=2:cluster[z[n:]==k]=Y[known].mean(0)
 for j in range(5):
  y=Y[:,j]
  if len(np.unique(y))<2:outputs.append(np.full(len(Q),y[0]));continue
  m=SVC(C=1,kernel='precomputed',class_weight='balanced').fit(K[:n,:n],y,sample_weight=gold)
  for fraction in [.1,1/3,1]:
   for _ in range(3):
    d=m.decision_function(K[n:,:m.n_features_in_]);pseudo=(d>=0).astype(int);w=np.full(len(u),.03*fraction)
    w[((cluster[:,j]>=.5)!=pseudo)&(np.maximum(cluster[:,j],1-cluster[:,j])>=.8)]*=.1
    m=SVC(C=1,kernel='precomputed',class_weight='balanced').fit(K,np.r_[y,pseudo],sample_weight=np.r_[gold,w])
  outputs.append(expit(2*m.decision_function(Kq)))
 p=np.array(outputs).T
 if return_unlabeled:return p[:len(B)],p[len(B):],u
 return p

e.predict=predict18

def configs():
 cs=[dict(model='weighted_ssl',weighting='year',slope=s) for s in [-1,.5,1,2]]
 cs += [dict(model='weighted_ssl',weighting='late',late_weight=w) for w in [1.5,3,5]]
 cs += [dict(model='weighted_ssl',weighting='domain',domain_C=C,power=p) for C in [.01,.1,1] for p in [.5,1]]
 cs += [dict(model='weighted_ssl',weighting='kmm',weight_gamma=g,blend=b) for g in [.001,.003,.01] for b in [.5,1]]
 return cs
if __name__=='__main__':
 cs=configs();e.save_json('experiments/phase18-configs.json',cs)
 for i,c in enumerate(cs):print('RUN',i+1,len(cs),c,flush=True);e.evaluate(c,tag='phase18')


Overwriting current_reproduction/phase18.py


In [34]:
%%writefile current_reproduction/phase19_calibration.py
"""Training-size matched nested calibration; 15-fold inner fits retain ~93% of gold."""
import phase18,experiment as e
cs=[dict(model='calibrated_ssl',calibration='thresholds',macro_weight=w,inner_splits=15) for w in [0,.25,.5]]
cs+=[dict(model='calibrated_ssl',calibration=kind,meta_C=C,inner_splits=15) for kind,C in [('independent',1),('joint',10),('patterns',1)]]
e.save_json('experiments/phase19-configs.json',cs)
for i,c in enumerate(cs):print('RUN',i+1,len(cs),c,flush=True);e.evaluate(c,tag='phase19')


Overwriting current_reproduction/phase19_calibration.py


In [35]:
%%writefile current_reproduction/phase20.py
"""Audited search gaps: separate feature kernels and effective pseudo-label weighting.
All transforms, kernel weights, clustering and pseudo-labels are learned within folds.
"""
import phase18,experiment as e
import numpy as np,json
from scipy.special import expit
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.preprocessing import normalize
from sklearn.svm import SVC
from sklearn.cluster import KMeans
legacy_predict=e.predict

def centered(K):return K-K.mean(0)[None,:]-K.mean(1)[:,None]+K.mean()

def feature_kernel(fitframe,query,block):
 if block=='fingerprint':
  A=e.chemical(fitframe,block);B=e.chemical(query,block)
  def tan(a,b):
   inter=a@b.T;den=a.sum(1)[:,None]+b.sum(1)[None,:]-inter
   return np.divide(inter,den,out=np.zeros_like(inter),where=den>0)
  return tan(A,A),tan(B,A),(A.sum(1)>0).astype(float),(B.sum(1)>0).astype(float)
 A,B=e.block_features(fitframe,query,block);A=normalize(A);B=normalize(B)
 return A@A.T,B@A.T,np.sum(A*A,axis=1),np.sum(B*B,axis=1)

def predict20(train_pos,query_frame,c,excluded_groups=None,return_unlabeled=False):
 if c['model']!='audited_ssl':return legacy_predict(train_pos,query_frame,c,excluded_groups,return_unlabeled)
 pos=np.asarray(train_pos);n=len(pos);u=e.UNLAB if excluded_groups is None else e.UNLAB[~np.isin(e.GROUP[e.UNLAB],excluded_groups)]
 idx=np.r_[e.LAB[pos],u];A,B=e.features(idx,query_frame,c);Y=e.Y[pos];Q=np.r_[B,A[n:]] if return_unlabeled else B
 Kbase=rbf_kernel(A,gamma=c.get('gamma',.001));Kqb=rbf_kernel(Q,A,gamma=c.get('gamma',.001))
 query=e.pd.concat([query_frame,e.DATA.iloc[u]],ignore_index=True) if return_unlabeled else query_frame
 blocks=c.get('aux_blocks',[]);kernels=[feature_kernel(e.DATA.iloc[idx],query,block) for block in blocks]
 if c.get('deduplicate'):
  _,inv,counts=np.unique(e.GROUP[idx],return_inverse=True,return_counts=True);rowweight=1/counts[inv].astype(float)
  rowweight[:n]/=rowweight[:n].mean();rowweight[n:]/=rowweight[n:].mean()
 else:rowweight=np.ones(len(A))
 cluster=None
 if c.get('cluster_gate',True):
  z=KMeans(n_clusters=10,n_init=10,random_state=42).fit_predict(A);cluster=np.full((len(u),5),.5)
  for k in np.unique(z):
   gold=z[:n]==k
   if gold.sum()>=2:cluster[z[n:]==k]=Y[gold].mean(0)
 outputs=[]
 for j in range(5):
  y=Y[:,j]
  if len(np.unique(y))<2:outputs.append(np.full(len(Q),y[0]));continue
  if kernels:
   weights=np.ones(len(kernels))/len(kernels)
   if c.get('alignment'):
    target=centered(np.outer(y*2-1,y*2-1));align=[]
    for kk,_,_,_ in kernels:
     h=centered(kk[:n,:n]);align.append(max(0,float(np.sum(h*target)/(np.linalg.norm(h)*np.linalg.norm(target)+1e-15))))
    weights=np.array(align)+1e-8;weights/=weights.sum()
   fraction=c.get('aux_weight',.2);K=(1-fraction)*Kbase;Kq=(1-fraction)*Kqb;diag=np.full(len(A),1-fraction);dq=np.full(len(Q),1-fraction)
   for w,(kk,kq,di,dqi) in zip(weights,kernels):K+=fraction*w*kk;Kq+=fraction*w*kq;diag+=fraction*w*di;dq+=fraction*w*dqi
   K/=np.sqrt(diag[:,None]*diag[None,:]);Kq/=np.sqrt(dq[:,None]*diag[None,:])
  else:K=Kbase;Kq=Kqb
  def fit(k,labels,sw):
   if c.get('balance','raw')=='effective':
    counts=np.bincount(labels,weights=sw,minlength=2);bal={v:sw.sum()/(2*counts[v]) for v in [0,1]}
   else:bal='balanced'
   return SVC(C=c.get('C',1),kernel='precomputed',class_weight=bal).fit(k,labels,sample_weight=np.ascontiguousarray(sw))
  m=fit(K[:n,:n],y,rowweight[:n])
  for fraction in [.1,1/3,1]:
   for _ in range(3):
    pseudo=(m.decision_function(K[n:,:m.n_features_in_])>=0).astype(int);w=rowweight[n:]*c.get('pseudo_weight',.03)*fraction
    if cluster is not None:w[((cluster[:,j]>=.5)!=pseudo)&(np.maximum(cluster[:,j],1-cluster[:,j])>=.8)]*=.1
    m=fit(K,np.r_[y,pseudo],np.r_[rowweight[:n],w])
  outputs.append(expit(2*m.decision_function(Kq)))
 p=np.array(outputs).T
 if return_unlabeled:return p[:len(B)],p[len(B):],u
 return p

e.predict=predict20

def configs():
 cs=[]
 for block in ['criteria','drugs','drug_targets','fingerprint']:
  for w in [.1,.3]:cs.append(dict(model='audited_ssl',aux_blocks=[block],aux_weight=w))
 for pair in [['criteria','drugs'],['criteria','fingerprint'],['diseases_cleaned','drug_targets'],['drugs','drug_targets'],['fingerprint','drug_targets']]:
  for C in [.5,1,2]:cs.append(dict(model='audited_ssl',aux_blocks=pair,aux_weight=.3,C=C))
 for w in [.2,.5]:cs.append(dict(model='audited_ssl',aux_blocks=['criteria','drugs','drug_targets','fingerprint'],aux_weight=w,alignment=True))
 for C in [.3,1,3]:
  for w in [.01,.03,.1]:cs.append(dict(model='audited_ssl',balance='effective',C=C,pseudo_weight=w))
 for balance in ['raw','effective']:cs.append(dict(model='audited_ssl',balance=balance,deduplicate=True))
 return cs
if __name__=='__main__':
 cs=configs();e.save_json('experiments/phase20-configs.json',cs)
 for i,c in enumerate(cs):print('RUN',i+1,len(cs),c,flush=True);e.evaluate(c,tag='phase20')


Overwriting current_reproduction/phase20.py


In [36]:
%%writefile current_reproduction/phase21.py
"""Binary-relevance tri-training with classical learners and fold-local guards.
Adapted from Zhou & Li (2005). Weighted pseudo-labels, cluster checks, stratified
bootstraps, confidence gates, and cross-fitted error floors are explicit variants;
we do not assert the original theoretical guarantees for these modifications.
"""
import phase20, experiment as e
import json
import numpy as np
from scipy.special import expit
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import rbf_kernel, polynomial_kernel

legacy_predict = e.predict
GUARD_CACHE = {}
LAST_DIAGNOSTICS = []


def member_specs(c):
    C = c.get('C', 1)
    base = dict(kind='rbf', C=C, gamma=.001)
    group = c.get('committee', 'rbf')
    if group == 'mixed':
        return [base, dict(kind='logistic', C=.3), dict(kind='ridge', alpha=10)]
    if group == 'scales':
        return [dict(kind='rbf', C=C, gamma=g) for g in [.0006, .001, .002]]
    if group == 'kernels':
        return [base, dict(kind='poly', C=C, gamma=.003), dict(kind='linear', C=C)]
    return [base.copy() for _ in range(3)]


def bootstrap(y, rng, stratified=True):
    # Preserve minority presence; bootstrap rows independently within each class.
    if not stratified:
        return rng.choice(len(y), size=len(y), replace=True)
    return np.concatenate([rng.choice(np.flatnonzero(y == v), (y == v).sum(), replace=True)
                           for v in np.unique(y)])


def update_size(error, previous_error, previous_size, available):
    """Return allowable pseudo-label count under the original error/size guards."""
    if not error < previous_error or available == 0:
        return 0, previous_size
    if previous_size == 0:
        previous_size = int(np.floor(error / (previous_error - error) + 1))
    if not previous_size < available:
        return 0, previous_size
    if error * available < previous_error * previous_size:
        return available, previous_size
    if error > 0 and previous_size > error / (previous_error - error):
        size = int(np.ceil(previous_error * previous_size / error - 1))
        if previous_size < size <= available:
            return size, previous_size
    return 0, previous_size


def predict21(train_pos, query_frame, c, excluded_groups=None, return_unlabeled=False):
    global LAST_DIAGNOSTICS
    if c['model'] == 'tri_labelwise':
        base = predict21(train_pos, query_frame, c['base'], excluded_groups, return_unlabeled)
        teacher = predict21(train_pos, query_frame, c['tri'], excluded_groups, return_unlabeled)
        def combine(bp, tp):
            hb = e.decisions(bp, c['base_rule'])
            ht = e.decisions(tp, c['tri_rule'])
            out = np.where(np.asarray(c['use_tri'], bool)[None, :], ht, hb)
            empty = out.sum(1) == 0
            out[empty] = hb[empty]
            return out.astype(float)
        if return_unlabeled:
            assert np.array_equal(base[2], teacher[2])
            return combine(base[0], teacher[0]), combine(base[1], teacher[1]), base[2]
        return combine(base, teacher)
    if c['model'] != 'tri_training':
        return legacy_predict(train_pos, query_frame, c, excluded_groups, return_unlabeled)
    pos = np.asarray(train_pos)
    n = len(pos)
    u = e.UNLAB if excluded_groups is None else e.UNLAB[~np.isin(e.GROUP[e.UNLAB], excluded_groups)]
    A, B = e.features(np.r_[e.LAB[pos], u], query_frame, c)
    Y = e.Y[pos]
    Q = np.r_[B, A[n:]] if return_unlabeled else B
    specs = member_specs(c)
    matrices = []
    for spec in specs:
        kind = spec['kind']
        if kind == 'rbf':
            matrices.append((rbf_kernel(A, gamma=spec['gamma']), rbf_kernel(Q, A, gamma=spec['gamma'])))
        elif kind == 'poly':
            matrices.append((polynomial_kernel(A, gamma=spec['gamma'], degree=2, coef0=1),
                             polynomial_kernel(Q, A, gamma=spec['gamma'], degree=2, coef0=1)))
        elif kind == 'linear':
            matrices.append((A @ A.T / A.shape[1], Q @ A.T / A.shape[1]))
        else:
            matrices.append(None)

    def fit(k, rows, yy, weights):
        spec = specs[k]
        if len(np.unique(yy)) < 2:
            return (float(yy[0]), rows)
        if matrices[k] is not None:
            model = SVC(C=spec['C'], kernel='precomputed', class_weight='balanced')
            model.fit(matrices[k][0][np.ix_(rows, rows)], yy, sample_weight=np.ascontiguousarray(weights))
        else:
            if spec['kind'] == 'logistic':
                model = LogisticRegression(C=spec['C'], class_weight='balanced', max_iter=1500)
            else:
                model = RidgeClassifier(alpha=spec['alpha'], class_weight='balanced')
            model.fit(A[rows], yy, sample_weight=np.ascontiguousarray(weights))
        return model, rows

    def probability(k, fitted, rows=None, query=False):
        model, used = fitted
        size = len(Q) if query else len(rows)
        if isinstance(model, float):
            return np.full(size, model)
        if matrices[k] is not None:
            X = matrices[k][1][:, used] if query else matrices[k][0][np.ix_(rows, used)]
        else:
            X = Q if query else A[rows]
        temperature = 1 if specs[k]['kind'] == 'logistic' else 2
        return expit(temperature * model.decision_function(X))

    cluster = None
    if c.get('cluster_gate', True):
        z = KMeans(n_clusters=10, n_init=10, random_state=42).fit_predict(A)
        cluster = np.full((len(u), 5), .5)
        for group in np.unique(z):
            gold = z[:n] == group
            if gold.sum() >= 2:
                cluster[z[n:] == group] = Y[gold].mean(0)

    guard_predictions = None
    if c.get('guard', 'textbook') == 'crossfit':
        geometry = {key: c.get(key) for key in ['scale', 'pca', 'white', 'norm', 'features', 'feature_weight']}
        key = (tuple(pos), tuple(u), Y.tobytes(), json.dumps(specs, sort_keys=True), json.dumps(geometry, sort_keys=True), c.get('stratified_bootstrap', True))
        if key not in GUARD_CACHE:
            gp = np.zeros((3, n, 5))
            for fi, (a, b) in enumerate(e.make_splits(Y, groups=e.GROUP[e.LAB[pos]], seed=210017, n_splits=3)):
                for j in range(5):
                    for k in range(3):
                        rng = np.random.default_rng(210000 + 100 * j + 10 * k + fi)
                        rows = a[bootstrap(Y[a, j], rng, c.get('stratified_bootstrap', True))]
                        model = fit(k, rows, Y[rows, j], np.ones(len(rows)))
                        gp[k, b, j] = probability(k, model, rows=b)
            GUARD_CACHE[key] = gp
        guard_predictions = GUARD_CACHE[key]

    outputs = []
    diagnostics = []
    unlabeled_rows = np.arange(n, len(A))
    all_gold = np.arange(n)
    for j in range(5):
        y = Y[:, j]
        if len(np.unique(y)) < 2:
            outputs.append(np.full(len(Q), y[0]))
            continue
        rngs = [np.random.default_rng(210123 + 100*j + k) for k in range(3)]
        models = []
        for k in range(3):
            rows = bootstrap(y, rngs[k], c.get('stratified_bootstrap', True))
            models.append(fit(k, rows, y[rows], np.ones(n)))
        previous_errors = np.full(3, .5)
        previous_sizes = np.zeros(3, int)
        for iteration in range(c.get('rounds', 8)):
            if c.get('no_teaching', False):
                break
            goldp = np.array([probability(k, models[k], rows=all_gold) for k in range(3)])
            up = np.array([probability(k, models[k], rows=unlabeled_rows) for k in range(3)])
            planned = []
            for k in range(3):
                peers = [v for v in range(3) if v != k]
                g0, g1 = goldp[peers]
                agree = (g0 >= .5) == (g1 >= .5)
                error = float(np.mean((g0[agree] >= .5) != y[agree])) if agree.any() else .5
                if guard_predictions is not None:
                    p0, p1 = guard_predictions[peers, :, j]
                    ok = (p0 >= .5) == (p1 >= .5)
                    # A fixed OOF floor prevents overly confident in-sample error estimates.
                    floor = (np.sum((p0[ok] >= .5) != y[ok]) + 1) / (ok.sum() + 2)
                    error = max(error, floor)
                p0, p1 = up[peers]
                match = (p0 >= .5) == (p1 >= .5)
                confidence = np.minimum(np.maximum(p0, 1-p0), np.maximum(p1, 1-p1))
                pool = np.flatnonzero(match & (confidence >= c.get('confidence', .5)))
                count, previous_sizes[k] = update_size(error, previous_errors[k], previous_sizes[k], len(pool))
                if count == 0:
                    continue
                selected = rngs[k].choice(pool, size=count, replace=False) if count < len(pool) else pool
                pseudo = (p0[selected] >= .5).astype(int)
                weights = np.full(count, c.get('pseudo_weight', .1))
                if cluster is not None:
                    cp = cluster[selected, j]
                    weights[((cp >= .5) != pseudo) & (np.maximum(cp, 1-cp) >= .8)] *= .1
                if c.get('cap_mass'):
                    for value in [0, 1]:
                        mask = pseudo == value
                        cap = c.get('mass_fraction', .5) * np.sum(y == value)
                        if weights[mask].sum() > cap:
                            weights[mask] *= cap / weights[mask].sum()
                planned.append((k, selected, pseudo, weights, error))
            if not planned:
                break
            # Simultaneous updates: no learner sees another's update from this round.
            for k, selected, pseudo, weights, error in planned:
                models[k] = fit(k, np.r_[all_gold, n+selected], np.r_[y, pseudo], np.r_[np.ones(n), weights])
                previous_errors[k] = error
                previous_sizes[k] = len(selected)
                diagnostics.append(dict(label=e.LABELS[j],round=iteration,learner=k,error=error,
                                        pseudo_count=len(selected),positive=int(pseudo.sum()),
                                        effective_weight=float(weights.sum())))
        qp = np.array([probability(k, models[k], query=True) for k in range(3)])
        if c.get('aggregation', 'mean') == 'vote':
            # Preserve majority decisions at 0.5 and use scores to break empty-output ties.
            p = .8 * (qp >= .5).mean(0) + .2 * qp.mean(0)
        else:
            p = qp.mean(0)
        outputs.append(p)
    LAST_DIAGNOSTICS = diagnostics
    p = np.array(outputs).T
    if return_unlabeled:
        return p[:len(B)], p[len(B):], u
    return p


e.predict = predict21


def configs():
    cs = []
    for committee in ['rbf', 'mixed', 'scales', 'kernels']:
        for guard in ['textbook', 'crossfit']:
            for weight in [.03, .1, 1.]:
                cs.append(dict(model='tri_training', committee=committee, guard=guard, pseudo_weight=weight))
        for confidence in [.7, .85]:
            cs.append(dict(model='tri_training', committee=committee, guard='crossfit', pseudo_weight=.1,
                           confidence=confidence, cap_mass=True))
        cs.append(dict(model='tri_training', committee=committee, no_teaching=True))
        cs.append(dict(model='tri_training', committee=committee, pseudo_weight=.1, aggregation='vote'))
    return cs


if __name__ == '__main__':
    cs = configs()
    e.save_json('experiments/phase21-configs.json', cs)
    for i, c in enumerate(cs):
        print('RUN', i+1, len(cs), c, flush=True)
        e.evaluate(c, tag='phase21')
        e.save_json(f'experiments/phase21-last-fold-diagnostics-{e.config_id(c)}.json', LAST_DIAGNOSTICS)


Overwriting current_reproduction/phase21.py


In [37]:
%%writefile current_reproduction/phase21_classical_hybrid.py
"""Apply the preregistered reason-combination rule unchanged to classical tri-training."""
import phase21 as tri,experiment as e
import numpy as np,pandas as pd,json,datetime
from pathlib import Path
from threadpoolctl import threadpool_limits
chosen=json.loads(Path('experiments/phase21c-validation-frozen.json').read_text())['candidates'][0]
base=json.loads(Path('experiments/phase21-prior-incumbent.json').read_text())['candidate']
weights=dict(np.load('experiments/phase21-weights.npz'));names=['propensity_logistic','propensity_tree','kmm'];w=np.array([weights[k]/weights[k].sum() for k in names])
ps=[];baseline=[]
for seed in [20261107,20261108,20261109]:
 bp=np.load(f"experiments/phase21_refit-{e.config_id(base['config'])}-{seed}.npz",allow_pickle=True)['p']
 tp=np.load(f"experiments/phase21c_repeat-{e.config_id(chosen['config'])}-{seed}.npz",allow_pickle=True)['p']
 hb=e.decisions(bp,base['rule']);ht=e.decisions(tp,chosen['rule']);ps.append((hb,ht))
 row=2*(hb*e.Y).sum(1)/(hb.sum(1)+e.Y.sum(1));baseline.append(float(row.mean()))
records=[]
for number in range(32):
 mask=((number>>np.arange(5))&1).astype(bool);weighted=[];ordinary=[]
 for hb,ht in ps:
  h=np.where(mask[None,:],ht,hb);empty=h.sum(1)==0;h[empty]=hb[empty]
  row=2*(h*e.Y).sum(1)/(h.sum(1)+e.Y.sum(1));ordinary.append(float(row.mean()));weighted.append(w@row)
 ordinary=np.array(ordinary);weighted=np.array(weighted);robust=.75*weighted.min(1)+.25*ordinary
 eligible=bool(np.all(ordinary>=np.array(baseline)-.005));score=float(robust.mean()-.2*robust.std()-.001*mask.sum())
 records.append(dict(use_tri=mask.astype(int).tolist(),ordinary_cv=float(ordinary.mean()),weighted_cv=dict(zip(names,weighted.mean(0).tolist())),selection_score=score,eligible=eligible))
e.save_json('experiments/phase21d-hybrid-screen.json',records)
best=max([r for r in records if r['eligible']],key=lambda r:r['selection_score'])
candidates=[]
if any(best['use_tri']):
 config=dict(model='tri_labelwise',base=base['config'],tri=chosen['config'],base_rule=base['rule'],tri_rule=chosen['rule'],use_tri=best['use_tri'])
 candidates.append(dict(name='classical_tri_reason_hybrid',config=config,rule='t0.5',**best))
e.save_json('experiments/phase21d-validation-frozen.json',dict(frozen_at=datetime.datetime.now(datetime.timezone.utc).isoformat(),hybrid_selection=best,candidates=candidates))
print('SELECTED',json.dumps(best),flush=True)
if candidates:
 V=pd.read_csv('data/part1_val.csv',usecols=lambda col:col not in e.LABELS)
 with threadpool_limits(limits=1):preds=[e.predict(np.arange(len(e.Y)),V,r['config']) for r in candidates]
 np.savez_compressed('experiments/phase21d-validation-predictions.npz',p=preds)
 y=pd.read_csv('data/part1_val.csv')[e.LABELS].to_numpy();results=[]
 for r,p in zip(candidates,preds):
  out={**r,'validation':e.metrics(y,p,r['rule'])};results.append(out);print('VALIDATION',json.dumps(out),flush=True)
else:results=[]
e.save_json('experiments/phase21d-validation-results.json',results)


Overwriting current_reproduction/phase21_classical_hybrid.py


In [38]:
%%writefile current_reproduction/phase21_fullsize_current.py
"""Fixed-policy leave-one-embedding-group-out audit for the new frozen hybrid."""
import phase21,experiment as e
import json,numpy as np
from pathlib import Path
from threadpoolctl import threadpool_limits
frozen=json.loads(Path('experiments/current-best.json').read_text());candidate=frozen['candidate'];c=candidate['config'];cid=e.config_id(c);groups=np.unique(e.GROUP[e.LAB]);allpos=np.arange(len(e.Y));p=np.zeros_like(e.Y,float)
with threadpool_limits(limits=1):
 for i,g in enumerate(groups):
  b=np.flatnonzero(e.GROUP[e.LAB]==g);a=allpos[~np.isin(allpos,b)]
  p[b]=e.predict(a,e.DATA.iloc[e.LAB[b]],c,excluded_groups=[g])
  if i%25==0:print('LEAVE-GROUP-OUT',i,len(groups),flush=True)
metrics=e.metrics(e.Y,p,candidate['rule']);record=dict(config=c,rule=candidate['rule'],metrics=metrics,n_groups=len(groups),interpretation='Development audit of the already selected fixed policy; not an unbiased rerun of the full search.')
np.savez_compressed(f'experiments/phase21-fullsize-{cid}.npz',p=p,record=record)
e.save_json('experiments/phase21-fullsize-current.json',record)
latest=json.loads(Path('experiments/current-best.json').read_text());assert e.config_id(latest['candidate']['config'])==cid
latest['leave_one_embedding_group_out_f1']=metrics['sample_f1'];e.save_json('experiments/current-best.json',latest);print(json.dumps(record,indent=2))


Overwriting current_reproduction/phase21_fullsize_current.py


In [39]:
%%writefile current_reproduction/phase21_hybrid.py
"""Training-only per-reason combination and an exactly matched no-teaching ablation."""
import phase21 as tri,experiment as e
import numpy as np,pandas as pd,json,datetime
from pathlib import Path
from threadpoolctl import threadpool_limits

frozen=json.loads(Path('experiments/phase21-validation-frozen.json').read_text())
chosen=next(r for r in frozen['candidates'] if r['name'].startswith('tri_ordinary_cv'))
base=json.loads(Path('experiments/phase21-prior-incumbent.json').read_text())['candidate']
weights=dict(np.load('experiments/phase21-weights.npz'));names=['propensity_logistic','propensity_tree','kmm']
w=np.array([weights[k]/weights[k].sum() for k in names]);ps=[];baseline=[]
for seed in [20261107,20261108,20261109]:
 bp=np.load(f"experiments/phase21_refit-{e.config_id(base['config'])}-{seed}.npz",allow_pickle=True)['p']
 tp=np.load(f"experiments/phase21_refit-{e.config_id(chosen['config'])}-{seed}.npz",allow_pickle=True)['p']
 hb=e.decisions(bp,base['rule']);ht=e.decisions(tp,chosen['rule']);ps.append((hb,ht))
 row=2*(hb*e.Y).sum(1)/(hb.sum(1)+e.Y.sum(1));baseline.append(float(row.mean()))
records=[]
for number in range(32):
 mask=((number>>np.arange(5))&1).astype(bool);weighted=[];ordinary=[]
 for hb,ht in ps:
  h=np.where(mask[None,:],ht,hb);empty=h.sum(1)==0;h[empty]=hb[empty]
  row=2*(h*e.Y).sum(1)/(h.sum(1)+e.Y.sum(1));ordinary.append(float(row.mean()));weighted.append(w@row)
 ordinary=np.array(ordinary);weighted=np.array(weighted)
 robust=.75*weighted.min(1)+.25*ordinary
 # Prefer the incumbent unless a stable training-only benefit survives a complexity penalty.
 eligible=bool(np.all(ordinary>=np.array(baseline)-.005))
 score=float(robust.mean()-.2*robust.std()-.001*mask.sum())
 records.append(dict(use_tri=mask.astype(int).tolist(),ordinary_cv=float(ordinary.mean()),weighted_cv=dict(zip(names,weighted.mean(0).tolist())),selection_score=score,eligible=eligible))
e.save_json('experiments/phase21b-hybrid-screen.json',records)
best=max([r for r in records if r['eligible']],key=lambda r:r['selection_score'])
candidates=[]
if any(best['use_tri']):
 config=dict(model='tri_labelwise',base=base['config'],tri=chosen['config'],base_rule=base['rule'],tri_rule=chosen['rule'],use_tri=best['use_tri'])
 candidates.append(dict(name='training_selected_reason_hybrid',config=config,rule='t0.5',**best))
# Match every setting and the frozen decoder; only cross-teaching is switched off.
control={**chosen['config'],'no_teaching':True};cv=[]
for seed in [20261107,20261108,20261109]:
 rr=e.evaluate(control,tag='phase21b_control',seed=seed)
 p=np.load(f"experiments/phase21b_control-{e.config_id(control)}-{seed}.npz",allow_pickle=True)['p'];cv.append(e.metrics(e.Y,p,chosen['rule'])['sample_f1'])
candidates.append(dict(name='exact_no_teaching_ablation',config=control,rule=chosen['rule'],ordinary_cv=float(np.mean(cv))))
e.save_json('experiments/phase21b-validation-frozen.json',dict(frozen_at=datetime.datetime.now(datetime.timezone.utc).isoformat(),purpose='A reason-wise combination selected solely from fresh training OOF predictions; an exact ablation fixes an aggregation mismatch in the first control.',hybrid_selection=best,candidates=candidates))
V=pd.read_csv('data/part1_val.csv',usecols=lambda col:col not in e.LABELS)
with threadpool_limits(limits=1):preds=[e.predict(np.arange(len(e.Y)),V,r['config']) for r in candidates]
np.savez_compressed('experiments/phase21b-validation-predictions.npz',p=preds)
y=pd.read_csv('data/part1_val.csv')[e.LABELS].to_numpy();results=[]
for r,p in zip(candidates,preds):
 out={**r,'validation':e.metrics(y,p,r['rule'])};results.append(out);print('VALIDATION',json.dumps(out),flush=True)
e.save_json('experiments/phase21b-validation-results.json',results)


Overwriting current_reproduction/phase21_hybrid.py


In [40]:
%%writefile current_reproduction/phase21_integrity.py
"""Audit the selected hybrid without tuning its configuration or decoder."""
from pathlib import Path
import hashlib,json
import numpy as np
from sklearn.metrics import f1_score
from threadpoolctl import threadpool_limits
import phase21,experiment as e

def run():
    candidate=json.loads(Path('experiments/current-best.json').read_text())['candidate']
    c=candidate['config']; rule=candidate['rule']
    train,held=e.make_splits(seed=20261107,n_splits=5)[0]
    excluded=e.GROUP[e.LAB[held]]
    query=e.DATA.iloc[e.LAB[held]].copy()
    original=e.Y.copy()
    with threadpool_limits(limits=1):
        before=e.predict(train,query,c,excluded_groups=excluded)
        try:
            e.Y[held]=1-e.Y[held]
            query.loc[:,e.LABELS]=1-query[e.LABELS].to_numpy()
            after=e.predict(train,query,c,excluded_groups=excluded)
        finally:
            e.Y[:]=original
        single=e.predict(train,query.iloc[:1],c,excluded_groups=excluded)
        with_masked,pu,ix=e.predict(train,query,c,excluded_groups=excluded,return_unlabeled=True)
    assert np.array_equal(before,after)
    assert np.array_equal(before[:1],single)
    assert np.array_equal(before,with_masked)
    assert np.array_equal(ix,e.UNLAB[~np.isin(e.GROUP[e.UNLAB],excluded)])
    assert not np.isin(e.GROUP[ix],excluded).any()
    assert np.isin(before,[0,1]).all() and (before.sum(1)>0).all()
    assert pu.shape==(len(ix),5) and np.isin(pu,[0,1]).all()
    assert (pu.sum(1)>0).all()
    weights=dict(np.load('experiments/phase21-weights.npz'))
    for w in weights.values():
        assert len(w)==len(e.Y) and np.isfinite(w).all() and (w>0).all()
        assert abs(w.mean()-1)<1e-12
    h=e.decisions(before,rule);y=e.Y[held]
    row=2*(h*y).sum(1)/(h.sum(1)+y.sum(1))
    assert abs(row.mean()-f1_score(y,h,average='samples'))<1e-12
    source=['phase21.py','phase21_weights.py','phase21_select.py','phase21_hybrid.py',
            'phase21_plain.py','phase21_classical_hybrid.py','phase21_integrity.py']
    checks=dict(selected_config=c,heldout_rows=len(held),
        selected_hybrid_holdout_label_invariance=True,
        selected_hybrid_query_batch_invariance=True,
        selected_hybrid_return_unlabeled_query_invariance=True,
        matching_masked_embedding_duplicates_excluded=True,
        all_predictions_binary_nonempty=True,
        positive_normalized_risk_weights=True,uniform_row_f1_matches_sklearn=True,
        max_label_perturbation=float(abs(before-after).max()),
        max_batch_difference=float(abs(before[:1]-single).max()),
        validation_labels_used_for_training=0,refit_with_validation_performed=False,
        source_sha256={f:hashlib.sha256(Path(f).read_bytes()).hexdigest() for f in source})
    e.save_json('experiments/phase21-integrity-checks.json',checks)
    print(json.dumps(checks,indent=2))

if __name__=='__main__':run()


Overwriting current_reproduction/phase21_integrity.py


In [41]:
%%writefile current_reproduction/phase21_plain.py
"""Classical majority-vote tri-training ablation without added cluster safeguards."""
import phase21 as tri,experiment as e
import numpy as np,pandas as pd,json,datetime
from pathlib import Path
from threadpoolctl import threadpool_limits
cs=[dict(model='tri_training',committee=group,cluster_gate=False,pseudo_weight=1.,aggregation='vote',stratified_bootstrap=False) for group in ['rbf','mixed','scales','kernels']]
e.save_json('experiments/phase21c-configs.json',cs)
rows=[]
for c in cs:
 r=e.evaluate(c,tag='phase21c');p=np.load(f"experiments/phase21c-{e.config_id(c)}-20261104.npz",allow_pickle=True)['p']
 rows.append(dict(config=c,metrics=e.metrics(e.Y,p,'t0.5')))
short={}
for key in ['sample_f1','macro_f1']:
 r=max(rows,key=lambda r:r['metrics'][key]);short[e.config_id(r['config'])]=r
summary=[]
for cid,r in short.items():
 ms=[]
 for seed in [20261107,20261108,20261109]:
  e.evaluate(r['config'],tag='phase21c_repeat',seed=seed)
  p=np.load(f'experiments/phase21c_repeat-{cid}-{seed}.npz',allow_pickle=True)['p'];ms.append(e.metrics(e.Y,p,'t0.5'))
 summary.append(dict(name='classical_tri_majority',config=r['config'],rule='t0.5',ordinary_cv=float(np.mean([m['sample_f1'] for m in ms])),cv_std=float(np.std([m['sample_f1'] for m in ms]))))
e.save_json('experiments/phase21c-selection.json',summary)
c=max(summary,key=lambda r:r['ordinary_cv']-.2*r['cv_std'])
e.save_json('experiments/phase21c-validation-frozen.json',dict(frozen_at=datetime.datetime.now(datetime.timezone.utc).isoformat(),purpose='Check whether conservative modifications suppressed classical tri-training: full-weight pseudo-labels, ordinary bootstrap, no cluster gate, and fixed majority decisions.',candidates=[c]))
V=pd.read_csv('data/part1_val.csv',usecols=lambda col:col not in e.LABELS)
with threadpool_limits(limits=1):p=e.predict(np.arange(len(e.Y)),V,c['config'])
np.savez_compressed('experiments/phase21c-validation-predictions.npz',p=p)
y=pd.read_csv('data/part1_val.csv')[e.LABELS].to_numpy();r={**c,'validation':e.metrics(y,p,c['rule'])}
e.save_json('experiments/phase21c-validation-results.json',[r]);print('VALIDATION',json.dumps(r),flush=True)


Overwriting current_reproduction/phase21_plain.py


In [42]:
%%writefile current_reproduction/phase21_select.py
"""Target-aware selection over existing and tri-training candidates, then a frozen audit."""
import phase21 as tri,experiment as e
from pathlib import Path
import json,datetime,hashlib
import numpy as np,pandas as pd
from threadpoolctl import threadpool_limits

RULES=['t0.25','t0.35','t0.45','t0.5','t0.55','t0.65','t0.75','expected_f1','argmax']
W=dict(np.load('experiments/phase21-weights.npz'))
NAMES=list(W);weight_matrix=np.array([W[k]/W[k].sum() for k in NAMES])
PRIMARY=NAMES.index('propensity_logistic');SHIFT=[NAMES.index(k) for k in ['propensity_logistic','propensity_tree','kmm']]
UNIFORM=NAMES.index('uniform')


def table(ps):
    # axes: rule, repeated split, weighting scheme
    out=[]
    for rule in RULES:
        rr=[]
        for p in ps:
            h=e.decisions(p,rule)
            row=2*(h*e.Y).sum(1)/(h.sum(1)+e.Y.sum(1))
            rr.append(weight_matrix@row)
        out.append(rr)
    return np.array(out)


def choice(t,objective):
    means=t.mean(1)
    if objective=='ordinary':values=t[:,:,UNIFORM]
    elif objective=='iw':values=t[:,:,PRIMARY]
    else:values=.75*t[:,:,SHIFT].min(2)+.25*t[:,:,UNIFORM]
    score=values.mean(1)-.2*values.std(1)
    best=max(range(len(RULES)),key=lambda j:(score[j],-abs(float(RULES[j][1:])-.5) if RULES[j].startswith('t') else -1))
    return dict(rule=RULES[best],selection_score=float(score[best]),ordinary_cv=float(means[best,UNIFORM]),
                weighted_cv={k:float(means[best,i]) for i,k in enumerate(NAMES)},
                split_score_std=float(values[best].std()))


def family(c):
    if c['model'] in ['lowdensity_svm','lowdensity_v2','lowdensity_v3','kernel_lowdensity']:return 'lowdensity_svm'
    return c['model']


def run():
    rows=[json.loads(s) for s in Path('experiments/trials.jsonl').read_text().splitlines()]
    first={}
    for r in rows:
        if r['status']=='ok' and r.get('seed')==20261104:first[r['id']]=r
    screen=[]
    with threadpool_limits(limits=1):
        for cid,r in first.items():
            path=Path(f"experiments/{r['tag']}-{cid}-20261104.npz")
            if not path.exists():continue
            z=np.load(path,allow_pickle=True);p=z['p']
            if p.shape!=(157,5) or not np.isfinite(p).all() or len(np.unique(z['fold']))!=5:continue
            t=table([p]);screen.append(dict(id=cid,config=r['config'],family=family(r['config']),
                                           **{objective:choice(t,objective) for objective in ['ordinary','iw','robust']}))
    e.save_json('experiments/phase21-risk-screen.json',screen)
    selected={}
    old=[r for r in screen if r['config']['model']!='tri_training']
    training=[r for r in screen if r['config']['model']=='tri_training' and not r['config'].get('no_teaching')]
    for objective in ['iw','robust']:
        seen=set()
        for r in sorted(old,key=lambda r:r[objective]['selection_score'],reverse=True):
            if r['family'] in seen:continue
            selected[r['id']]=r;seen.add(r['family'])
            if len(seen)==3:break
    for objective in ['ordinary','iw','robust']:
        r=max(training,key=lambda r:r[objective]['selection_score']);selected[r['id']]=r
    # Retain one guarded teacher variant independently of in-sample peer error scores.
    r=max([r for r in training if r['config'].get('guard')=='crossfit'],key=lambda r:r['iw']['selection_score']);selected[r['id']]=r
    incumbent=json.loads(Path('experiments/phase21-prior-incumbent.json').read_text())['candidate']['config']
    cid=e.config_id(incumbent);selected[cid]=dict(id=cid,config=incumbent,family=family(incumbent))
    # Matched bootstrap-only controls make the effect of cross-teaching assessable.
    committees={r['config']['committee'] for r in selected.values() if r['config']['model']=='tri_training'}
    for r in screen:
        if r['config']['model']=='tri_training' and r['config'].get('no_teaching') and r['config']['committee'] in committees:selected[r['id']]=r
    e.save_json('experiments/phase21-refit-shortlist.json',dict(frozen_at=datetime.datetime.now(datetime.timezone.utc).isoformat(),weight_sha256=hashlib.sha256(Path('experiments/phase21-weights.npz').read_bytes()).hexdigest(),fresh_seeds=[20261107,20261108,20261109],candidates=list(selected.values())))
    summary=[]
    with threadpool_limits(limits=1):
        for i,r in enumerate(selected.values()):
            print('REFIT',i+1,len(selected),r['config'],flush=True)
            ps=[]
            for seed in [20261107,20261108,20261109]:
                record=e.evaluate(r['config'],seed=seed,tag='phase21_refit')
                if record['status']!='ok':raise RuntimeError(record)
                ps.append(np.load(f"experiments/phase21_refit-{r['id']}-{seed}.npz",allow_pickle=True)['p'])
            t=table(ps);summary.append(dict(id=r['id'],config=r['config'],**{objective:choice(t,objective) for objective in ['ordinary','iw','robust']}))
            print('SUMMARY',json.dumps(summary[-1]),flush=True)
    e.save_json('experiments/phase21-selection.json',summary)
    candidates=[]
    def add(r,objective,purpose):
        candidate=dict(name=purpose,config=r['config'],objective=objective,**r[objective])
        key=(e.config_id(candidate['config']),candidate['rule'])
        for existing in candidates:
            if (e.config_id(existing['config']),existing['rule'])==key:
                existing['name']+=' + '+purpose;return
        candidates.append(candidate)
    teaching=[r for r in summary if r['config']['model']=='tri_training' and not r['config'].get('no_teaching')]
    ordinary=max(teaching,key=lambda r:r['ordinary']['selection_score']);add(ordinary,'ordinary','tri_ordinary_cv')
    add(max(teaching,key=lambda r:r['iw']['selection_score']),'iw','tri_target_weighted_cv')
    old=[r for r in summary if r['config']['model']!='tri_training']
    add(max(old,key=lambda r:r['iw']['selection_score']),'iw','existing_target_weighted_cv')
    add(max(summary,key=lambda r:r['robust']['selection_score']),'robust','robust_weighting_cv')
    control=next(r for r in summary if r['config'].get('no_teaching') and r['config']['committee']==ordinary['config']['committee'])
    add(control,'ordinary','matched_no_teaching_control')
    assert len(candidates)<=5
    e.save_json('experiments/phase21-validation-frozen.json',dict(frozen_at=datetime.datetime.now(datetime.timezone.utc).isoformat(),weighting_file='phase21-weighting.json',candidates=candidates))
    V=pd.read_csv('data/part1_val.csv',usecols=lambda col:col not in e.LABELS);preds=[]
    with threadpool_limits(limits=1):
        for candidate in candidates:
            preds.append(e.predict(np.arange(len(e.Y)),V,candidate['config']))
            if candidate['config']['model']=='tri_training':
                e.save_json('experiments/phase21-fullfit-diagnostics-'+e.config_id(candidate['config'])+'.json',tri.LAST_DIAGNOSTICS)
    np.savez_compressed('experiments/phase21-validation-predictions.npz',p=preds)
    # Validation labels are first read only after predictions and decisions are frozen.
    y=pd.read_csv('data/part1_val.csv')[e.LABELS].to_numpy();results=[]
    for candidate,p in zip(candidates,preds):
        r={**candidate,'validation':e.metrics(y,p,candidate['rule'])};results.append(r);print('VALIDATION',json.dumps(r),flush=True)
    e.save_json('experiments/phase21-validation-results.json',results)

if __name__=='__main__':run()


Overwriting current_reproduction/phase21_select.py


In [43]:
%%writefile current_reproduction/phase21_weights.py
"""Cross-fitted selection-propensity weights and KMM for model-selection risk.
No reason-label values, validation rows, or test rows are used to estimate weights.
"""
import experiment as e
import numpy as np,json,datetime
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score,log_loss
from sklearn.metrics.pairwise import rbf_kernel
from scipy.optimize import minimize
from threadpoolctl import threadpool_limits


def stabilized(w):
    w=np.clip(w,.2,5.)
    w=w/w.mean()
    return .5+.5*w


def run():
    X=e.DATA[e.EMB].to_numpy();year=e.DATA.start_year.to_numpy()[:,None]
    selected=np.zeros(len(X),int);selected[e.LAB]=1
    rate=selected.mean();weights={'uniform':np.ones(len(e.LAB))};diagnostics={}
    splits=list(StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=210719).split(X,selected,e.GROUP))
    with threadpool_limits(limits=1):
        for kind in ['logistic','tree']:
            p=np.zeros(len(X))
            for a,b in splits:
                sc=StandardScaler().fit(X[a]);pc=PCA(n_components=20,svd_solver='full').fit(sc.transform(X[a]))
                za=np.column_stack([pc.transform(sc.transform(X[a])),year[a]])
                zb=np.column_stack([pc.transform(sc.transform(X[b])),year[b]])
                scale=StandardScaler().fit(za);za=scale.transform(za);zb=scale.transform(zb)
                if kind=='logistic':model=LogisticRegression(C=.1,max_iter=1500)
                else:model=HistGradientBoostingClassifier(max_iter=120,max_leaf_nodes=7,min_samples_leaf=30,learning_rate=.04,l2_regularization=10,random_state=42)
                model.fit(za,selected[a]);p[b]=model.predict_proba(zb)[:,1]
            # P(X) / P(X | selected) = P(selected) / P(selected | X).
            raw=rate/np.maximum(p[e.LAB],.02)
            weights['propensity_'+kind]=stabilized(raw)
            diagnostics[kind]=dict(domain_oof_auc=float(roc_auc_score(selected,p)),domain_oof_logloss=float(log_loss(selected,p)),gold_propensity_quantiles=np.quantile(p[e.LAB],[0,.1,.5,.9,1]).tolist(),raw_weight_quantiles=np.quantile(raw,[0,.1,.5,.9,1]).tolist(),clipped_fraction=float(np.mean((raw<.2)|(raw>5))))
        # Match the full observed training population, not external validation/test.
        A=StandardScaler().fit_transform(X);n=len(e.LAB)
        K=rbf_kernel(A[e.LAB],gamma=.003);kappa=n*rbf_kernel(A[e.LAB],A,gamma=.003).mean(1)
        ridge=.1
        result=minimize(lambda w:.5*w@K@w-kappa@w+.5*ridge*np.sum((w-1)**2),np.ones(n),jac=lambda w:K@w-kappa+ridge*(w-1),method='SLSQP',bounds=[(.2,5)]*n,constraints=[dict(type='eq',fun=lambda w:w.sum()-n,jac=lambda w:np.ones(n))],options={'maxiter':400,'ftol':1e-9})
        if not result.success:raise RuntimeError(result.message)
        weights['kmm']=stabilized(result.x)
        diagnostics['kmm']=dict(gamma=.003,ridge=ridge,optimizer_success=bool(result.success),iterations=int(result.nit))
    for name,w in weights.items():
        diagnostics.setdefault(name,{})
        diagnostics[name].update(effective_sample_size=float(w.sum()**2/(w@w)),minimum=float(w.min()),maximum=float(w.max()),mean=float(w.mean()))
    metadata=dict(created_at=datetime.datetime.now(datetime.timezone.utc).isoformat(),source='157 gold-status training rows',target='all 1171 training rows; only features and gold/masked status',validation_used=False,test_used=False,reason_labels_used=False,clipping=[.2,5],shrinkage_to_uniform=.5,diagnostics=diagnostics,limitations='Estimated covariate-shift weights are approximate. Label-dependent sampling and missing support can violate the necessary assumptions. These are development-risk weights, not guaranteed unbiased target risks.')
    e.save_json('experiments/phase21-weighting.json',metadata)
    np.savez_compressed('experiments/phase21-weights.npz',**weights)
    frame=e.pd.DataFrame({'ID':e.DATA.iloc[e.LAB].ID.to_numpy(),**weights});frame.to_csv('experiments/phase21-weights.csv',index=False)
    print(json.dumps(metadata,indent=2),flush=True)

if __name__=='__main__':run()


Overwriting current_reproduction/phase21_weights.py


In [44]:
%%writefile current_reproduction/phase22.py
"""Phase 22 refinement copied explicitly from the verified phase21 core.
Defaults reproduce phase21; phase21 itself remains unchanged.
Binary-relevance tri-training with classical learners and fold-local guards.
Adapted from Zhou & Li (2005). Weighted pseudo-labels, cluster checks, stratified
bootstraps, confidence gates, and cross-fitted error floors are explicit variants;
we do not assert the original theoretical guarantees for these modifications.
"""
import phase21, experiment as e
import json
import numpy as np
from scipy.special import expit
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import rbf_kernel, polynomial_kernel

legacy_predict = e.predict
GUARD_CACHE = {}
LAST_DIAGNOSTICS = []


def member_specs(c):
    C = c.get('C', 1)
    base = dict(kind='rbf', C=C, gamma=c.get('gamma', .001))
    group = c.get('committee', 'rbf')
    if group == 'mixed':
        return [base, dict(kind='logistic', C=c.get('logistic_C', .3)), dict(kind='ridge', alpha=c.get('ridge_alpha', 10))]
    if group == 'scales':
        return [dict(kind='rbf', C=C, gamma=g) for g in [.0006, .001, .002]]
    if group == 'kernels':
        return [base, dict(kind='poly', C=C, gamma=.003), dict(kind='linear', C=C)]
    return [base.copy() for _ in range(3)]


def bootstrap(y, rng, stratified=True):
    # Preserve minority presence; bootstrap rows independently within each class.
    if not stratified:
        return rng.choice(len(y), size=len(y), replace=True)
    return np.concatenate([rng.choice(np.flatnonzero(y == v), (y == v).sum(), replace=True)
                           for v in np.unique(y)])


def update_size(error, previous_error, previous_size, available):
    """Return allowable pseudo-label count under the original error/size guards."""
    if not error < previous_error or available == 0:
        return 0, previous_size
    if previous_size == 0:
        previous_size = int(np.floor(error / (previous_error - error) + 1))
    if not previous_size < available:
        return 0, previous_size
    if error * available < previous_error * previous_size:
        return available, previous_size
    if error > 0 and previous_size > error / (previous_error - error):
        size = int(np.ceil(previous_error * previous_size / error - 1))
        if previous_size < size <= available:
            return size, previous_size
    return 0, previous_size


def predict_core(train_pos, query_frame, c, excluded_groups=None, return_unlabeled=False):
    global LAST_DIAGNOSTICS
    if c['model'] != 'tri_refined':
        return legacy_predict(train_pos, query_frame, c, excluded_groups, return_unlabeled)
    pos = np.asarray(train_pos)
    n = len(pos)
    u = e.UNLAB if excluded_groups is None else e.UNLAB[~np.isin(e.GROUP[e.UNLAB], excluded_groups)]
    A, B = e.features(np.r_[e.LAB[pos], u], query_frame, c)
    Y = e.Y[pos]
    Q = np.r_[B, A[n:]] if return_unlabeled else B
    specs = member_specs(c)
    matrices = []
    for spec in specs:
        kind = spec['kind']
        if kind == 'rbf':
            matrices.append((rbf_kernel(A, gamma=spec['gamma']), rbf_kernel(Q, A, gamma=spec['gamma'])))
        elif kind == 'poly':
            matrices.append((polynomial_kernel(A, gamma=spec['gamma'], degree=2, coef0=1),
                             polynomial_kernel(Q, A, gamma=spec['gamma'], degree=2, coef0=1)))
        elif kind == 'linear':
            matrices.append((A @ A.T / A.shape[1], Q @ A.T / A.shape[1]))
        else:
            matrices.append(None)

    def fit(k, rows, yy, weights):
        spec = specs[k]
        if len(np.unique(yy)) < 2:
            return (float(yy[0]), rows)
        if matrices[k] is not None:
            model = SVC(C=spec['C'], kernel='precomputed', class_weight=c.get('class_weight', 'balanced'))
            model.fit(matrices[k][0][np.ix_(rows, rows)], yy, sample_weight=np.ascontiguousarray(weights))
        else:
            if spec['kind'] == 'logistic':
                model = LogisticRegression(C=spec['C'], class_weight=c.get('class_weight', 'balanced'), max_iter=1500)
            else:
                model = RidgeClassifier(alpha=spec['alpha'], class_weight=c.get('class_weight', 'balanced'))
            model.fit(A[rows], yy, sample_weight=np.ascontiguousarray(weights))
        return model, rows

    def probability(k, fitted, rows=None, query=False):
        model, used = fitted
        size = len(Q) if query else len(rows)
        if isinstance(model, float):
            return np.full(size, model)
        if matrices[k] is not None:
            X = matrices[k][1][:, used] if query else matrices[k][0][np.ix_(rows, used)]
        else:
            X = Q if query else A[rows]
        temperature = 1 if specs[k]['kind'] == 'logistic' else 2
        return expit(temperature * model.decision_function(X))

    cluster = None
    if c.get('cluster_gate', True):
        z = KMeans(n_clusters=10, n_init=10, random_state=42).fit_predict(A)
        cluster = np.full((len(u), 5), .5)
        for group in np.unique(z):
            gold = z[:n] == group
            if gold.sum() >= 2:
                cluster[z[n:] == group] = Y[gold].mean(0)

    guard_predictions = None
    if c.get('guard', 'textbook') == 'crossfit':
        geometry = {key: c.get(key) for key in ['scale', 'pca', 'white', 'norm', 'features', 'feature_weight']}
        key = (tuple(pos), tuple(u), Y.tobytes(), json.dumps(specs, sort_keys=True), json.dumps(geometry, sort_keys=True), c.get('stratified_bootstrap', True))
        if key not in GUARD_CACHE:
            gp = np.zeros((3, n, 5))
            for fi, (a, b) in enumerate(e.make_splits(Y, groups=e.GROUP[e.LAB[pos]], seed=210017, n_splits=3)):
                for j in range(5):
                    for k in range(3):
                        rng = np.random.default_rng(210000 + 100 * j + 10 * k + fi)
                        rows = a[bootstrap(Y[a, j], rng, c.get('stratified_bootstrap', True))]
                        model = fit(k, rows, Y[rows, j], np.ones(len(rows)))
                        gp[k, b, j] = probability(k, model, rows=b)
            GUARD_CACHE[key] = gp
        guard_predictions = GUARD_CACHE[key]

    outputs = []
    diagnostics = []
    unlabeled_rows = np.arange(n, len(A))
    all_gold = np.arange(n)
    for j in range(5):
        y = Y[:, j]
        if len(np.unique(y)) < 2:
            outputs.append(np.full(len(Q), y[0]))
            continue
        rngs = [np.random.default_rng(c.get('bootstrap_seed', 210123) + 100*j + k) for k in range(3)]
        models = []
        for k in range(3):
            rows = bootstrap(y, rngs[k], c.get('stratified_bootstrap', True))
            models.append(fit(k, rows, y[rows], np.ones(n)))
        previous_errors = np.full(3, .5)
        previous_sizes = np.zeros(3, int)
        for iteration in range(c.get('rounds', 8)):
            if c.get('no_teaching', False):
                break
            goldp = np.array([probability(k, models[k], rows=all_gold) for k in range(3)])
            up = np.array([probability(k, models[k], rows=unlabeled_rows) for k in range(3)])
            planned = []
            for k in range(3):
                peers = [v for v in range(3) if v != k]
                g0, g1 = goldp[peers]
                agree = (g0 >= .5) == (g1 >= .5)
                error = float(np.mean((g0[agree] >= .5) != y[agree])) if agree.any() else .5
                if guard_predictions is not None:
                    p0, p1 = guard_predictions[peers, :, j]
                    ok = (p0 >= .5) == (p1 >= .5)
                    # A fixed OOF floor prevents overly confident in-sample error estimates.
                    floor = (np.sum((p0[ok] >= .5) != y[ok]) + 1) / (ok.sum() + 2)
                    error = max(error, floor)
                p0, p1 = up[peers]
                match = (p0 >= .5) == (p1 >= .5)
                confidence = np.minimum(np.maximum(p0, 1-p0), np.maximum(p1, 1-p1))
                pool = np.flatnonzero(match & (confidence >= c.get('confidence', .5)))
                count, previous_sizes[k] = update_size(error, previous_errors[k], previous_sizes[k], len(pool))
                if count == 0:
                    continue
                selected = rngs[k].choice(pool, size=count, replace=False) if count < len(pool) else pool
                pseudo = (p0[selected] >= .5).astype(int)
                weights = np.full(count, c.get('pseudo_weight', .1))
                if cluster is not None:
                    cp = cluster[selected, j]
                    weights[((cp >= .5) != pseudo) & (np.maximum(cp, 1-cp) >= .8)] *= .1
                if c.get('cap_mass'):
                    for value in [0, 1]:
                        mask = pseudo == value
                        cap = c.get('mass_fraction', .5) * np.sum(y == value)
                        if weights[mask].sum() > cap:
                            weights[mask] *= cap / weights[mask].sum()
                planned.append((k, selected, pseudo, weights, error))
            if not planned:
                break
            # Simultaneous updates: no learner sees another's update from this round.
            for k, selected, pseudo, weights, error in planned:
                models[k] = fit(k, np.r_[all_gold, n+selected], np.r_[y, pseudo], np.r_[np.ones(n), weights])
                previous_errors[k] = error
                previous_sizes[k] = len(selected)
                diagnostics.append(dict(label=e.LABELS[j],round=iteration,learner=k,error=error,
                                        pseudo_count=len(selected),positive=int(pseudo.sum()),
                                        effective_weight=float(weights.sum())))
        qp = np.array([probability(k, models[k], query=True) for k in range(3)])
        if c.get('aggregation', 'mean') == 'vote':
            # Preserve majority decisions at 0.5 and use scores to break empty-output ties.
            p = .8 * (qp >= .5).mean(0) + .2 * qp.mean(0)
        else:
            p = qp.mean(0)
        outputs.append(p)
    LAST_DIAGNOSTICS = diagnostics
    p = np.array(outputs).T
    if return_unlabeled:
        return p[:len(B)], p[len(B):], u
    return p


# Cache predictions, never fitted query-dependent transforms. Label bytes in the
# key prevent stale caches during explicit training-label perturbation checks.
PREDICTION_CACHE = {}

def cached(train_pos, query_frame, c, excluded_groups=None, return_unlabeled=False):
    from pandas import util
    pos=np.asarray(train_pos)
    columns=[x for x in query_frame.columns if x not in e.LABELS]
    query_hash=util.hash_pandas_object(query_frame[columns],index=False).to_numpy().tobytes()
    excluded=tuple(sorted([] if excluded_groups is None else excluded_groups))
    key=(tuple(pos),e.Y[pos].tobytes(),query_hash,e.config_id(c),excluded,return_unlabeled)
    if key not in PREDICTION_CACHE:
        PREDICTION_CACHE[key]=predict22(pos,query_frame,c,excluded_groups,return_unlabeled)
    return PREDICTION_CACHE[key]

def predict22(train_pos, query_frame, c, excluded_groups=None, return_unlabeled=False):
    kind=c['model']
    if kind=='tri_refined':
        return predict_core(train_pos,query_frame,c,excluded_groups,return_unlabeled)
    if kind=='tri_seed_ensemble':
        members=[cached(train_pos,query_frame,dict(c['member'],bootstrap_seed=seed),excluded_groups,return_unlabeled)
                 for seed in c['seeds']]
        if return_unlabeled:
            assert all(np.array_equal(m[2],members[0][2]) for m in members)
            return np.mean([m[0] for m in members],0),np.mean([m[1] for m in members],0),members[0][2]
        return np.mean(members,0)
    if kind=='hybrid22':
        base=cached(train_pos,query_frame,c['base'],excluded_groups,return_unlabeled)
        tri=cached(train_pos,query_frame,c['tri'],excluded_groups,return_unlabeled)
        def combine(bp,tp):
            if c.get('combination','hard')=='soft':
                # Align each component's frozen decision boundary before averaging.
                from scipy.special import logit
                b=expit(logit(np.clip(bp,1e-7,1-1e-7))-logit(float(c['base_rule'][1:])))
                t=expit(logit(np.clip(tp,1e-7,1-1e-7))-logit(float(c['tri_rule'][1:])))
                return (1-c['tri_weight'])*b+c['tri_weight']*t
            hb=e.decisions(bp,c['base_rule']);ht=e.decisions(tp,c['tri_rule'])
            out=np.where(np.asarray(c['use_tri'],bool)[None,:],ht,hb)
            empty=out.sum(1)==0;out[empty]=hb[empty]
            return out.astype(float)
        if return_unlabeled:
            assert np.array_equal(base[2],tri[2])
            return combine(base[0],tri[0]),combine(base[1],tri[1]),base[2]
        return combine(base,tri)
    return legacy_predict(train_pos,query_frame,c,excluded_groups,return_unlabeled)

e.predict=predict22



Overwriting current_reproduction/phase22.py


In [45]:
%%writefile current_reproduction/phase22_run.py
"""Prespecified refinements, grouped-CV selection, then one small validation batch."""
from pathlib import Path
import copy,json,datetime
import numpy as np
from threadpoolctl import threadpool_limits
import phase22 as m,experiment as e

E=Path('experiments')
frozen=json.loads((E/'phase22-baseline.json').read_text())
inc=frozen['candidate']['config'];base=inc['base'];original=inc['tri']
refined={**original,'model':'tri_refined'}
seeds=[210123,220123,230123,240123,250123]
W=dict(np.load(E/'phase21-weights.npz'))
w=np.array([W[n]/W[n].sum() for n in ['propensity_logistic','propensity_tree','kmm']])

def hybrid(tri,**extra):
    return dict(model='hybrid22',base=base,tri=tri,base_rule=inc['base_rule'],
                tri_rule=inc['tri_rule'],use_tri=inc['use_tri'],**extra)

def ensemble(count,**extra):
    return dict(model='tri_seed_ensemble',member={**refined,**extra},seeds=seeds[:count])

def configurations():
    out=[]
    def add(bucket,c):out.append(dict(bucket=bucket,config=c))
    add('control',hybrid(refined))
    for count in [3,5]:add('bootstrap',hybrid(ensemble(count)))
    for key,values in [('C',[.3,3]),('gamma',[.0005,.002]),
                       ('logistic_C',[.1,1]),('ridge_alpha',[3,30]),
                       ('pseudo_weight',[.1,.3]),('confidence',[.6,.75])]:
        for v in values:add('parameters',hybrid({**refined,key:v}))
    for scale in ['global','raw']:add('parameters',hybrid({**refined,'scale':scale}))
    for n in [50,100]:add('parameters',hybrid({**refined,'pca':n}))
    for block in ['criteria','criteria_char','diseases_cleaned','drugs','icdcodes',
                  'drug_targets','drug_actions','drug_pathways','fingerprint','descriptors','year','counts']:
        for weight in [.15,.4]:add('features',hybrid({**refined,'features':[block],'feature_weight':weight}))
    for tri in [refined,ensemble(5)]:
        for weight in [.25,.5,.75]:add('soft',hybrid(tri,combination='soft',tri_weight=weight))
    return out

def scores(p):
    h=e.decisions(p,'t0.5');row=2*(h*e.Y).sum(1)/(h.sum(1)+e.Y.sum(1))
    weighted=w@row;ordinary=float(row.mean())
    return dict(ordinary=ordinary,robust=float(.75*weighted.min()+.25*ordinary),
                weighted=weighted.tolist(),macro=e.metrics(e.Y,p,'t0.5')['macro_f1'])

def evaluate(r,seed,tag):
    result=e.evaluate(r['config'],seed=seed,tag=tag)
    if result['status']!='ok':return None
    p=np.load(E/f"{tag}-{e.config_id(r['config'])}-{seed}.npz",allow_pickle=True)['p']
    return scores(p)

def run():
    cs=configurations();e.save_json(E/'phase22-configs.json',cs)
    # Default-core equivalence is required before relying on the copied core.
    a,b=e.make_splits(seed=20261110)[0];q=e.DATA.iloc[e.LAB[b]];excluded=e.GROUP[e.LAB[b]]
    with threadpool_limits(limits=1):
        old=m.legacy_predict(a,q,original,excluded)
        new=m.predict_core(a,q,refined,excluded)
    assert np.allclose(old,new,atol=1e-12,rtol=0)
    e.save_json(E/'phase22-core-equivalence.json',dict(max_difference=float(abs(old-new).max()),passed=True))
    screen=[]
    for i,r in enumerate(cs):
        print('SCREEN',i+1,len(cs),r['bucket'],r['config']['tri'],flush=True)
        s=evaluate(r,20261110,'phase22')
        if s is not None:screen.append({**r,**s})
        e.save_json(E/'phase22-screen.json',screen)
    shortlist={}
    for bucket in ['bootstrap','parameters','features','soft']:
        available=[r for r in screen if r['bucket']==bucket]
        if not available:continue
        best=max(available,key=lambda r:(r['robust'],r['ordinary']))
        shortlist[e.config_id(best['config'])]=best
    selected=list(shortlist.values())
    e.save_json(E/'phase22-shortlist-frozen.json',dict(frozen_at=datetime.datetime.now(datetime.timezone.utc).isoformat(),candidates=selected,repeat_seeds=[20261111,20261112,20261113]))
    repeats=[]
    for r in [cs[0]]+selected:
        ss=[]
        for seed in [20261111,20261112,20261113]:
            print('REPEAT',r['bucket'],seed,flush=True)
            s=evaluate(r,seed,'phase22_repeat')
            if s is None:break
            ss.append(s)
        if len(ss)<3:continue
        out={**r,'ordinary_cv':float(np.mean([s['ordinary'] for s in ss])),
             'robust_cv':float(np.mean([s['robust'] for s in ss])),
             'cv_std':float(np.std([s['ordinary'] for s in ss])),
             'robust_std':float(np.std([s['robust'] for s in ss])),
             'per_repeat':ss,'rule':'t0.5'}
        out['selection_score']=out['robust_cv']-.2*out['robust_std']
        repeats.append(out);e.save_json(E/'phase22-repeat-results.json',repeats)
    baseline=next(r for r in repeats if r['bucket']=='control')
    eligible=[r for r in repeats if r['bucket']!='control' and r['ordinary_cv']>=baseline['ordinary_cv']-.003 and r['robust_cv']>=baseline['robust_cv']-.003]
    chosen={}
    if eligible:
        for metric in ['selection_score','ordinary_cv']:
            r=max(eligible,key=lambda r:r[metric]);chosen[e.config_id(r['config'])]=r
        feature=[r for r in eligible if r['bucket']=='features']
        if feature:chosen[e.config_id(feature[0]['config'])]=feature[0]
    candidates=[{**r,'name':'phase22_'+r['bucket']} for r in chosen.values()]
    assert len(candidates)<=3
    e.save_json(E/'phase22-validation-frozen.json',dict(frozen_at=datetime.datetime.now(datetime.timezone.utc).isoformat(),baseline_cv=baseline,candidates=candidates))
    results=[]
    if candidates:
        V=e.pd.read_csv('data/part1_val.csv',usecols=lambda col:col not in e.LABELS)
        with threadpool_limits(limits=1):
            predictions=[e.predict(np.arange(len(e.Y)),V,r['config']) for r in candidates]
        np.savez_compressed(E/'phase22-validation-predictions.npz',p=predictions)
        y=e.pd.read_csv('data/part1_val.csv')[e.LABELS].to_numpy()
        for r,p in zip(candidates,predictions):
            out={**r,'validation':e.metrics(y,p,r['rule'])};results.append(out)
            print('VALIDATION',json.dumps(out),flush=True)
    e.save_json(E/'phase22-validation-results.json',results)
    print('COMPLETE',len(results),'frozen validation candidates',flush=True)

if __name__=='__main__':run()


Overwriting current_reproduction/phase22_run.py


In [46]:
%%writefile current_reproduction/phase23.py
"""Nested out-of-fold score specialists and a joint sample-F1 decoder.
Only original training labels are used. Each outer fold owns its inner fits.
"""
from pathlib import Path
import json,hashlib
import numpy as np
from scipy.special import logit,expit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler,PolynomialFeatures
import phase22 as p22,experiment as e

legacy_predict=e.predict
INNER_CACHE={}

def component_scores(pos,query,c,excluded,return_unlabeled=False):
    b=p22.cached(pos,query,c['base'],excluded,return_unlabeled)
    t=p22.cached(pos,query,c['tri'],excluded,return_unlabeled)
    if return_unlabeled:
        assert np.array_equal(b[2],t[2])
        return np.c_[b[0],t[0]],np.c_[b[1],t[1]],b[2]
    return np.c_[b,t]

def meta_predict(p,y,q,c):
    X=logit(np.clip(p,1e-5,1-1e-5));Q=logit(np.clip(q,1e-5,1-1e-5))
    sc=StandardScaler();X=sc.fit_transform(X);Q=sc.transform(Q)
    if c.get('interactions'):
        poly=PolynomialFeatures(2,interaction_only=True,include_bias=False)
        X=poly.fit_transform(X);Q=poly.transform(Q)
    kind=c['meta'];C=c.get('meta_C',.3)
    if kind=='patterns':
        patterns,target=np.unique(y,axis=0,return_inverse=True)
        if len(patterns)==1:return np.tile(patterns[0],(len(q),1)).astype(float)
        model=LogisticRegression(C=C,max_iter=2000).fit(X,target)
        pp=model.predict_proba(Q);patterns=patterns[model.classes_]
        choices=((np.arange(1,32)[:,None]>>np.arange(5))&1)
        utility=2*(choices@patterns.T)/(choices.sum(1)[:,None]+patterns.sum(1)[None,:])
        return choices[(pp@utility.T).argmax(1)].astype(float)
    # Preserve the previous hybrid's original label decisions outside the specialist.
    bp=q[:,:5];tp=q[:,5:]
    hb=e.decisions(bp,c['base_rule']);ht=e.decisions(tp,c['tri_rule'])
    h=np.where(np.array(c['use_tri'],bool)[None,:],ht,hb)
    empty=h.sum(1)==0;h[empty]=hb[empty]
    fallback=np.where(np.array(c['use_tri'],bool)[None,:],tp,
                      expit(logit(np.clip(bp,1e-7,1-1e-7))-logit(float(c['base_rule'][1:]))))
    indices={'other':[4],'pair':[3,4],'all':list(range(5))}[kind]
    for j in indices:
        if len(np.unique(y[:,j]))==1:prob=np.full(len(q),y[0,j])
        else:
            model=LogisticRegression(C=C,max_iter=2000).fit(X,y[:,j])
            prob=model.predict_proba(Q)[:,1]
        strength=c.get('strength',1.)
        prob=strength*prob+(1-strength)*fallback[:,j]
        h[:,j]=(prob>=.5);fallback[:,j]=prob
    empty=h.sum(1)==0;h[np.flatnonzero(empty),fallback[empty].argmax(1)]=1
    return h.astype(float)

def predict23(train_pos,query_frame,c,excluded_groups=None,return_unlabeled=False):
    if c['model']!='joint_specialist':
        return legacy_predict(train_pos,query_frame,c,excluded_groups,return_unlabeled)
    pos=np.asarray(train_pos);excluded=np.asarray([] if excluded_groups is None else excluded_groups)
    inner=c.get('inner_splits',15)
    components={k:c[k] for k in ['base','tri']}
    key=(tuple(pos),e.Y[pos].tobytes(),tuple(sorted(excluded)),e.config_id(components),inner)
    if key not in INNER_CACHE:
        # No validation/test inputs are needed to construct these training features.
        p=np.zeros((len(pos),10))
        for a,b in e.make_splits(e.Y[pos],groups=e.GROUP[e.LAB[pos]],seed=230117,n_splits=inner):
            exclusions=np.r_[excluded,e.GROUP[e.LAB[pos[b]]]]
            p[b]=component_scores(pos[a],e.DATA.iloc[e.LAB[pos[b]]],c,exclusions)
        INNER_CACHE[key]=p
    p=INNER_CACHE[key]
    q=component_scores(pos,query_frame,c,excluded_groups,return_unlabeled)
    if return_unlabeled:
        query,unlabeled,ix=q
        out=meta_predict(p,e.Y[pos],np.r_[query,unlabeled],c)
        return out[:len(query)],out[len(query):],ix
    return meta_predict(p,e.Y[pos],q,c)

e.predict=predict23


Overwriting current_reproduction/phase23.py


In [47]:
%%writefile current_reproduction/phase23_integrity.py
"""Uncached perturbation check of nested specialist fitting and query isolation."""
from pathlib import Path
import json,hashlib
import numpy as np
from threadpoolctl import threadpool_limits
import phase23 as m,phase22 as p22,experiment as e

def run():
    c=json.loads(Path('experiments/phase22-baseline.json').read_text())['candidate']['config']
    c={**c,'model':'joint_specialist','meta':'other','meta_C':.3,'inner_splits':15}
    a,b=e.make_splits(seed=20261110)[0];q=e.DATA.iloc[e.LAB[b]].copy();excluded=e.GROUP[e.LAB[b]]
    original=e.Y.copy()
    def clear():m.INNER_CACHE.clear();p22.PREDICTION_CACHE.clear()
    with threadpool_limits(limits=1):
        clear();p=e.predict(a,q,c,excluded)
        try:
            e.Y[b]=1-e.Y[b];q.loc[:,e.LABELS]=1-q[e.LABELS].to_numpy()
            clear();altered=e.predict(a,q,c,excluded)
        finally:e.Y[:]=original
        one=e.predict(a,q.iloc[:1],c,excluded)
    assert np.array_equal(p,altered)
    assert np.array_equal(p[:1],one)
    assert np.isin(p,[0,1]).all() and (p.sum(1)>0).all()
    checks=dict(nested_heldout_label_invariance=True,caches_cleared_before_perturbation=True,
                query_batch_invariance=True,binary_nonempty=True,max_difference=float(abs(p-altered).max()),
                heldout_rows=len(b),validation_used_for_training=False,
                source_sha256={f:hashlib.sha256(Path(f).read_bytes()).hexdigest() for f in ['phase22.py','phase23.py']})
    e.save_json('experiments/phase23-integrity-checks.json',checks)
    print(json.dumps(checks,indent=2))

if __name__=='__main__':run()


Overwriting current_reproduction/phase23_integrity.py


In [48]:
%%writefile current_reproduction/phase23_run.py
"""Evaluate nested specialists without hand-selecting validation errors."""
from pathlib import Path
import json,datetime
import numpy as np
from threadpoolctl import threadpool_limits
import phase23,experiment as e
import phase22_run as prior

E=Path('experiments')
inc=json.loads((E/'phase22-baseline.json').read_text())['candidate']['config']

def run():
    cs=[]
    for kind in ['other','pair','all','patterns']:
        for C in [.1,.3,1]:
            c={**inc,'model':'joint_specialist','meta':kind,'meta_C':C,'inner_splits':15}
            cs.append(dict(bucket=kind,config=c))
    for C in [.1,.3,1]:
        cs.append(dict(bucket='other',config={**inc,'model':'joint_specialist','meta':'other',
                       'meta_C':C,'inner_splits':15,'interactions':True}))
    for strength in [.5,.75]:
        cs.append(dict(bucket='other',config={**inc,'model':'joint_specialist','meta':'other',
                       'meta_C':.3,'inner_splits':15,'strength':strength}))
    e.save_json(E/'phase23-protocol.json',dict(frozen_at=datetime.datetime.now(datetime.timezone.utc).isoformat(),
        candidates=cs,outer_screen_seed=20261110,repeat_seeds=[20261111,20261112,20261113],
        inner_folds=15,selection='Select strongest ordinary/robust CV candidates; at most three distinct configs receive repeats, at most two receive validation.',
        validation_fit=False,reason='Other has the most errors on baseline training OOF predictions.',
        sources=['https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.StackingClassifier.html',
                 'https://proceedings.neurips.cc/paper/2011/hash/71ad16ad2c4d81f348082ff6c4b20768-Abstract.html']))
    screen=[]
    for i,r in enumerate(cs):
        print('SCREEN',i+1,len(cs),r['config']['meta'],r['config']['meta_C'],flush=True)
        s=prior.evaluate(r,20261110,'phase23')
        if s is not None:screen.append({**r,**s})
        e.save_json(E/'phase23-screen.json',screen)
    short={}
    for metric in ['ordinary','robust']:
        if screen:
            r=max(screen,key=lambda r:r[metric]);short[e.config_id(r['config'])]=r
    other=[r for r in screen if r['bucket']=='other']
    if other:
        r=max(other,key=lambda r:r['robust']);short[e.config_id(r['config'])]=r
    e.save_json(E/'phase23-shortlist-frozen.json',dict(frozen_at=datetime.datetime.now(datetime.timezone.utc).isoformat(),candidates=list(short.values())))
    repeats=[]
    for r in short.values():
        ss=[]
        for seed in [20261111,20261112,20261113]:
            print('REPEAT',r['bucket'],seed,flush=True)
            s=prior.evaluate(r,seed,'phase23_repeat')
            if s is None:break
            ss.append(s)
        if len(ss)<3:continue
        out={**r,'ordinary_cv':float(np.mean([s['ordinary'] for s in ss])),
             'robust_cv':float(np.mean([s['robust'] for s in ss])),
             'cv_std':float(np.std([s['ordinary'] for s in ss])),
             'robust_std':float(np.std([s['robust'] for s in ss])),
             'per_repeat':ss,'rule':'t0.5'}
        out['selection_score']=out['robust_cv']-.2*out['robust_std'];repeats.append(out)
        e.save_json(E/'phase23-repeat-results.json',repeats)
    baseline=next(r for r in json.loads((E/'phase22-repeat-results.json').read_text()) if r['bucket']=='control')
    eligible=[r for r in repeats if r['ordinary_cv']>=baseline['ordinary_cv']-.003 and r['robust_cv']>=baseline['robust_cv']-.003]
    picked={}
    for metric in ['selection_score','ordinary_cv']:
        if eligible:
            r=max(eligible,key=lambda r:r[metric]);picked[e.config_id(r['config'])]=r
    candidates=[{**r,'name':'phase23_'+r['bucket']} for r in picked.values()]
    e.save_json(E/'phase23-validation-frozen.json',dict(frozen_at=datetime.datetime.now(datetime.timezone.utc).isoformat(),candidates=candidates))
    results=[]
    if candidates:
        V=e.pd.read_csv('data/part1_val.csv',usecols=lambda col:col not in e.LABELS)
        with threadpool_limits(limits=1):
            ps=[e.predict(np.arange(len(e.Y)),V,r['config']) for r in candidates]
        np.savez_compressed(E/'phase23-validation-predictions.npz',p=ps)
        y=e.pd.read_csv('data/part1_val.csv')[e.LABELS].to_numpy()
        for r,p in zip(candidates,ps):
            out={**r,'validation':e.metrics(y,p,r['rule'])};results.append(out)
            print('VALIDATION',json.dumps(out),flush=True)
    e.save_json(E/'phase23-validation-results.json',results)
    print('COMPLETE',len(results),'validation candidates',flush=True)

if __name__=='__main__':run()


Overwriting current_reproduction/phase23_run.py


In [49]:
%%writefile current_reproduction/phase24.py
"""Two-view co-training and reason-specific metric learning with classical SVMs.
All model fitting uses supplied training rows only. Queries are transformed only.
"""
from pathlib import Path
import json
import numpy as np
from scipy import sparse
from scipy.special import expit
from sklearn.cluster import KMeans
from sklearn.covariance import LedoitWolf
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import pairwise_distances
from sklearn.neighbors import NeighborhoodComponentsAnalysis
from sklearn.preprocessing import normalize
from sklearn.svm import SVC
import phase23,experiment as e

legacy_predict=e.predict
METRIC_CACHE={}
VIEW_CACHE={}
BASE_CACHE={}
LAST_DIAGNOSTICS=[]

def indices(pos,excluded_groups):
    u=e.UNLAB if excluded_groups is None else e.UNLAB[~np.isin(e.GROUP[e.UNLAB],excluded_groups)]
    return np.r_[e.LAB[pos],u],u

def query_key(frame):
    return e.pd.util.hash_pandas_object(frame.drop(columns=e.LABELS,errors='ignore'),index=False).to_numpy().tobytes()

def metadata_view(idx,query,blocks):
    # Cache only label-free features, with exact fit rows and query content keys.
    key=(tuple(idx),tuple(blocks),query_key(query))
    if key not in VIEW_CACHE:
        aa=[];bb=[]
        for block in blocks:
            a,b=e.block_features(e.DATA.iloc[idx],query,block)
            aa.append(sparse.csr_matrix(a));bb.append(sparse.csr_matrix(b))
        A=normalize(sparse.hstack(aa,format='csr'))*np.sqrt(150)
        B=normalize(sparse.hstack(bb,format='csr'))*np.sqrt(150)
        if len(VIEW_CACHE)>=16:VIEW_CACHE.clear()
        VIEW_CACHE[key]=(A,B)
    return VIEW_CACHE[key]

def co_training(pos,query,c,excluded_groups=None,return_unlabeled=False):
    global LAST_DIAGNOSTICS
    idx,u=indices(pos,excluded_groups);n=len(pos);Y=e.Y[pos]
    A,B=e.features(idx,query,dict(scale='standard'))
    M,N=metadata_view(idx,query,c['views'])
    Q=np.r_[B,A[n:]] if return_unlabeled else B
    MQ=sparse.vstack([N,M[n:]],format='csr') if return_unlabeled else N
    K=np.exp(-.001*pairwise_distances(A,metric='sqeuclidean'))
    Kq=np.exp(-.001*pairwise_distances(Q,A,metric='sqeuclidean'))
    groups=KMeans(n_clusters=10,n_init=10,random_state=42).fit_predict(A)
    cluster=np.full((len(u),5),.5)
    for group in np.unique(groups):
        gold=groups[:n]==group
        if gold.sum()>=2:cluster[groups[n:]==group]=Y[gold].mean(0)
    outputs=[];diagnostics=[]
    for j in range(5):
        y=Y[:,j]
        if len(np.unique(y))<2:outputs.append(np.full(len(Q),y[0]));continue
        used=[{},{}]
        def fit(view):
            extra=np.array(sorted(used[view]),int)
            rows=np.r_[np.arange(n),n+extra]
            yy=np.r_[y,[used[view][int(i)] for i in extra]].astype(int)
            weight=np.r_[np.ones(n),np.full(len(extra),c.get('pseudo_weight',.1))]
            if len(extra):
                cp=cluster[extra,j];disagree=(cp>=.5)!=yy[n:]
                weight[n:][disagree&(np.maximum(cp,1-cp)>=.8)]*=.1
            if view==0:
                model=SVC(C=c.get('C',1),kernel='precomputed',class_weight='balanced')
                model.fit(K[np.ix_(rows,rows)],yy,sample_weight=weight)
            else:
                model=LogisticRegression(C=c.get('meta_C',.3),class_weight='balanced',max_iter=1500)
                model.fit(M[rows],yy,sample_weight=weight)
            return model,rows
        models=[fit(0),fit(1)]
        for round_number in range(0 if c.get('no_exchange') else c.get('rounds',8)):
            emb=expit(2*models[0][0].decision_function(K[n:][:,models[0][1]]))
            meta=models[1][0].predict_proba(M[n:])[:,1]
            predictions=[emb,meta];planned=[]
            for student in [0,1]:
                peer=1-student;p=predictions[peer]
                already=np.array(sorted(used[student]),int)
                eligible=np.ones(len(u),bool);eligible[already]=False
                if c.get('agreement',False):
                    eligible&=(predictions[0]>=.5)==(predictions[1]>=.5)
                for label in [0,1]:
                    confidence=p if label else 1-p
                    pool=np.flatnonzero(eligible&(confidence>=c.get('confidence',.8))&((p>=.5)==label))
                    # A bounded quota per class prevents majority pseudo-label avalanches.
                    quota=min(c.get('quota',3),max(1,int(np.sum(y==label)/3)))
                    chosen=pool[np.argsort(-confidence[pool],kind='stable')[:quota]]
                    planned.extend((student,int(i),label) for i in chosen)
            if not planned:break
            for student,i,label in planned:used[student][i]=label
            models=[fit(0),fit(1)]
            diagnostics.append(dict(label=e.LABELS[j],round=round_number,
                embedding_received=len(used[0]),metadata_received=len(used[1]),
                metadata_to_embedding_positive=sum(used[0].values())))
        pe=expit(2*models[0][0].decision_function(Kq[:,models[0][1]]))
        pm=models[1][0].predict_proba(MQ)[:,1]
        weight=c.get('metadata_vote',0.)
        outputs.append((1-weight)*pe+weight*pm)
    LAST_DIAGNOSTICS=diagnostics;p=np.array(outputs).T
    if return_unlabeled:return p[:len(B)],p[len(B):],u
    return p

def learned_projection(A,pos,j,c,idx):
    n=len(pos);y=e.Y[pos,j]
    params={k:c.get(k) for k in ['metric','metric_pca','components','subtypes']}
    key=(tuple(idx),tuple(pos),y.tobytes(),j,e.config_id(params))
    if key not in METRIC_CACHE:
        pc=PCA(n_components=min(c.get('metric_pca',30),A.shape[1],len(A)-1),svd_solver='full')
        Z=pc.fit_transform(A);gold=Z[:n]
        if c['metric']=='nca':
            model=NeighborhoodComponentsAnalysis(n_components=c.get('components',5),init='pca',
                max_iter=100,tol=1e-5,random_state=240117)
            model.fit(gold,y);projection=model.components_.T
            diag=dict(metric='nca',iterations=int(model.n_iter_))
        else:
            # Shrink within-class covariance; one or several positive subtype contrasts.
            residual=gold.copy()
            for v in [0,1]:residual[y==v]-=gold[y==v].mean(0)
            covariance=LedoitWolf().fit(residual).covariance_
            k=min(c.get('subtypes',1),max(1,int((y==1).sum()/3)))
            positive=gold[y==1];negative=gold[y==0].mean(0)
            centers=positive.mean(0)[None,:] if k==1 else KMeans(n_clusters=k,n_init=10,random_state=240117).fit(positive).cluster_centers_
            projection=np.linalg.solve(covariance+1e-5*np.eye(len(covariance)),(centers-negative).T)
            projection/=np.maximum(np.linalg.norm(projection,axis=0),1e-12)
            diag=dict(metric='fisher',subtypes=k)
        METRIC_CACHE[key]=(pc,projection,diag)
    return METRIC_CACHE[key]

def metric_ssl(pos,query,c,excluded_groups=None,return_unlabeled=False):
    global LAST_DIAGNOSTICS
    idx,u=indices(pos,excluded_groups);n=len(pos);Y=e.Y[pos]
    A,B=e.features(idx,query,dict(scale='standard'));Q=np.r_[B,A[n:]] if return_unlabeled else B
    D=pairwise_distances(A,metric='sqeuclidean');Dq=pairwise_distances(Q,A,metric='sqeuclidean')
    # KMeans remains the unsupervised step of this augmented-label pipeline.
    groups=KMeans(n_clusters=10,n_init=10,random_state=42).fit_predict(A)
    cluster=np.full((len(u),5),.5)
    for group in np.unique(groups):
        gold=groups[:n]==group
        if gold.sum()>=2:cluster[groups[n:]==group]=Y[gold].mean(0)
    outputs=[];diagnostics=[]
    for j in range(5):
        y=Y[:,j]
        if len(np.unique(y))<2:outputs.append(np.full(len(Q),y[0]));continue
        blend=c.get('metric_weight',.3)
        if blend:
            pc,projection,diag=learned_projection(A,pos,j,c,idx)
            Z=pc.transform(A)@projection;Zq=pc.transform(Q)@projection
            learned=pairwise_distances(Z,metric='sqeuclidean')
            dq=pairwise_distances(Zq,Z,metric='sqeuclidean')
            # Scale learned geometry using fitting rows only, never query statistics.
            scale=float(np.mean(D)/max(np.mean(learned),1e-12))
            DK=(1-blend)*D+blend*scale*learned
            DQ=(1-blend)*Dq+blend*scale*dq
            diagnostics.append(dict(label=e.LABELS[j],scale=scale,**diag))
        else:DK=D;DQ=Dq
        K=np.exp(-c.get('gamma',.001)*DK);Kq=np.exp(-c.get('gamma',.001)*DQ)
        model=SVC(C=c.get('C',1),kernel='precomputed',class_weight='balanced').fit(K[:n,:n],y)
        for fraction in [.1,1/3,1]:
            for _ in range(3):
                pseudo=(model.decision_function(K[n:,:model.n_features_in_])>=0).astype(int)
                sw=np.full(len(u),c.get('pseudo_weight',.03)*fraction)
                sw[((cluster[:,j]>=.5)!=pseudo)&(np.maximum(cluster[:,j],1-cluster[:,j])>=.8)]*=.1
                model=SVC(C=c.get('C',1),kernel='precomputed',class_weight='balanced').fit(K,np.r_[y,pseudo],sample_weight=np.r_[np.ones(n),sw])
        outputs.append(expit(2*model.decision_function(Kq)))
    LAST_DIAGNOSTICS=diagnostics;p=np.array(outputs).T
    if return_unlabeled:return p[:len(B)],p[len(B):],u
    return p

def apply_policy(p,baseline,rule,scope):
    h=e.decisions(p,rule)
    if scope=='all':return h.astype(float)
    original=e.decisions(baseline,'t0.5')
    out=original.copy();cols=[4] if scope=='other' else [3,4]
    out[:,cols]=h[:,cols]
    empty=out.sum(1)==0;out[empty]=original[empty]
    return out.astype(float)

def predict24(train_pos,query_frame,c,excluded_groups=None,return_unlabeled=False):
    pos=np.asarray(train_pos)
    if c['model']=='co_training':return co_training(pos,query_frame,c,excluded_groups,return_unlabeled)
    if c['model']=='metric_ssl':return metric_ssl(pos,query_frame,c,excluded_groups,return_unlabeled)
    if c['model']=='policy24':
        p=predict24(pos,query_frame,c['learner'],excluded_groups,return_unlabeled)
        key=(tuple(pos),e.Y[pos].tobytes(),query_key(query_frame),tuple(sorted([] if excluded_groups is None else excluded_groups)),return_unlabeled,e.config_id(c['baseline']))
        if key not in BASE_CACHE:BASE_CACHE[key]=legacy_predict(pos,query_frame,c['baseline'],excluded_groups,return_unlabeled)
        b=BASE_CACHE[key]
        if return_unlabeled:
            assert np.array_equal(p[2],b[2])
            return apply_policy(p[0],b[0],c['decision_rule'],c['scope']),apply_policy(p[1],b[1],c['decision_rule'],c['scope']),p[2]
        return apply_policy(p,b,c['decision_rule'],c['scope'])
    return legacy_predict(pos,query_frame,c,excluded_groups,return_unlabeled)

e.predict=predict24


Overwriting current_reproduction/phase24.py


In [50]:
%%writefile current_reproduction/phase24_control.py
"""Exact no-exchange ablation of the training-selected co-training policy."""
from pathlib import Path
import copy,json,datetime
import numpy as np
from threadpoolctl import threadpool_limits
import phase24,experiment as e
import phase24_run as runner

def run():
    frozen=json.loads(Path('experiments/phase24-validation-frozen.json').read_text())
    selected=next(r for r in frozen['candidates'] if r['config']['learner']['model']=='co_training')
    c=copy.deepcopy(selected['config']);c['learner']['no_exchange']=True
    ss=[]
    for seed in [20261115,20261116,20261117]:
        p=runner.load_predictions(c,seed,'phase24b_control');ss.append(runner.score(p))
    candidate=dict(name='phase24_exact_no_exchange',config=c,rule='t0.5',
        ordinary_cv=float(np.mean([s['ordinary'] for s in ss])),
        robust_cv=float(np.mean([s['robust'] for s in ss])),
        cv_std=float(np.std([s['ordinary'] for s in ss])),per_repeat=ss)
    e.save_json('experiments/phase24b-validation-frozen.json',dict(frozen_at=datetime.datetime.now(datetime.timezone.utc).isoformat(),candidates=[candidate],purpose='Matched control of the training-selected co-training recipe; no tuning of scope or decoder.'))
    V=e.pd.read_csv('data/part1_val.csv',usecols=lambda col:col not in e.LABELS)
    with threadpool_limits(limits=1):p=e.predict(np.arange(len(e.Y)),V,c)
    np.savez_compressed('experiments/phase24b-validation-predictions.npz',p=[p])
    y=e.pd.read_csv('data/part1_val.csv')[e.LABELS].to_numpy()
    result={**candidate,'validation':e.metrics(y,p,'t0.5')}
    e.save_json('experiments/phase24b-validation-results.json',[result]);print(json.dumps(result,indent=2))

if __name__=='__main__':run()


Overwriting current_reproduction/phase24_control.py


In [51]:
%%writefile current_reproduction/phase24_integrity.py
"""Uncached label/query isolation checks for both new learning approaches."""
from pathlib import Path
import json,hashlib
import numpy as np
from threadpoolctl import threadpool_limits
import phase24 as m,experiment as e

def clear():
    m.METRIC_CACHE.clear();m.VIEW_CACHE.clear();m.BASE_CACHE.clear()

def run():
    configs=[dict(model='co_training',views=['drugs','drug_targets','drug_actions','diseases_cleaned'],agreement=True,pseudo_weight=.1),
             dict(model='metric_ssl',metric='nca',metric_pca=25,components=5,metric_weight=.4)]
    a,b=e.make_splits(seed=20261114)[0];q=e.DATA.iloc[e.LAB[b]].copy();ex=e.GROUP[e.LAB[b]]
    original=e.Y.copy();checks=[]
    with threadpool_limits(limits=1):
        for c in configs:
            clear();p=e.predict(a,q,c,ex)
            try:
                e.Y[b]=1-e.Y[b];changed=q.copy();changed.loc[:,e.LABELS]=1-changed[e.LABELS].to_numpy()
                clear();p2=e.predict(a,changed,c,ex)
            finally:e.Y[:]=original
            single=e.predict(a,q.iloc[:1],c,ex)
            p3,pu,ix=e.predict(a,q,c,ex,return_unlabeled=True)
            delta=float(abs(p-p2).max());batch=float(abs(p[:1]-single).max())
            assert delta<1e-10 and batch<1e-10
            assert np.allclose(p,p3,atol=1e-10,rtol=0)
            assert not np.isin(e.GROUP[ix],ex).any()
            assert np.array_equal(ix,e.UNLAB[~np.isin(e.GROUP[e.UNLAB],ex)])
            assert pu.shape==(len(ix),5) and np.isfinite(pu).all()
            assert np.isfinite(p).all() and ((p>=0)&(p<=1)).all()
            checks.append(dict(model=c['model'],heldout_label_invariance=True,query_label_invariance=True,
                query_batch_invariance=True,return_unlabeled_query_invariance=True,
                masked_duplicates_excluded=True,max_label_difference=delta,max_batch_difference=batch))
    result=dict(checks=checks,caches_cleared_for_label_perturbation=True,validation_fit=False,
                source_sha256={f:hashlib.sha256(Path(f).read_bytes()).hexdigest() for f in ['phase24.py','phase24_run.py']})
    e.save_json('experiments/phase24-integrity-checks.json',result);print(json.dumps(result,indent=2))

if __name__=='__main__':run()


Overwriting current_reproduction/phase24_integrity.py


In [52]:
%%writefile current_reproduction/phase24_run.py
"""Frozen grouped-CV screen for the two user-requested learning approaches."""
from pathlib import Path
import json,datetime
import numpy as np
from threadpoolctl import threadpool_limits
import phase24 as m,experiment as e

E=Path('experiments')
BASE=json.loads((E/'phase24-baseline.json').read_text())['candidate']['config']
WEIGHTS=dict(np.load(E/'phase21-weights.npz'))
W=np.array([WEIGHTS[k]/WEIGHTS[k].sum() for k in ['propensity_logistic','propensity_tree','kmm']])
RULES=['t0.45','t0.5','t0.55','t0.65'];SCOPES=['all','other','recruitment_other']

def configurations():
    cs=[]
    views=[['criteria'],['drugs','drug_targets','drug_actions','diseases_cleaned'],
           ['criteria','drugs','drug_targets','drug_actions','diseases_cleaned']]
    for blocks in views:
        for agreement in [False,True]:
            for weight in [.1,.3]:
                for vote in [0.,.2]:
                    cs.append(dict(model='co_training',views=blocks,agreement=agreement,pseudo_weight=weight,metadata_vote=vote))
        cs.append(dict(model='co_training',views=blocks,no_exchange=True,metadata_vote=0.))
        cs.append(dict(model='co_training',views=blocks,no_exchange=True,metadata_vote=1.))
    for pca in [25,50]:
        for components in [2,5,10]:
            for weight in [.15,.4,.7]:
                cs.append(dict(model='metric_ssl',metric='nca',metric_pca=pca,components=components,metric_weight=weight))
        for subtypes in [1,2,3]:
            for weight in [.15,.4,.7]:
                cs.append(dict(model='metric_ssl',metric='fisher',metric_pca=pca,subtypes=subtypes,metric_weight=weight))
    return cs

def load_predictions(c,seed,tag):
    r=e.evaluate(c,seed=seed,tag=tag)
    if r['status']!='ok':return None
    return np.load(E/f'{tag}-{e.config_id(c)}-{seed}.npz',allow_pickle=True)['p']

def score(h):
    row=2*(h*e.Y).sum(1)/(h.sum(1)+e.Y.sum(1));ordinary=float(row.mean())
    weighted=W@row
    return dict(ordinary=ordinary,robust=float(.75*weighted.min()+.25*ordinary),weighted=weighted.tolist())

def select_policy(c,p,b):
    choices=[]
    for rule in RULES:
        for scope in SCOPES:
            s=score(m.apply_policy(p,b,rule,scope))
            pc=dict(model='policy24',learner=c,baseline=BASE,decision_rule=rule,scope=scope)
            choices.append(dict(config=pc,**s))
    return choices

def run():
    cs=configurations();e.save_json(E/'phase24-configs.json',cs)
    # Numerical equivalence when the learned metric has zero weight.
    a,b=e.make_splits(seed=20261114)[0];q=e.DATA.iloc[e.LAB[b]];ex=e.GROUP[e.LAB[b]]
    with threadpool_limits(limits=1):
        p=m.legacy_predict(a,q,BASE['base'],ex)
        z=m.metric_ssl(a,q,dict(model='metric_ssl',metric='nca',metric_weight=0.),ex)
    delta=float(abs(p-z).max());assert delta<1e-5,delta
    e.save_json(E/'phase24-equivalence.json',dict(original_metric_max_difference=delta,passed=True))
    control=load_predictions(BASE,20261114,'phase24_control')
    screen=[]
    for i,c in enumerate(cs):
        print('SCREEN',i+1,len(cs),c,flush=True)
        p=load_predictions(c,20261114,'phase24')
        if p is None:continue
        choices=select_policy(c,p,control)
        screen.extend(choices);e.save_json(E/'phase24-screen.json',screen)
        e.save_json(E/f'phase24-diagnostics-{e.config_id(c)}.json',m.LAST_DIAGNOSTICS)
    short={}
    for family in ['co_training','metric_ssl']:
        pool=[r for r in screen if r['config']['learner']['model']==family and not r['config']['learner'].get('no_exchange')]
        for metric in ['ordinary','robust']:
            r=max(pool,key=lambda r:(r[metric],r['ordinary'],r['config']['scope']=='other'))
            short[e.config_id(r['config'])]=r
    e.save_json(E/'phase24-shortlist-frozen.json',dict(frozen_at=datetime.datetime.now(datetime.timezone.utc).isoformat(),candidates=list(short.values()),repeat_seeds=[20261115,20261116,20261117]))
    baseline=[]
    for seed in [20261115,20261116,20261117]:baseline.append(score(load_predictions(BASE,seed,'phase24_control')))
    repeats=[]
    for r in short.values():
        ss=[]
        for seed in [20261115,20261116,20261117]:
            print('REPEAT',r['config']['learner'],seed,flush=True)
            p=load_predictions(r['config'],seed,'phase24_repeat')
            if p is None:break
            ss.append(score(p))
        if len(ss)<3:continue
        out={**r,'ordinary_cv':float(np.mean([s['ordinary'] for s in ss])),
            'robust_cv':float(np.mean([s['robust'] for s in ss])),
            'cv_std':float(np.std([s['ordinary'] for s in ss])),
            'robust_std':float(np.std([s['robust'] for s in ss])),
            'rule':'t0.5','per_repeat':ss}
        out['selection_score']=out['ordinary_cv']-.2*out['cv_std']
        repeats.append(out);e.save_json(E/'phase24-repeat-results.json',repeats)
    # Report both requested approaches, even if they lose to the baseline.
    # Within each family only the repeated-CV winner is evaluated on validation.
    selected=[]
    for family in ['co_training','metric_ssl']:
        pool=[r for r in repeats if r['config']['learner']['model']==family]
        if pool:
            best=max(pool,key=lambda r:(r['selection_score'],r['robust_cv']))
            selected.append({**best,'name':'phase24_'+family})
    e.save_json(E/'phase24-validation-frozen.json',dict(frozen_at=datetime.datetime.now(datetime.timezone.utc).isoformat(),baseline_cv=baseline,candidates=selected))
    V=e.pd.read_csv('data/part1_val.csv',usecols=lambda col:col not in e.LABELS)
    with threadpool_limits(limits=1):ps=[e.predict(np.arange(len(e.Y)),V,r['config']) for r in selected]
    np.savez_compressed(E/'phase24-validation-predictions.npz',p=ps)
    y=e.pd.read_csv('data/part1_val.csv')[e.LABELS].to_numpy();results=[]
    for r,p in zip(selected,ps):
        out={**r,'validation':e.metrics(y,p,r['rule'])};results.append(out)
        print('VALIDATION',json.dumps(out),flush=True)
    e.save_json(E/'phase24-validation-results.json',results)
    print('COMPLETE',len(results),'validation finalists',flush=True)

if __name__=='__main__':run()


Overwriting current_reproduction/phase24_run.py


In [53]:
%%writefile current_reproduction/phase25.py
"""Nested per-label tuning, calibrated pseudo-label reliability, local SVM experts.
All fit transformations and label use are confined to the current training fold.
"""
import json
import numpy as np
from scipy.special import expit,logit
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score,pairwise_distances
from sklearn.svm import SVC
import phase24,experiment as e

legacy_predict=e.predict
BASE=dict(model='lowdensity_v2',cluster_gate=True,clusters=10,C=1,gamma=.001,pseudo_weight=.03)
INNER_CACHE={};PRED_CACHE={};LAST_DIAGNOSTICS={}

def parameter_grid():
    options=[BASE.copy()]
    for key,values in [('C',[.3,3]),('gamma',[.0005,.002]),('pseudo_weight',[.01,.1])]:
        options += [{**BASE,key:v} for v in values]
    options += [{**BASE,'C':3,'gamma':.0005},{**BASE,'C':.3,'gamma':.002}]
    return options

def cached_predict(pos,query,c,excluded=None,return_unlabeled=False):
    key=(tuple(pos),e.Y[pos].tobytes(),phase24.query_key(query),e.config_id(c),
         tuple(sorted([] if excluded is None else excluded)),return_unlabeled)
    if key not in PRED_CACHE:
        PRED_CACHE[key]=predict25(pos,query,c,excluded,return_unlabeled)
    return PRED_CACHE[key]

def inner_predictions(pos,config,excluded,folds=5):
    exclusion=[] if excluded is None else list(excluded)
    key=(tuple(pos),e.Y[pos].tobytes(),tuple(sorted(exclusion)),e.config_id(config),folds)
    if key not in INNER_CACHE:
        p=np.zeros((len(pos),5))
        for a,b in e.make_splits(e.Y[pos],groups=e.GROUP[e.LAB[pos]],seed=250117,n_splits=folds):
            p[b]=cached_predict(pos[a],e.DATA.iloc[e.LAB[pos[b]]],config,
                np.r_[exclusion,e.GROUP[e.LAB[pos[b]]]])
        INNER_CACHE[key]=p
    return INNER_CACHE[key]

def data(pos,query,excluded,return_unlabeled):
    idx,u=phase24.indices(pos,excluded);A,B=e.features(idx,query,dict(scale='standard'))
    Q=np.r_[B,A[len(pos):]] if return_unlabeled else B
    K=np.exp(-.001*pairwise_distances(A,metric='sqeuclidean'))
    Kq=np.exp(-.001*pairwise_distances(Q,A,metric='sqeuclidean'))
    return idx,u,A,B,Q,K,Kq

def gold_svm(pos,query,c,excluded=None,return_unlabeled=False):
    _,u,A,B,Q,K,Kq=data(pos,query,excluded,return_unlabeled);n=len(pos);out=[]
    for j in range(5):
        y=e.Y[pos,j]
        if len(np.unique(y))<2:out.append(np.full(len(Q),y[0]));continue
        model=SVC(C=1,kernel='precomputed',class_weight='balanced').fit(K[:n,:n],y)
        out.append(expit(2*model.decision_function(Kq[:,:n])))
    p=np.array(out).T
    return (p[:len(B)],p[len(B):],u) if return_unlabeled else p

def tuned(pos,query,c,excluded=None,return_unlabeled=False):
    global LAST_DIAGNOSTICS
    options=parameter_grid();ps=np.array([inner_predictions(pos,o,excluded,c.get('inner_splits',5)) for o in options])
    y=e.Y[pos];chosen=np.zeros(5,int);penalty=c.get('penalty',.005)
    def matrix(selection):return np.column_stack([ps[selection[j],:,j] for j in range(5)])
    def objective(selection):
        h=e.decisions(matrix(selection),'t0.55')
        value=f1_score(y,h,average='samples',zero_division=0)
        return value-penalty*np.count_nonzero(selection)/5
    if c.get('selection','sample')=='binary':
        for j in range(5):
            vals=[f1_score(y[:,j],ps[k,:,j]>=.55,zero_division=0)-penalty*(k!=0) for k in range(len(options))]
            chosen[j]=int(np.argmax(vals))
    else:
        for _ in range(3):
            previous=chosen.copy()
            for j in range(5):
                candidates=[]
                for k in range(len(options)):
                    trial=chosen.copy();trial[j]=k
                    candidates.append((objective(trial),k==0,-k,k))
                chosen[j]=max(candidates)[-1]
            if np.array_equal(previous,chosen):break
    outputs={k:cached_predict(pos,query,options[k],excluded,return_unlabeled) for k in np.unique(chosen)}
    LAST_DIAGNOSTICS=dict(selected_by_label={e.LABELS[j]:options[chosen[j]] for j in range(5)},inner_rows=len(pos))
    if return_unlabeled:
        indices=[x[2] for x in outputs.values()];assert all(np.array_equal(ix,indices[0]) for ix in indices)
        return np.column_stack([outputs[chosen[j]][0][:,j] for j in range(5)]),np.column_stack([outputs[chosen[j]][1][:,j] for j in range(5)]),indices[0]
    return np.column_stack([outputs[chosen[j]][:,j] for j in range(5)])

def cluster_prior(A,Y):
    n=len(Y);z=KMeans(n_clusters=10,n_init=10,random_state=42).fit_predict(A)
    prior=np.full((len(A)-n,5),.5)
    for group in np.unique(z):
        gold=z[:n]==group
        if gold.sum()>=2:prior[z[n:]==group]=Y[gold].mean(0)
    return prior

def reliability(pos,query,c,excluded=None,return_unlabeled=False):
    global LAST_DIAGNOSTICS
    teacher=BASE if c.get('teacher','ssl')=='ssl' else dict(model='gold_svm25')
    pquery,pu,ix=cached_predict(pos,query,teacher,excluded,True)
    _,u,A,B,Q,K,Kq=data(pos,query,excluded,return_unlabeled)
    assert np.array_equal(ix,u);n=len(pos);Y=e.Y[pos]
    d=logit(np.clip(pu,1e-7,1-1e-7))/2;pseudo=(d>=0).astype(int)
    kind=c.get('reliability','per_label');quality=np.ones_like(pu)
    if kind!='uniform':
        po=inner_predictions(pos,teacher,excluded,c.get('inner_splits',5))
        do=logit(np.clip(po,1e-7,1-1e-7))/2
        if kind=='per_label':
            for j in range(5):
                if len(np.unique(Y[:,j]))<2:quality[:,j]=1;continue
                model=LogisticRegression(C=1,max_iter=1000).fit(do[:,j:j+1],Y[:,j])
                pr=model.predict_proba(d[:,j:j+1])[:,1]
                quality[:,j]=np.where(pseudo[:,j],pr,1-pr)
        else:
            def features(margins):
                v=margins.ravel();labels=np.tile(np.arange(5),len(margins))
                return np.c_[np.abs(v),v,np.eye(5)[labels]]
            correct=((do>=0)==Y).ravel().astype(int)
            if len(np.unique(correct))<2:quality[:]=float(correct[0])
            else:
                model=LogisticRegression(C=1,max_iter=1000).fit(features(do),correct)
                quality=model.predict_proba(features(d))[:,1].reshape(d.shape)
        quality=np.where(quality>=c.get('min_reliability',.5),np.clip(2*quality-1,0,1)**2,0)
    priors=cluster_prior(A,Y);out=[];diagnostics=[]
    for j in range(5):
        y=Y[:,j]
        if len(np.unique(y))<2:out.append(np.full(len(Q),y[0]));continue
        sw=c.get('pseudo_weight',.1)*quality[:,j]
        sw[((priors[:,j]>=.5)!=pseudo[:,j])&(np.maximum(priors[:,j],1-priors[:,j])>=.8)]*=.1
        # Zero-weight rows are removed so they do not affect automatic class counts.
        kept=np.flatnonzero(sw>1e-9);rows=np.r_[np.arange(n),n+kept]
        model=SVC(C=1,kernel='precomputed',class_weight='balanced').fit(K[np.ix_(rows,rows)],np.r_[y,pseudo[kept,j]],sample_weight=np.r_[np.ones(n),sw[kept]])
        out.append(expit(2*model.decision_function(Kq[:,rows])))
        diagnostics.append(dict(label=e.LABELS[j],accepted=len(kept),pseudo_mass=float(sw.sum())))
    LAST_DIAGNOSTICS=dict(reliability=kind,teacher=c.get('teacher','ssl'),labels=diagnostics)
    p=np.array(out).T
    return (p[:len(B)],p[len(B):],u) if return_unlabeled else p

def local_experts(pos,query,c,excluded=None,return_unlabeled=False):
    global LAST_DIAGNOSTICS
    gq,gu,ix=cached_predict(pos,query,BASE,excluded,True)
    idx,u,A,B,Q,K,Kq=data(pos,query,excluded,return_unlabeled);n=len(pos);Y=e.Y[pos]
    assert np.array_equal(ix,u)
    global_scores=np.r_[gq,gu] if return_unlabeled else gq
    km=KMeans(n_clusters=c.get('regions',3),n_init=10,random_state=250117).fit(A)
    distances=km.transform(A)**2;query_distances=km.transform(Q)**2
    width=max(float(np.median(np.min(distances,axis=1))),1e-6)*c.get('bandwidth',1.)
    routing=np.exp(-(query_distances-query_distances.min(1)[:,None])/(2*width))
    routing/=routing.sum(1)[:,None]
    gamma=c.get('gamma',.001)
    if gamma!=.001:
        K=np.exp(-gamma*pairwise_distances(A,metric='sqeuclidean'))
        Kq=np.exp(-gamma*pairwise_distances(Q,A,metric='sqeuclidean'))
    output=np.zeros_like(global_scores);diagnostics=[]
    for region in range(km.n_clusters):
        gold_w=np.exp(-(distances[:n,region]-distances[:n,region].min())/(2*width))
        gold_w=np.maximum(gold_w,.05);gold_w/=gold_w.mean()
        chosen=np.argsort(distances[n:,region])[:min(c.get('masked_neighbors',250),len(u))]
        rows=np.r_[np.arange(n),n+chosen];local=[]
        for j in range(5):
            y=Y[:,j]
            if len(np.unique(y))<2:local.append(np.full(len(Q),y[0]));continue
            pseudo=(gu[chosen,j]>=.5).astype(int)
            confidence=np.clip(np.abs(logit(np.clip(gu[chosen,j],1e-7,1-1e-7))/2),.1,1)
            sw=np.r_[gold_w,.03*confidence]
            model=SVC(C=c.get('C',1),kernel='precomputed',class_weight='balanced').fit(K[np.ix_(rows,rows)],np.r_[y,pseudo],sample_weight=sw)
            pred=expit(2*model.decision_function(Kq[:,rows]))
            support=min(gold_w[y==0].sum(),gold_w[y==1].sum())
            mix=c.get('local_mix',.5)*min(1.,support/5)
            local.append((1-mix)*global_scores[:,j]+mix*pred)
        output+=routing[:,region,None]*np.array(local).T
        diagnostics.append(dict(region=region,masked_neighbors=len(chosen),gold_weight_range=[float(gold_w.min()),float(gold_w.max())]))
    LAST_DIAGNOSTICS=dict(regions=diagnostics)
    return (output[:len(B)],output[len(B):],u) if return_unlabeled else output

def apply_policy(p,baseline,rule,scope):
    if scope!='rare':return phase24.apply_policy(p,baseline,rule,scope)
    original=e.decisions(baseline,'t0.5');out=original.copy()
    out[:,:3]=e.decisions(p,rule)[:,:3]
    empty=out.sum(1)==0;out[empty]=original[empty]
    return out.astype(float)

def predict25(train_pos,query_frame,c,excluded_groups=None,return_unlabeled=False):
    pos=np.asarray(train_pos)
    methods={'gold_svm25':gold_svm,'label_tuned_ssl':tuned,'reliable_ssl':reliability,'local_ssl':local_experts}
    if c['model'] in methods:return methods[c['model']](pos,query_frame,c,excluded_groups,return_unlabeled)
    if c['model']=='policy25':
        pred=cached_predict(pos,query_frame,c['learner'],excluded_groups,return_unlabeled)
        base=cached_predict(pos,query_frame,c['baseline'],excluded_groups,return_unlabeled)
        if return_unlabeled:
            assert np.array_equal(pred[2],base[2])
            return apply_policy(pred[0],base[0],c['decision_rule'],c['scope']),apply_policy(pred[1],base[1],c['decision_rule'],c['scope']),pred[2]
        return apply_policy(pred,base,c['decision_rule'],c['scope'])
    return legacy_predict(pos,query_frame,c,excluded_groups,return_unlabeled)

e.predict=predict25


Overwriting current_reproduction/phase25.py


In [54]:
%%writefile current_reproduction/phase25_controls.py
"""Matched untuned/unweighted controls of the frozen training-selected policies."""
from pathlib import Path
import json,copy,datetime
import numpy as np
from threadpoolctl import threadpool_limits
import phase25 as m,phase25_run as runner,experiment as e

def main():
    frozen=json.loads(Path('experiments/phase25-validation-frozen.json').read_text())
    controls=[]
    for r in frozen['candidates']:
        family=r['config']['learner']['model']
        if family not in ['label_tuned_ssl','reliable_ssl']:continue
        c=copy.deepcopy(r['config'])
        if family=='label_tuned_ssl':c['learner']=m.BASE.copy()
        else:c['learner']['reliability']='uniform'
        ss=[]
        for seed in [20261119,20261120,20261121]:
            p=runner.load_predictions(c,seed,'phase25b_control');ss.append(runner.score(p))
        controls.append(dict(name='phase25_control_'+family,config=c,rule='t0.5',
            ordinary_cv=float(np.mean([s['ordinary'] for s in ss])),
            robust_cv=float(np.mean([s['robust'] for s in ss])),
            cv_std=float(np.std([s['ordinary'] for s in ss])),per_repeat=ss))
    e.save_json('experiments/phase25b-validation-frozen.json',dict(frozen_at=datetime.datetime.now(datetime.timezone.utc).isoformat(),candidates=controls,purpose='Fixed matched controls; no control-specific threshold or scope tuning.'))
    V=e.pd.read_csv('data/part1_val.csv',usecols=lambda col:col not in e.LABELS)
    with threadpool_limits(limits=1):ps=[e.predict(np.arange(len(e.Y)),V,r['config']) for r in controls]
    np.savez_compressed('experiments/phase25b-validation-predictions.npz',p=ps)
    y=e.pd.read_csv('data/part1_val.csv')[e.LABELS].to_numpy();results=[]
    for r,p in zip(controls,ps):
        out={**r,'validation':e.metrics(y,p,r['rule'])};results.append(out);print('VALIDATION',json.dumps(out),flush=True)
    e.save_json('experiments/phase25b-validation-results.json',results)

if __name__=='__main__':main()


Overwriting current_reproduction/phase25_controls.py


In [55]:
%%writefile current_reproduction/phase25_integrity.py
"""Uncached perturbation tests for all three requested techniques."""
from pathlib import Path
import json,hashlib
import numpy as np
from threadpoolctl import threadpool_limits
import phase25 as m,experiment as e

def clear():
    m.PRED_CACHE.clear();m.INNER_CACHE.clear()

def main():
    cs=[dict(model='label_tuned_ssl',selection='sample',penalty=.005,inner_splits=5),
        dict(model='reliable_ssl',teacher='ssl',reliability='pooled',pseudo_weight=.1,min_reliability=.8),
        dict(model='local_ssl',regions=3,C=1,gamma=.001,local_mix=.5)]
    a,b=e.make_splits(seed=20261118)[0];q=e.DATA.iloc[e.LAB[b]].copy();ex=e.GROUP[e.LAB[b]];original=e.Y.copy();results=[]
    with threadpool_limits(limits=1):
        for c in cs:
            clear();p=e.predict(a,q,c,ex)
            try:
                e.Y[b]=1-e.Y[b];altered=q.copy();altered.loc[:,e.LABELS]=1-altered[e.LABELS].to_numpy()
                clear();p2=e.predict(a,altered,c,ex)
            finally:e.Y[:]=original
            one=e.predict(a,q.iloc[:1],c,ex)
            p3,pu,ix=e.predict(a,q,c,ex,return_unlabeled=True)
            assert np.allclose(p,p2,atol=1e-10,rtol=0)
            assert np.allclose(p[:1],one,atol=1e-10,rtol=0)
            assert np.allclose(p,p3,atol=1e-10,rtol=0)
            assert np.array_equal(ix,e.UNLAB[~np.isin(e.GROUP[e.UNLAB],ex)])
            assert pu.shape==(len(ix),5) and np.isfinite(pu).all()
            assert np.isfinite(p).all() and ((p>=-1e-12)&(p<=1+1e-12)).all()
            results.append(dict(model=c['model'],heldout_label_invariance=True,query_label_invariance=True,
                query_batch_invariance=True,unlabeled_output_consistency=True,masked_duplicates_excluded=True,
                max_label_difference=float(abs(p-p2).max()),max_batch_difference=float(abs(p[:1]-one).max())))
        control=e.predict(a,q,{**cs[-1],'local_mix':0},ex)
        base=m.legacy_predict(a,q,m.BASE,ex)
        assert np.allclose(control,base,atol=1e-10,rtol=0)
    result=dict(checks=results,local_zero_mix_matches_global=True,caches_cleared_before_perturbation=True,
                validation_fit=False,source_sha256={f:hashlib.sha256(Path(f).read_bytes()).hexdigest() for f in ['phase25.py','phase25_run.py']})
    e.save_json('experiments/phase25-integrity-checks.json',result);print(json.dumps(result,indent=2))

if __name__=='__main__':main()


Overwriting current_reproduction/phase25_integrity.py


In [56]:
%%writefile current_reproduction/phase25_run.py
"""Training-only selection and a frozen validation batch for three refinements."""
from pathlib import Path
import json,datetime
import numpy as np
from threadpoolctl import threadpool_limits
import phase25 as m,experiment as e

E=Path('experiments');BASE=json.loads((E/'phase25-baseline.json').read_text())['candidate']['config']
W0=dict(np.load(E/'phase21-weights.npz'))
W=np.array([W0[k]/W0[k].sum() for k in ['propensity_logistic','propensity_tree','kmm']])
RULES=['t0.45','t0.5','t0.55','t0.65'];SCOPES=['all','other','recruitment_other','rare']
FAMILIES=['label_tuned_ssl','reliable_ssl','local_ssl']

def configs():
    cs=[dict(model='label_tuned_ssl',selection=mode,penalty=penalty,inner_splits=5)
        for mode in ['sample','binary'] for penalty in [0,.005,.02]]
    for teacher in ['supervised','ssl']:
        for weight in [.03,.1,.3]:
            for kind in ['per_label','pooled']:
                for floor in [.5,.8]:
                    cs.append(dict(model='reliable_ssl',teacher=teacher,pseudo_weight=weight,
                        reliability=kind,min_reliability=floor,inner_splits=5))
            cs.append(dict(model='reliable_ssl',teacher=teacher,pseudo_weight=weight,reliability='uniform'))
    for regions in [3,6]:
        for C in [.3,1,3]:
            for gamma in [.001,.003]:
                for mix in [.25,.5]:cs.append(dict(model='local_ssl',regions=regions,C=C,gamma=gamma,local_mix=mix))
    return cs

def score(h):
    row=2*(h*e.Y).sum(1)/(h.sum(1)+e.Y.sum(1));ordinary=float(row.mean());weighted=W@row
    return dict(ordinary=ordinary,robust=float(.75*weighted.min()+.25*ordinary),weighted=weighted.tolist())

def load_predictions(c,seed,tag):
    r=e.evaluate(c,seed=seed,tag=tag)
    if r['status']!='ok':return None
    return np.load(E/f'{tag}-{e.config_id(c)}-{seed}.npz',allow_pickle=True)['p']

def main():
    cs=configs();e.save_json(E/'phase25-configs.json',cs)
    base=load_predictions(BASE,20261118,'phase25_control');screen=[]
    for i,c in enumerate(cs):
        print('SCREEN',i+1,len(cs),c,flush=True)
        p=load_predictions(c,20261118,'phase25')
        if p is None:continue
        for rule in RULES:
            for scope in SCOPES:
                cfg=dict(model='policy25',learner=c,baseline=BASE,decision_rule=rule,scope=scope)
                screen.append(dict(config=cfg,**score(m.apply_policy(p,base,rule,scope))))
        e.save_json(E/'phase25-screen.json',screen)
        e.save_json(E/f'phase25-diagnostics-{e.config_id(c)}.json',m.LAST_DIAGNOSTICS)
    short={}
    for family in FAMILIES:
        pool=[r for r in screen if r['config']['learner']['model']==family and r['config']['learner'].get('reliability')!='uniform']
        for metric in ['ordinary','robust']:
            r=max(pool,key=lambda r:(r[metric],r['ordinary'],r['config']['scope']=='rare'))
            short[e.config_id(r['config'])]=r
    e.save_json(E/'phase25-shortlist-frozen.json',dict(frozen_at=datetime.datetime.now(datetime.timezone.utc).isoformat(),candidates=list(short.values()),repeat_seeds=[20261119,20261120,20261121]))
    baseline=[]
    for seed in [20261119,20261120,20261121]:baseline.append(score(load_predictions(BASE,seed,'phase25_control')))
    repeats=[]
    for r in short.values():
        ss=[]
        for seed in [20261119,20261120,20261121]:
            print('REPEAT',r['config']['learner'],seed,flush=True)
            p=load_predictions(r['config'],seed,'phase25_repeat')
            if p is None:break
            ss.append(score(p))
        if len(ss)<3:continue
        out={**r,'ordinary_cv':float(np.mean([s['ordinary'] for s in ss])),
             'robust_cv':float(np.mean([s['robust'] for s in ss])),
             'cv_std':float(np.std([s['ordinary'] for s in ss])),
             'robust_std':float(np.std([s['robust'] for s in ss])),
             'per_repeat':ss,'rule':'t0.5'}
        out['selection_score']=out['ordinary_cv']-.2*out['cv_std']
        repeats.append(out);e.save_json(E/'phase25-repeat-results.json',repeats)
    selected=[]
    for family in FAMILIES:
        pool=[r for r in repeats if r['config']['learner']['model']==family]
        if pool:selected.append({**max(pool,key=lambda r:(r['selection_score'],r['robust_cv'])),'name':'phase25_'+family})
    e.save_json(E/'phase25-validation-frozen.json',dict(frozen_at=datetime.datetime.now(datetime.timezone.utc).isoformat(),baseline_cv=baseline,candidates=selected))
    V=e.pd.read_csv('data/part1_val.csv',usecols=lambda col:col not in e.LABELS)
    with threadpool_limits(limits=1):ps=[e.predict(np.arange(len(e.Y)),V,r['config']) for r in selected]
    np.savez_compressed(E/'phase25-validation-predictions.npz',p=ps)
    y=e.pd.read_csv('data/part1_val.csv')[e.LABELS].to_numpy();results=[]
    for r,p in zip(selected,ps):
        out={**r,'validation':e.metrics(y,p,r['rule'])};results.append(out)
        print('VALIDATION',json.dumps(out),flush=True)
    e.save_json(E/'phase25-validation-results.json',results)
    print('COMPLETE',len(results),'validation candidates',flush=True)

if __name__=='__main__':main()


Overwriting current_reproduction/phase25_run.py


In [57]:
%%writefile current_reproduction/phase26.py
"""Label-shift correction, finite-kernel LapSVMs, and latent Other subclasses.

Queries never participate in fitting, calibration, graph construction or prior
estimation. Every learned object belongs to the current training fold.
"""
from pathlib import Path
import json
import numpy as np
from scipy.linalg import cholesky, solve_triangular
from scipy.special import expit, logit
from scipy.sparse import csr_matrix, diags
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.kernel_approximation import Nystroem
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import pairwise_distances
from sklearn.svm import SVC
import phase25 as prior, experiment as e

legacy_predict=e.predict
INC=json.loads((Path(__file__).resolve().parent/'experiments/phase26-baseline.json').read_text())['candidate']['config']
PRED_CACHE={}; INNER_CACHE={}; GEOMETRY_CACHE={}; LAST_DIAGNOSTICS={}

def clip(p):return np.clip(p,1e-6,1-1e-6)

def cached(pos,q,c,excluded=None,return_unlabeled=False):
    key=(tuple(pos),e.Y[pos].tobytes(),prior.phase24.query_key(q),e.config_id(c),
         tuple(sorted([] if excluded is None else excluded)),return_unlabeled)
    if key not in PRED_CACHE:PRED_CACHE[key]=predict26(pos,q,c,excluded,return_unlabeled)
    return PRED_CACHE[key]

def soft_incumbent(pos,q,excluded=None,return_unlabeled=False):
    b=cached(pos,q,INC['base'],excluded,return_unlabeled)
    t=cached(pos,q,INC['tri'],excluded,return_unlabeled)
    def combine(bp,tp):
        bp=expit(logit(clip(bp))-logit(float(INC['base_rule'][1:])))
        return np.where(np.array(INC['use_tri'],bool)[None,:],tp,bp)
    if return_unlabeled:
        assert np.array_equal(b[2],t[2])
        return combine(b[0],t[0]),combine(b[1],t[1]),b[2]
    return combine(b,t)

def inner(pos,excluded):
    ex=[] if excluded is None else list(excluded)
    key=(tuple(pos),e.Y[pos].tobytes(),tuple(sorted(ex)))
    if key not in INNER_CACHE:
        p=np.zeros((len(pos),5))
        for a,b in e.make_splits(e.Y[pos],groups=e.GROUP[e.LAB[pos]],seed=260117,n_splits=5):
            p[b]=soft_incumbent(pos[a],e.DATA.iloc[e.LAB[pos[b]]],np.r_[ex,e.GROUP[e.LAB[pos[b]]]])
        INNER_CACHE[key]=p
    return INNER_CACHE[key]

def estimate_prevalence(po,y,pu,method):
    source=float(y.mean());source=float(np.clip(source,.01,.99))
    if method=='bbse':
        neg=float(po[y==0].mean());slope=float(po[y==1].mean()-neg)
        # Ridge toward source prevalence; weakly informative confusion estimates
        # cannot produce an arbitrarily large adjustment.
        target=(slope*(float(pu.mean())-neg)+.01*source)/(slope*slope+.01)
        diag=dict(conditional_score_separation=slope)
    else:
        target=source
        for iteration in range(200):
            r=expit(logit(clip(pu))+logit(target)-logit(source))
            new=float(np.clip((r.sum()+20*source)/(len(r)+20),.01,.99))
            if abs(new-target)<1e-8:target=new;break
            target=new
        diag=dict(em_iterations=iteration+1)
    return float(np.clip(target,.01,.99)),diag

def label_shift(pos,q,c,excluded=None,return_unlabeled=False):
    global LAST_DIAGNOSTICS
    po=inner(pos,excluded);pq,pu,u=soft_incumbent(pos,q,excluded,True)
    combined=np.r_[pq,pu] if return_unlabeled else pq
    y=e.Y[pos];out=[];diagnostics=[]
    for j in range(5):
        if len(np.unique(y[:,j]))<2:out.append(np.full(len(combined),y[0,j]));continue
        # Calibration uses OOF teacher scores, never predictions from fitting the
        # same row's label. Source confusion summaries use these training rows.
        cal=LogisticRegression(C=c.get('calibration_C',1),max_iter=1000)
        cal.fit(logit(clip(po[:,j:j+1])),y[:,j])
        source_scores=cal.predict_proba(logit(clip(po[:,j:j+1])))[:,1]
        masked_scores=cal.predict_proba(logit(clip(pu[:,j:j+1])))[:,1]
        target,diag=estimate_prevalence(source_scores,y[:,j],masked_scores,c['method'])
        source=float(np.clip(y[:,j].mean(),.01,.99))
        delta=c.get('strength',.5)*np.clip(logit(target)-logit(source),-1.5,1.5)
        raw=combined[:,j]
        if c.get('output','calibrated')=='calibrated':raw=cal.predict_proba(logit(clip(combined[:,j:j+1])))[:,1]
        out.append(expit(logit(clip(raw))+delta))
        diagnostics.append(dict(label=e.LABELS[j],source_prevalence=source,
            estimated_masked_prevalence=target,applied_log_odds_shift=float(delta),**diag))
    LAST_DIAGNOSTICS=dict(labels=diagnostics,assumption='Stable feature distribution conditional on each binary reason; not guaranteed for these data.')
    p=np.array(out).T
    return (p[:len(q)],p[len(q):],u) if return_unlabeled else p

def geometry(pos,q,c,excluded):
    idx,u=prior.phase24.indices(pos,excluded)
    key=(tuple(idx),c.get('gamma',.001),c.get('neighbors',15),c.get('components',192))
    if key not in GEOMETRY_CACHE:
        A,_=e.features(idx,e.DATA.iloc[idx[:1]],dict(scale='standard'))
        # Preserve the exact fold-local scaling for arbitrary future queries.
        from sklearn.preprocessing import StandardScaler
        sc=StandardScaler().fit(e.DATA.iloc[idx][e.EMB].to_numpy())
        A=sc.transform(e.DATA.iloc[idx][e.EMB].to_numpy())
        ny=Nystroem(kernel='rbf',gamma=c.get('gamma',.001),n_components=min(c.get('components',192),len(A)),random_state=260117)
        F=ny.fit_transform(A)
        D=pairwise_distances(A,metric='sqeuclidean');np.fill_diagonal(D,np.inf)
        k=min(c.get('neighbors',15),len(A)-1);nn=np.argsort(D,axis=1)[:,:k]
        bandwidth=np.maximum(np.sqrt(np.take_along_axis(D,nn,axis=1)[:,-1]),1e-6)
        distances=np.take_along_axis(D,nn,axis=1)
        weights=np.exp(-distances/np.maximum(bandwidth[:,None]*bandwidth[nn],1e-12))
        W=csr_matrix((weights.ravel(),(np.repeat(np.arange(len(A)),k),nn.ravel())),shape=(len(A),len(A)))
        W=W.maximum(W.T);L=diags(np.asarray(W.sum(1)).ravel())-W
        G=F.T@(L@F)/len(A);G=(G+G.T)/2
        scale=max(float(np.trace(G)/len(G)),1e-12);G/=scale
        GEOMETRY_CACHE[key]=(sc,ny,F,G,dict(nodes=len(A),undirected_edges=int(W.nnz//2),graph_penalty_scale=scale))
    sc,ny,F,G,diag=GEOMETRY_CACHE[key]
    Q=ny.transform(sc.transform(q[e.EMB].to_numpy()))
    return F,Q,G,u,diag

def lap_svm(pos,q,c,excluded=None,return_unlabeled=False):
    global LAST_DIAGNOSTICS
    F,Q,G,u,diag=geometry(pos,q,c,excluded);n=len(pos)
    # Objective: .5*w'(I+lambda*G)w + C*sum_i weighted_hinge(y_i,f_i).
    # Cholesky whitening converts it to an ordinary finite-kernel SVM.
    R=cholesky(np.eye(len(G))+c.get('graph_weight',1)*G,lower=True)
    Z=solve_triangular(R,F.T,lower=True).T
    query=solve_triangular(R,Q.T,lower=True).T
    if return_unlabeled:query=np.r_[query,Z[n:]]
    K=Z[:n]@Z[:n].T;Kq=query@Z[:n].T;out=[]
    for j in range(5):
        y=e.Y[pos,j]
        if len(np.unique(y))<2:out.append(np.full(len(query),y[0]));continue
        model=SVC(C=c.get('C',1),kernel='precomputed',class_weight='balanced')
        model.fit(K,y);out.append(expit(2*model.decision_function(Kq)))
    LAST_DIAGNOSTICS=dict(**diag,approximation='192-dimensional Nystroem RBF map',graph_weight=c.get('graph_weight',1))
    p=np.array(out).T
    return (p[:len(q)],p[len(q):],u) if return_unlabeled else p

def other_subclasses(pos,q,c,excluded=None,return_unlabeled=False):
    global LAST_DIAGNOSTICS
    idx,u=prior.phase24.indices(pos,excluded);n=len(pos);y=e.Y[pos,4]
    base,pu,iu=soft_incumbent(pos,q,excluded,True);assert np.array_equal(u,iu)
    scores=np.r_[base,pu].copy() if return_unlabeled else base.copy()
    if len(np.unique(y))<2:scores[:,4]=y[0];return (scores[:len(q)],scores[len(q):],u) if return_unlabeled else scores
    A,B=e.features(idx,q,dict(scale='standard'));query=np.r_[B,A[n:]] if return_unlabeled else B
    pc=PCA(n_components=min(25,A.shape[1],len(A)-1),svd_solver='full').fit(A)
    gold=pc.transform(A[:n]);masked=pc.transform(A[n:]);positive=gold[y==1]
    k=min(c.get('subgroups',2),max(1,len(positive)//4))
    while True:
        km=KMeans(n_clusters=k,n_init=20,random_state=260117).fit(positive)
        if k==1 or np.bincount(km.labels_,minlength=k).min()>=3:break
        k-=1
    target=np.zeros(n,int);target[y==1]=km.labels_+1
    # Negative examples stay a single class; positive subclasses map back to Other.
    selected=np.flatnonzero((pu[:,4]>=.8)|(pu[:,4]<=.2)) if c.get('pseudo_weight',.03)>0 else np.array([],int)
    yp=np.zeros(len(selected),int);positive_u=pu[selected,4]>=.8
    yp[positive_u]=km.predict(masked[selected[positive_u]])+1
    rows=np.r_[np.arange(n),n+selected];yy=np.r_[target,yp]
    # Balance aggregate Other versus non-Other, not each small subtype equally.
    balance=np.where(yy>0,n/(2*np.sum(y==1)),n/(2*np.sum(y==0)))
    sw=balance*np.r_[np.ones(n),np.full(len(selected),c.get('pseudo_weight',.03))]
    if c.get('representation','kernel')=='kernel':
        ny=Nystroem(kernel='rbf',gamma=.001,n_components=min(192,len(A)),random_state=260117)
        A=ny.fit_transform(A);query=ny.transform(query)
    model=LogisticRegression(C=c.get('C',1),max_iter=2000).fit(A[rows],yy,sample_weight=sw)
    p=model.predict_proba(query)[:,model.classes_>0].sum(1)
    mix=c.get('mix',1);scores[:,4]=(1-mix)*scores[:,4]+mix*p
    LAST_DIAGNOSTICS=dict(requested_subgroups=c.get('subgroups',2),fitted_subgroups=k,
        gold_subgroup_sizes=np.bincount(km.labels_,minlength=k).tolist(),pseudo_rows=len(selected),
        pseudo_other_rows=int(positive_u.sum()),iterations=model.n_iter_.tolist())
    return (scores[:len(q)],scores[len(q):],u) if return_unlabeled else scores

def apply_policy(p,base,rule,scope):
    if scope!='efficacy_other':return prior.apply_policy(p,base,rule,scope)
    original=e.decisions(base,'t0.5');h=original.copy();h[:,[2,4]]=e.decisions(p,rule)[:,[2,4]]
    empty=h.sum(1)==0;h[empty]=original[empty]
    return h.astype(float)

def predict26(train_pos,query_frame,c,excluded_groups=None,return_unlabeled=False):
    pos=np.asarray(train_pos)
    methods={'label_shift26':label_shift,'lap_svm26':lap_svm,'other_subclasses26':other_subclasses}
    if c['model'] in methods:return methods[c['model']](pos,query_frame,c,excluded_groups,return_unlabeled)
    if c['model']=='policy26':
        p=cached(pos,query_frame,c['learner'],excluded_groups,return_unlabeled)
        b=cached(pos,query_frame,c['baseline'],excluded_groups,return_unlabeled)
        if return_unlabeled:
            assert np.array_equal(p[2],b[2])
            return apply_policy(p[0],b[0],c['decision_rule'],c['scope']),apply_policy(p[1],b[1],c['decision_rule'],c['scope']),p[2]
        return apply_policy(p,b,c['decision_rule'],c['scope'])
    return legacy_predict(pos,query_frame,c,excluded_groups,return_unlabeled)

e.predict=predict26


Overwriting current_reproduction/phase26.py


In [58]:
%%writefile current_reproduction/phase26_integrity.py
"""Leakage, inference consistency, and objective/control checks for phase 26."""
from pathlib import Path
import hashlib,json
import numpy as np
from scipy.linalg import cholesky,solve_triangular
from sklearn.svm import SVC
from scipy.special import expit
from threadpoolctl import threadpool_limits
import phase26 as m,experiment as e

def clear():
    for module in [m,m.prior]:
        for name in ['PRED_CACHE','INNER_CACHE','GEOMETRY_CACHE']:
            if hasattr(module,name):getattr(module,name).clear()

def main():
    configs=[dict(model='label_shift26',method='em',strength=.5,output='raw',calibration_C=.3),
        dict(model='lap_svm26',C=1,gamma=.001,neighbors=10,graph_weight=1),
        dict(model='other_subclasses26',subgroups=3,representation='kernel',C=1,mix=.5,pseudo_weight=.03)]
    a,b=e.make_splits(seed=20261122)[0];q=e.DATA.iloc[e.LAB[b]].copy();ex=e.GROUP[e.LAB[b]];original=e.Y.copy();results=[]
    with threadpool_limits(limits=1):
        for c in configs:
            clear();p=e.predict(a,q,c,ex)
            try:
                e.Y[b]=1-e.Y[b];altered=q.copy();altered.loc[:,e.LABELS]=1-altered[e.LABELS].to_numpy()
                clear();p2=e.predict(a,altered,c,ex)
            finally:e.Y[:]=original
            one=e.predict(a,q.iloc[:1],c,ex);p3,pu,ix=e.predict(a,q,c,ex,True)
            assert np.allclose(p,p2,atol=1e-9,rtol=0)
            assert np.allclose(p[:1],one,atol=1e-9,rtol=0)
            assert np.allclose(p,p3,atol=1e-9,rtol=0)
            assert np.array_equal(ix,e.UNLAB[~np.isin(e.GROUP[e.UNLAB],ex)])
            assert pu.shape==(len(ix),5) and np.isfinite(pu).all()
            assert np.isfinite(p).all() and ((p>=0)&(p<=1)).all()
            results.append(dict(model=c['model'],heldout_label_invariance=True,query_batch_invariance=True,
                masked_duplicate_exclusion=True,query_label_invariance=True,unlabeled_output_consistency=True))
        F,Q,G,u,diag=m.geometry(a,q,configs[1],ex)
        R=cholesky(np.eye(len(G))+G,lower=True);beta=np.random.default_rng(26).normal(size=len(G))
        w=solve_triangular(R.T,beta,lower=False)
        assert np.allclose(w@(np.eye(len(G))+G)@w,beta@beta)
        zero=e.predict(a,q,{**configs[1],'graph_weight':0},ex)
        manual=[]
        for j in range(5):
            model=SVC(C=1,kernel='precomputed',class_weight='balanced').fit(F[:len(a)]@F[:len(a)].T,e.Y[a,j])
            manual.append(expit(2*model.decision_function(Q@F[:len(a)].T)))
        assert np.allclose(zero,np.array(manual).T,atol=1e-9)
        zero_shift=e.predict(a,q,{**configs[0],'strength':0},ex)
        raw=m.soft_incumbent(a,q,ex)
        assert np.allclose(zero_shift,m.clip(raw),atol=1e-9)
        for method in ['em','bbse']:
            sy=np.r_[np.zeros(60),np.ones(40)];sp=np.where(sy, .9,.1)
            target,_=m.estimate_prevalence(sp,sy,np.r_[np.full(800,.1),np.full(200,.9)],method)
            assert .1<target<.4
    result=dict(passed=True,checks=results,graph_objective_equivalence=True,zero_graph_matches_finite_kernel_svm=True,
        zero_shift_matches_raw_scores=True,synthetic_prior_shift_direction=True,caches_cleared=True,validation_fit=False,
        source_sha256={f:hashlib.sha256(Path(f).read_bytes()).hexdigest() for f in ['phase26.py','phase26_run.py']})
    e.save_json('experiments/phase26-integrity-checks.json',result);print(json.dumps(result,indent=2))

if __name__=='__main__':main()


Overwriting current_reproduction/phase26_integrity.py


In [59]:
%%writefile current_reproduction/phase26_run.py
"""Predeclared screen, repeated grouped CV, matched controls, frozen validation."""
from pathlib import Path
import copy,datetime,json
import numpy as np
from threadpoolctl import threadpool_limits
import phase26 as m,experiment as e
import phase25_run as prior

E=Path('experiments');BASE=m.INC
SCREEN_SEED=20261122;SEEDS=[20261123,20261124,20261125]
RULES=['t0.4','t0.5','t0.6','t0.7']
FAMILIES=['label_shift26','lap_svm26','other_subclasses26']

def configs():
    cs=[]
    for method in ['em','bbse']:
        for strength in [.25,.5,1]:
            for output in ['raw','calibrated']:
                for C in [.3,3]:cs.append(dict(model='label_shift26',method=method,strength=strength,output=output,calibration_C=C))
    for C in [.3,1,3]:
        for gamma in [.001,.003]:
            for neighbors in [10,25]:
                for weight in [.1,1,10]:cs.append(dict(model='lap_svm26',C=C,gamma=gamma,neighbors=neighbors,graph_weight=weight))
    for k in [2,3]:
        for representation in ['linear','kernel']:
            for C in [.3,1,3]:
                for mix in [.5,1]:cs.append(dict(model='other_subclasses26',subgroups=k,representation=representation,C=C,mix=mix,pseudo_weight=.03))
    return cs

def load(c,seed,tag):
    r=e.evaluate(c,seed=seed,tag=tag)
    if r['status']!='ok':return None
    return np.load(E/f'{tag}-{e.config_id(c)}-{seed}.npz',allow_pickle=True)['p']

def scopes(family):
    if family=='other_subclasses26':return ['other']
    return ['all','rare','efficacy_other','recruitment_other']

def repeat(r,tag):
    ss=[]
    for seed in SEEDS:
        p=load(r['config'],seed,tag)
        if p is None:return None
        ss.append(prior.score(p))
    out={**r,'ordinary_cv':float(np.mean([s['ordinary'] for s in ss])),
        'robust_cv':float(np.mean([s['robust'] for s in ss])),
        'cv_std':float(np.std([s['ordinary'] for s in ss])),
        'per_repeat':ss,'rule':'t0.5'}
    out['selection_score']=out['ordinary_cv']-.2*out['cv_std']
    return out

def main():
    cs=configs();e.save_json(E/'phase26-configs.json',cs)
    protocol=dict(created_at=datetime.datetime.now(datetime.timezone.utc).isoformat(),
        raw_configurations=len(cs),screen_seed=SCREEN_SEED,repeat_seeds=SEEDS,
        selection='Up to two policies per family nominated by ordinary and robust CV; repeat selection uses mean sample-F1 minus .2 times split standard deviation.',
        validation_cap=6,validation_fit=False,final_train_plus_validation_refit=False,
        controls='For each training-selected finalist: zero prior adjustment, zero graph penalty, or one Other subgroup; preserve remaining settings and decoder.',
        graph_approximation='192-component Nystroem RBF features with a fold-local self-tuning nearest-neighbor graph.',
        label_shift_assumption='Binary label shift assumes conditional score distributions are stable; source calibration summaries are training estimates, not an independent guarantee.',
        reported_leader='0.93xx; exact value unknown')
    e.save_json(E/'phase26-protocol.json',protocol)
    base=load(BASE,SCREEN_SEED,'phase26_control');screen=[]
    for i,c in enumerate(cs):
        print('SCREEN',i+1,len(cs),c,flush=True)
        p=load(c,SCREEN_SEED,'phase26')
        if p is None:continue
        for rule in RULES:
            for scope in scopes(c['model']):
                cfg=dict(model='policy26',learner=c,baseline=BASE,decision_rule=rule,scope=scope)
                screen.append(dict(config=cfg,**prior.score(m.apply_policy(p,base,rule,scope))))
        e.save_json(E/'phase26-screen.json',screen)
        e.save_json(E/f'phase26-diagnostics-{e.config_id(c)}.json',m.LAST_DIAGNOSTICS)
    short={}
    for family in FAMILIES:
        pool=[r for r in screen if r['config']['learner']['model']==family]
        for metric in ['ordinary','robust']:
            r=max(pool,key=lambda r:(r[metric],r['ordinary']))
            short[e.config_id(r['config'])]=r
    e.save_json(E/'phase26-shortlist-frozen.json',dict(frozen_at=datetime.datetime.now(datetime.timezone.utc).isoformat(),candidates=list(short.values()),repeat_seeds=SEEDS))
    baseline=[prior.score(load(BASE,seed,'phase26_control')) for seed in SEEDS]
    repeats=[]
    for r in short.values():
        print('REPEAT',r['config'],flush=True)
        out=repeat(r,'phase26_repeat')
        if out:repeats.append(out);e.save_json(E/'phase26-repeat-results.json',repeats)
    selected=[]
    for family in FAMILIES:
        pool=[r for r in repeats if r['config']['learner']['model']==family]
        if pool:selected.append({**max(pool,key=lambda r:(r['selection_score'],r['robust_cv'])),'name':'phase26_'+family,'matched_control':False})
    controls=[]
    for r in selected:
        c=copy.deepcopy(r['config']);family=c['learner']['model']
        if family=='label_shift26':c['learner']['strength']=0
        elif family=='lap_svm26':c['learner']['graph_weight']=0
        else:c['learner']['subgroups']=1
        out=repeat(dict(config=c,name='phase26_control_'+family,matched_control=True),'phase26_matched')
        if out:controls.append(out)
    e.save_json(E/'phase26-validation-frozen.json',dict(frozen_at=datetime.datetime.now(datetime.timezone.utc).isoformat(),baseline_cv=baseline,candidates=selected+controls))
    # Integrity must have passed before accessing validation at all.
    integrity=json.loads((E/'phase26-integrity-checks.json').read_text())
    assert integrity['passed']
    V=e.pd.read_csv('data/part1_val.csv',usecols=lambda col:col not in e.LABELS)
    with threadpool_limits(limits=1):
        ps=[]
        for r in selected+controls:
            ps.append(e.predict(np.arange(len(e.Y)),V,r['config']))
            e.save_json(E/f"phase26-final-diagnostics-{r['name']}.json",m.LAST_DIAGNOSTICS)
    np.savez_compressed(E/'phase26-validation-predictions.npz',p=ps)
    y=e.pd.read_csv('data/part1_val.csv')[e.LABELS].to_numpy();results=[]
    for r,p in zip(selected+controls,ps):
        out={**r,'validation':e.metrics(y,p,r['rule'])};results.append(out)
        print('VALIDATION',json.dumps(out),flush=True)
    e.save_json(E/'phase26-validation-results.json',results)
    print('COMPLETE',len(results),'validation comparisons',flush=True)

if __name__=='__main__':main()


Overwriting current_reproduction/phase26_run.py


In [60]:
%%writefile current_reproduction/phase3.py
"""Classical extensions for phase 3. Keeps the earlier engine and its provenance intact."""
import experiment as e
import numpy as np,json
from pathlib import Path
from scipy.linalg import cho_factor,cho_solve
from scipy.special import expit,softmax
from sklearn.metrics.pairwise import euclidean_distances,rbf_kernel
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC,LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.semi_supervised import LabelSpreading
from threadpoolctl import threadpool_limits
legacy_predict=e.predict

def predict3(train_pos,query_frame,c,excluded_groups=None,return_unlabeled=False):
    new={'dense_graph','kernel_neighbors','margin_svm','weighted_supervised','spectral_logistic','iterative_ssl'}
    if c['model'] not in new:return legacy_predict(train_pos,query_frame,c,excluded_groups,return_unlabeled)
    pos=np.asarray(train_pos);n=len(pos);actual=e.LAB[pos]
    u=e.UNLAB if excluded_groups is None else e.UNLAB[~np.isin(e.GROUP[e.UNLAB],excluded_groups)]
    idx=np.r_[actual,u]
    A,B=e.features(idx,query_frame,c);y=e.Y[pos]
    Q=np.r_[B,A[n:]] if return_unlabeled else B
    kind=c['model']
    if kind in ['dense_graph','kernel_neighbors']:
        scale=np.sqrt(np.mean(np.sum(A*A,axis=1)));A=A/scale;Q=Q/scale
        gamma=c.get('bandwidth',10.)
        if kind=='kernel_neighbors':
            d=euclidean_distances(Q,A[:n],squared=True)
            if c.get('k',0):
                ix=np.argsort(d,axis=1)[:,:c['k']];mask=np.ones_like(d,bool);np.put_along_axis(mask,ix,False,axis=1);d[mask]=np.inf
            w=np.exp(-gamma*(d-d.min(1)[:,None]));p=w@y/w.sum(1)[:,None]
        else:
            W=rbf_kernel(A,gamma=gamma);np.fill_diagonal(W,0)
            degree=np.maximum(W.sum(1),1e-30);S=W/np.sqrt(degree[:,None]*degree[None,:])
            Z=np.zeros((len(A),6));Z[:n,:5]=y;Z[:n,5]=1
            F=cho_solve(cho_factor(np.eye(len(A))-c.get('alpha',.2)*S,lower=True),Z)
            Wq=rbf_kernel(Q,A,gamma=gamma)
            if c.get('message_mass',False):p=(Wq@F[:,:5])/np.maximum(Wq@F[:,5,None],1e-30)
            else:
                F=np.divide(F[:,:5],F[:,5,None],out=np.tile(y.mean(0),(len(A),1)),where=F[:,5,None]>1e-30)
                if c.get('clamp',False):F[:n]=y
                p=Wq@F/np.maximum(Wq.sum(1)[:,None],1e-30)
    elif kind=='spectral_logistic':
        # Smooth covariance reweighting rather than discarding small-eigenvalue components.
        from sklearn.decomposition import PCA
        pca=PCA(n_components=c.get('components',150),svd_solver='full')
        A=pca.fit_transform(A);Q=pca.transform(Q)
        shrink=c.get('cov_ridge',.1)*pca.explained_variance_.mean()
        scale=(pca.explained_variance_+shrink)**(c.get('power',.5)/2)
        A/=scale;Q/=scale
        p=e.fit_supervised(A[:n],y,Q,{**c,'model':'logistic'})
    elif kind=='margin_svm':
        out=[]
        for j in range(5):
            if len(np.unique(y[:,j]))<2:out.append(np.full(len(Q),y[0,j]));continue
            if c.get('kernel','rbf')=='linear':m=LinearSVC(C=c.get('C',1),class_weight=c.get('balanced'),max_iter=10000)
            else:m=SVC(C=c.get('C',1),gamma=c.get('gamma',.001),kernel=c.get('kernel','rbf'),class_weight=c.get('balanced'),degree=2)
            m.fit(A[:n],y[:,j]);out.append(expit(c.get('temperature',2)*m.decision_function(Q)))
        p=np.array(out).T
    elif kind=='weighted_supervised':
        years=e.DATA.iloc[actual].start_year.to_numpy()
        if c.get('late_weight'):w=np.where(years>2002,c['late_weight'],1.)
        else:w=np.clip(np.exp((years-years.max())/c.get('time_scale',5)),.1,1)
        w=w/w.mean()
        p=e.fit_supervised(A[:n],y,Q,{**c,'model':c.get('base','logistic')},w)
    else:
        # Iterate weighted pseudo-labels; optionally require independent cluster agreement.
        teacher=c.get('teacher')
        teacherp=None
        if teacher:
            T,_=e.preprocess(e.DATA.iloc[idx][e.EMB],query_frame[e.EMB],teacher)
            yy=np.full((len(A),5),-1);yy[:n]=y
            teacherp=e.Cluster(teacher).fit(T,yy).predict(T[n:])
        outs=[]
        for j in range(5):
            gold=y[:,j];sel=np.zeros(len(u),bool);pseudo=np.zeros(len(u),int)
            for iteration in range(c.get('iterations',5)+1):
                xx=np.r_[A[:n],A[n:][sel]];yy=np.r_[gold,pseudo[sel]]
                pw=np.full(sel.sum(),c.get('pseudo_weight',.2))
                if c.get('cap_mass',False):
                    for v in [0,1]:
                        m=pseudo[sel]==v;pw[m]*=min(1.,c.get('mass_fraction',.5)*np.sum(gold==v)/max(pw[m].sum(),1e-12))
                weights=np.r_[np.ones(n),pw]
                m=LogisticRegression(C=c.get('C',.3),max_iter=2000,class_weight=c.get('balanced'))
                if len(np.unique(yy))<2:break
                m.fit(xx,yy,sample_weight=weights)
                if iteration==c.get('iterations',5):break
                pu=m.predict_proba(A[n:])[:,1];new=(np.maximum(pu,1-pu)>=c.get('confidence',.9))
                if teacherp is not None:new&=(pu>=.5)==(teacherp[:,j]>=.5)
                # Monotonic inclusion; labels can update only within already confident assignments.
                new|=sel
                if np.array_equal(new,sel):break
                sel=new;pseudo=(pu>=.5).astype(int)
            outs.append(m.predict_proba(Q)[:,1] if len(np.unique(gold))>1 else np.full(len(Q),gold[0]))
        p=np.array(outs).T
    p=np.clip(p,0,1)
    if return_unlabeled:return p[:len(B)],p[len(B):],u
    return p

e.predict=predict3

def configs():
    cs=[]
    for pc in [0,25,75]:
        for bandwidth in [2,5,10,20,50]:
            for alpha in [.2,.8]:
                cs.append(dict(model='dense_graph',pca=pc,bandwidth=bandwidth,alpha=alpha))
    for bandwidth in [5,20,50]:
        for alpha in [.2,.8,.99]:cs.append(dict(model='dense_graph',bandwidth=bandwidth,alpha=alpha,message_mass=True))
    for pc in [0,25]:
        for bandwidth in [2,5,10,20,50]:cs.append(dict(model='kernel_neighbors',pca=pc,bandwidth=bandwidth))
    for kernel in ['linear','rbf']:
        for C in [.03,.1,.3,1,3,10]:
            for balanced in [None,'balanced']:cs.append(dict(model='margin_svm',kernel=kernel,C=C,balanced=balanced,gamma=.001))
    for base in ['logistic','svc']:
        for weight in [2,4,8]:
            for C in ([.03,.1,.3,1] if base=='logistic' else [1,10]):cs.append(dict(model='weighted_supervised',base=base,late_weight=weight,C=C,gamma=.001))
    for power in [-1,-.5,.5,1]:
        for C in [.03,.1,.3,1]:cs.append(dict(model='spectral_logistic',power=power,C=C,cov_ridge=.1))
    for C in [.1,.3,1]:
        for confidence in [.8,.9,.95]:
            for cap in [True,False]:cs.append(dict(model='iterative_ssl',C=C,confidence=confidence,cap_mass=cap,pseudo_weight=.2))
    teacher=dict(model='cluster',cluster='hdbscan',min_cluster=5,pca=25)
    for C in [.1,.3,1]:
        for conf in [.8,.95]:cs.append(dict(model='iterative_ssl',C=C,confidence=conf,cap_mass=True,pseudo_weight=.2,teacher=teacher))
    return cs

if __name__=='__main__':
    import argparse
    parser=argparse.ArgumentParser();parser.add_argument('--configs');parser.add_argument('--tag',default='phase3');parser.add_argument('--seed',type=int,default=20261104);args=parser.parse_args()
    cs=json.loads(Path(args.configs).read_text()) if args.configs else configs()
    e.save_json('experiments/'+args.tag+'-configs.json',cs)
    for i,c in enumerate(cs):
        print('RUN',i+1,len(cs),c,flush=True);e.evaluate(c,seed=args.seed,tag=args.tag)


Overwriting current_reproduction/phase3.py


In [61]:
%%writefile current_reproduction/phase3_select.py
"""Repeat training CV, temporal audits, then freeze a small family-diverse validation batch."""
import phase3 as p3
import experiment as e
import numpy as np,pandas as pd,json,datetime
from pathlib import Path
from threadpoolctl import threadpool_limits
rows=[json.loads(s) for s in Path('experiments/trials.jsonl').read_text().splitlines()]
rs=[r for r in rows if r['tag']=='phase3' and r['status']=='ok'];models=sorted(set(r['model'] for r in rs));candidates=[]
for model in models:
 matches=[r for r in rs if r['model']==model]
 candidates.extend([max(matches,key=lambda r:r['score']),max(matches,key=lambda r:r['macro_f1'])])
candidates=list({r['id']:r for r in candidates}.values())
summary=[]
with threadpool_limits(limits=1):
 for base in candidates:
  c=base['config'];cid=base['id'];preds=[]
  for seed in [20261104,20261105,20261106]:
   tag='phase3' if seed==20261104 else 'phase3_repeat'
   r=e.evaluate(c,seed=seed,tag=tag);preds.append(np.load(f'experiments/{tag}-{cid}-{seed}.npz',allow_pickle=True)['p'])
  mean=np.mean(preds,axis=0);rule,score=e.choose_rule(e.Y,mean)
  avg=np.mean([e.metrics(e.Y,x,rule)['sample_f1'] for x in preds]);macro=np.mean([e.metrics(e.Y,x,rule)['macro_f1'] for x in preds])
  times=[]
  for cutoff in [2002,2005]:
   yr=e.DATA.iloc[e.LAB].start_year.to_numpy();a=np.flatnonzero(yr<=cutoff);b=np.flatnonzero(yr>cutoff)
   a=a[~np.isin(e.GROUP[e.LAB[a]],e.GROUP[e.LAB[b]])]
   pp=e.predict(a,e.DATA.iloc[e.LAB[b]],c,excluded_groups=e.GROUP[e.LAB[b]])
   times.append(e.metrics(e.Y[b],pp,'t0.5'))
  temporal=np.mean([x['sample_f1'] for x in times])
  record=dict(config=c,id=cid,rule=rule,mean_cv_sample=avg,mean_cv_macro=macro,averaged_oof_sample=score,temporal=times,selection_score=.45*avg+.20*macro+.35*temporal)
  summary.append(record);np.savez_compressed('experiments/phase3-mean-'+cid+'.npz',p=mean)
  print('SUMMARY',json.dumps(record),flush=True)
e.save_json('experiments/phase3-selection.json',summary)
chosen=[]
for model in models:chosen.append(max([x for x in summary if x['config']['model']==model],key=lambda x:x['selection_score']))
# One training-only calibrated threshold alternative for the strongest sample-F1 candidate.
from sklearn.metrics import f1_score
best=max(summary,key=lambda x:x['mean_cv_sample']);P=np.load('experiments/phase3-mean-'+best['id']+'.npz')['p'];thresholds=[]
for j in range(5):
 grid=np.arange(.1,.651,.05);fs=np.array([f1_score(e.Y[:,j],P[:,j]>=t,zero_division=0) for t in grid]);optimal=float(np.median(grid[fs>=fs.max()-1e-12]))
 # Shrink rare-label cutoffs toward 0.35; never use validation labels.
 weight=e.Y[:,j].sum()/(e.Y[:,j].sum()+10);thresholds.append(float(weight*optimal+(1-weight)*.35))
chosen.append({**best,'thresholds':thresholds,'rule':'per_label_training_f1_shrunk'})
e.save_json('experiments/phase3-validation-frozen.json',dict(frozen_at=datetime.datetime.now(datetime.timezone.utc).isoformat(),selection_criterion='0.45 repeated grouped CV sample F1 + 0.20 macro F1 + 0.35 fixed-threshold temporal sample F1; one representative per family, plus one training-only per-label calibration alternative.',candidates=chosen))
V=pd.read_csv('data/part1_val.csv',usecols=lambda c:c not in e.LABELS)
with threadpool_limits(limits=1):predictions=[e.predict(np.arange(len(e.Y)),V,c['config']) for c in chosen]
np.savez_compressed('experiments/phase3-validation-predictions.npz',p=predictions)
y=pd.read_csv('data/part1_val.csv')[e.LABELS].to_numpy();results=[]
for candidate,pp in zip(chosen,predictions):
 if 'thresholds' in candidate:
  hh=(pp>=candidate['thresholds']).astype(int);empty=hh.sum(1)==0;hh[np.flatnonzero(empty),pp[empty].argmax(1)]=1
 else:hh=e.decisions(pp,candidate['rule'])
 result={**candidate,'validation_sample_f1':float(f1_score(y,hh,average='samples')),'validation_macro_f1':float(f1_score(y,hh,average='macro'))}
 results.append(result);print('VALIDATION',json.dumps(result),flush=True)
e.save_json('experiments/phase3-validation-results.json',results)


Overwriting current_reproduction/phase3_select.py


In [62]:
%%writefile current_reproduction/phase4.py
"""Classical manifold-SVM and synthetic embedding composition; no neural models."""
import phase3 as p3
import experiment as e
import numpy as np,json
from pathlib import Path
from scipy.special import expit
from scipy.linalg import solve
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.neighbors import NearestNeighbors
legacy_predict=e.predict

def predict4(train_pos,query_frame,c,excluded_groups=None,return_unlabeled=False):
    if c['model'] not in ['manifold_svm','lowdensity_svm','composition']:
        return legacy_predict(train_pos,query_frame,c,excluded_groups,return_unlabeled)
    pos=np.asarray(train_pos);n=len(pos);u=e.UNLAB if excluded_groups is None else e.UNLAB[~np.isin(e.GROUP[e.UNLAB],excluded_groups)]
    idx=np.r_[e.LAB[pos],u];A,B=e.features(idx,query_frame,c);Y=e.Y[pos]
    Q=np.r_[B,A[n:]] if return_unlabeled else B
    kind=c['model'];rng=np.random.default_rng(42)
    if kind=='manifold_svm':
        gamma=c.get('gamma',.003);K=rbf_kernel(A,gamma=gamma);Kq=rbf_kernel(Q,A,gamma=gamma)
        d,ix=NearestNeighbors(n_neighbors=11).fit(A).kneighbors(A);d=d[:,1:];ix=ix[:,1:]
        W=np.zeros_like(K);np.put_along_axis(W,ix,np.exp(-d*d/np.median(d[:,-1])**2),axis=1);W=(W+W.T)*.5
        D=np.maximum(W.sum(1),1e-12);L=np.eye(len(A))-W/np.sqrt(D[:,None]*D[None,:])
        E=np.zeros((len(A),n));E[:n]=np.eye(n)
        V=solve(np.eye(len(A))+c.get('mu',1)*(L@K),E,assume_a='gen')
        Ktrain=(K@V)[:n];Ktrain=(Ktrain+Ktrain.T)*.5;Kquery=Kq@V;out=[]
        for j in range(5):
            if len(np.unique(Y[:,j]))<2:out.append(np.full(len(Q),Y[0,j]));continue
            m=SVC(C=c.get('C',10),kernel='precomputed',class_weight=c.get('balanced'),probability=c.get('proba',True),random_state=42).fit(Ktrain,Y[:,j])
            out.append(m.predict_proba(Kquery)[:,1] if c.get('proba',True) else expit(2*m.decision_function(Kquery)))
        p=np.array(out).T
    elif kind=='lowdensity_svm':
        out=[]
        for j in range(5):
            y=Y[:,j]
            if len(np.unique(y))<2:out.append(np.full(len(Q),y[0]));continue
            m=SVC(C=c.get('C',3),gamma=c.get('gamma',.003),class_weight=c.get('balanced')).fit(A[:n],y)
            for w in [c.get('pseudo_weight',.1)/10,c.get('pseudo_weight',.1)/3,c.get('pseudo_weight',.1)]:
                for _ in range(3):
                    labels=(m.decision_function(A[n:])>=0).astype(int)
                    sw=np.r_[np.ones(n),np.full(len(u),w)]
                    m=SVC(C=c.get('C',3),gamma=c.get('gamma',.003),class_weight=c.get('balanced')).fit(A,np.r_[y,labels],sample_weight=sw)
            out.append(expit(2*m.decision_function(Q)))
        p=np.array(out).T
    else:
        # Vicinal augmentation: random convex combinations, soft mixup, same-label SMOTE,
        # or semantic multi-reason union. All parents come from this training fold only.
        count=c.get('synthetic',500);left=rng.integers(n,size=count);right=rng.integers(n,size=count)
        if c.get('mix','union')=='same':
            for i in range(count):
                matches=np.flatnonzero((Y==Y[left[i]]).all(1));right[i]=rng.choice(matches)
        alpha=c.get('mix_alpha',1.);lam=rng.beta(alpha,alpha,size=count)[:,None]
        if c.get('mix','union')=='union':lam=np.clip(lam,.25,.75)
        Xnew=lam*A[left]+(1-lam)*A[right]
        if c.get('mix','union')=='soft':ynew=lam*Y[left]+(1-lam)*Y[right]
        elif c.get('mix','union')=='same':ynew=Y[left]
        else:ynew=np.maximum(Y[left],Y[right])
        xx=np.r_[A[:n],Xnew];yy=np.r_[Y,ynew];sw=np.r_[np.ones(n),np.full(count,c.get('synthetic_weight',.1))]
        # Optional cluster-supported masked training rows ensure real semi-supervised augmentation.
        if c.get('teacher'):
            t=c['teacher'];T,_=e.preprocess(e.DATA.iloc[idx][e.EMB],query_frame[e.EMB],t);ys=np.full((len(A),5),-1);ys[:n]=Y
            pt=e.Cluster(t).fit(T,ys).predict(T[n:]);sel=np.min(np.maximum(pt,1-pt),axis=1)>=.95
            xx=np.r_[xx,A[n:][sel]];yy=np.r_[yy,(pt[sel]>=.5).astype(int)];sw=np.r_[sw,np.full(sel.sum(),.05)]
        out=[]
        for j in range(5):
            if len(np.unique(Y[:,j]))<2:out.append(np.full(len(Q),Y[0,j]));continue
            if c.get('base','logistic')=='logistic':m=LogisticRegression(C=c.get('C',.3),max_iter=2000,class_weight=c.get('balanced'))
            else:m=SVC(C=c.get('C',10),gamma=c.get('gamma',.001),probability=True,random_state=42,class_weight=c.get('balanced'))
            if c.get('mix')=='soft':
                # Binary weighted likelihood representation of fractional targets, no neural model.
                xfit=np.r_[xx,xx];yfit=np.r_[np.ones(len(xx)),np.zeros(len(xx))];wfit=np.r_[sw*yy[:,j],sw*(1-yy[:,j])]
                good=wfit>1e-8;m.fit(xfit[good],yfit[good],sample_weight=wfit[good])
            else:m.fit(xx,yy[:,j].astype(int),sample_weight=sw)
            out.append(m.predict_proba(Q)[:,1])
        p=np.array(out).T
    p=np.clip(p,0,1)
    if return_unlabeled:return p[:len(B)],p[len(B):],u
    return p

e.predict=predict4

def configs():
    out=[]
    for gamma in [.001,.003,.01]:
        for C in [1,10]:
            for mu in [.1,1,10]:out.append(dict(model='manifold_svm',gamma=gamma,C=C,mu=mu))
    for gamma in [.001,.003]:
        for C in [1,10]:
            for w in [.03,.1,.3]:out.append(dict(model='lowdensity_svm',gamma=gamma,C=C,pseudo_weight=w,balanced='balanced'))
    for base in ['logistic','svc']:
        for mix in ['union','soft','same']:
            for C in ([.1,.3,1] if base=='logistic' else [1,10]):
                for weight in [.03,.1,.3]:out.append(dict(model='composition',mix=mix,base=base,C=C,gamma=.001,synthetic_weight=weight))
    return out

if __name__=='__main__':
    cs=configs();e.save_json('experiments/phase4-configs.json',cs)
    for i,c in enumerate(cs):print('RUN',i+1,len(cs),c,flush=True);e.evaluate(c,tag='phase4')


Overwriting current_reproduction/phase4.py


In [63]:
%%writefile current_reproduction/phase45_validate.py
import phase5,experiment as e
import json,numpy as np,pandas as pd,datetime
from pathlib import Path
from threadpoolctl import threadpool_limits
rows=[json.loads(s) for s in Path('experiments/trials.jsonl').read_text().splitlines()]
rows=[r for r in rows if r['status']=='ok' and r['tag'] in ['phase4','phase5']]
def family(r):return r['model']+'_'+r['config']['mix'] if r['model']=='composition' else r['model']
chosen=[]
with threadpool_limits(limits=1):
 for fam in sorted(set(family(r) for r in rows)):
  rr=[r for r in rows if family(r)==fam];short=list({r['id']:r for r in [max(rr,key=lambda r:r['score']),max(rr,key=lambda r:r['macro_f1'])]}.values());summaries=[]
  for r in short:
   c=r['config'];ps=[]
   for seed in [20261104,20261105,20261106]:
    tag=r['tag'] if seed==20261104 else 'phase45_repeat';res=e.evaluate(c,seed=seed,tag=tag);ps.append(np.load(f"experiments/{tag}-{r['id']}-{seed}.npz",allow_pickle=True)['p'])
   avg=np.mean(ps,axis=0);rule,score=e.choose_rule(e.Y,avg);sample=np.mean([e.metrics(e.Y,p,rule)['sample_f1'] for p in ps]);macro=np.mean([e.metrics(e.Y,p,rule)['macro_f1'] for p in ps])
   record=dict(family=fam,config=c,id=r['id'],rule=rule,averaged_oof_sample=score,mean_cv_sample=sample,mean_cv_macro=macro,selection=.7*sample+.3*macro);summaries.append(record)
  chosen.append(max(summaries,key=lambda r:r['selection']))
e.save_json('experiments/phase45-validation-frozen.json',dict(frozen_at=datetime.datetime.now(datetime.timezone.utc).isoformat(),candidates=chosen))
V=pd.read_csv('data/part1_val.csv',usecols=lambda c:c not in e.LABELS)
with threadpool_limits(limits=1):ps=[e.predict(np.arange(len(e.Y)),V,r['config']) for r in chosen]
np.savez_compressed('experiments/phase45-validation-predictions.npz',p=ps)
y=pd.read_csv('data/part1_val.csv')[e.LABELS].to_numpy();results=[]
for r,p in zip(chosen,ps):
 result={**r,'validation':e.metrics(y,p,r['rule'])};results.append(result);print('VALIDATION',json.dumps(result),flush=True)
e.save_json('experiments/phase45-validation-results.json',results)


Overwriting current_reproduction/phase45_validate.py


In [64]:
%%writefile current_reproduction/phase5.py
"""Sparse semantic reconstruction and Bayesian multilabel neighbors."""
import phase4
import experiment as e
import numpy as np,json
from pathlib import Path
from sklearn.preprocessing import normalize
from sklearn.decomposition import sparse_encode
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from scipy.optimize import nnls
legacy_predict=e.predict

def predict5(train_pos,query_frame,c,excluded_groups=None,return_unlabeled=False):
    if c['model'] not in ['sparse_examples','sparse_prototypes','mlknn']:
        return legacy_predict(train_pos,query_frame,c,excluded_groups,return_unlabeled)
    pos=np.asarray(train_pos);n=len(pos);u=e.UNLAB if excluded_groups is None else e.UNLAB[~np.isin(e.GROUP[e.UNLAB],excluded_groups)]
    A,B=e.features(np.r_[e.LAB[pos],u],query_frame,c);Y=e.Y[pos];Q=np.r_[B,A[n:]] if return_unlabeled else B
    kind=c['model'];X=A[:n]
    if kind=='mlknn':
        k=min(c.get('k',10),n-1);s=c.get('smoothing',1.)
        neigh=NearestNeighbors(n_neighbors=k+1).fit(X)
        ix=neigh.kneighbors(X)[1]
        ix=np.array([row[row!=i][:k] for i,row in enumerate(ix)])
        count=Y[ix].sum(1)
        cp=np.zeros((5,k+1));cn=np.zeros_like(cp)
        prior=(Y.sum(0)+s)/(n+2*s)
        for j in range(5):
            cp[j]=(np.bincount(count[Y[:,j]==1,j],minlength=k+1)+s)/(Y[:,j].sum()+s*(k+1))
            cn[j]=(np.bincount(count[Y[:,j]==0,j],minlength=k+1)+s)/((n-Y[:,j].sum())+s*(k+1))
        qi=neigh.kneighbors(Q,n_neighbors=k)[1];z=Y[qi].sum(1)
        pp=np.column_stack([prior[j]*cp[j,z[:,j]] for j in range(5)])
        pn=np.column_stack([(1-prior[j])*cn[j,z[:,j]] for j in range(5)])
        p=pp/np.maximum(pp+pn,1e-12)
    else:
        if kind=='sparse_prototypes':
            D=[];targets=[]
            for j in range(5):
                mask=Y[:,j]>0
                if c.get('singletons',True) and np.sum(mask&(Y.sum(1)==1))>=2:mask&=Y.sum(1)==1
                if not mask.any():continue
                count=min(c.get('clusters',2),mask.sum())
                centers=KMeans(n_clusters=count,n_init=10,random_state=42).fit(X[mask]).cluster_centers_
                D.extend(centers);targets.extend(np.eye(5)[[j]*count])
            D=np.array(D);targets=np.array(targets)
        else:D=X;targets=Y
        if c.get('normalize_dictionary',True):D=normalize(D);Q=normalize(Q)
        alpha=c.get('alpha',.05)
        if alpha>0:coeff=sparse_encode(Q,D,algorithm='lasso_lars',alpha=alpha,positive=True,max_iter=1000,n_jobs=1)
        else:coeff=np.array([nnls(D.T,q)[0] for q in Q])
        p=coeff@targets/np.maximum(coeff.sum(1)[:,None],1e-12)
        missing=coeff.sum(1)<1e-10
        if missing.any():p[missing]=Y[NearestNeighbors(n_neighbors=1).fit(X).kneighbors(Q[missing])[1][:,0]]
    p=np.clip(p,0,1)
    if return_unlabeled:return p[:len(B)],p[len(B):],u
    return p

e.predict=predict5

def configs():
    cs=[]
    for pc in [0,25,75]:
        for alpha in [.005,.02,.05,.1,.2]:cs.append(dict(model='sparse_examples',pca=pc,alpha=alpha,scale='global'))
    for pc in [0,25]:
        for k in [1,2,4]:
            for alpha in [0,.05,.1]:cs.append(dict(model='sparse_prototypes',pca=pc,clusters=k,alpha=alpha,scale='global'))
    for pc in [0,25,75]:
        for k in [3,5,10,20]:
            for s in [.1,1]:cs.append(dict(model='mlknn',pca=pc,k=k,smoothing=s))
    return cs

if __name__=='__main__':
    cs=configs();e.save_json('experiments/phase5-configs.json',cs)
    for i,c in enumerate(cs):print('RUN',i+1,len(cs),c,flush=True);e.evaluate(c,tag='phase5')


Overwriting current_reproduction/phase5.py

In [65]:
%%writefile current_reproduction/phase6.py
"""Efficient low-density SVM refinement; parameters selected without validation labels."""
import phase5,experiment as e
import numpy as np,json
from pathlib import Path
from scipy.special import expit
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.svm import SVC
from sklearn.cluster import KMeans
legacy_predict=e.predict

def predict6(train_pos,query_frame,c,excluded_groups=None,return_unlabeled=False):
    if c['model']!='lowdensity_v2':return legacy_predict(train_pos,query_frame,c,excluded_groups,return_unlabeled)
    pos=np.asarray(train_pos);n=len(pos);u=e.UNLAB if excluded_groups is None else e.UNLAB[~np.isin(e.GROUP[e.UNLAB],excluded_groups)]
    A,B=e.features(np.r_[e.LAB[pos],u],query_frame,c);Y=e.Y[pos];Q=np.r_[B,A[n:]] if return_unlabeled else B
    K=rbf_kernel(A,gamma=c.get('gamma',.001));Kq=rbf_kernel(Q,A,gamma=c.get('gamma',.001));outputs=[]
    cluster=None
    if c.get('cluster_gate'):
        # Clusters fit without labels; gold label agreements control unreliable pseudo-label weight.
        z=KMeans(n_clusters=c.get('clusters',20),n_init=10,random_state=42).fit_predict(A)
        cluster=np.full((len(u),5),.5)
        for group in np.unique(z):
            gold=(z[:n]==group)
            if gold.sum()>=2:cluster[z[n:]==group]=Y[gold].mean(0)
    for j in range(5):
        y=Y[:,j]
        if len(np.unique(y))<2:outputs.append(np.full(len(Q),y[0]));continue
        m=SVC(C=c.get('C',1),kernel='precomputed',class_weight=c.get('balanced','balanced'),tol=c.get('tol',.001)).fit(K[:n,:n],y)
        previous=None
        for w in [c.get('pseudo_weight',.03)/10,c.get('pseudo_weight',.03)/3,c.get('pseudo_weight',.03)]:
            for iteration in range(c.get('iterations',3)):
                d=m.decision_function(K[n:,:m.n_features_in_]);labels=(d>=0).astype(int)
                if c.get('early_stop',False) and previous is not None and np.array_equal(labels,previous):break
                previous=labels.copy();puw=np.full(len(u),w)
                if c.get('confidence_weight',False):puw*=np.clip(np.abs(d),.1,1)
                if cluster is not None:
                    confident=np.maximum(cluster[:,j],1-cluster[:,j])>=.8
                    disagree=(cluster[:,j]>=.5)!=labels
                    puw[confident&disagree]*=.1
                balance=c.get('balanced','balanced')
                if c.get('balance_power') is not None:
                    counts=np.bincount(np.r_[y,labels],minlength=2);power=c['balance_power']
                    balance={j:(len(A)/(2*counts[j]))**power for j in [0,1]}
                m=SVC(C=c.get('C',1),kernel='precomputed',class_weight=balance,tol=c.get('tol',.001)).fit(K,np.r_[y,labels],sample_weight=np.r_[np.ones(n),puw])
        outputs.append(expit(c.get('temperature',2)*m.decision_function(Kq[:,:m.n_features_in_])))
    p=np.array(outputs).T
    if return_unlabeled:return p[:len(B)],p[len(B):],u
    return p

e.predict=predict6

def configs():
    cs=[]
    for C in [.3,.6,1,2,3]:
        for gamma in [.0003,.0006,.001,.002]:
            for w in [.01,.03,.08]:cs.append(dict(model='lowdensity_v2',C=C,gamma=gamma,pseudo_weight=w))
    for pc in [50,75,100,125]:
        for C in [.6,1,2]:cs.append(dict(model='lowdensity_v2',pca=pc,C=C,gamma=.001,pseudo_weight=.03))
    for power in [.25,.5,.75,1.25]:
        for C in [.6,1,2]:cs.append(dict(model='lowdensity_v2',balance_power=power,C=C,gamma=.001,pseudo_weight=.03))
    for C in [.6,1,2]:
        for count in [10,20,40]:cs.append(dict(model='lowdensity_v2',cluster_gate=True,clusters=count,C=C,gamma=.001,pseudo_weight=.03))
    for C in [.6,1,2]:cs.append(dict(model='lowdensity_v2',confidence_weight=True,C=C,gamma=.001,pseudo_weight=.03))
    return cs

if __name__=='__main__':
    import argparse
    parser=argparse.ArgumentParser();parser.add_argument('--configs');parser.add_argument('--tag',default='phase6');parser.add_argument('--seed',type=int,default=20261104);args=parser.parse_args()
    cs=json.loads(Path(args.configs).read_text()) if args.configs else configs();e.save_json('experiments/'+args.tag+'-configs.json',cs)
    for i,c in enumerate(cs):print('RUN',i+1,len(cs),c,flush=True);e.evaluate(c,seed=args.seed,tag=args.tag)


Overwriting current_reproduction/phase6.py


In [66]:
%%writefile current_reproduction/phase6_select.py
"""Training-only selection of low-density variants and label-specific specialists."""
import phase6,experiment as e
from pathlib import Path
from collections import defaultdict
import json,numpy as np,pandas as pd,datetime
from sklearn.metrics import f1_score
from threadpoolctl import threadpool_limits
rows=[json.loads(s) for s in Path('experiments/trials.jsonl').read_text().splitlines()];rs=[r for r in rows if r['status']=='ok' and r['tag'] in ['phase6','phase6_features']]
def family(c):
 if c.get('features'):return 'auxiliary_features'
 if c.get('pca'):return 'pca'
 if c.get('cluster_gate'):return 'cluster_gate'
 if c.get('balance_power') is not None:return 'class_balance'
 if c.get('confidence_weight'):return 'confidence_weight'
 return 'plain'
chosen={}
for fam in sorted({family(r['config']) for r in rs}):
 subset=[r for r in rs if family(r['config'])==fam]
 for r in [max(subset,key=lambda r:r['score']),max(subset,key=lambda r:r['macro_f1'])]:chosen[r['id']]=r
# Include the champions at a fixed threshold for each reason to explore independent settings.
for j,label in enumerate(e.LABELS):
 eligible=[]
 for r in rs:
  if r['score']<.85:continue
  p=np.load(f"experiments/{r['tag']}-{r['id']}-20261104.npz",allow_pickle=True)['p'];eligible.append((f1_score(e.Y[:,j],p[:,j]>=.5,zero_division=0),r))
 r=max(eligible,key=lambda x:x[0])[1];chosen[r['id']]=r
summary=[];probabilities={}
with threadpool_limits(limits=1):
 for r in chosen.values():
  c=r['config'];cid=r['id'];ps=[]
  for seed in [20261104,20261105,20261106]:
   tag=r['tag'] if seed==20261104 else 'phase6_repeat';rr=e.evaluate(c,seed=seed,tag=tag)
   ps.append(np.load(f'experiments/{tag}-{cid}-{seed}.npz',allow_pickle=True)['p'])
  p=np.mean(ps,axis=0);probabilities[cid]=p;rule,score=e.choose_rule(e.Y,p)
  f1s=np.array([[f1_score(e.Y[:,j],z[:,j]>=.5,zero_division=0) for j in range(5)] for z in ps])
  record=dict(id=cid,config=c,family=family(c),rule=rule,averaged_oof_sample=score,mean_cv_sample=float(np.mean([e.metrics(e.Y,z,rule)['sample_f1'] for z in ps])),mean_cv_macro=float(np.mean([e.metrics(e.Y,z,rule)['macro_f1'] for z in ps])),label_mean=f1s.mean(0).tolist(),label_std=f1s.std(0).tolist())
  summary.append(record);print('SUMMARY',json.dumps(record),flush=True)
e.save_json('experiments/phase6-selection.json',summary)
# Freeze five mechanism-diverse candidates. Rules come only from mean OOF predictions.
candidates=[]
for fam in ['plain','cluster_gate','pca','class_balance','auxiliary_features']:
 rr=max([r for r in summary if r['family']==fam],key=lambda r:.8*r['mean_cv_sample']+.2*r['mean_cv_macro'])
 candidates.append(dict(name=fam,config=rr['config'],rule=rr['rule'],cv=rr['averaged_oof_sample']))
# Labelwise specialist voting; each label uses top three stable candidates, avoiding one winner per eight-positive label.
bylabel=[];p=np.zeros_like(e.Y,float)
for j in range(5):
 rr=sorted(summary,key=lambda r:r['label_mean'][j]-.5*r['label_std'][j],reverse=True)[:3]
 bylabel.append([r['config'] for r in rr]);p[:,j]=np.mean([probabilities[r['id']][:,j] for r in rr],axis=0)
rule,score=e.choose_rule(e.Y,p);candidates.append(dict(name='labelwise_specialists',members_by_label=bylabel,rule=rule,cv=score))
e.save_json('experiments/phase6-validation-frozen.json',dict(frozen_at=datetime.datetime.now(datetime.timezone.utc).isoformat(),candidates=candidates))
V=pd.read_csv('data/part1_val.csv',usecols=lambda c:c not in e.LABELS);cache={};preds=[]
with threadpool_limits(limits=1):
 def get(c):
  cid=e.config_id(c)
  if cid not in cache:cache[cid]=e.predict(np.arange(len(e.Y)),V,c)
  return cache[cid]
 for candidate in candidates:
  if 'config' in candidate:preds.append(get(candidate['config']))
  else:preds.append(np.column_stack([np.mean([get(m)[:,j] for m in ms],axis=0) for j,ms in enumerate(candidate['members_by_label'])]))
np.savez_compressed('experiments/phase6-validation-predictions.npz',p=preds)
y=pd.read_csv('data/part1_val.csv')[e.LABELS].to_numpy();results=[]
for c,p in zip(candidates,preds):
 r={**c,'validation':e.metrics(y,p,c['rule'])};results.append(r);print('VALIDATION',json.dumps(r),flush=True)
e.save_json('experiments/phase6-validation-results.json',results)


Overwriting current_reproduction/phase6_select.py


In [67]:
%%writefile current_reproduction/phase7.py
"""Low-density SVM convergence, annealing, and explicitly transductive inference."""
import phase6,experiment as e
import numpy as np,json
from pathlib import Path
from scipy.special import expit
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.svm import SVC
from sklearn.cluster import KMeans
legacy_predict=e.predict
LAST_HISTORY=[]

def predict7(train_pos,query_frame,c,excluded_groups=None,return_unlabeled=False):
    global LAST_HISTORY
    if c['model']!='lowdensity_v3':return legacy_predict(train_pos,query_frame,c,excluded_groups,return_unlabeled)
    pos=np.asarray(train_pos);n=len(pos);u=e.UNLAB if excluded_groups is None else e.UNLAB[~np.isin(e.GROUP[e.UNLAB],excluded_groups)]
    A,B=e.features(np.r_[e.LAB[pos],u],query_frame,c);Y=e.Y[pos]
    Q=np.r_[B,A[n:]] if return_unlabeled else B
    if c.get('transductive',False):A=np.r_[A,B]
    K=rbf_kernel(A,gamma=c.get('gamma',.001));Kq=rbf_kernel(Q,A,gamma=c.get('gamma',.001));outputs=[];history=[]
    cluster=None
    if c.get('cluster_gate'):
        z=KMeans(n_clusters=c.get('clusters',10),n_init=10,random_state=42).fit_predict(A)
        cluster=np.full((len(A)-n,5),.5)
        for group in np.unique(z):
            gold=z[:n]==group
            if gold.sum()>=2:cluster[z[n:]==group]=Y[gold].mean(0)
    for j in range(5):
        y=Y[:,j]
        if len(np.unique(y))<2:outputs.append(np.full(len(Q),y[0]));continue
        m=SVC(C=c.get('C',1),kernel='precomputed',class_weight='balanced').fit(K[:n,:n],y)
        initial=(m.decision_function(K[n:,:n])>=0).astype(int)
        source=c.get('balance_source','joint')
        counts=np.bincount(y if source=='gold' else np.r_[y,initial],minlength=2)
        fixed={v:(counts.sum()/(2*counts[v]))**c.get('balance_power',1.) for v in [0,1]}
        schedule=([.003,.01,.03,.1,.3,1.] if c.get('gradual',False) else [.1,1/3,1.])
        for fraction in schedule:
            w=c.get('pseudo_weight',.03)*fraction;previous=None
            for iteration in range(c.get('iterations',10)):
                d=m.decision_function(K[n:,:m.n_features_in_]);labels=(d>=0).astype(int)
                changes=len(labels) if previous is None else np.sum(previous!=labels)
                history.append(dict(label=e.LABELS[j],weight=w,iteration=iteration,changed=int(changes),positive=int(labels.sum())))
                if previous is not None and changes==0:break
                previous=labels.copy();puw=np.full(len(labels),w)
                if cluster is not None:
                    disagree=(cluster[:,j]>=.5)!=labels;confident=np.maximum(cluster[:,j],1-cluster[:,j])>=.8
                    puw[confident&disagree]*=c.get('gate_weight',.1)
                if source=='joint':
                    count=np.bincount(np.r_[y,labels],minlength=2);balance={v:(count.sum()/(2*count[v]))**c.get('balance_power',1.) for v in [0,1]}
                else:balance=fixed
                m=SVC(C=c.get('C',1),kernel='precomputed',class_weight=balance).fit(K,np.r_[y,labels],sample_weight=np.r_[np.ones(n),puw])
        outputs.append(expit(2*m.decision_function(Kq)))
    LAST_HISTORY=history;p=np.array(outputs).T
    if return_unlabeled:return p[:len(B)],p[len(B):],u
    return p

e.predict=predict7

def configs():
    cs=[]
    for transductive in [False,True]:
        for source in ['joint','initial','gold']:
            for gradual in [False,True]:cs.append(dict(model='lowdensity_v3',C=1,gamma=.001,pseudo_weight=.03,transductive=transductive,balance_source=source,gradual=gradual,iterations=15))
        for C in [.6,1,2]:
            for weight in [.01,.03,.08]:cs.append(dict(model='lowdensity_v3',C=C,gamma=.001,pseudo_weight=weight,transductive=transductive,gradual=True,iterations=15))
        for gate in [0,.1,.5]:cs.append(dict(model='lowdensity_v3',C=1,gamma=.001,pseudo_weight=.03,transductive=transductive,gradual=True,iterations=15,cluster_gate=True,clusters=10,gate_weight=gate))
    return list({e.config_id(c):c for c in cs}.values())

if __name__=='__main__':
    cs=configs();e.save_json('experiments/phase7-configs.json',cs)
    for i,c in enumerate(cs):print('RUN',i+1,len(cs),c,flush=True);e.evaluate(c,tag='phase7')


Overwriting current_reproduction/phase7.py


In [68]:
%%writefile current_reproduction/phase7_select.py
import phase7,experiment as e
import json,datetime,numpy as np,pandas as pd
from pathlib import Path
from threadpoolctl import threadpool_limits
rs=[json.loads(s) for s in Path('experiments/trials.jsonl').read_text().splitlines()];rs=[r for r in rs if r['tag']=='phase7' and r['status']=='ok']
def family(c):return ('transductive' if c.get('transductive') else 'inductive')+('_cluster' if c.get('cluster_gate') else '')
chosen={}
for fam in sorted({family(r['config']) for r in rs}):
 rows=[r for r in rs if family(r['config'])==fam]
 for r in [max(rows,key=lambda r:r['score']),max(rows,key=lambda r:r['macro_f1'])]:chosen[r['id']]=r
summary=[]
with threadpool_limits(limits=1):
 for r in chosen.values():
  ps=[]
  for seed in [20261104,20261105,20261106]:
   tag='phase7' if seed==20261104 else 'phase7_repeat';e.evaluate(r['config'],seed=seed,tag=tag)
   ps.append(np.load(f"experiments/{tag}-{r['id']}-{seed}.npz",allow_pickle=True)['p'])
  p=np.mean(ps,axis=0);rule,score=e.choose_rule(e.Y,p)
  rr=dict(name=family(r['config']),config=r['config'],rule=rule,cv=score,cv_mean=float(np.mean([e.metrics(e.Y,z,rule)['sample_f1'] for z in ps])),cv_macro=float(np.mean([e.metrics(e.Y,z,rule)['macro_f1'] for z in ps])))
  summary.append(rr);print('SUMMARY',json.dumps(rr),flush=True)
e.save_json('experiments/phase7-selection.json',summary)
candidates=[max([r for r in summary if r['name']==fam],key=lambda r:.8*r['cv_mean']+.2*r['cv_macro']) for fam in sorted({r['name'] for r in summary})]
e.save_json('experiments/phase7-validation-frozen.json',dict(frozen_at=datetime.datetime.now(datetime.timezone.utc).isoformat(),candidates=candidates))
V=pd.read_csv('data/part1_val.csv',usecols=lambda c:c not in e.LABELS)
with threadpool_limits(limits=1):preds=[e.predict(np.arange(len(e.Y)),V,c['config']) for c in candidates]
np.savez_compressed('experiments/phase7-validation-predictions.npz',p=preds)
y=pd.read_csv('data/part1_val.csv')[e.LABELS].to_numpy();results=[]
for c,p in zip(candidates,preds):
 r={**c,'validation':e.metrics(y,p,c['rule'])};results.append(r);print('VALIDATION',json.dumps(r),flush=True)
e.save_json('experiments/phase7-validation-results.json',results)


Overwriting current_reproduction/phase7_select.py


In [69]:
%%writefile current_reproduction/phase8.py
"""Training-only hierarchy for mutually exclusive rare reasons plus Recruitment.
Uses no validation labels; every fitted component is scoped to the outer fold.
"""
import phase7,experiment as e
import numpy as np,json
from scipy.special import expit,softmax
from sklearn.svm import SVC
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.cluster import KMeans
legacy_predict=e.predict
RARE=[0,1,2,4]

def binary(K,ki,y,ui,Kq,c,weight=None):
    if len(np.unique(y))<2:return np.full(len(K),2*y[0]-1.),np.full(len(Kq),2*y[0]-1.)
    ix=np.r_[ki,ui];n=len(ki)
    m=SVC(C=c.get('C',1),kernel='precomputed',class_weight=c.get('balanced','balanced')).fit(K[np.ix_(ki,ki)],y)
    active=ki
    if c.get('pseudo_weight',.03)>0 and len(ui):
        for fraction in [.1,1/3,1]:
            for _ in range(c.get('iterations',3)):
                pseudo=(m.decision_function(K[np.ix_(ui,active)])>=0).astype(int)
                uw=np.ones(len(ui)) if weight is None else weight
                m=SVC(C=c.get('C',1),kernel='precomputed',class_weight=c.get('balanced','balanced')).fit(K[np.ix_(ix,ix)],np.r_[y,pseudo],sample_weight=np.r_[np.ones(n),uw*c.get('pseudo_weight',.03)*fraction]);active=ix
    return m.decision_function(K[:,active]),m.decision_function(Kq[:,active])

def predict8(train_pos,query_frame,c,excluded_groups=None,return_unlabeled=False):
    if c['model'] not in ['hierarchical_svm','multiclass_ssl']:return legacy_predict(train_pos,query_frame,c,excluded_groups,return_unlabeled)
    pos=np.asarray(train_pos);n=len(pos);u=e.UNLAB if excluded_groups is None else e.UNLAB[~np.isin(e.GROUP[e.UNLAB],excluded_groups)]
    A,B=e.features(np.r_[e.LAB[pos],u],query_frame,c);Y=e.Y[pos];Q=np.r_[B,A[n:]] if return_unlabeled else B
    K=rbf_kernel(A,gamma=c.get('gamma',.001));Kq=rbf_kernel(Q,A,gamma=c.get('gamma',.001));ki=np.arange(n);ui=np.arange(n,len(A))
    rec,recq=binary(K,ki,Y[:,3],ui,Kq,c);p=np.zeros((len(Q),5));p[:,3]=expit(2*recq)
    rare=Y[:,RARE].any(axis=1)
    if c['model']=='hierarchical_svm':
        gate,gateq=binary(K,ki,rare.astype(int),ui,Kq,c)
        sel=ui[expit(2*gate[ui])>=c.get('gate_threshold',.5)];ri=ki[rare]
        conditional=[]
        cc={**c,'C':c.get('conditional_C',c.get('C',1)),'pseudo_weight':c.get('conditional_weight',c.get('pseudo_weight',.03))}
        for j in RARE:
            _,q=binary(K,ri,Y[rare,j],sel,Kq,cc);conditional.append(q)
        d=np.array(conditional).T
        if c.get('conditional_mode','softmax')=='softmax':r=softmax(d*c.get('temperature',2),axis=1)
        else:r=expit(2*d)
        p[:,RARE]=r*expit(2*gateq)[:,None]**c.get('gate_power',1.)
    else:
        # Each gold row has at most one rare reason; retain Recruitment independently.
        assert (Y[:,RARE].sum(1)<=1).all()
        y=np.where(rare,np.argmax(Y[:,RARE],axis=1),4)
        m=SVC(C=c.get('C',1),kernel='precomputed',class_weight=c.get('balanced','balanced'),decision_function_shape='ovr').fit(K[:n,:n],y);active=ki
        cluster=None
        if c.get('cluster_gate'):
            z=KMeans(n_clusters=c.get('clusters',10),n_init=10,random_state=42).fit_predict(A);cluster=np.full(len(u),-1)
            for k in np.unique(z):
                known=z[:n]==k
                if known.sum()>=2:
                    count=np.bincount(y[known],minlength=5)
                    if count.max()/count.sum()>=.8:cluster[z[n:]==k]=count.argmax()
        if c.get('pseudo_weight',.03)>0:
            for fraction in [.1,1/3,1]:
                for _ in range(c.get('iterations',3)):
                    pseudo=m.predict(K[n:,active]);uw=np.full(len(u),c.get('pseudo_weight',.03)*fraction)
                    if cluster is not None:uw[(cluster>=0)&(cluster!=pseudo)]*=.1
                    m=SVC(C=c.get('C',1),kernel='precomputed',class_weight=c.get('balanced','balanced'),decision_function_shape='ovr').fit(K,np.r_[y,pseudo],sample_weight=np.r_[np.ones(n),uw]);active=np.arange(len(A))
        d=m.decision_function(Kq[:,active]);r=softmax(d*c.get('temperature',2),axis=1)
        for j,label in enumerate(m.classes_):
            if label<4:p[:,RARE[label]]=r[:,j]
        if c.get('recruitment_mode')=='multiclass':
            p[:,3]=r[:,np.flatnonzero(m.classes_==4)[0]]
    if return_unlabeled:return p[:len(B)],p[len(B):],u
    return p

e.predict=predict8

def configs():
 cs=[]
 for C in [.3,1,3]:
  for gamma in [.0003,.001,.003]:
   for w in [0,.03]:
    cs.append(dict(model='hierarchical_svm',C=C,gamma=gamma,pseudo_weight=w))
    cs.append(dict(model='multiclass_ssl',C=C,gamma=gamma,pseudo_weight=w))
 for mode in ['softmax','sigmoid']:
  for gate in [0,.5,1.]:
   for C in [1,3]:cs.append(dict(model='hierarchical_svm',C=1,conditional_C=C,gamma=.001,pseudo_weight=.03,conditional_mode=mode,gate_power=gate))
 for weight in [.01,.1]:
  for C in [1,3]:cs.append(dict(model='hierarchical_svm',C=C,gamma=.001,pseudo_weight=.03,conditional_weight=weight,gate_threshold=.7))
 for C in [.3,1,3]:
  for w in [.01,.1]:cs.append(dict(model='multiclass_ssl',C=C,gamma=.001,pseudo_weight=w,cluster_gate=True))
 return cs
if __name__=='__main__':
 cs=configs();e.save_json('experiments/phase8-configs.json',cs)
 for i,c in enumerate(cs):print('RUN',i+1,len(cs),c,flush=True);e.evaluate(c,tag='phase8')


Overwriting current_reproduction/phase8.py


In [70]:
%%writefile current_reproduction/phase9.py
"""Alternative classical kernels and training-only embedding geometry for SSL SVM."""
import phase8,experiment as e
import numpy as np,json
from scipy.special import expit
from sklearn.metrics.pairwise import rbf_kernel,laplacian_kernel,polynomial_kernel,linear_kernel,pairwise_distances
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize
from sklearn.svm import SVC
from sklearn.cluster import KMeans
legacy_predict=e.predict

def predict9(train_pos,query_frame,c,excluded_groups=None,return_unlabeled=False):
 if c['model']!='kernel_lowdensity':return legacy_predict(train_pos,query_frame,c,excluded_groups,return_unlabeled)
 pos=np.asarray(train_pos);n=len(pos);u=e.UNLAB if excluded_groups is None else e.UNLAB[~np.isin(e.GROUP[e.UNLAB],excluded_groups)]
 A,B=e.features(np.r_[e.LAB[pos],u],query_frame,c);Y=e.Y[pos]
 if c.get('remove_top'):
  pc=PCA(n_components=c['remove_top'],svd_solver='full').fit(A)
  A=A-pc.inverse_transform(pc.transform(A));B=B-pc.inverse_transform(pc.transform(B))
 if c.get('postnorm'):A=normalize(A)*np.sqrt(150);B=normalize(B)*np.sqrt(150)
 Q=np.r_[B,A[n:]] if return_unlabeled else B
 kind=c.get('kernel','rbf');gamma=c.get('gamma',.001)
 def kernel(a,b):
  if kind=='rbf':return rbf_kernel(a,b,gamma=gamma)
  if kind=='laplacian':return laplacian_kernel(a,b,gamma=gamma)
  if kind=='poly':return polynomial_kernel(a,b,gamma=gamma,degree=c.get('degree',2),coef0=c.get('coef0',1.))
  if kind=='linear':return linear_kernel(a,b)/150
  if kind=='rq':return (1+gamma*pairwise_distances(a,b,metric='sqeuclidean')/c.get('alpha',1.))**(-c.get('alpha',1.))
  raise ValueError(kind)
 K=kernel(A,A);Kq=kernel(Q,A);cluster=None
 if c.get('cluster_gate'):
  z=KMeans(n_clusters=c.get('clusters',10),n_init=10,random_state=42).fit_predict(A);cluster=np.full((len(u),5),.5)
  for k in np.unique(z):
   known=z[:n]==k
   if known.sum()>=2:cluster[z[n:]==k]=Y[known].mean(0)
 outputs=[]
 for j in range(5):
  y=Y[:,j]
  if len(np.unique(y))<2:outputs.append(np.full(len(Q),y[0]));continue
  m=SVC(C=c.get('C',1),kernel='precomputed',class_weight='balanced').fit(K[:n,:n],y)
  for fraction in [.1,1/3,1]:
   for _ in range(c.get('iterations',3)):
    d=m.decision_function(K[n:,:m.n_features_in_]);pseudo=(d>=0).astype(int);w=np.full(len(u),c.get('pseudo_weight',.03)*fraction)
    if cluster is not None:w[((cluster[:,j]>=.5)!=pseudo)&(np.maximum(cluster[:,j],1-cluster[:,j])>=.8)]*=.1
    m=SVC(C=c.get('C',1),kernel='precomputed',class_weight='balanced').fit(K,np.r_[y,pseudo],sample_weight=np.r_[np.ones(n),w])
  outputs.append(expit(2*m.decision_function(Kq)))
 p=np.array(outputs).T
 if return_unlabeled:return p[:len(B)],p[len(B):],u
 return p

e.predict=predict9

def configs():
 cs=[]
 for scale in ['global','raw','standard']:
  for norm in [False,True]:
   for remove in [0,1,3,5]:cs.append(dict(model='kernel_lowdensity',C=1,gamma=.001,scale=scale,postnorm=norm,remove_top=remove))
 for white in [.25,.5,1.]:
  for dim in [25,75,125,150]:
   for C in [.3,1,3]:cs.append(dict(model='kernel_lowdensity',C=C,gamma=.001,pca=dim,white=white))
 for kind,gs in [('laplacian',[.001,.003,.01]),('poly',[.001,.003,.01]),('rq',[.001,.003,.01]),('linear',[.001])]:
  for C in [.3,1,3]:
   for gamma in gs:cs.append(dict(model='kernel_lowdensity',kernel=kind,C=C,gamma=gamma))
 for degree in [3,4]:
  for C in [.3,1,3]:cs.append(dict(model='kernel_lowdensity',kernel='poly',degree=degree,C=C,gamma=.003))
 return cs
if __name__=='__main__':
 cs=configs();e.save_json('experiments/phase9-configs.json',cs)
 for i,c in enumerate(cs):print('RUN',i+1,len(cs),c,flush=True);e.evaluate(c,tag='phase9')


Overwriting current_reproduction/phase9.py


In [71]:
%%writefile current_reproduction/prepare_consensus.py
import experiment as e
configs=[]
teachers=[dict(model='graph',pca=25,k=5,alpha=.8),dict(model='cluster',cluster='hdbscan',min_cluster=5,pca=25)]
for t in teachers:
 for model in ['logistic','svc']:
  for c in ([.1,.3] if model=='logistic' else [3,10]):
   for w in [.05,.2]:
    for conf in [.8,.95]:
     configs.append(dict(model='pseudo',teacher=t,student=model,C=c,gamma=.001,scale='global',pseudo_weight=w,confidence=conf,labelwise=True,consensus=True))
e.save_json('experiments/consensus-input.json',configs)


Overwriting current_reproduction/prepare_consensus.py


In [72]:
%%writefile current_reproduction/prepare_metric.py
import experiment as e
configs=[]
for model in ['logistic','svc']:
 for k in [5,15,40]:
  for residual in [0,.5,2]:
   configs.append(dict(model='metric_graph',metric_model=model,C=.3 if model=='logistic' else 10,gamma=.001,scale='global',k=k,alpha=.8,residual=residual))
for n in [2,3,5,10,20,40]:configs.append(dict(model='pls',components=n))
e.save_json('experiments/metric-input.json',configs)


Overwriting current_reproduction/prepare_metric.py


In [73]:
%%writefile current_reproduction/prepare_refinement.py
import experiment as e,json
from pathlib import Path
rows=[json.loads(x) for x in Path('experiments/trials.jsonl').read_text().splitlines()];ok=[r for r in rows if r['status']=='ok'];configs=[]
for scale in ['standard','global']:
 for C in [.03,.05,.2,.3,.5,2,3,30]:configs.append(dict(model='logistic',scale=scale,C=C))
 for C in [1,3,10,30]:
  for gamma in [.0001,.0003,.0005,.002,.005]:configs.append(dict(model='svc',scale=scale,C=C,gamma=gamma))
for C in [1,3,30,100]:
 for scale in ['global','raw']:configs.append(dict(model='logistic',scale=scale,norm=True,C=C))
for alpha in [.001,.01,.1,1]:
 for gamma in [.0003,.001,.003,.01]:configs.append(dict(model='krr',alpha=alpha,gamma=gamma))
for amplitude in [1,10]:
 for length in [5,10,20,40]:configs.append(dict(model='gp',amplitude=amplitude,length=length))
for pc in [2,3,5]:configs.append(dict(model='qda',pca=pc,reg=.9))
for pc in [5,15,30]:
 for k in [3,7]:configs.append(dict(model='nca',pca=30,components=pc,k=k))
# Refine graph geometry, especially sharper local neighborhoods and less noisy dimensions.
for pc in [5,10,15,40]:
 for k in [3,7,12]:configs.append(dict(model='graph',pca=pc,k=k,alpha=.8))
for scale in ['global','raw']:
 for norm in [True,False]:
  for k in [3,7,15]:configs.append(dict(model='graph',scale=scale,norm=norm,k=k,alpha=.9))
for typ in ['kmeans','gmm']:
 for pc in [5,10,50]:
  for n in [5,15,35]:configs.append(dict(model='cluster',cluster=typ,pca=pc,clusters=n))
# Pseudo-labels generated by graph or density-cluster teachers; small weight prevents false labels dominating.
teachers=[max([r for r in ok if r['model']==t],key=lambda r:r['score'])['config'] for t in ['graph','cluster']]
teachers.append(dict(model='cluster',cluster='kmeans',clusters=12,pca=25))
for teacher in teachers:
 for student in ['svc','logistic']:
  for weight in [.05,.2,.5]:
   for confidence in [.8,.95]:configs.append(dict(model='pseudo',teacher=teacher,student=student,C=10 if student=='svc' else .1,gamma=.001,pseudo_weight=weight,confidence=confidence,labelwise=True))
e.save_json('experiments/refine-input.json',configs)
# Stage independently adds each semantic group to the embedding baseline.
blocks=['year','criteria','criteria_char','diseases_cleaned','drugs','icdcodes','matched_drugs','drug_targets','drug_actions','drug_pathways','pathway_level0','pathway_level2','pathway_levels','fingerprint','descriptors','missingness','counts']
configs=[]
for base in [dict(model='svc',C=10,gamma=.001,scale='standard'),dict(model='logistic',C=.1,scale='global')]:
 for block in blocks:
  for w in [.15,.5,1.]:configs.append({**base,'features':[block],'feature_weight':w})
e.save_json('experiments/features-input.json',configs)


Overwriting current_reproduction/prepare_refinement.py


In [74]:
%%writefile current_reproduction/reproduce_current.py
"""Reproduce the frozen current-best candidate; never selects using validation labels.
Run from the project or portable midterm directory. All training data stay unchanged.
"""
from pathlib import Path
import argparse,hashlib,json,datetime
import numpy as np,pandas as pd
import phase21,experiment as e
from threadpoolctl import threadpool_limits

def main():
 parser=argparse.ArgumentParser();parser.add_argument('--output',default='predictions/current_best');args=parser.parse_args()
 out=Path(args.output);out.mkdir(parents=True,exist_ok=True)
 frozen=json.loads(Path('experiments/current-best.json').read_text());candidate=frozen['candidate'];config=candidate['config'];rule=candidate['rule']
 # Freeze/test predictions first, then read validation labels solely for reporting.
 test=pd.read_csv('data/part1_test.csv');V=pd.read_csv('data/part1_val.csv',usecols=lambda col:col not in e.LABELS)
 with threadpool_limits(limits=1):
  pt,pu,ix=e.predict(np.arange(len(e.Y)),test,config,return_unlabeled=True)
  pv=e.predict(np.arange(len(e.Y)),V,config)
 assert np.array_equal(ix,e.UNLAB)
 ht=e.decisions(pt,rule);hu=e.decisions(pu,rule);hv=e.decisions(pv,rule)
 sub=pd.DataFrame({'ID':test.ID,'Predicted':[' '.join(label for label,v in zip(e.LABELS,row) if v) for row in ht]});sub.to_csv(out/'part1_submission.csv',index=False)
 aug=e.DATA.copy();aug.loc[e.UNLAB,e.LABELS]=hu;aug['label_source']='original_gold';aug.loc[e.UNLAB,'label_source']='predicted_missing';aug.to_csv(out/'part1_train_augmented.csv',index=False)
 for name,ids,prob in [('test',test.ID,pt),('unlabeled',e.DATA.iloc[e.UNLAB].ID,pu),('validation',V.ID,pv)]:
  frame=pd.DataFrame(prob,columns=['score_'+l for l in e.LABELS]);frame.insert(0,'ID',ids.to_numpy());frame.to_csv(out/f'{name}_scores.csv',index=False)
 yv=pd.read_csv('data/part1_val.csv')[e.LABELS].to_numpy();val=e.metrics(yv,pv,rule)
 assert abs(val['sample_f1']-frozen['best_validation_f1'])<1e-12,(val,frozen['best_validation_f1'])
 audit=pd.DataFrame({'ID':V.ID,'true':[' '.join(l for l,v in zip(e.LABELS,row) if v) for row in yv],'predicted':[' '.join(l for l,v in zip(e.LABELS,row) if v) for row in hv]})
 audit['sample_f1']=2*np.sum(yv*hv,axis=1)/(yv.sum(1)+hv.sum(1));audit.to_csv(out/'validation_audit.csv',index=False)
 assert sub.ID.tolist()==pd.read_csv('data/part1_sample_submission.csv').ID.tolist()
 assert len(sub)==312 and sub.ID.is_unique and sub.Predicted.map(lambda s:bool(s) and set(s.split())<=set(e.LABELS)).all()
 assert np.array_equal(aug.loc[e.LAB,e.LABELS].to_numpy(),e.Y)
 assert aug[e.LABELS].isin([0,1]).all().all() and len(aug)==len(e.DATA)
 raw=json.loads(Path('experiments/data-audit.json').read_text())['files']
 for name,record in raw.items():assert hashlib.sha256((Path('data')/name).read_bytes()).hexdigest()==record['sha256']
 source_files=['experiment.py']+[f'phase{i}.py' for i in [3,4,5,6,7,8,9,10,11,14,16,17,18,20,21]]
 manifest=dict(created_at=datetime.datetime.now(datetime.timezone.utc).isoformat(),candidate=candidate,validation=val,target=.92735,target_met=val['sample_f1']>.92735,kaggle_score=None,gold_rows=157,predicted_missing_rows=1014,test_rows=312,submission_sha256=hashlib.sha256((out/'part1_submission.csv').read_bytes()).hexdigest(),raw_data_hashes_verified=True,gold_labels_preserved=True,validation_labels_used_for_training=0,final_train_plus_validation_refit_performed=False,source_sha256={f:hashlib.sha256(Path(f).read_bytes()).hexdigest() for f in source_files},score_interpretation='Validation is repeatedly used for selection; CV values are development scores. Neither establishes a Kaggle leaderboard result. Final hybrid scores are binary decisions; component margin/vote scores are not calibrated probabilities.')
 e.save_json(out/'manifest.json',manifest);print(json.dumps({k:manifest[k] for k in ['validation','target_met','submission_sha256','gold_labels_preserved']},indent=2))
if __name__=='__main__':main()


Overwriting current_reproduction/reproduce_current.py


In [75]:
%%writefile current_reproduction/robustness.py
"""Fresh-seed checks and predeclared feature combinations; no external validation."""
import json
from pathlib import Path
import experiment as e
rs=[json.loads(s) for s in Path('experiments/trials.jsonl').read_text().splitlines()]
ok=[r for r in rs if r['status']=='ok' and r['tag'] in ['screen','refine','features','consensus']]
chosen=[]
for model in ['svc','logistic','lda','chain','powerset','graph','cluster','pseudo']:
 matches=[r for r in ok if r['model']==model and not r['config'].get('features')]
 chosen.extend(sorted(matches,key=lambda x:x['score'],reverse=True)[:(3 if model in ['svc','logistic','pseudo'] else 1)])
for block in ['drugs','pathway_levels','pathway_level0','descriptors','fingerprint','criteria','drug_pathways','drug_targets','year']:
 matches=[r for r in ok if r['config'].get('features')==[block]]
 chosen.append(max(matches,key=lambda x:x['score']))
configs=list({e.config_id(r['config']):r['config'] for r in chosen}.values())
e.save_json('experiments/robustness-input.json',configs)
for seed in [20261105,20261106,20261107]:
 for i,c in enumerate(configs):
  print('REPEAT',seed,i+1,len(configs),c,flush=True);e.evaluate(c,seed=seed,tag='robustness')
base=dict(model='svc',C=10,gamma=.001,scale='standard')
combos=[['drugs','pathway_levels'],['drugs','descriptors'],['drugs','pathway_level0'],['drugs','criteria'],['pathway_levels','descriptors'],['drugs','pathway_levels','descriptors'],['drugs','pathway_levels','criteria'],['drugs','pathway_levels','descriptors','criteria'],['criteria','diseases_cleaned','drugs','icdcodes','matched_drugs','drug_targets','drug_actions','drug_pathways','fingerprint','descriptors','year','missingness','counts']]
configs=[{**base,'features':b,'feature_weight':w} for b in combos for w in [.15,.5]]
e.save_json('experiments/combinations-input.json',configs)
for seed in [20261104,20261105,20261106]:
 for c in configs:e.evaluate(c,seed=seed,tag='combinations')


Overwriting current_reproduction/robustness.py


In [76]:
%%writefile current_reproduction/select_ensembles.py
import experiment as e,json,numpy as np
from pathlib import Path
from collections import defaultdict
by=defaultdict(dict)
for f in Path('experiments').glob('*.npz'):
 z=np.load(f,allow_pickle=True);r=z['record'].item()
 if r['tag'] in ['screen','refine','features','consensus','robustness','combinations']:by[r['id']][r['seed']]=(r,z['p'])
rank=json.loads(Path('experiments/robust-ranking.json').read_text())
# Broad mechanisms rather than many near-identical hyperparameter fits.
proto=[
 dict(model='logistic',scale='global',C=.2),
 dict(model='logistic',scale='global',C=.3),
 dict(model='svc',C=10,gamma=.001,scale='standard',features=['drug_targets'],feature_weight=1.),
 dict(model='svc',C=10,gamma=.001,scale='standard',features=['fingerprint'],feature_weight=.5),
 dict(model='svc',C=10,gamma=.001,scale='standard',features=['criteria'],feature_weight=1.),
 next(x['config'] for x in rank if x['config']['model']=='pseudo'),
 next(x['config'] for x in rank if x['config']['model']=='pseudo' and x['config']['teacher']['model']=='graph'),
]
ensembles=[]
for j in [2,3,4,5,6]:
 for w in [.25,.5,.75]:ensembles.append(dict(model='ensemble',members=[proto[0],proto[j]],weights=[1-w,w]))
for ids in [[0,2,5],[0,3,5],[0,2,4,5],[0,2,3,4,5],[0,5,6],[2,4,5],[0,1,2,3,4,5,6]]:
 ensembles.append(dict(model='ensemble',members=[proto[j] for j in ids]))
results=[]
for c in ensembles:
 ids=[e.config_id(m) for m in c['members']];seeds=sorted(set.intersection(*(set(by[cid]) for cid in ids)))
 ps=[];weights=np.asarray(c.get('weights',np.ones(len(ids))),float);weights/=weights.sum()
 for seed in seeds:ps.append(sum(w*by[cid][seed][1] for cid,w in zip(ids,weights)))
 p=np.mean(ps,axis=0);rule,score=e.choose_rule(e.Y,p)
 r=dict(config=c,id=e.config_id(c),rule=rule,averaged_oof_score=score,mean_split_score=float(np.mean([e.metrics(e.Y,x,rule)['sample_f1'] for x in ps])),n_seeds=len(seeds))
 results.append(r)
 np.savez_compressed('experiments/ensemble-'+r['id']+'.npz',p=p,record=r)
results.sort(key=lambda r:(r['averaged_oof_score'],r['mean_split_score']),reverse=True)
e.save_json('experiments/ensemble-ranking.json',results)
for r in results[:8]:print(r['averaged_oof_score'],r['mean_split_score'],r['rule'],r['config'])
# Direct SSL candidate with feature additions, evaluated across fresh seeds.
base=proto[5];configs=[]
for block in ['drug_targets','drugs','criteria','fingerprint','pathway_levels']:
 for w in [.15,.5]:configs.append({**base,'features':[block],'feature_weight':w})
e.save_json('experiments/ssl-features-input.json',configs)


Overwriting current_reproduction/select_ensembles.py

In [77]:
%%writefile current_reproduction/update_validation_docs.py
"""Update deliverables with the requested frozen-candidate validation measurement."""
from pathlib import Path
import json,shutil,zipfile
import nbformat as nbf
from nbconvert import HTMLExporter
r=json.loads(Path('experiments/final-validation.json').read_text());score=r['sample_f1']
notice=f'''LATEST FINAL-CANDIDATE VALIDATION RESULT
Sample-wise F1: {score:.8f}; macro F1: {r['macro_f1']:.8f}; 78 trials.
This is the exact delivered full-fit candidate with the same model settings and 0.35 threshold. Its test probabilities and submission predictions were verified unchanged. Its training CV F1 is 0.93631; that did not transfer to validation. Improvement over the user's prior model has not been demonstrated. The first bagged comparison coincidentally had the same best sample-wise F1; the per-label and macro scores differ.
Details: experiments/final-validation.json and predictions/fullfit_validation_audit.csv.
'''
p=Path('READ_ME_FIRST.txt');s=p.read_text()
if not s.startswith('LATEST FINAL-CANDIDATE'):s=notice+'\n'+s
s=s.replace('This is NOT a Kaggle score or a new external validation result.','This CV value is not a Kaggle score. The exact full-fit candidate subsequently scored 0.850427 on validation with its settings unchanged.')
s=s.replace('External validation was used once for a frozen 3-candidate comparison. Later work did not recheck it.','External validation was first used for a frozen 3-candidate comparison. At the user’s explicit request, the final full-fit candidate was then evaluated once with its settings unchanged; it scored 0.850427.')
p.write_text(s)
p=Path('midterm/01_experiment_report.ipynb');nb=nbf.read(p,as_version=4)
nb.cells[0].source=f'''# AML Midterm — Part 1 experiment report

**Final candidate validation F1: {score:.5f}.** Macro F1: {r['macro_f1']:.5f}, measured on all 78 validation rows. The model configuration and threshold were unchanged. Its test probabilities match the delivered submission.

The candidate's **0.93631 training-only grouped leave-one-out F1 did not transfer to validation**. No improvement over the user's existing model has been demonstrated, and a leaderboard win is not established. The initial bagged-model comparison also had a best sample-F1 of 0.85043; these are separate measurements. Historical search and first-comparison details follow.
'''
for cell in nb.cells:
 if cell.cell_type!='markdown':continue
 cell.source=cell.source.replace('These are the only three external-validation model evaluations.','These are the three models in the initial frozen external-validation comparison. The final full-fit candidate was evaluated separately at the user’s request; see the latest result above.')
 cell.source=cell.source.replace('After its disappointing result, later experiments remained training-only; no individual validation-error retuning or repeated validation scoring occurred.','After its disappointing result, subsequent model development remained training-only. The final candidate was later evaluated at the user’s request, with no threshold or model retuning.')
 cell.source=cell.source.replace('The final candidate was **not re-evaluated on the used validation set**. Its 0.93631 leave-group-out score remains a selection-biased development estimate, awaiting external confirmation.','The exact final candidate was subsequently evaluated on validation at the user’s request: **0.85043 sample-F1**. Its 0.93631 leave-group-out score remains a selection-biased development estimate and is substantially higher than the observed validation result.')
 cell.source=cell.source.replace('Submit `predictions/part1_fullfit_candidate.csv` once and record its Kaggle result before claiming improvement.','The final candidate does not demonstrate an improvement on validation; do not infer a leaderboard improvement from its CV score.')
 cell.source=cell.source.replace('The final full-fit regime addresses training-size differences but does not erase that evidence.','The full-fit candidate also scored 0.85043 on validation, confirming that the training-size change did not solve the measured gap.')
# Add the exact measurement and its source once, leaving historical outputs intact.
if not any(c.cell_type=='markdown' and c.source.startswith('## Requested final-candidate evaluation') for c in nb.cells):
 table='| Reason | Validation F1 |\n|---|---:|\n'+'\n'.join(f"| {l} | {r['f1_'+l]:.5f} |" for l in ['Safety','Funding','Efficacy','Recruitment','Other'])
 nb.cells.insert(1,nbf.v4.new_markdown_cell('## Requested final-candidate evaluation\n\n'+table+'\n\nSource: `experiments/final-validation.json`. Reproducible evaluation: `evaluate_final_validation.py`. This evaluation does not alter any training labels, settings, threshold, or Kaggle predictions.'))
nbf.validate(nb);nbf.write(nb,p)
body,_=HTMLExporter().from_notebook_node(nb);p.with_suffix('.html').write_text(body)
p=Path('midterm/02_reproduce.ipynb');nb=nbf.read(p,as_version=4)
if not any(c.cell_type=='markdown' and c.source.startswith('## Exact final validation evaluation') for c in nb.cells):
 nb.cells.append(nbf.v4.new_markdown_cell('## Exact final validation evaluation\n\nAt the user’s request, the frozen full-fit candidate was evaluated: sample-F1 0.85043, macro-F1 0.68841. The script below reproduces that measurement without changing the model or threshold and verifies the original test predictions. It is saved here but is not automatically executed.'))
 nb.cells.append(nbf.v4.new_code_cell('evaluation_source = '+repr(Path('evaluate_final_validation.py').read_text())+'\n(replica / "evaluate_final_validation.py").write_text(evaluation_source)\n'))
nbf.validate(nb);nbf.write(nb,p)
checks=json.loads(Path('experiments/deliverable-checks.json').read_text());checks.update(external_validation_comparisons=2,external_validation_candidate_count=4,final_candidate_validation_f1=score,final_candidate_test_predictions_verified_unchanged=True)
Path('experiments/deliverable-checks.json').write_text(json.dumps(checks,indent=2))
for name in ['READ_ME_FIRST.txt','evaluate_final_validation.py','update_validation_docs.py','make_candidate.py','build_deliverables.py']:
 shutil.copy2(name,Path('midterm')/name)
for name in ['final-candidate.json','final-validation.json','deliverable-checks.json']:
 shutil.copy2(Path('experiments')/name,Path('midterm/experiments')/name)
shutil.copy2('predictions/fullfit_validation_audit.csv','midterm/predictions/fullfit_validation_audit.csv')
with zipfile.ZipFile('AMLMidterm_Part1.zip','w',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
 for p in Path('midterm').rglob('*'):
  if p.is_file() and not any(part in {'reproduction','__pycache__','.DS_Store'} for part in p.parts):z.write(p,p.as_posix())
print('Updated report, notebooks, result records and archive. Final validation sample-F1:',score)


Overwriting current_reproduction/update_validation_docs.py


In [78]:
%%writefile current_reproduction/validate_representatives.py
import experiment as e,json,numpy as np,pandas as pd,datetime
from pathlib import Path
from threadpoolctl import threadpool_limits
out=Path('experiments/phase3-reference-validation.json')
if out.exists():raise SystemExit('Already evaluated')
rows=[json.loads(s) for s in Path('experiments/trials.jsonl').read_text().splitlines()]
models=['logistic','svc','knn','cluster','graph','softmax','cluster_features','catboost']
chosen=[]
for model in models:
 eligible=[r for r in rows if r['status']=='ok' and r['config']['model']==model and not r['config'].get('features') and r['seed']==20261104]
 r=max(eligible,key=lambda r:r['score'])
 chosen.append({'config':r['config'],'rule':r['rule'],'cv_score':r['score'],'model':model})
e.save_json('experiments/phase3-reference-frozen.json',chosen)
V=pd.read_csv('data/part1_val.csv',usecols=lambda c:c not in e.LABELS);preds=[]
with threadpool_limits(limits=1):
 for r in chosen:
  p=e.predict(np.arange(len(e.Y)),V,r['config']);preds.append(p);print('PREDICTED',r['model'],flush=True)
np.savez_compressed('experiments/phase3-reference-predictions.npz',p=np.array(preds))
y=pd.read_csv('data/part1_val.csv')[e.LABELS].to_numpy()
results=[]
for r,p in zip(chosen,preds):
 result={**r,**e.metrics(y,p,r['rule'])};results.append(result);print(json.dumps(result),flush=True)
e.save_json(out,dict(evaluated_at=datetime.datetime.now(datetime.timezone.utc).isoformat(),results=results,protocol='Frozen configurations and decision rules chosen using training CV; one representative per mechanism, no validation retuning.'))


Overwriting current_reproduction/validate_representatives.py


In [79]:
%%writefile current_reproduction/verify_benchmarks.py
"""Audit stored predictions, frozen recipes and original-model preservation."""
from pathlib import Path
import json,numpy as np
import benchmark_cv as b
import benchmark_pooled_cv as pooled

def verify_predictions(folder,summary,pooled_mode=False):
    p=json.loads((folder/'protocol.json').read_text())
    registry=json.loads((folder/'registry.json').read_text())
    records={r['name']:r for r in registry}
    for row in summary['models']:
        key=row['pooled_result_key'] if pooled_mode else row['result_key']
        result=json.loads((folder/(key+'.json')).read_text())
        assert result['protocol_sha256']==b.digest(p)
        r=records[row['candidate']]
        assert result['config']==r['config'] and result['rule']==r['rule']
        a=np.load(folder/(key+'.npz'),allow_pickle=True)
        truth=a['truth'];h=a['predictions'];scores=a['scores']
        assert np.array_equal(truth,b.e.Y) and h.shape==(3,len(truth),5)
        assert a['ids'].tolist()==b.e.DATA.iloc[b.e.LAB].ID.tolist()
        assert np.array_equal(h,np.array([b.e.decisions(s,r['rule']) for s in scores]))
        per_repeat=(2*(h*truth[None,:,:]).sum(2)/(h.sum(2)+truth.sum(1)[None,:])).mean(1)
        mean_col='pooled_cv_mean' if pooled_mode else 'cv_mean'
        std_col='pooled_cv_std' if pooled_mode else 'repeat_std'
        assert abs(float(per_repeat.mean())-row[mean_col])<1e-12
        assert abs(float(per_repeat.std())-row[std_col])<1e-12
        assert len(result['fold_scores'])==15
        assert all(b.file_hash(b.ROOT/name)==value for name,value in result['source_sha256'].items())
    return dict(models_verified=len(registry),identical_protocol_for_all=True,explicit_frozen_decoders_verified=True,
        pooled_sample_f1_verified=True,source_hashes_verified=True,complete_oof_coverage=True)

def main():
    b.protocol();primary=b.OUT;summary=json.loads((primary/'summary.json').read_text())
    primary_checks=verify_predictions(primary,summary)
    primary_checks.update(saved_folds_valid=True,grouped_folds=15,gold_rows=157,gold_embedding_groups=150,
        validation_evaluations_performed=0,current_submission_unchanged=True)
    before=pooled.protected_hashes()
    with pooled.pooled_context():
        protocol=pooled.protocol();registry=pooled.registry()
        gold=b.e.DATA.iloc[b.e.LAB].copy();gold['embedding_group']=b.e.GROUP[b.e.LAB]
        conflicts=(gold.groupby('embedding_group')[b.e.LABELS].nunique().max(axis=1)>1).sum()
        assert conflicts==0
        second=json.loads((pooled.OUT/'summary.json').read_text())
        secondary_checks=verify_predictions(pooled.OUT,second,True)
        secondary_checks.update(saved_folds_valid=True,grouped_folds=15,gold_rows=235,gold_embedding_groups=222,
            no_conflicting_gold_embedding_groups=True,all_configs_and_rules_unchanged=True,
            no_final_all_label_refit=True,current_submission_unchanged=True)
        previous={r['candidate']:r for r in summary['models']}
        for r in second['models']:
            assert abs(r['training_cv_mean']-previous[r['candidate']]['cv_mean'])<1e-12
            assert r['original_validation_f1']==previous[r['candidate']]['validation_f1']
    assert pooled.protected_hashes()==before and len(b.e.Y)==157
    for name,sha in json.loads((pooled.OUT/'preservation.json').read_text())['protected_sha256'].items():
        assert b.file_hash(b.ROOT/name)==sha
    submission=b.file_hash(b.ROOT/'predictions/current_best/part1_submission.csv')
    assert submission=='f9a287fb1d28fd9a6bbc0c583ac84ec2061e72945e2fbeb5aced9ff5b15ff44a'
    primary_checks['submission_sha256']=submission;secondary_checks['submission_sha256']=submission
    b.save(primary/'checks.json',primary_checks);b.save(pooled.OUT/'checks.json',secondary_checks)
    print(json.dumps(dict(training_only=primary_checks,pooled=secondary_checks),indent=2))

if __name__=='__main__':main()


Writing current_reproduction/verify_benchmarks.py


In [80]:
%%writefile current_reproduction/verify_engine.py
"""Mathematical and leakage checks for custom experiment plumbing."""
import numpy as np
import experiment as e
from threadpoolctl import threadpool_limits
from sklearn.metrics import f1_score
rng=np.random.default_rng(42)
with threadpool_limits(limits=1):
    for a,b in e.make_splits():
        assert len(set(e.GROUP[e.LAB[a]])&set(e.GROUP[e.LAB[b]]))==0
    # Deterministic posteriors must produce precisely their nonempty labels.
    patterns=((np.arange(1,32)[:,None]>>np.arange(5))&1)
    assert np.array_equal(e.decisions(patterns,'expected_f1'),patterns)
    # Graph must respect two well-separated components, and probabilities sum appropriately.
    X=np.r_[rng.normal(-10,.1,(20,3)),rng.normal(10,.1,(20,3))]
    Y=np.full((40,5),-1);Y[0]=[1,0,0,0,0];Y[20]=[0,1,0,0,0]
    for hard in [False,True]:
        m=e.Graph({'k':5,'alpha':.95,'hard':hard}).fit(X,Y)
        p=m.predict(np.array([[-10]*3,[10]*3]))
        # Hard mode deliberately shrinks toward the global prior, unlike normalized spreading.
        assert p[0,0]>(.6 if hard else .95) and p[1,1]>(.6 if hard else .95)
        assert np.all((p>=0)&(p<=1))
    a,b=e.make_splits()[0]
    c={'model':'logistic','C':.1}
    p=e.predict(a,e.DATA.iloc[e.LAB[b]],c,excluded_groups=e.GROUP[e.LAB[b]])
    original=e.Y[b].copy();e.Y[b]=1-original
    q=e.predict(a,e.DATA.iloc[e.LAB[b]],c,excluded_groups=e.GROUP[e.LAB[b]])
    e.Y[b]=original
    assert np.allclose(p,q), 'Held-out labels affected fitting'
    query=e.DATA.iloc[e.LAB[b]].copy();query['ID']='arbitrary'
    assert np.allclose(p,e.predict(a,query,c,excluded_groups=e.GROUP[e.LAB[b]])), 'ID affected predictions'
    # Scaler fit is independent of query batch; changing one query cannot change the others.
    altered=query.copy();altered.loc[altered.index[0],e.EMB]=100000
    r=e.predict(a,altered,c,excluded_groups=e.GROUP[e.LAB[b]])
    assert np.allclose(p[1:],r[1:]), 'Query contaminated fit'
print('PASS: grouped folds, expected-F1 decoder, graph diffusion, held-out-label isolation, ID exclusion, preprocessing isolation')


Overwriting current_reproduction/verify_engine.py


In [81]:
result=subprocess.run([sys.executable,'reproduce_current.py','--output','result'],cwd=replay,check=True,text=True,capture_output=True)
print(result.stdout)
expected=json.loads((root/'predictions/current_best/manifest.json').read_text())['submission_sha256']
actual=hashlib.sha256((replay/'result/part1_submission.csv').read_bytes()).hexdigest()
assert actual==expected,(actual,expected)
print('Exact current submission reproduced:',actual)

{
  "validation": {
    "sample_f1": 0.9123931623931625,
    "macro_f1": 0.81044356609574,
    "f1_Safety": 0.9,
    "f1_Funding": 0.6666666666666666,
    "f1_Efficacy": 0.8,
    "f1_Recruitment": 0.98989898989899,
    "f1_Other": 0.6956521739130435
  },
  "target_met": false,
  "submission_sha256": "f9a287fb1d28fd9a6bbc0c583ac84ec2061e72945e2fbeb5aced9ff5b15ff44a",
  "gold_labels_preserved": true
}

Exact current submission reproduced: f9a287fb1d28fd9a6bbc0c583ac84ec2061e72945e2fbeb5aced9ff5b15ff44a


To replay searches, use the phase scripts and their frozen config JSON files. `experiment.py --help` lists the original search entry point. Most later `phaseN.py` scripts run their own training-CV screen; `continued_select.py phaseN` performs documented repeat-CV selection and validation. Running these intentionally expands the experiment log and repeats validation; it is not required to reproduce the frozen candidate.